In [1]:
import os
import re
import keyring
import pyodbc
import pandas as pd
from datetime import datetime, date
from tqdm import tqdm
from itertools import product
import numpy as np
import io
from IPython.utils import io as ipython_io
from dateutil.relativedelta import relativedelta
from defs_utils import remove_test_patients
import tempfile
import requests
from redcap import Project
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE  # já vem pronta
import xml.etree.ElementTree as ET
from collections import defaultdict
import json
import time


In [2]:
# Recuperando as credenciais do chaveiro
server = keyring.get_password("database", "server")
username = keyring.get_password("database", "username")
password = keyring.get_password("database", "password")
database = 'DATADANTE'  # Você pode também armazenar e recuperar isso usando keyring se desejar


# String de conexão
conn_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};PWD={password};"
    "Encrypt=yes;TrustServerCertificate=yes;"
)


# Configurações do pandas
pd.set_option('display.max_columns', None, 'display.max_rows', 10)

In [3]:
# Função para executar query e retornar dataframe
def query_sql_to_dataframe(conn_string, query):
    try:
        conexao = pyodbc.connect(conn_string)
        print("Conexão estabelecida com sucesso!")
        df = pd.read_sql(query, conexao)
    except pyodbc.Error as ex:
        print(f"Ocorreu um erro ao conectar ao banco de dados: {ex}")
        df = None
    finally:
        if conexao:
            conexao.close()
    return df

In [7]:
# api_key
from __credenciais_redcap import RedcapTokens

# Chama o atributo diretamente
api_key = RedcapTokens.REGVALV
api_url = RedcapTokens.API_URL

def import_redcap(df_sorted, api_url, api_key, batch_size=500, overwrite=False):
    batch_error = []

    # Substitua as colunas de data por suas versões string no formato 'YYYY-MM-DD'
    for col in df_sorted.select_dtypes(include=['datetime']):
        df_sorted[col] = df_sorted[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

    #df_sorted = df_sorted.fillna('')

    total_imported = 0
    total_batches = len(df_sorted) // batch_size + (1 if len(df_sorted) % batch_size > 0 else 0)

    # Dividir o DataFrame em partes com no máximo 'batch_size' linhas

    for i, start_row in enumerate(range(0, len(df_sorted), batch_size), 1):
        batch = df_sorted.iloc[start_row:start_row + batch_size]

        with tempfile.NamedTemporaryFile(mode='w+', delete=True, suffix='.csv') as temp_file:

            # Converter o batch do DataFrame para CSV e salvar no arquivo temporário
            batch.to_csv(temp_file, index=False, sep=",")

            # Voltar ao início do arquivo para leitura
            temp_file.seek(0)

             # Preparar os dados para a requisição POST
            data = {
                'token': api_key,
                'content': 'record',
                'action': 'import',
                'format': 'csv',
                'type': 'flat',
                'overwriteBehavior': 'overwrite' if overwrite else 'normal',  #normal - blank/empty values will be ignored; overwrite - blank/empty values are valid and will overwrite data
                'forceAutoNumber': 'false',
                'data': temp_file.read(),
                'returnContent': 'count',
                'returnFormat': 'json',
                'dateFormat': 'DMY' # Opcional: Ajuda se tiver datas no padrão BR
            }

             # Enviar a requisição POST para o REDCap
            response = requests.post(api_url, data=data)

            # Verificar se a resposta é um JSON válido
            try:
                response_json = response.json()
                if 'count' in response_json:
                    batch_imported = response_json['count']
                    total_imported += batch_imported
                    print(f'Batch {i}/{total_batches} imported successfully. Records in this batch: {batch_imported}. Total records imported: {total_imported}')
                else:
                    batch_error.append(f'Error occurred during import of batch {i}/{total_batches}. Response: {response.text}')
            except ValueError:
                print(f'Error occurred during import of batch {i}/{total_batches}. Response is not valid JSON: {response.text}')

        # Adicionar uma pausa entre as solicitações para evitar atingir limites de taxa de solicitação
        time.sleep(1)

    return batch_error
    
# USO
# batch_error = import_redcap(df_final, api_url, api_key)

In [8]:
LOCK_MSG = "This field is located on a form that is locked. You must first unlock this form for this record."

def extract_locked_ids_only(batch_errors):
    ids = []
    for msg in batch_errors:
        # 1) Tenta extrair o JSON após "Response: {...}"
        m = re.search(r'\{.*\}\s*$', msg)
        if m:
            try:
                obj = json.loads(m.group(0))
                err = obj.get("error", "")
                if err:
                    # 'err' é um CSV com linhas do tipo:
                    # "1104380","dados_demograficos_complete","1","This field is located on a form..."
                    reader = csv.reader(io.StringIO(err), delimiter=',', quotechar='"')
                    for row in reader:
                        # row esperado: [record_id, field_name, value, error_message, ...]
                        if len(row) >= 4 and row[3] == LOCK_MSG:
                            rid = row[0]
                            if rid.isdigit() and rid not in ids:
                                ids.append(rid)
                    continue
            except Exception:
                pass

        # 2) Fallback: caça diretamente em string escapada (sem parse do JSON)
        # Linha escapada começa com \\"<id>\", e deve conter a mensagem LOCK_MSG
        esc_pat = re.compile(
            r'(?:^|\\\\n)\\"(\d+)\\"[^\\n]*?\\"This field is located on a form that is locked\. You must first unlock this form for this record\.\\"'
        )
        for rid in esc_pat.findall(msg):
            if rid not in ids:
                ids.append(rid)
    return ids

# Uso:
#ids = extract_locked_ids_only(batch_error)
# ids agora contém SOMENTE os record_id com esse erro específico
# print(len(ids), ids[:15])

In [9]:

# --- FUNÇÃO DE LEITURA BLINDADA ---
def ler_csv_flexivel(texto_csv):
    if not texto_csv or len(texto_csv.strip()) == 0: return pd.DataFrame()
    df = pd.DataFrame()
    try: df = pd.read_csv(io.StringIO(texto_csv))
    except:
        try: df = pd.read_csv(io.StringIO(texto_csv), sep=';')
        except: 
            try: df = pd.read_csv(io.StringIO(texto_csv), sep=None, engine='python')
            except: pass
    if not df.empty: df.columns = df.columns.str.strip()
    return df

# --- 1. BAIXAR ESTRUTURAS ---
print("--- Baixando Estruturas... ---")
df_meta = ler_csv_flexivel(requests.post(api_url, data={'token': api_key, 'content': 'metadata', 'format': 'csv'}).text)
# df_map baixado apenas para compatibilidade, pois usaremos o DICT manual
df_map = ler_csv_flexivel(requests.post(api_url, data={'token': api_key, 'content': 'formEventMapping', 'format': 'csv'}).text)

# --- 2. DEFINIÇÃO MANUAL DE FASES (DICT FORNECIDO) ---
dict_fases_manual = {
    'acompanhamento_nutricional': 1,
    'alta_hospitalar': 1,
    'ambulatorio': 1,
    'anestesia': 1,
    'dados_demograficos': 1,
    'ecocardiograma': 1,
    'evento_intrahospitalar': 1,
    'exames_laboratoriais': 1,
    'fatores_risco': 1,
    'ficha_complicacoes': 1,
    'ficha_satisfacao': 1,
    'fisioterapia': 2,
    'odontologia': 1,
    'pos_operatorio': 1,
    'pre_operatoria': 1,
    'procedimento': 1,
    'psicologia': 2,
    'qfca': 1,
    'qfca_internacao': 2,
    'qualidade_vida': 2,
}

print("--- Usando Definição Manual de Fases ---")
for k, v in dict_fases_manual.items():
    if v > 1: print(f"Formulário '{k}' configurado com {v} fases.")

# --- 3. BAIXAR DADOS ---
print("\n--- Baixando Registros... ---")
r_data = requests.post(api_url, data={
    'token': api_key, 'content': 'record', 'format': 'csv', 'type': 'flat',
    'rawOrLabel': 'raw', 'exportSurveyFields': 'false', 'exportDataAccessGroups': 'false',
})
df_data = ler_csv_flexivel(r_data.text)

# --- 4. DEFINIR OS UNIVERSOS (BASES) ---

# Verificação se existe a variável chave para contagem de procedimentos
coluna_chave_procedimento = 'procedimento_complete' # Ajuste se necessário, ex: 'cd_atendimento'
if coluna_chave_procedimento not in df_data.columns:
    # Tenta usar cd_atendimento se procedimento_complete não existir
    if 'cd_atendimento' in df_data.columns:
        coluna_chave_procedimento = 'cd_atendimento'
    else:
        print(f"ERRO: Nem 'procedimento_complete' nem 'cd_atendimento' encontrados. Usando record_id.")
        coluna_chave_procedimento = 'record_id'

# 1. Identificar Pacientes Válidos (que têm procedimento)
pacientes_com_procedimento = df_data[df_data[coluna_chave_procedimento].notna()]['record_id'].unique().tolist()

# 2. Filtrar Dados (Trabalhar apenas com pacientes cirúrgicos)
df_data = df_data[df_data['record_id'].isin(pacientes_com_procedimento)]

# 3. Calcular Bases
# Base Pacientes (Únicos)
qtde_pacientes_reais = len(pacientes_com_procedimento) 

# Base Procedimentos (Total de cirurgias realizadas)
qtde_procedimentos_reais = df_data[coluna_chave_procedimento].count()

print(f"Base Pacientes (Únicos): {qtde_pacientes_reais}")
print(f"Base Procedimentos ({coluna_chave_procedimento}): {qtde_procedimentos_reais}")

# --- 5. GERAÇÃO DO RELATÓRIO ---
relatorio = []
tem_coluna_repeticao = 'redcap_repeat_instrument' in df_data.columns

for index, row in df_meta.iterrows():
    var_name = row['field_name']
    form_name = row['form_name']
    field_type = row['field_type']
    label_base = str(row['field_label'])[:60]
    
    if field_type == 'descriptive': continue

    # === APLICANDO LÓGICA DO DICT MANUAL ===
    
    # 1. Pega o Multiplicador do seu DICT
    # Se o formulário não estiver no dict, assume 1 por segurança
    mult_eventos = dict_fases_manual.get(form_name, 1)
    
    # 2. Define a Base (Paciente ou Procedimento?) e Recorte de Dados
    base_calculo = 0
    df_slice = pd.DataFrame()
    
    # Lógica simples: Se for dados_demograficos é Paciente, o resto é Procedimento
    if form_name == 'dados_demograficos':
        base_calculo = qtde_pacientes_reais
        
        # Filtro de dados (Demográficos não repete)
        if tem_coluna_repeticao:
            df_slice = df_data[df_data['redcap_repeat_instrument'].isna()]
        else:
            df_slice = df_data.copy()
            
    else:
        # Todo o resto usa base de procedimentos
        base_calculo = qtde_procedimentos_reais
        
        # Verifica se é repetitivo nos DADOS
        e_repetitivo = False
        if tem_coluna_repeticao:
            e_repetitivo = form_name in df_data['redcap_repeat_instrument'].values
            
        if e_repetitivo:
            # Se for repetitivo, filtramos pelo nome do instrumento na coluna de repetição
            df_slice = df_data[df_data['redcap_repeat_instrument'] == form_name]
        else:
            # Se não for repetitivo (ex: está na linha principal do evento), filtramos linhas não repetitivas
            # Nota: O filtro aqui é amplo (todas as linhas base dos pacientes cirurgicos), 
            # pois o numerador deve contar todas as ocorrências
            if tem_coluna_repeticao:
                df_slice = df_data[df_data['redcap_repeat_instrument'].isna()]
            else:
                df_slice = df_data.copy()

    # 3. CALCULA O TOTAL ESPERADO FINAL
    # Ex Fisioterapia: 430 (base) * 2 (fases) = 860 esperados
    total_esperado_final = base_calculo * mult_eventos

    # === CONTABILIZAÇÃO DO NUMERADOR ===
    lista_vars = []
    if field_type == 'checkbox':
        cols = [c for c in df_data.columns if c.startswith(f"{var_name}___")]
        for c in cols:
            opt = c.split('___')[-1]
            lista_vars.append({'col': c, 'lbl': f"{label_base} (Op.)"})
    else:
        lista_vars.append({'col': var_name, 'lbl': label_base})

    for item in lista_vars:
        col_real = item['col']
        lbl_final = item['lbl']
        
        preenchidos = 0
        if col_real in df_slice.columns:
            # Conta valores não nulos
            preenchidos = df_slice[col_real].count()
        
        nulos = total_esperado_final - preenchidos
        pct = (preenchidos / total_esperado_final * 100) if total_esperado_final > 0 else 0
        if nulos < 0: nulos = 0

        relatorio.append({
            'Instrumento': form_name,
            'Variavel_ID': col_real,
            'Pergunta': lbl_final,
            'Fases_Dict': mult_eventos, # Para conferência
            'Total_Esperado': total_esperado_final,
            'Qtd_Preenchidos': preenchidos,
            'Qtd_Vazios': nulos,
            'Preenchimento (%)': round(pct, 2)
        })

# --- EXIBIÇÃO ---
df_resultado = pd.DataFrame(relatorio)



--- Baixando Estruturas... ---
--- Usando Definição Manual de Fases ---
Formulário 'fisioterapia' configurado com 2 fases.
Formulário 'psicologia' configurado com 2 fases.
Formulário 'qfca_internacao' configurado com 2 fases.
Formulário 'qualidade_vida' configurado com 2 fases.

--- Baixando Registros... ---


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_16528\761324737.py:5: DtypeWarning: Columns (36,49,51,61,64,76,107,108,109,190,191,192,253,285,286,287,288,289,290,342,359,361,367,373,377,379,380,381,382,383,384,392,409,411,419,425,437,438,439,457,471,692,697,699,761,762,763,766,767,768,769,770,771,772,774,779,780,785,786,787,788,789,790,791,792,794,795,797,800,817,825,830,831,835,836,837,845,851,861,863,879,903,911,918,926,931,933,934,935,936,1094,1095,1096,1097,1098,1099,1100,1101,1102,1103,1104,1105,1106,1107,1108,1110,1111,1112,1113,1114,1115,1117,1119,1122,1123,1125,1127,1128,1129,1130,1132,1134,1135,1136,1139,1140,1141,1142,1143,1144,1145,1147,1151,1154,1155,1156,1160,1161,1162,1163,1171,1172,1173,1174,1175,1176,1177,1178,1179,1180,1181,1182,1183,1184,1185,1187,1188,1189,1190,1191,1192,1194,1196,1197,1198,1199,1200,1201,1202,1203,1204,1205,1206,1207,1208,1209,1210,1211,1212,1213,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1226,1227,1228,1229,1230,1231,1232

Base Pacientes (Únicos): 543
Base Procedimentos (procedimento_complete): 549


In [10]:
# Obtém a data atual no formato desejado (dd.mm.aaaa)
data_atual = datetime.now().strftime('%d.%m.%Y')

# Define o caminho do arquivo com a data no nome
caminho_dicio = f"C://Users/priscilla.sequetin/Downloads/dicionario_regvalv_{data_atual}.xlsx"

# Salva o DataFrame no arquivo parquet
df_resultado.to_excel(caminho_dicio, index=False)

print(f"Arquivo salvo como: {caminho_dicio}")


Arquivo salvo como: C://Users/priscilla.sequetin/Downloads/dicionario_regvalv_25.02.2026.xlsx


In [ ]:
# import random
# from datetime import datetime, timedelta

# def gerar_base_tcc_redcap(caminho_dicionario, n_registros=150):
#     # 1. Carregar o dicionário (CSV com separador ;)
#     try:
#         df_dic = pd.read_csv(caminho_dicionario, sep=';')
#     except Exception as e:
#         print(f"Erro ao ler o dicionário: {e}")
#         return None

#     def parse_choices(choice_str):
#         """Extrai os códigos numéricos de strings como '0, Nao | 1, Sim'"""
#         if pd.isna(choice_str) or not str(choice_str).strip() or '[' in str(choice_str):
#             return None
        
#         # Divide por '|' e pega o valor antes da vírgula
#         parts = str(choice_str).split('|')
#         keys = []
#         for p in parts:
#             if ',' in p:
#                 key = p.split(',')[0].strip()
#                 keys.append(key)
#         return keys if keys else None

#     data = []
    
#     # 2. Geração de registros
#     for i in range(1, n_registros + 1):
#         registro = {}
        
#         for _, row in df_dic.iterrows():
#             var_name = row['Variable / Field Name']
#             field_type = row['Field Type']
#             choices_raw = row['Choices, Calculations, OR Slider Labels']
#             validation = row['Text Validation Type OR Show Slider Number']
            
#             # Identificador único
#             if var_name == 'record_id':
#                 registro[var_name] = i
#                 continue
            
#             # Lógica baseada nas Escolhas (Choices)
#             choices = parse_choices(choices_raw)
#             if choices:
#                 registro[var_name] = random.choice(choices)
            
#             # Lógica baseada no Tipo de Campo e Validação
#             elif field_type == 'yesno':
#                 registro[var_name] = random.choice(['0', '1'])
            
#             elif field_type == 'text':
#                 if validation == 'date_dmy':
#                     # Gera datas aleatórias entre 1950 e 2024
#                     inicio = datetime(1950, 1, 1)
#                     fim = datetime(2024, 1, 1)
#                     dt = inicio + timedelta(days=random.randint(0, (fim - inicio).days))
#                     registro[var_name] = dt.strftime('%d/%m/%Y')
#                 elif validation in ['integer', 'number']:
#                     # Tenta pegar min/max do dicionário ou usa padrão
#                     try:
#                         vmin = int(row['Text Validation Min']) if not pd.isna(row['Text Validation Min']) else 0
#                         vmax = int(row['Text Validation Max']) if not pd.isna(row['Text Validation Max']) else 100
#                     except: vmin, vmax = 0, 100
#                     registro[var_name] = random.randint(vmin, vmax)
#                 else:
#                     registro[var_name] = f"Dado_{i}"
            
#             elif field_type == 'calc':
#                 registro[var_name] = round(random.uniform(10, 50), 1) # Ex: IMC
            
#             else:
#                 registro[var_name] = np.nan
        
#         data.append(registro)

#     df_result = pd.DataFrame(data)

#     # 3. Simulação de Missing Values (15% de falha em colunas que não são ID)
#     for col in df_result.columns:
#         if col != 'record_id':
#             df_result.loc[df_result.sample(frac=0.15).index, col] = np.nan

#     return df_result

# # Execução
# caminho_dic = r'C://Users/priscilla.sequetin/Downloads/RegValvDANTE_DataDictionary_2026-02-04.csv'
# df_teste = gerar_base_tcc_redcap(caminho_dic, 200)

# # Salvar e conferir
# if df_teste is not None:
    # df_teste.to_csv(r'C://Users/priscilla.sequetin/Downloads/base_teste_sintetica.csv', index=False)
    # print(f"Base gerada com {df_teste.shape[0]} registros e {df_teste.shape[1]} colunas.")
    # print(df_teste[['record_id', 'genero', 'etnia', 'classe_nyha']].head())

In [ ]:
# import_redcap(df_pct, api_url, api_key, overwrite=False)
# import_redcap(relatrio_de_teste_cardiopulmonar_3, api_url, api_key, overwrite=False, batch_size=100)

In [11]:
def organizar_dados_paciente(df, chaves_agrupamento=None, colunas_fixas=None, colunas_desejadas=None):
    """
    Função unificada e otimizada para organizar dados de pacientes.
    
    Parâmetros:
    - df: DataFrame de entrada
    - chaves_agrupamento: lista de colunas para agrupar (ex: ['CD_ATENDIMENTO', 'record_id', 'DH_DOCUMENTO'])
    - colunas_fixas: lista de colunas extras fixas a incluir no agrupamento (ex: ['NM_PRESTADOR'])
    - colunas_desejadas: lista específica de colunas a manter (ignora as chaves; usa .first() nelas)
    
    Se colunas_desejadas=None, mantém todas as colunas.
    """
    # Definir chaves padrão baseadas no seu contexto de fisioterapia/REDcap
    if chaves_agrupamento is None:
        chaves_agrupamento = ['CD_ATENDIMENTO', 'record_id', 'DH_DOCUMENTO']
    
    # Verificar se todas as chaves existem
    chaves_faltando = [k for k in chaves_agrupamento if k not in df.columns]
    if chaves_faltando:
        raise ValueError(f"Chaves faltando no DataFrame: {chaves_faltando}")
    
    # Colunas fixas extras (não variam por grupo)
    if colunas_fixas is None:
        colunas_fixas = []
    else:
        colunas_fixas = [c for c in colunas_fixas if c in df.columns and c not in chaves_agrupamento]
    
    todas_chaves = chaves_agrupamento + colunas_fixas
    
    # Se colunas_desejadas fornecida, filtrar DataFrame primeiro para eficiência
    if colunas_desejadas is not None:
        # Garantir que chaves estão incluídas
        colunas_validas = list(set(chaves_agrupamento + [c for c in colunas_desejadas if c in df.columns]))
        df = df[colunas_validas].copy()
    else:
        colunas_desejadas = [c for c in df.columns if c not in todas_chaves]
    
    # Agrupamento vetorizado: .first() pega primeiro não-NaN automaticamente!
    df_organizado = df.groupby(todas_chaves, dropna=False, as_index=False).first()
    
    # Reordenar colunas: chaves primeiro, depois desejadas
    cols_finais = todas_chaves + sorted([c for c in colunas_desejadas if c in df_organizado.columns])
    df_organizado = df_organizado[cols_finais]
    
    # Substituir restantes NaN por 0 apenas nas colunas desejadas (como no seu 1º código)
    for col in colunas_desejadas:
        if col in df_organizado.columns:
            df_organizado[col] = df_organizado[col].fillna(0)
    
    return df_organizado


# # ------------- COMO USAR-----------------------

# # Informando colunas desejadas para avaliar
# cols_df = []

# df_organizado = organizar_dados_paciente(
#     df, 
#     chaves_agrupamento=['CD_ATENDIMENTO', 'record_id', 'DH_DOCUMENTO'],
#     colunas_desejadas=cols_df[3:]  # só as variáveis clínicas
# )

# # Informando colunas chaves para agrupar
# df_organizado = organizar_dados_paciente(
#     df,
#     chaves_agrupamento=['CD_DOCUMENTO', 'CD_PACIENTE', 'CD_ATENDIMENTO', 'DH_DOCUMENTO'],
#     colunas_fixas=['NM_PRESTADOR']
#     # Sem colunas_desejadas: mantém todas!
# )

In [12]:
def organizar_df_otimizado(df):
    """
    Agrupa os dados pelo ID do Documento.
    Se houver linhas duplicadas para o mesmo documento (ex: estrutura EAV),
    pega o primeiro valor não nulo encontrado.
    """
    # 1. Definir as colunas que identificam unicamente o formulário
    # Adicionamos CD_DOCUMENTO para garantir que os 5 docs do dia não se misturem
    chaves_agrupamento = ['CD_DOCUMENTO', 'CD_PACIENTE', 'CD_ATENDIMENTO', 'DH_DOCUMENTO']
    
    # Adicione colunas que não variam por documento, mas você quer manter (ex: prestador)
    colunas_extras_fixas = ['NM_PRESTADOR']
    for col in colunas_extras_fixas:
        if col in df.columns and col not in chaves_agrupamento:
            chaves_agrupamento.append(col)

    # 2. Agrupamento Otimizado (Vectorized)
    # O .first() ignora NaNs automaticamente. Ele pega o primeiro valor válido.
    # Isso substitui todo aquele seu loop 'for' manual.
    df_organizado = df.groupby(chaves_agrupamento, as_index=False).first()
    
    return df_organizado

In [13]:
def vincular_referencia_temporal(
    df_principal, 
    df_referencia, 
    col_data_principal='DH_DOCUMENTO',
    col_data_referencia='data_cirurgia',
    col_instancia='redcap_repeat_instance',
    col_id='record_id',
    col_atend_principal=None,  # ex: 'CD_ATENDIMENTO'
    col_atend_referencia=None,  # ex: 'cd_atendimento'
    tolerancia_dias=90,
    direction='nearest'
):
    """
    Vincula instância e data_cirurgia de df_referencia ao df_principal.
    Prioridade: 1) Match exato por ID + atendimento (se cols informadas).
                2) Match temporal via merge_asof (data_cirurgia como marco).
    Regras de instância: ref > original > '1'.
    
    Retorna df_principal com colunas adicionadas/atualizadas: data_cirurgia, redcap_repeat_instance.
    """
    # 1. Cópias seguras e padronização (IDs/decimais/strings)
    df_main = df_principal.copy()
    df_ref = df_referencia.copy()
    
    # Datas como datetime
    df_main[col_data_principal] = pd.to_datetime(df_main[col_data_principal], errors='coerce')
    df_ref[col_data_referencia] = pd.to_datetime(df_ref[col_data_referencia], errors='coerce')
    
    # Padronizar ID e atendimento (remove .0, strip)
    for col in [col_id] + ([col_atend_principal, col_atend_referencia] if col_atend_principal else []):
        if col and col in df_main.columns:
            df_main[col] = df_main[col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        if col and col in df_ref.columns:
            df_ref[col] = df_ref[col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    
    # Separar main sem data (não processam)
    df_main_valid = df_main[df_main[col_data_principal].notna()].copy()
    df_main_sem_data = df_main[df_main[col_data_principal].isna()].copy()
    
    df_ref_valid = df_ref[df_ref[col_data_referencia].notna()].copy()
    cols_ref = [col_id, col_data_referencia, col_instancia]
    if col_atend_referencia:
        cols_ref.append(col_atend_referencia)
    
    df_result = df_main_valid.copy()
    
    # 2. FASE 1: Match EXATO por atendimento (se aplicável)
    usar_atend = col_atend_principal and col_atend_referencia and col_atend_principal in df_main_valid.columns
    if usar_atend and not df_main_valid.empty and not df_ref_valid.empty:
        df_ref_dedup = df_ref_valid.drop_duplicates(subset=[col_id, col_atend_referencia])
        df_merged_atend = pd.merge(
            df_main_valid, df_ref_dedup[cols_ref],
            left_on=[col_id, col_atend_principal], right_on=[col_id, col_atend_referencia],
            how='left', suffixes=('', '_ref')
        )
        mask_match = df_merged_atend[f'{col_data_referencia}_ref'].notna()
        df_result = df_merged_atend[~mask_match][df_main_valid.columns].copy()  # Sem match vão para temporal
        df_exato = df_merged_atend[mask_match].copy()
    else:
        df_exato = pd.DataFrame()
    
    # 3. FASE 2: Match TEMPORAL nos restantes
    df_para_temporal = df_result if 'df_result' in locals() else df_main_valid
    if not df_para_temporal.empty and not df_ref_valid.empty:
        df_para_temporal = df_para_temporal.sort_values(col_data_principal)
        df_temporal = pd.merge_asof(
            df_para_temporal, df_ref_valid[cols_ref],
            left_on=col_data_principal, right_on=col_data_referencia,
            by=col_id, direction=direction, tolerance=pd.Timedelta(days=tolerancia_dias),
            suffixes=('', '_ref')
        )
    else:
        df_temporal = df_para_temporal.copy()
    
    # 4. Concat e resolver instância (prioridade ref > original > 1)
    df_final = pd.concat([df_exato, df_temporal, df_main_sem_data], ignore_index=True)
    
    def resolver_instancia(row):
        ref_inst = row.get(f'{col_instancia}_ref')
        if pd.notna(ref_inst):
            return str(int(float(ref_inst)))
        orig_inst = row.get(col_instancia)
        if pd.notna(orig_inst):
            return str(int(float(orig_inst)))
        return '1'
    
    df_final[col_instancia] = df_final.apply(resolver_instancia, axis=1)
    df_final[f'{col_data_referencia}'] = df_final[f'{col_data_referencia}_ref']  # Traz data_cirurgia
    
    # Limpa auxiliares
    df_final.drop(columns=[f'{col_instancia}_ref', f'{col_data_referencia}_ref'], inplace=True, errors='ignore')
    
    return df_final


In [14]:
def classificar_eventos_flexivel(
    df_entrada,
    tipo_fluxo='cirurgia_fisio',  # 'cirurgia_fisio' | 'geronto_odonto_psico' | custom
    col_data_doc='DH_DOCUMENTO',
    col_instancia='redcap_repeat_instance',
    col_atend=None,  # 'CD_ATENDIMENTO' para isolar
    col_setor='CD_DOCUMENTO',  # ou 'NM_SETOR'
    col_data_marco1=None,  # ex: 'data_cirurgia' ou 'data_internacao'
    col_data_marco2=None,   # ex: 'data_alta' para pós-alta
    evento_pre='pre_internacao',
    evento_internacao='durante_internacao',
    evento_pos_intern='pos_internacao',
    evento_amb='ambulatorio'
):
    """
    Classifica para QUALQUER fluxo clínico:
    - 'cirurgia_fisio': Pré-cirurgia → UTI/ENF → AMB (seu caso original).
    - 'geronto_odonto_psico': Pré-internação → Internação → Pós-intern → AMB.
    - Custom: passe marcos (data_cirurgia/alta) e eventos.
    
    Detecta fase por: atendimento → data_marco → setor.
    """
    df = df_entrada.copy()
    df[col_data_doc] = pd.to_datetime(df[col_data_doc], errors='coerce')
    
    # Datas marco (opcional)
    if col_data_marco1: df[col_data_marco1] = pd.to_datetime(df[col_data_marco1], errors='coerce')
    if col_data_marco2: df[col_data_marco2] = pd.to_datetime(df[col_data_marco2], errors='coerce')
    
    tem_doc = df[col_data_doc].notna()
    mask_atend = pd.Series([True] * len(df), index=df.index)
    if col_atend and col_atend in df:
        mask_atend = df[col_atend].notna()
    
    # Máscaras setor (padrão fisio, override por tipo)
    def mascara_setor(vals):
        if isinstance(vals, (list, tuple)):
            return pd.to_numeric(df[col_setor], errors='coerce').isin(vals)
        return df[col_setor].astype(str).str.contains(vals, case=False, na=False)
    
    # Config por tipo_fluxo
    if tipo_fluxo == 'cirurgia_fisio':
        # UTI/ENF/AMB como original
        is_uti = mascara_setor((1140, 1145, 1137))
        is_enf = mascara_setor((1136, 1141))
        is_amb = mascara_setor('AMBULATORIO')
        tem_marco1 = df[col_data_marco1].notna() if col_data_marco1 else pd.Series([False]*len(df))
        
        conds = [
            tem_doc & ~tem_marco1 & mask_atend,  # Pré-cirurgia
            tem_marco1 & (df[col_data_doc] < df[col_data_marco1]) & mask_atend,  # Pré real
            tem_marco1 & (df[col_data_doc] >= df[col_data_marco1]) & is_uti & mask_atend,
            tem_marco1 & (df[col_data_doc] >= df[col_data_marco1]) & is_enf & mask_atend,
            tem_marco1 & (df[col_data_doc] >= df[col_data_marco1]) & is_amb & mask_atend
        ]
        escolhas = [evento_pre, evento_pre, evento_internacao, evento_pos_intern, evento_amb]
        
    elif tipo_fluxo == 'geronto_odonto_psico':
        # Fluxo completo: pré → intern → pós-intern → amb (usa data_internacao + data_alta)
        tem_intern = df[col_data_marco1].notna() if col_data_marco1 else pd.Series([False]*len(df))
        tem_alta = df[col_data_marco2].notna() if col_data_marco2 else pd.Series([False]*len(df))
        is_amb = mascara_setor('AMBULATORIO')
        
        conds = [
            tem_doc & ~tem_intern & mask_atend,  # Pré-internação
            tem_intern & (df[col_data_doc] < df[col_data_marco1]) & mask_atend,  # Pré-intern real
            tem_intern & (df[col_data_marco1] <= df[col_data_doc]) & (~tem_alta | (df[col_data_doc] < df[col_data_marco2])) & mask_atend,  # Durante intern
            tem_alta & (df[col_data_doc] >= df[col_data_marco2]) & ~is_amb & mask_atend,  # Pós-alta internado
            is_amb & mask_atend  # Ambulatorial (qualquer tempo)
        ]
        escolhas = [evento_pre, evento_pre, evento_internacao, evento_pos_intern, evento_amb]
        
    else:  # Custom: use marcos informados
        conds = [
            tem_doc & mask_atend,  # Default pré se sem marcos
            (df[col_data_marco1] <= df[col_data_doc]) & mask_atend  # Pós-marco1
        ]
        escolhas = [evento_pre, evento_pos_intern]
    
    df['redcap_event_name'] = np.select(conds, escolhas, default=evento_pre)
    df[col_instancia] = pd.to_numeric(df[col_instancia], errors='coerce').fillna(1).astype(int)
    
    return df



In [12]:
## FORMAS DE USAR A FUNÇÃO
# # 1. SEMPRE vincule primeiro (mesma função anterior)
#df_vinculado = vincular_referencia_temporal(df_fisio, df_cirurgia, 
#                                           col_atend_principal='CD_ATENDIMENTO')

# # 2. AGORA use a UNIFICADA para QUALQUER tipo
# # Fisio (comportamento original):
# df_fisio_final = classificar_eventos_flexivel(
#     df_vinculado,
#     tipo_fluxo='cirurgia_fisio',  # ← Seu caso original
#     col_data_marco1='data_cirurgia'
# )

# # Geronto (novo fluxo completo):
# df_geronto_final = classificar_eventos_flexivel(
#     df_geronto_vinc,
#     tipo_fluxo='geronto_odonto_psico',
#     col_data_marco1='data_internacao',
#     col_data_marco2='data_alta'
# )

# # 1. FISIO/CIRURGIA (seu caso original)
# df_fisio_class = classificar_eventos_flexivel(
#     df_fisio_vinculada,
#     tipo_fluxo='cirurgia_fisio',
#     col_data_marco1='data_cirurgia',
#     col_atend='CD_ATENDIMENTO'
# )

# # 2. GERONTO (pré-intern → intern → pós-alta → amb)
# df_geronto_class = classificar_eventos_flexivel(
#     df_geronto_vinc,
#     tipo_fluxo='geronto_odonto_psico',
#     col_data_marco1='data_internacao',  # Ou data_admissao
#     col_data_marco2='data_alta',
#     col_atend='CD_ATENDIMENTO',
#     evento_pre='geronto_pre_arm_1',
#     evento_internacao='geronto_intern_arm_1',
#     evento_pos_intern='geronto_posalta_arm_1',
#     evento_amb='geronto_amb_arm_1'
# )

# # 3. ODONTO/PSICO (custom, só ambulatorial pós-alta)
# df_odonto_class = classificar_eventos_flexivel(
#     df_odonto,
#     tipo_fluxo='custom',
#     col_data_marco1='data_alta',
#     evento_pre='odonto_pre',
#     evento_pos_intern='odonto_posalta_amb'
# )

In [15]:

def vincular_instancias_temporal(
    df_principal, 
    df_referencia, 
    col_data_principal='DH_DOCUMENTO', 
    col_data_referencia='data_cirurgia',
    col_instancia='redcap_repeat_instance',
    col_id='record_id',
    tolerancia_dias=90,
    direction='nearest',
    col_atend_principal=None, 
    col_atend_referencia=None
):
    """
    Vincula instâncias do REDCap usando estratégia híbrida:
    1. Match exato por atendimento (se informado)
    2. Match temporal (merge_asof)

    Regras de instância:
    - Prioridade: instância do df_referencia (ex: regvalv_cirurgia)
    - Fallback: instância já existente no df_principal
    - Último fallback: 1
    """

    # ------------------------------------------------------------
    # 1) Cópias e tipagem segura
    # ------------------------------------------------------------
    df_main = df_principal.copy()
    df_ref = df_referencia.copy()

    def _norm_id(s: pd.Series) -> pd.Series:
        return (
            s.astype(str)
             .str.replace(r'\.0$', '', regex=True)
             .str.strip()
        )

    # Normaliza IDs
    df_main[col_id] = _norm_id(df_main[col_id])
    df_ref[col_id]  = _norm_id(df_ref[col_id])

    # Normaliza atendimentos (se informados)
    usar_atendimento = (col_atend_principal is not None and col_atend_referencia is not None)
    if usar_atendimento:
        if col_atend_principal in df_main.columns:
            df_main[col_atend_principal] = _norm_id(df_main[col_atend_principal])
        else:
            df_main[col_atend_principal] = ''
        if col_atend_referencia in df_ref.columns:
            df_ref[col_atend_referencia] = _norm_id(df_ref[col_atend_referencia])
        else:
            df_ref[col_atend_referencia] = ''

    # Datas coerentes e tz-naive
    def _to_naive_dt(s: pd.Series) -> pd.Series:
        s = pd.to_datetime(s, errors='coerce')
        # Se vier tz-aware, remove timezone
        try:
            if hasattr(s.dt, 'tz') and s.dt.tz is not None:
                s = s.dt.tz_localize(None)
        except Exception:
            pass
        return s

    df_main[col_data_principal] = _to_naive_dt(df_main[col_data_principal])
    df_ref[col_data_referencia]  = _to_naive_dt(df_ref[col_data_referencia])

    # ------------------------------------------------------------
    # 2) Separação por presença de data
    # ------------------------------------------------------------
    df_main_validos   = df_main.dropna(subset=[col_id, col_data_principal]).copy()
    df_main_sem_data  = df_main[df_main[col_data_principal].isna()].copy()
    df_ref_validos    = df_ref.dropna(subset=[col_id, col_data_referencia]).copy()

    cols_ref_merge = [col_id, col_data_referencia, col_instancia]
    if 'redcap_event_name' in df_ref_validos.columns:
        cols_ref_merge.append('redcap_event_name')

    # ------------------------------------------------------------
    # 3) FASE 1 — Match por atendimento (exato)
    # ------------------------------------------------------------
    df_resolvidos_exato = pd.DataFrame()
    df_para_temporal = df_main_validos.copy()

    if usar_atendimento and not df_main_validos.empty and not df_ref_validos.empty:
        # Deduplica referência por (id, atendimento), mantendo a MAIS RECENTE pela data
        df_ref_dedup = (
            df_ref_validos
            .sort_values(col_data_referencia, kind='mergesort')
            .drop_duplicates(subset=[col_id, col_atend_referencia], keep='last')
        )

        df_merge_atend = pd.merge(
            df_main_validos,
            df_ref_dedup[cols_ref_merge + [col_atend_referencia]],
            left_on=[col_id, col_atend_principal],
            right_on=[col_id, col_atend_referencia],
            how='left',
            suffixes=('', '_ref')
        )

        mask_match = df_merge_atend[col_data_referencia].notna()

        df_resolvidos_exato = df_merge_atend[mask_match].copy()
        df_para_temporal    = df_merge_atend.loc[~mask_match, df_main_validos.columns].copy()

    # ------------------------------------------------------------
    # 4) FASE 2 — Match temporal (merge_asof) com ordenação garantida
    # ------------------------------------------------------------
    def _merge_asof_seguro(left, right):
        """
        Executa merge_asof garantindo:
        - Ordenação estável por (col_id, data)
        - Tolerância em dias
        - Fallback por grupo (se ainda der erro de ordenação)
        """
        if left.empty or right.empty:
            return left.copy()

        left2 = (
            left.dropna(subset=[col_id, col_data_principal])
                .sort_values([col_id, col_data_principal], kind='mergesort')
        )
        right2 = (
            right.dropna(subset=[col_id, col_data_referencia])
                 .sort_values([col_id, col_data_referencia], kind='mergesort')
        )

        try:
            out = pd.merge_asof(
                left2,
                right2[cols_ref_merge],
                left_on=col_data_principal,
                right_on=col_data_referencia,
                by=col_id,
                direction=direction,
                tolerance=pd.Timedelta(days=tolerancia_dias),
                suffixes=('', '_ref')
            )
            return out
        except ValueError:
            # Fallback por grupo (mais lento, porém robusto)
            chunks = []
            for rid, gleft in left2.groupby(col_id, sort=False):
                gright = right2[right2[col_id] == rid]
                if gright.empty:
                    chunks.append(gleft.assign(**{
                        col_data_referencia: pd.NaT,
                        f'{col_instancia}__ref': pd.NA
                    }))
                else:
                    merged = pd.merge_asof(
                        gleft.sort_values(col_data_principal, kind='mergesort'),
                        gright[cols_ref_merge].sort_values(col_data_referencia, kind='mergesort'),
                        left_on=col_data_principal,
                        right_on=col_data_referencia,
                        direction=direction,
                        tolerance=pd.Timedelta(days=tolerancia_dias),
                        suffixes=('', '_ref')
                    )
                    chunks.append(merged)
            return pd.concat(chunks, ignore_index=True)

    if not df_para_temporal.empty and not df_ref_validos.empty:
        df_resolvidos_temporal = _merge_asof_seguro(df_para_temporal, df_ref_validos)
    else:
        df_resolvidos_temporal = df_para_temporal.copy()

    # ------------------------------------------------------------
    # 5) Reconstrução do DataFrame
    # ------------------------------------------------------------
    df_final = pd.concat(
        [df_resolvidos_exato, df_resolvidos_temporal, df_main_sem_data],
        ignore_index=True
    )

    # ------------------------------------------------------------
    # 6) Resolução DEFINITIVA da instância (mantém como STRING)
    # ------------------------------------------------------------
    def resolver_instancia(row):
        # Prioridade 1 — instância do df_referencia
        inst_ref = row.get(f'{col_instancia}_ref')
        if pd.notna(inst_ref):
            try:
                return str(int(float(inst_ref)))
            except Exception:
                pass

        # Prioridade 2 — instância original
        inst_ori = row.get(col_instancia)
        if pd.notna(inst_ori):
            try:
                return str(int(float(inst_ori)))
            except Exception:
                pass

        # Fallback
        return '1'

    df_final[col_instancia] = df_final.apply(resolver_instancia, axis=1)

    # Remove coluna auxiliar (se houver)
    df_final.drop(columns=[f'{col_instancia}_ref'], inplace=True, errors='ignore')

    return df_final


In [16]:
def vincular_cirurgia_por_atendimento(
    df_itens, 
    df_cirurgia, 
    col_id='record_id',
    col_atendimento_item='CD_ATENDIMENTO',   # Atendimento na tabela de fisio
    col_atendimento_cirurgia='cd_atendimento', # Atendimento na tabela de cirurgia (atenção ao case sensitive)
    col_data_item='DH_DOCUMENTO',
    col_data_cirurgia='data_cirurgia',
    col_instancia_cirurgia='redcap_repeat_instance'
):
    """
    Vincula os dados da cirurgia (Data e Instância) ao dataframe de itens (Fisio)
    usando a chave composta: record_id + CD_ATENDIMENTO.
    """
    
    # --- 1. Cópias de segurança e Padronização ---
    df_main = df_itens.copy()
    df_ref = df_cirurgia.copy()

    # Converter datas
    df_main[col_data_item] = pd.to_datetime(df_main[col_data_item], errors='coerce')
    df_ref[col_data_cirurgia] = pd.to_datetime(df_ref[col_data_cirurgia], errors='coerce')

    # Limpar e padronizar IDs e Atendimentos para string (remove decimais .0 se houver)
    for df_temp, c_id, c_atend in [(df_main, col_id, col_atendimento_item), 
                                   (df_ref, col_id, col_atendimento_cirurgia)]:
        df_temp[c_id] = df_temp[c_id].astype(str).str.replace(r'\.0$', '', regex=True)
        df_temp[c_atend] = df_temp[c_atend].astype(str).str.replace(r'\.0$', '', regex=True)

    # --- 2. Seleção de colunas úteis da cirurgia ---
    # Queremos trazer a data da cirurgia e a instância correta
    cols_ref = [col_id, col_atendimento_cirurgia, col_data_cirurgia, col_instancia_cirurgia]
    
    # Remove duplicatas na tabela de cirurgia (caso haja linhas sujas para o mesmo atendimento)
    # Assume-se que 1 Atendimento = 1 Cirurgia Principal
    df_ref_unique = df_ref[cols_ref].drop_duplicates(subset=[col_id, col_atendimento_cirurgia])

    # --- 3. Merge (Left Join) ---
    # Mantemos todas as evoluções (left), trazendo dados da cirurgia onde der match
    df_final = pd.merge(
        df_main,
        df_ref_unique,
        left_on=[col_id, col_atendimento_item],
        right_on=[col_id, col_atendimento_cirurgia],
        how='inner',
        suffixes=('', '_cirurgia')
    )

    # Se a coluna de atendimento tiver nomes diferentes, remove a duplicada criada pelo merge
    if col_atendimento_item != col_atendimento_cirurgia:
        df_final.drop(columns=[col_atendimento_cirurgia], inplace=True, errors='ignore')

    return df_final

def classificar_eventos_por_atendimento(
    df_entrada, 
    evento_pre, 
    evento_uti, 
    evento_enfermaria,
    evento_amb,
    col_data_referencia='data_cirurgia',   
    col_data_documento='DH_DOCUMENTO',
    col_instancia='redcap_repeat_instance',
    # --- Parâmetros Flexíveis ---
    col_criterio_local='CD_DOCUMENTO',     # Pode ser 'CD_DOCUMENTO' ou 'NM_SETOR'
    val_uti=(1140, 1145, 1137),            # Pode ser tupla de IDs ou string 'UTI'
    val_enfermaria=(1136, 1141),           # Pode ser tupla de IDs ou string 'ENF'
    val_amb='AMBULATORIO'				   # Pode ser tupla de IDs ou string 'AMBULATORIO'
):
    """
    Classifica em Pré ou Pós (UTI/Enfermaria) ou Ambulatório (Alta/Follow-up) de forma flexível.
    Default (caso não case com nada): evento_pre.
    Aceita classificação por IDs (usando lista/tupla) ou por texto (usando string parcial).
    """
    df = df_entrada.copy()

    # 1. Garantia de datetime
    df[col_data_documento] = pd.to_datetime(df[col_data_documento], errors='coerce')
    df[col_data_referencia] = pd.to_datetime(df[col_data_referencia], errors='coerce')
    
    # Validação
    if col_criterio_local not in df.columns:
        raise ValueError(f"A coluna '{col_criterio_local}' não existe no DataFrame.")

    # 2. Função interna para criar máscara
    def criar_mascara(coluna, valores):
        if valores is None:
            return pd.Series([False] * len(df), index=df.index)
        
        if isinstance(valores, (list, tuple, set)):
            return pd.to_numeric(coluna, errors='coerce').isin(valores)
        elif isinstance(valores, str):
            return coluna.astype(str).str.contains(valores, case=False, na=False, regex=False)
        else:
            return pd.Series([False] * len(df), index=df.index)

    # 3. Criação das Máscaras Booleanas
    is_uti = criar_mascara(df[col_criterio_local], val_uti)
    is_enf = criar_mascara(df[col_criterio_local], val_enfermaria)
    is_amb = criar_mascara(df[col_criterio_local], val_amb)
    
    tem_cirurgia = df[col_data_referencia].notna()
    tem_doc = df[col_data_documento].notna()

    # 4. Condições (Prioridade Sequencial)
    condicoes = [
        # 1. Sem data de cirurgia -> Pré
        (tem_doc & ~tem_cirurgia),

        # 2. Pré-operatório Real (Data Doc < Data Cirurgia) -> Pré
        (tem_cirurgia & (df[col_data_documento] < df[col_data_referencia])),

        # 3. Pós-operatório UTI
        (tem_cirurgia & (df[col_data_documento] >= df[col_data_referencia]) & is_uti),

        # 4. Pós-operatório Enfermaria
        (tem_cirurgia & (df[col_data_documento] >= df[col_data_referencia]) & is_enf),

        # 5. Pós-operatório Ambulatório
        (tem_cirurgia & (df[col_data_documento] >= df[col_data_referencia]) & is_amb)
    ]

    # 5. Escolhas correspondentes
    escolhas = [
        evento_pre,        
        evento_pre,        
        evento_uti,        
        evento_enfermaria, 
        evento_amb         
    ]

    # Aplica a seleção
    # CORREÇÃO: Default agora é o evento_pre (cobre casos indefinidos)
    df['redcap_event_name'] = np.select(condicoes, escolhas, default=evento_pre)

    # --- Tratamento da Instância ---
    df[col_instancia] = df[col_instancia].fillna(1).astype(int)

    return df


In [17]:
def classificar_eventos_redcap(df_entrada, evento_antes, evento_depois, 
                               col_data_referencia='data_cirurgia', 
                               col_data_documento='DH_DOCUMENTO'):
    """
    Classifica o evento do REDCap baseando-se na comparação entre duas datas.

    Parâmetros:
    -----------
    df_entrada : pd.DataFrame
        DataFrame contendo as colunas de datas.
    evento_antes : str
        Nome do evento se (Data Documento < Data Referência) ou se Data Referência for nula.
    evento_depois : str
        Nome do evento se (Data Documento >= Data Referência).
    col_data_referencia : str (Padrão: 'data_cirurgia')
        Nome da coluna que serve de marco divisor (ex: cirurgia, alta).
    col_data_documento : str (Padrão: 'DH_DOCUMENTO')
        Nome da coluna com a data do documento/atendimento a ser classificado.
    """
    
    # Cria cópia para segurança
    df_final = df_entrada.copy()
    
    # 1. Validação básica: Verifica se as colunas existem
    if col_data_documento not in df_final.columns:
        raise ValueError(f"A coluna '{col_data_documento}' não foi encontrada no DataFrame.")
    if col_data_referencia not in df_final.columns:
        # Nota: Se a coluna de referência não existir, o código quebraria. 
        # Aqui assumimos que ela deve existir, mesmo que tenha valores nulos.
        raise ValueError(f"A coluna de referência '{col_data_referencia}' não foi encontrada.")

    # 2. Conversão de Datas (Garante datetime)
    df_final[col_data_documento] = pd.to_datetime(df_final[col_data_documento], errors='raise')
    df_final[col_data_referencia] = pd.to_datetime(df_final[col_data_referencia], errors='raise')

    # 3. Lógica de Classificação
    def definir_evento(row):
        # Sem data do documento -> Impossível classificar
        if pd.isnull(row[col_data_documento]):
            return np.nan 

        # Sem data de referência -> Assume evento ANTERIOR (Regra de negócio padrão)
        if pd.isnull(row[col_data_referencia]):
            return evento_antes
        
        # --- COMPARAÇÃO ---
        
        # Caso A: Documento aconteceu ANTES do marco de referência
        if row[col_data_documento] < row[col_data_referencia]:
            return evento_antes
        
        # Caso B: Documento aconteceu DEPOIS ou no MESMO MOMENTO
        else:
            return evento_depois

    # 4. Aplicação
    df_final['redcap_event_name'] = df_final.apply(definir_evento, axis=1)

    return df_final

In [18]:
def limpar_dados_para_importacao(df_input, cols_decimais=None):
    """
    LIMPA e NORMALIZA DataFrame para importação no REDCap.
    
    REGRAS:
    - Tudo sai como STRING
    - Decimais apenas nas colunas explicitadas
    - Inteiros permanecem inteiros (sem .0)
    - 'altura' é tratada como cm (remove ponto e vírgula)
    - Remove caracteres ilegais
    - Padroniza nulos
    """
    if cols_decimais is None:
        cols_decimais = []

    df = df_input.copy()

    # --------------------------------------------------
    # 1️⃣ Padronização inicial de nulos
    # --------------------------------------------------
    df = df.replace(
        ['None', 'nan', 'NaN', 'NaT', '<NA>', ' ', ''],
        np.nan
    )

    # Regex de caracteres ilegais para REDCap
    illegal_chars_re = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")

    # --------------------------------------------------
    # 2️⃣ Tratamento especial: ALTURA (cm inteiro)
    # --------------------------------------------------
    if 'altura' in df.columns:
        df['altura'] = (
            df['altura']
            .astype(str)
            .str.strip()
            .str.replace(',', '.', regex=False)
            .str.replace('[^0-9.]', '', regex=True)
            .replace('', np.nan)
            .astype(float)
            .mul(100)                # converte para cm
            .round(0)
            .astype('Int64')
        )

    # --------------------------------------------------
    # 3️⃣ Tratamento de colunas DECIMAIS conhecidas
    # --------------------------------------------------
    for col in cols_decimais:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(',', '.', regex=False)
                .replace('', np.nan)
                .pipe(pd.to_numeric, errors='coerce')
                .round(2)
            )

    # --------------------------------------------------
    # 4️⃣ Demais colunas: numérico simples (inteiros)
    # --------------------------------------------------
    for col in df.columns:
        if col in cols_decimais or col == 'altura':
            continue

        df[col] = pd.to_numeric(df[col], errors='ignore')

    # --------------------------------------------------
    # 5️⃣ Limpeza textual final + conversão para string
    # --------------------------------------------------
    for col in df.columns:
        df[col] = df[col].astype(str)

        df[col] = (
            df[col]
            .str.replace(';', '-', regex=False)
            .str.replace(illegal_chars_re, '', regex=True)
            .replace(['nan', 'None', '<NA>'], '')
        )

        # Remove .0 residuais (segurança extra)
        if col not in cols_decimais:
            df[col] = df[col].str.replace(r'\.0$', '', regex=True)

    return df


In [19]:
def filtrar_registros_concluidos(df_novos_dados, df_status_redcap, col_status_form):
    """
    Remove do DataFrame de novos dados os registros que já estão com status:
    - '2' (Completo)
    - '0' (Incompleto - Já iniciado manualmente)
    
    Só permite atualização se o status for '1' (Unverified) ou se o registro for novo (NaN).
    """
    
    # 1. Preparação
    df_novos = df_novos_dados.copy()
    df_status = df_status_redcap.copy()
    
    # Garante tipagem de string para o merge
    df_novos['record_id'] = df_novos['record_id'].astype(str)
    df_status['record_id'] = df_status['record_id'].astype(str)
    
    # Define colunas de junção (record_id + evento se houver)
    chaves_merge = ['record_id']
    if 'redcap_event_name' in df_novos.columns and 'redcap_event_name' in df_status.columns:
        chaves_merge.append('redcap_event_name')
        
    # 2. Cruzamento (Merge Left)
    df_merge = pd.merge(
        df_novos,
        df_status[chaves_merge + [col_status_form]],
        on=chaves_merge,
        how='left'
    )
    
    # 3. Lógica de Blindagem Ajustada
    # Normaliza a coluna de status para string limpa ('2.0' vira '2')
    status_normalizado = df_merge[col_status_form].astype(str).str.replace('.0', '', regex=False)
    
    # DEFINIÇÃO DO BLOQUEIO:
    # Bloqueia se for '2' (Completo) OU '0' (Incompleto)
    # Apenas '1' (Unverified) ou 'nan' (Novo) passarão.
    valores_bloqueados = ['0', '2']
    
    condicao_blindagem = status_normalizado.isin(valores_bloqueados)
    
    # Aplica o filtro (Inverte a condição com ~ para pegar os NÃO bloqueados)
    df_filtrado = df_merge[~condicao_blindagem].copy()
    
    # Limpeza final (remove coluna auxiliar de status)
    if col_status_form in df_filtrado.columns:
        del df_filtrado[col_status_form]
        
    # --- Relatório ---
    total_entrada = len(df_novos)
    total_saida = len(df_filtrado)
    bloqueados = total_entrada - total_saida
    
    print(f"--- RELATÓRIO DE BLINDAGEM ({col_status_form}) ---")
    print(f"Entrada: {total_entrada}")
    print(f"Bloqueados (Status 0 ou 2): {bloqueados}")
    print(f"Liberados (Novos ou Status 1): {total_saida}")
    
    return df_filtrado

In [20]:
def unificar_colunas_sim_nao(df, col_destino, col_nao, col_sim, valor_positivo='SIM'):
    """
    Consolida duas colunas (Sim/Nao) em uma única (0/1).
    
    Lógica:
    - Se col_sim == 'SIM' -> 1
    - Se col_nao == 'SIM' -> 0
    - Se ambas vazias -> NaN
    - Se ambas 'SIM' (erro de cadastro) -> Prioriza 1 (ou você pode alterar a ordem)
    """
    
    # Condições (A ordem importa: a primeira que for True vence)
    condicoes = [
        df[col_sim] == valor_positivo,  # Prioridade 1: Se for SIM, vale 1
        df[col_nao] == valor_positivo   # Prioridade 2: Se for NÃO, vale 0
    ]
    
    valores = [1, 0]
    
    # Aplica a lógica
    df[col_destino] = np.select(condicoes, valores, default=np.nan)
    
    return df

In [21]:
def adicionar_textos_concatenados_por_fase(
    df, 
    cols_agrupamento, # AGORA É UMA LISTA: ['cod_atendimento', 'redcap_event_name']
    mapa_regras, 
    separador=' // '
):
    """
    1. Gera os textos concatenados agrupando por Atendimento E Fase.
    2. Devolve o df original com as novas colunas, respeitando o evento.
    """
    
    # Lista para guardar os resumos gerados
    lista_series = []
    
    for col_final, (col_bool, col_texto) in mapa_regras.items():
        
        # 1. Filtra apenas onde a booleana é verdadeira
        # Isso garante que só vamos pegar o texto se ele realmente existiu naquela fase
        mask = (df[col_bool] == 1) | (df[col_bool] == '1')
        df_temp = df[mask].copy()
        
        # 2. Limpa texto
        df_temp[col_texto] = df_temp[col_texto].astype(str).fillna('')
        df_temp = df_temp[df_temp[col_texto].str.strip() != '']
        
        # 3. Agrupa por ID E FASE (Isola o contexto)
        # Ex: Paciente 100 na Fase 4 -> "Dobuta"
        # Ex: Paciente 100 na Fase 5 -> "Dipirona" (sem misturar com Dobuta)
        serie_agrupada = df_temp.groupby(cols_agrupamento)[col_texto].apply(
            lambda x: separador.join(sorted(set(x)))
        )
        serie_agrupada.name = col_final
        lista_series.append(serie_agrupada)
    
    # 4. Se não houver regras, devolve original
    if not lista_series:
        return df 
        
    # 5. Cria o DataFrame de resumo (Multi-Index: Atendimento + Evento)
    df_resumo = pd.concat(lista_series, axis=1).reset_index()
    
    # 6. Merge de volta usando AS DUAS CHAVES
    # O texto da Fase 4 só será colado nas linhas da Fase 4
    df_final = pd.merge(
        df, 
        df_resumo, 
        on=cols_agrupamento, 
        how='left'
    )
    
    return df_final

In [22]:
def filtrar_novos_registros(df_novo, chaves_proibidas):
    """
    Filtra registros que já existem no REDCap para evitar duplicidade.
    Assume que chaves_proibidas é um set de tuplas (record_id, event, instance).
    """
    if df_novo.empty:
        return df_novo
    
    df_novo = df_novo.copy()
    
    # Padronização para comparação (String é mais seguro para a API)
    df_novo['record_id'] = df_novo['record_id'].astype(str)
    df_novo['redcap_repeat_instance'] = df_novo['redcap_repeat_instance'].astype(str)
    
    # Criamos a máscara de filtragem
    # Nota: O zip deve bater com a estrutura do seu 'set' de registros_existentes
    filtro = []
    ignorado_count = 0
    
    for rid, evt, inst in zip(df_novo['record_id'], 
                               df_novo['redcap_event_name'], 
                               df_novo['redcap_repeat_instance']):
        
        # A chave de busca deve ter a mesma 'forma' da chave_proibida
        chave_busca = (rid, evt, inst)
        
        if chave_busca not in chaves_proibidas:
            filtro.append(True)
        else:
            filtro.append(False)
            ignorado_count += 1
    
    df_filtrado = df_novo[filtro].copy()
    
    print(f"📊 Filtro aplicado:")
    print(f"   - Registros originais no DataFrame: {len(df_novo)}")
    print(f"   - Registros ignorados (já existem no REDCap): {ignorado_count}")
    print(f"   - Novos registros para importar: {len(df_filtrado)}")
    
    return df_filtrado

In [23]:
# Função para limpar: converte 1.0 -> '1', 0.0 -> '0', e mantém vazios como vazios
def limpar_formato_redcap(val):
    if pd.isna(val) or str(val).strip() == '':
        return ''
    try:
        # Converte para float primeiro (para pegar '1.0'), depois int, depois string
        return str(int(float(val)))
    except:
        return str(val)

In [24]:
def extrair_instrumento(df, lista_cols, nome_instrumento):
    """
    Extrai colunas específicas para instrumento REDCap com formatação padrão.
    """
    # Filtra apenas colunas que existem
    cols_existentes = [c for c in lista_cols if c in df.columns]
    
    # Sub-df com colunas base + específicas
    sub_df = df[cols_base + cols_existentes].copy()
    
    # Colunas obrigatórias REDCap
    sub_df['redcap_repeat_instrument'] = nome_instrumento
    sub_df[f'{nome_instrumento}_complete'] = 0  # 2=Complete
    
    # Ordem REDCap padrão
    cols_finais = (['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance'] + 
                   cols_existentes + [f'{nome_instrumento}_complete'])
    return sub_df[cols_finais]


In [25]:
def mesclar_preservando_redcap(lista_datasets, chaves):
    """
    lista_datasets: Lista de tuplas [(df_redcap_1, df_sql_1), (df_redcap_2, df_sql_2), ...]
    chaves: Lista de colunas de identificação
    """

    resultados = []

    for df_redcap, df_sql in lista_datasets:
        # 1. Cópias seguras
        df_r = df_redcap.copy()
        df_s = df_sql.copy()

        # 2. Garantir tipos iguais nas chaves
        for c in chaves:
            if c in df_r.columns:
                df_r[c] = df_r[c].astype(str)
            if c in df_s.columns:
                df_s[c] = df_s[c].astype(str)

        # 3. Definir chaves válidas
        chaves_existentes = [
            c for c in chaves 
            if c in df_r.columns and c in df_s.columns
        ]

        df_base = df_r.set_index(chaves_existentes)
        df_novos = df_s.set_index(chaves_existentes)

        # 4. Garantir que ambos tenham mesmas colunas
        for col in df_novos.columns:
            if col not in df_base.columns:
                df_base[col] = pd.NA

        for col in df_base.columns:
            if col not in df_novos.columns:
                df_novos[col] = pd.NA

        # 5. Atualização segura: valor novo sobrescreve se não for NaN nem vazio
        for col in df_novos.columns:
            mask = (
                df_novos[col].notna() &
                (df_novos[col].astype(str).str.strip() != "")
            )
            df_base.loc[mask, col] = df_novos.loc[mask, col]

        df_final = df_base.reset_index()

        resultados.append(df_final)

    return resultados

In [26]:
def aplicar_cheque_seguranca_prioritaria(df_redcap, df_sql, cols_exames):
    rc = df_redcap.copy()
    sql = df_sql.copy()
    
    # 1. TRATAMENTO DE CHAVES E DATAS ANTES DO MERGE
    keys = ['record_id', 'redcap_event_name', 'redcap_repeat_instance']
    
    for df in [rc, sql]:
        # Normaliza chaves como string limpa
        for k in keys:
            if k in df.columns:
                df[k] = df[k].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        
        # FORÇA dt_lab PARA DATETIME (Garante que ambos falem a mesma língua)
        if 'dt_lab' in df.columns:
            df['dt_lab'] = pd.to_datetime(df['dt_lab'], errors='coerce')

    # 2. OUTER MERGE
    merged = rc.merge(sql[keys + cols_exames], on=keys, how='outer', suffixes=('', '_sql'))

    # 3. LOGICA DE COMPARAÇÃO COLUNA POR COLUNA
    for col in cols_exames:
        col_sql = f"{col}_sql"
        if col_sql not in merged.columns:
            continue
            
        # TRATAMENTO ESPECIAL PARA A COLUNA DE DATA
        if col == 'dt_lab':
            # Se o REDCap está vazio, usa a data do SQL
            merged['dt_lab'] = merged['dt_lab'].fillna(merged['dt_lab_sql'])
            continue

        # TRATAMENTO PARA CAMPOS NUMÉRICOS/BINÁRIOS (hb_pre, creatinina, etc.)
        v_rc = pd.to_numeric(merged[col], errors='coerce')
        v_sql = pd.to_numeric(merged[col_sql], errors='coerce')

        # Regra: Upgrade (0->1) e Preenchimento de vazios
        temp_res = np.fmax(v_rc.fillna(-1), v_sql.fillna(-1))
        merged[col] = temp_res.replace(-1, np.nan)
        merged[col] = merged[col].fillna(v_sql)

    # 4. FORMATAÇÃO FINAL PARA O REDCAP
    # Converte dt_lab de volta para date (sem H:M:S) para o importador
    if 'dt_lab' in merged.columns:
        merged['dt_lab'] = pd.to_datetime(merged['dt_lab']).dt.date

    # Retorna apenas colunas do REDCap
    return merged[df_redcap.columns].copy()

In [ ]:
#rg_pct = ', '.join("'" + str(id) + "'" for id in df_angio['CD_PACIENTE_HIS'].astype('Int64').dropna().unique())

In [ ]:
# # Caminho do arquivo
# caminho_csv = r"C://Users/priscilla.sequetin/Downloads/RegValvDANTE_DATA_2026-02-09_1209.csv"

# # Leitura do CSV
# # Nota: O REDCap costuma exportar com codificação 'utf-8', 
# # mas se der erro de caracteres, tente 'latin1'
# def carregar_e_limpar_redcap(caminho_csv, separador='|'):
#     # 1. Carregar a base (lendo record_id como string para garantir)
#     try:
#         # Tentamos ler normalmente primeiro
#         df = pd.read_csv(caminho_csv, sep=separador, low_memory=False, encoding='utf-8')
#     except UnicodeDecodeError:
#         df = pd.read_csv(caminho_csv, sep=separador, low_memory=False, encoding='latin1')

#     print(f"Base carregada: {df.shape[1]} colunas detectadas.")

#     # 2. Percorrer todas as colunas automaticamente
#     for col in df.columns:
#         # Se for uma coluna de números decimais (onde aparece o .0)
#         if df[col].dtype == 'float64':
#             # Removemos os nulos temporariamente para checar se o que sobrou é inteiro
#             valores_nao_nulos = df[col].dropna()
            
#             # Se a coluna não estiver totalmente vazia e todos os valores forem "redondos"
#             if not valores_nao_nulos.empty and valores_nao_nulos.apply(lambda x: float(x).is_integer()).all():
#                 # Converte para Int64 (Inteiro do Pandas que aceita NaN e não põe .0)
#                 df[col] = df[col].astype('Int64')
        
#         # Garante que o record_id e campos de ID fiquem sem .0 e como texto
#         if 'id' in col.lower() or 'record' in col.lower():
#             # Remove .0 se o pandas tiver interpretado como float e transforma em string
#             df[col] = df[col].astype(str).str.replace('\.0$', '', regex=True)

#     return df

# # --- Execução ---
# df = carregar_e_limpar_redcap(caminho_csv)

# # # Verificar o instrumento específico
# # id_alvo = "exames_laboratoriais"
# # instrumento = df[df['redcap_repeat_instrument'] == id_alvo]

# # print("Colunas encontradas:", df.columns.tolist())
# # print(f"\nDados do registro {id_alvo}:")
# # #print(instrumento)

<>:32: SyntaxWarning: invalid escape sequence '\.'
<>:32: SyntaxWarning: invalid escape sequence '\.'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1389659404.py:32: SyntaxWarning: invalid escape sequence '\.'
  df[col] = df[col].astype(str).str.replace('\.0$', '', regex=True)


Base carregada: 1250 colunas detectadas.


In [ ]:
# # Verificar o instrumento específico
# id_alvo = "ecocardiograma"
# instrumento = df[df['redcap_repeat_instrument'] == id_alvo]

# print("Colunas encontradas:", df.columns.tolist())
# print(f"\nDados do registro {id_alvo}:")
# #print(instrumento)

Colunas encontradas: ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance', 'cd_paciente', 'data_inclusao', 'nome_paciente', 'data_nascimento', 'genero', 'etnia', 'nr_ddd_fone', 'nr_fone_1', 'nr_ddd_celular', 'nr_celular', 'nr_ddd_fone_comercial', 'nr_fone_comercial', 'consentimento_informado', 'import_tcle', 'screening', 'regvalv_continue', 'dados_demograficos_complete', 'altura', 'peso', 'imc', 'imc_classificacao', 'assintomatico', 'carga_sintomas_dispneia', 'classe_nyha', 'angina', 'ccs', 'carga_sintomas_sincope', 'palpitacoes', 'carga_sintomas_palpitacoes', 'fadiga', 'carga_sintomas_fadiga', 'hipertensao', 'diabetes', 'tipo_diabetes', 'diabetes_controle', 'tabagismo', 'carga_tabagica', 'etilista', 'dislipidemia', 'drc_clcr', 'doenca_renal_cronica', 'estagio_drc', 'dpoc', 'dpoc_medicamentos', 'iam_menor_90dias', 'hipertensao_pulmonar_encam', 'valor_psap_encam', 'avc_ait_previo', 'sequela_neurologica', 'data_avc_ait', 'endocardite_ativa', 'data_endoc

In [ ]:
# instrumento1 = instrumento[['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance', 'dt_lab', 'hb_pre',
#         'leucocitos_pre', 'plaquetas_pre', 'creatinina_pre', 'ureia_pre', 'bnp_pre',  'hba1c_encam', 'tsh_encam', 'sorologias_encam',
#         'exames_laboratoriais_complete']].copy()


In [ ]:
# instrumento2 = instrumento[['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance', 'data_do_eco', 'raiz_aorta',
#         'aorta_ascendente', 'disfuncao_ventricular_encam', 'disfuncao_ventricular_encam_2', 'feve_encam', 'fac_encam',  'tapse_encam', 'resultado_eco_psap',
#         'resultado_eco_dimensao_ve', 'resultado_eco_dimensao_ae',   'ecocardiograma_complete']].copy()


In [ ]:
#instrumento2.to_excel('C://Users/priscilla.sequetin/Downloads/instrumento3.xlsx', index=False)

In [ ]:
# import xml.etree.ElementTree as ET

# def xml_to_dataframe(caminho_xml):
#     # Namespaces padrão do REDCap
#     ns = {'odm': 'http://www.cdisc.org/ns/odm/v1.3'}
    
#     tree = ET.parse(caminho_xml)
#     root = tree.getroot()
    
#     lista_registros = []
    
#     # Percorre cada paciente (SubjectData)
#     for subject in root.findall('.//odm:SubjectData', ns):
#         row = {'record_id': subject.get('StudySubjectID')}
        
#         # Percorre todos os campos (ItemData) de cada paciente
#         for item in subject.findall('.//odm:ItemData', ns):
#             var_nome = item.get('ItemOID')
#             valor = item.get('Value')
#             row[var_nome] = valor
            
#         lista_registros.append(row)
    
#     return pd.DataFrame(lista_registros)

# # 1. Carregar o XML para um DataFrame
# caminho = r"C:\Users\priscilla.sequetin\Downloads\RegValvDANTE_2026-02-05_0904.REDCap.xml"
# df_ajuste = xml_to_dataframe(caminho)

# 2. Localizar e Alterar o registro específico
# Usamos .loc[filtro, coluna] para garantir a alteração no DF original
#id_alvo = "458634"

# Exemplo de alteração em múltiplas colunas para esse record_id
# df.loc[df['record_id'] == id_alvo, 'idade'] = '65'
# df.loc[df['record_id'] == id_alvo, 'sexo'] = '1'
# df.loc[df['record_id'] == id_alvo, 'status_vivos'] = 'Sim'

# 3. Verificar se a alteração funcionou
#print(f"Dados atualizados do paciente {id_alvo}:")
#print(df_ajuste[df_ajuste['record_id'] == id_alvo])

# 4. Agora você pode salvar como CSV ou Excel, que é mais fácil de lidar
# df.to_csv("base_final_tcc.csv", index=False)

# REDCAP RegValv, Pacientes, Procedimentos, Geronto, Fisio

In [27]:
## Check Input ID Paciente
# Define the form and the field you want to export
form_name = ['dados_demograficos']  # Replace with the form (instrument) name
field_name = ['record_id']  # Replace with the field name you want to export
 
# Prepare the data payload for the request
data = {
    'token': api_key,
    'content': 'record',
    'format': 'json',  # or 'csv', 'xml', depending on the format you need
    'type': 'flat',  # you can change this to 'eav' or 'nonform' depending on your need
    'forms': [form_name],  # Specify the form (instrument) name
    'fields': [field_name],  # Specify the field name
    'returnFormat': 'json'
}
 
# # Send the POST request
response = requests.post(api_url, data=data)
 
regvalv_pcte = pd.DataFrame(response.json())

regvalv_pcte = regvalv_pcte[regvalv_pcte['redcap_event_name']=='fase1preoperatorio_arm_1']

In [28]:
def exportar_instrumento_redcap(nome_instrumento, api_url, api_key, campos_especificos=None):
    """
    Exporta um instrumento específico do REDCap e filtra pelo repeat_instrument.
    
    Args:
        nome_instrumento (str): O nome técnico do formulário no REDCap (ex: 'odontologia').
        api_url (str): URL da API.
        api_key (str): Token de acesso.
        campos_especificos (list): (Opcional) Lista de campos extras. Por padrão traz record_id + form.
    
    Returns:
        pd.DataFrame: DataFrame processado e filtrado.
    """
    
    # Configuração básica do payload
    data = {
        'token': api_key,
        'content': 'record',
        'format': 'json',
        'type': 'flat',
        'forms[0]': nome_instrumento, # Solicita o formulário inteiro
        'fields[0]': 'record_id',     # Garante que o record_id venha junto
        'returnFormat': 'json'
    }

    # Se quiser campos específicos além do formulário, adiciona aqui
    if campos_especificos:
        for i, campo in enumerate(campos_especificos, start=1):
            data[f'fields[{i}]'] = campo

    try:
        # Envia a requisição
        response = requests.post(api_url, data=data)
        
        if response.status_code != 200:
            print(f"❌ Erro na API para '{nome_instrumento}': {response.text}")
            return pd.DataFrame() # Retorna vazio em caso de erro

        # Cria o DataFrame
        df = pd.DataFrame(response.json())
        
        # Se o DataFrame estiver vazio, retorna logo
        if df.empty:
            print(f"⚠️ Aviso: Nenhum dado encontrado para '{nome_instrumento}'.")
            return df

        # --- APLICAÇÃO DO FILTRO SOLICITADO ---
        # Verifica se é um instrumento de repetição antes de filtrar
        if 'redcap_repeat_instrument' in df.columns:
            df_filtrado = df[df['redcap_repeat_instrument'] == nome_instrumento].copy()
            
            # Se o filtro zerou o dataframe (ex: dados existem, mas não como repetição),
            # pode ser que o dado esteja na linha base (sem repeat_instrument preenchido).
            # Mas seguindo sua regra estrita:
            df = df_filtrado
        
        # Opcional: Já padronizar o record_id para Int64 como conversamos antes
        if 'record_id' in df.columns:
            df['record_id'] = df['record_id'].astype('Int64')

        print(f"✅ '{nome_instrumento}' : {len(df)} registros.")
        return df

    except Exception as e:
        print(f"❌ Erro crítico ao processar '{nome_instrumento}': {e}")
        return pd.DataFrame()

# --- COMO USAR ---

# 1. Defina suas credenciais uma única vez
api_key
api_url

# 2. Crie uma lista com os nomes dos instrumentos que você quer baixar
instrumentos_para_baixar = [
    'procedimento',
    'qualidade_vida',
    'fisioterapia',
    'psicologia',
    'odontologia',
    'pre_operatoria',
    'ficha_complicacoes',
    'fatores_risco',
    'ecocardiograma',
    'coronarias',
    'anestesia',
    'pos_operatorio',
    'alta_hospitalar',
    'ambulatorio',
    'ficha_satisfacao',
    'exames_laboratoriais',
    'acompanhamento_nutricional'
]

# 3. Dicionário para guardar os resultados (Melhor organização)
dados_redcap = {}

for instrumento in instrumentos_para_baixar:
    # Chama a função e guarda no dicionário com o nome do instrumento
    dados_redcap[instrumento] = exportar_instrumento_redcap(instrumento, api_url, api_key)

# 4. Acessando os dados individuais depois:
regvalv_cirurgia = dados_redcap['procedimento']
regvalv_geronto = dados_redcap['qualidade_vida']
regvalv_fisio = dados_redcap['fisioterapia']
regvalv_psico = dados_redcap['psicologia']
regvalv_odonto = dados_redcap['odontologia']
regvalv_preop = dados_redcap['pre_operatoria']
regvalv_obito = dados_redcap['ficha_complicacoes']
regvalv_ftrisk = dados_redcap['fatores_risco']
regvalv_eco = dados_redcap['ecocardiograma']
regvalv_coron = dados_redcap['coronarias']
regvalv_anest = dados_redcap['anestesia']
regvalv_posop = dados_redcap['pos_operatorio']
regvalv_altah = dados_redcap['alta_hospitalar']
regvalv_ambval = dados_redcap['ambulatorio']
regvalv_fcsatis = dados_redcap['ficha_satisfacao']
regvalv_labs = dados_redcap['exames_laboratoriais']
regvalv_dnutri = dados_redcap['acompanhamento_nutricional']
print(f"✅ 'dados_demograficos': {len(regvalv_pcte['record_id'].unique())} registros.")


✅ 'procedimento' : 549 registros.
✅ 'qualidade_vida' : 746 registros.
✅ 'fisioterapia' : 860 registros.
✅ 'psicologia' : 601 registros.
✅ 'odontologia' : 479 registros.
✅ 'pre_operatoria' : 552 registros.
✅ 'ficha_complicacoes' : 553 registros.
✅ 'fatores_risco' : 554 registros.
✅ 'ecocardiograma' : 546 registros.
✅ 'coronarias' : 552 registros.
✅ 'anestesia' : 352 registros.
✅ 'pos_operatorio' : 452 registros.
✅ 'alta_hospitalar' : 429 registros.
✅ 'ambulatorio' : 225 registros.
✅ 'ficha_satisfacao' : 94 registros.
✅ 'exames_laboratoriais' : 550 registros.
✅ 'acompanhamento_nutricional' : 357 registros.
✅ 'dados_demograficos': 548 registros.


In [ ]:
#regvalv_geronto.to_excel(f"C://Users/priscilla.sequetin/Downloads/regvalv_geronto.xlsx", index=False)


In [ ]:
#aPct1.to_excel(f"C://Users/priscilla.sequetin/Downloads/demo1.xlsx", index=False)

In [29]:
def padronizar_tipo(lista_dfs, coluna, novo_tipo):
    """
    Percorre uma lista de DataFrames. Se a coluna existir no DataFrame,
    converte para o novo_tipo.
    
    Args:
        lista_dfs (list): Lista contendo os objetos DataFrame.
        coluna (str): Nome da coluna a ser convertida.
        novo_tipo (str ou type): Tipo alvo (ex: 'Int64', 'str', 'float', 'datetime64[ns]').
    """
    contador = 0
    for df in lista_dfs:
        # Só tenta converter se a coluna existir neste dataframe
        if coluna in df.columns:
            try:
                df[coluna] = df[coluna].astype(novo_tipo)
                contador += 1
            except Exception as e:
                print(f"⚠️ Erro ao converter '{coluna}' no dataframe (índice {lista_dfs.index(df)}): {e}")

    print(f"✅ Coluna '{coluna}' padronizada para '{novo_tipo}' em {contador} tabelas.")

# --- COMO USAR ---

# 1. Crie sua lista de instrumentos (basta fazer isso uma vez)
meus_instrumentos = [
    regvalv_pcte, 
    regvalv_cirurgia, 
    regvalv_geronto, 
    regvalv_fisio, 
    regvalv_psico, 
    regvalv_odonto,
    regvalv_preop,
    regvalv_obito,
    regvalv_ftrisk,
    regvalv_eco,
    regvalv_coron
]

# 2. Exemplo 1: Padronizando o ID (o que você pediu)
padronizar_tipo(meus_instrumentos, 'record_id', 'Int64')

# 3. Exemplo 2: Padronizando Datas (muito útil)
# Digamos que vários tenham a coluna 'dt_coleta'
#padronizar_tipo(meus_instrumentos, 'dt_coleta', 'datetime64[ns]')

# 4. Exemplo 3: Transformando Flags em Categoria (economiza memória)
#padronizar_tipo(meus_instrumentos, 'sexo', 'category')


✅ Coluna 'record_id' padronizada para 'Int64' em 11 tabelas.


In [ ]:
# # Converte a coluna e a lista para conjuntos (sets)
# set_dataframe = set(regvalv_pcte['record_id'])
# set_lista = set(lst)

# # Pergunta: O que tem na lista MENOS o que tem no dataframe?
# diferenca = set_lista - set_dataframe

# print(f"IDs que estão sobrando na lista: {diferenca}")

In [30]:
regvalv_cirurgia['procedimento_complete'].value_counts()

procedimento_complete
2    459
1     80
0     10
Name: count, dtype: int64

In [ ]:
#regvalv_cirurgia.to_excel(f"C://Users/priscilla.sequetin/Downloads/regvalv_cirurgia_1.xlsx", index=False)

In [31]:
# --- 1. DEFINIÇÃO DA "BLACKLIST" (O que já foi verificado e deve ser ignorado) ---

# lista de pendencias de procedimentos checadas em 05/01/2026 por cd_atendimento
lista_anterior = [1225508, 2043066, 884401, 1958664, 2152009, 2110943, 101997, 1354471, 1934634, 
                2141839, 1872848]

# Estes são os 12 record_id que você já sabe que não tem info e não quer processar de novo.
lista_ja_verificados = [587546, 712755, 884401, 910528, 985224, 
                        1023415, 1056201, 1089242, 1135603, 1145248, 1147643]

# --- 2. CAPTURA O CENÁRIO ATUAL (Todos os '0' do momento e os cd_atendimentos (registros com mais de 1 cirurgia)) ---
# Pegamos TODOS que estão incompletos ('0') no dataframe atual
check_cirurgia_1 = regvalv_cirurgia[(regvalv_cirurgia['procedimento_complete']=='0') & 
    (~regvalv_cirurgia['cd_atendimento'].isin(lista_anterior))]
check_cirurgia_2 = regvalv_cirurgia[(regvalv_cirurgia['procedimento_complete']=='1') & 
    (~regvalv_cirurgia['cd_atendimento'].isin(lista_anterior))]

lista_check_cir1 = check_cirurgia_1['record_id'].to_list()
lista_check_cir2 = check_cirurgia_2['record_id'].to_list()

# --- 3. A MÁGICA (FILTRO DO NOVO REGISTRO) ---
# Lógica: Tudo que está incompleto HOJE - Tudo que eu JÁ CONHEÇO
# O resultado será APENAS os novos IDs que entraram nessa condição recentemente
#teste = [324016, 1045208]

# Criamos sets para operações matemáticas de conjunto
set_incompletos = set(lista_check_cir1)
set_emespera = set(lista_check_cir2)
set_ignorados = set(lista_ja_verificados)
#set_teste = set(teste)

lista_a_processar = list((set_incompletos | set_emespera) - set_ignorados) #| set_teste)

print(f"IDs verificados (ignorados): {len(lista_ja_verificados)} , {lista_ja_verificados}")
print(f"IDs em espera (cirurgia): {len(lista_check_cir2)} , {lista_check_cir2}")
print(f"IDs incompletos encontrados hoje: {len(lista_a_processar)} , {lista_a_processar}")



IDs verificados (ignorados): 11 , [587546, 712755, 884401, 910528, 985224, 1023415, 1056201, 1089242, 1135603, 1145248, 1147643]
IDs em espera (cirurgia): 80 , [335458, 336893, 341220, 349574, 391604, 401562, 440359, 446949, 472525, 550953, 556037, 571572, 584721, 593227, 600762, 612689, 618437, 677960, 678566, 741733, 793042, 822752, 834779, 846884, 883945, 887664, 903485, 904606, 939793, 949704, 950526, 988625, 989233, 999140, 1005872, 1020833, 1043905, 1050096, 1073969, 1075661, 1085090, 1103315, 1104809, 1106694, 1110802, 1111616, 1112267, 1114593, 1121141, 1129508, 1129915, 1131889, 1133920, 1134420, 1140434, 1140620, 1143719, 1144992, 1145734, 1145924, 1146111, 1146922, 1148693, 1149962, 1150098, 1151700, 1152195, 1152246, 1154874, 1155826, 1156346, 1156393, 1156847, 1157043, 1158377, 1159923, 1160564, 1161235, 1161988, 1163216]
IDs incompletos encontrados hoje: 83 , [1161988, 556037, 1106694, 1149962, 584721, 1128722, 1161235, 939793, 1110802, 1148693, 846884, 1129508, 440359, 5

In [32]:
# Informe o ano para as consultas

ano = 2026

In [33]:
pct_cir_anterior = [587546, 458634, 1056201, 1089242, 1082666]


In [ ]:
## 1156 Ficha Pré Valvula
## 1157 Ficha Alta Hospitalar
## 1158 Ficha Pós Valvula
## 1030	UTI - COMPLICAÇÕES GERAIS
## 1182	FICHA DE COMPLICAÇÕES

# FICHA VALVULA PRE-OP

In [55]:
# Buscando docText.VALV

sql_VALV_texto = f"""
SET DATEFORMAT ymd;

WITH doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)
SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID,
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    fde.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,
    /* Fallback de setor:
       - Internação: usa unidade de passagem associada ao leito e ao instante
       - Ambulatorial: usa origem do atendimento (doa) */
    COALESCE(
        fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, 
        doa.DS_ORI_ATE
    ) AS NM_SETOR
FROM ft_doc_eletronico_texto fde
LEFT JOIN dim_campo_documento dcd ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt       ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

/* Internação (só tenta casar leito se o campo está preenchido) */
LEFT JOIN ft_internacao_unidade fiu 
       ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
      AND fde.DS_LEITO IS NOT NULL 
      AND fde.DS_LEITO <> ''
      AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
      AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
      AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

/* Ambulatorial */
LEFT JOIN ft_atendimento ate 
       ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
      AND ate.TP_ATENDIMENTO <> 'I'
LEFT JOIN doa 
       ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

WHERE 
    fde.CD_DOCUMENTO IN (1156)
    AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
    --AND (fde.DS_RESPOSTA IS NOT NULL AND fde.DS_RESPOSTA <> 'false')
    --AND YEAR(fde.DH_DOCUMENTO) >= {ano}
    AND YEAR(fde.DH_DOCUMENTO) >= 2025;


""" 

# Buscando docRadio.VALV

sql_VALV_radio = f"""
SET DATEFORMAT ymd;

WITH doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)
SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    fde.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.RESPOSTA AS DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,
    COALESCE(
        fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, 
        doa.DS_ORI_ATE
    ) AS NM_SETOR
FROM FT_DOC_ELETRONICO fde 
LEFT JOIN dim_campo_documento dcd ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt       ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

LEFT JOIN ft_internacao_unidade fiu 
       ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
      AND fde.DS_LEITO IS NOT NULL 
      AND fde.DS_LEITO <> ''
      AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
      AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
      AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

/* Mantive a mesma lógica de fallback do docText (para consistência) */
LEFT JOIN ft_atendimento ate 
       ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
LEFT JOIN doa 
       ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

WHERE 
    fde.CD_DOCUMENTO IN (1156)
    AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
    --AND fde.RESPOSTA = 'SIM'
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND YEAR(fde.DH_DOCUMENTO) >= 2025

ORDER BY fde.DH_DOCUMENTO DESC;

""" 


In [ ]:
#Organiza as colunas mapeadas

preop_ls = {
	'diabetes': ['RB_DM_NAO_1', 'RB_DM_SIM_1'],
    'hipertensao': ['RB_HAS_NAO_1', 'RB_HAS_SIM_1'],
	'origem_do_paciente': ['RB_PS_UTI_ORIGEM_PACIENTE_1', 'RB_ELETICO_ORIGEM_PACIENTE_1'],
    'sintomas_duracao': ['RB_ANOS_SINTOMAS_FICHA_PRE_1', 'RB_DIAS_AO_SINTOMAS_FICHA_PRE_1', 'RB_MESES_SINTOMAS_FICHA_PRE_1'],
    'hipertensao_pulmonar_encam': ['RB_HIPERTENSAO_PULMONAR_NAO_1', 'RB_HIPERTENSAO_PULMONAR_SIM_1', 
                                   'RB_HIPERTENSAO_ARTERIAL_PULMONAR_NAO_COMORBIDADES_PRINCIPAIS_1', 'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_COMORBIDADES_PRINCIPAIS_1'],
	'valor_psap_encam': ['RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_PSAP_ENTRE_31_55_MMHG_1', 'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_PSAP_MAIOR_55_MMHG_1', 
                      'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_PSAP_MENOR_31_MMHG_1'],
    'disfuncao_ventricular_encam': ['RB_DISFUNCAO_VENTRICULAR_VE_NAO_1', 'RB_DISFUNCAO_VENTRICULAR_VE_SIM_1'],
    'disfuncao_ventricular_encam_2': ['RB_DISFUNCAO_VENTRICULAR_VD_NAO_1',  'RB_DISFUNCAO_VENTRICULAR_VD_SIM_1'],
    'aceita_transfusao': ['RB_ACEITA_TRANSFUSAO_NAO_1', 'RB_ACEITA_TRANSFUSAO_SIM_1'],
    'anticoagulacao': ['RB_ANTICOAGULACAO_DOAC_VALVOPATIAS_1', 'RB_ANTICOAGULACAO_VARFARINA_VALVOPATIAS_1'],
    'assintomatico': ['RB_ASSINTOMATICO_NAO_VALPATIAS_1', 'RB_ASSINTOMATICO_SIM_VALPATIAS_1'],
    'fadiga': ['RB_FADIGA_NAO_VALPATIAS_1', 'RB_FADIGA_SIM_VALPATIAS_1'],
    'palpitacoes': ['RB_PALPITACOES_NAO_VALVOPATIAS_1', 'RB_PALPITACOES_SIM_VALVOPATIAS_1'],
    'carga_sintomas_sincope': ['RB_SINCOPE_NAO_VALVOPATIAS_1', 'RB_SINCOPE_SIM_VALVOPATIAS_1'],
	'angina': ['RB_ANGINA_NAO_VALPATIAS_1',  'RB_ANGINA_SIM_VALPATIAS_1'],
    'ccs': ['RB_ANGINA_SIM_CCS1_SINTOMAS_PRINCIPAL_1', 'RB_ANGINA_SIM_CCS2_SINTOMAS_PRINCIPAL_1', 'RB_ANGINA_SIM_CCS3_SINTOMAS_PRINCIPAL_1', 'RB_ANGINA_SIM_CCS4_SINTOMAS_PRINCIPAL_1'],
    'classe_nyha': ['RB_DISPNEIA_CF_NYHA_I_1', 'RB_DISPNEIA_CF_NYHA_II_1', 'RB_DISPNEIA_CF_NYHA_III_1', 'RB_DISPNEIA_CF_NYHA_IV_1'],
    'cirurgia_proposta': ['PRIMEIRA_CIRURGIA_VALPATIAS_1', 'RB_REOPERACAO_PRIMEIRA_VALPATIAS_1', 'RB_REOPERACAO_SEGUNDA_VALPATIAS_1', 'RB_REOPERACAO_TERCEIRO_VALPATIAS_1', 
                          'RB_REOPERACAO_QUARTA_VALPATIAS_1', 'RB_REOPERACAO_QUINTO_VALPATIAS_1', 'RB_REOPERACAO_SEXTO_VALPATIAS_1'], 
    'tipo_operacao': ['RB_ELETIVA_OPERACAO_1', 'RB_URGENCIA_OPERACAO_1', 'RB_EMERGENCIA_OPERACAO_1', 'RB_SALVAMENTO_OPERACAO_1'],
    'heart_team_encam': ['RB_NAO_DISCUTIDO_HEART_TEAM_1', 'RB_SIM_DISCUTIDO_HEART_TEAM_1'],
    'drc_clcr': ['RB_DOENCA_RENAL_DIALISE_1', 'RB_DOENCA_RENAL_MODERADA_CLCR_ENTRE_50_85_1', 'RB_DOENCA_RENAL_NORMAL_CLCR_MAIOR_85_1', 'RB_DOENCA_RENAL_SEVERA_CLCR_MENOR_50_1'],
    'estado_pr_operat_rio_cr_ti': ['RB_ESTADO_PRE_OPERATORIO_CRITICO_NAO_1', 'RB_ESTADO_PRE_OPERATORIO_CRITICO_SIM_1'], 
    'mobilidade_severa_preju': ['RB_MOBILIDADE_SEVERAMENTE_PREJUDICADA_NAO_1', 'RB_MOBILIDADE_SEVERAMENTE_PREJUDICADA_SIM_1'],
    'arteriopatia_extracardiaca': ['RB_ARTERIOPATIA_EXTRACARDIACA_NAO_1', 'RB_ARTERIOPATIA_EXTRACARDIACA_SIM_1'],
    'iam_menor_90dias': ['RB_INFARTO_MIOCARDIO_MENOR_90_DIAS_NAO_1', 'RB_INFARTO_MIOCARDIO_MENOR_90_DIAS_SIM_1'],
    'dpoc_medicamentos': ['RB_USO_CORTICOIDE_BRONCODILATADOR_LONGA_DATA_NAO_1', 'RB_USO_CORTICOIDE_BRONCODILATADOR_LONGA_DATA_SIM_1'],
    'dpoc': ['RB_DOENCA_PULMONAR_CRONICA_NAO_COMORBIDADES_PRINCIPAIS_1', 'RB_DOENCA_PULMONAR_CRONICA_SIM_COMORBIDADES_PRINCIPAIS_1'],
    'sequela_neurologica': [ 'RB_DATA_EVENTO_MAIS_RECENTE_SEQUELA_NEUROLOGICA_NAO_1', 'RB_DATA_EVENTO_MAIS_RECENTE_SEQUELA_NEUROLOGICA_SIM_1'],
    'avc_ait_previo': ['RB_DOENCA_NEUROLOGICA_PREVIA_AVC_PREVIO_DEFICIT_1', 'RB_DOENCA_NEUROLOGICA_PREVIA_AIT_PREVIO_1', 'RB_DOENCA_NEUROLOGICA_PREVIA_NAO_1', 'RB_DOENCA_NEUROLOGICA_PREVIA_SIM_1'],
    'tabagismo': ['RB_EX_TABAGISTA_1', 'RB_TABAGISMO_NAO_1', 'RB_TABAGISMO_SIM_1'],
    'diabetes_controle': ['RB_DIETA_COMORBIDADES_PRINCIPAIS_1', 'RB_HIPOGLICEMIANTE_ORAL_COMORBIDADES_PRINCIPAIS_1', 'RB_INSULINA_COMORBIDADES_PRINCIPAIS_1'],
    'etilista': ['RB_ETILISTA_ATIVO_COMORBIDADES_PRINCIPAIS_1', 'RB_ETILISTA_NAO_COMORBIDADES_PRINCIPAIS_1', 'RB_EX_ETILISTA_COMORBIDADES_PRINCIPAIS_1'],
    'massa_corporal_critica': ['RB_MASSA_CORPORAL_CRITICA_NAO_1', 'RB_MASSA_CORPORAL_CRITICA_SIM_1'], 
    'avaliacao_corporal': ['RB_MASSA_CORPORAL_CRITICA_CAQUEXIA_IMC_MENOR_18_1', 'RB_MASSA_CORPORAL_CRITICA_OBESIDADE_MORBIDA_IMC_MAIOR_30_1'],
    'endocardite_ativa': ['RB_ENDOCARDITE_ATIVA_NAO_1', 'RB_ENDOCARDITE_ATIVA_SIM_1'],
    'tipo_de_arteriopatia': ['RB_CLAUDICACAO_LIMITANTE_1', 'RB_DOENCA_CAROTIDEA_MAIOR_IGUAL_50_1', 'RB_AMPUTACAO_POR_DAOP_1', 'RB_INTERVENCAO_PLANEJADA_PREVIA_AORTA_ABDOMINAL_MEMBROS_CAROTIDAS_1'],
    'imunossupressao': ['RB_IMUNOSSUPRESSAO_NAO_COMORBIDADES_PRINCIPAIS_1', 'RB_IMUNOSSUPRESSAO_SIM_COMORBIDADES_PRINCIPAIS_1'],
    'tipo_de_imunossupressao': ['RB_DOENCA_COMORBIDADES_PRINCIPAIS_1', 'RB_MEDICACAO_COMORBIDADES_PRINCIPAIS_1'],
    'cancer': ['RB_CANCER_NAO_COMORBIDADES_PRINCIPAIS_1', 'RB_CANCER_SIM_COMORBIDADES_PRINCIPAIS_1'],
    'tratamento_de_cancer': ['RB_TIPO_CANCER_ATIVO_NAO_TRATADO_COMORBIDADES_PRINCIPAIS_1', 'RB_TIPO_CANCER_TRATADO_MAIOR_5_COMORBIDADES_PRINCIPAIS_1', 'RB_TIPO_CANCER_TRATADO_MENOR_5_COMORBIDADES_PRINCIPAIS_1'],
    'doenca_hepatica': ['RB_DOENCA_NAO_HEPATICA_COMORBIDADES_PRINCIPAIS_1', 'RB_DOENCA_SIM_HEPATICA_COMORBIDADES_PRINCIPAIS_1'],
    'cirurgia_anterior': ['RB_CIRURGIA_CARDIACA_PREVIA_NAO_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1', 'RB_CIRURGIA_CARDIACA_PREVIA_SIM_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1'],
    'tipo_de_fa': ['RB_FIBRILACAO_ATRIAL_PAROXISTICA_RITMO_CARDIACO_ATUAL_1', 'RB_FIBRILACAO_ATRIAL_PERMANENTE_RITMO_CARDIACO_ATUAL_1', 'RB_FIBRILACAO_ATRIAL_PERSISTENTE_RITMO_CARDIACO_ATUAL_1'],
    'tipo_mp_trc_cdi': ['RB_MP_DEFINITIVO_RITMO_CARDIACO_ATUAL_1', 'RB_MP_PROVISORIO_RITMO_CARDIACO_ATUAL_1', 'RB_TRC_RITMO_CARDIACO_ATUAL_1', 'RB_CDI_RITMO_CARDIACO_ATUAL_1'],
    'coronaria_exame': ['RB_ANGIOTC_CORONARIAS_ESTUDO_CORONARIAS_1', 'RB_CATETERISMO_CARDIACO_ESTUDO_CORONARIAS_1', 'RB_NAO_REALIZADO_ESTUDO_CORONARIAS_1'],
    'dac_exame':['RB_DOENCA_ARTERIAL_CORONARIANA_DAC_OBSTRUTIVA_NAO_ESTUDO_CORONARIAS_1', 'RB_DOENCA_ARTERIAL_CORONARIANA_DAC_OBSTRUTIVA_SIM_ESTUDO_CORONARIAS_1'],
    'qtde_vasos_acometidos': ['RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_0_ESTUDO_CORONARIAS_1', 'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_1_ESTUDO_CORONARIAS_1', 
        	'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_2_ESTUDO_CORONARIAS_1', 'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_3_ESTUDO_CORONARIAS_1'],
    'tronco_de_coron_ria_esquer': ['RB_TRONCO_CORONARIA_ESQUERDA_MAIOR_50_NAO_ESTUDO_CORONARIAS_1', 'RB_TRONCO_CORONARIA_ESQUERDA_MAIOR_50_SIM_ESTUDO_CORONARIAS_1'],
    'lib_cirurgia':['RB_ODONTOLOGIA_LIBERADO_NAO_OUTROS_EXAMES_1', 'RB_ODONTOLOGIA_LIBERADO_SIM_OUTROS_EXAMES_1'],
    
    




    
    
    
    #'etiologia_valva_aortica': ['ETIOLOGIA_ANEURISMA_DISSECCAO_VALVA_AORTICA_1', 'ETIOLOGIA_BIVALVULAR_VALVA_AORTICA_1', 'ETIOLOGIA_REUMATICA_VALVA_AORTICA_1', 'ETIOLOGIA_CALCIFICA_VALVA_AORTICA_1', 'ETIOLOGIA_ENDOCARDITE_VALVA_AORTICA_1'],
    #'etiologia_valva_mitral': ['ETIOLOGIA_REUMATICA_1', 'ETIOLOGIA_CALCIFICA_1', 'ETIOLOGIA_FUNCIONAL_1', 'ETIOLOGIA_FIBROELASTICA_PROLAPSO_BARLOW_1', 'ETIOLOGIA_ENDOCARDITE_1'],
    #'etiologia_valva_tricuspide': ['PRIMARIA_ETIOLOGIA_NATIVA_TRICUSPIDE_1', 'FUNCIONAL_ETIOLOGIA_NATIVA_TRICUSPIDE_1', 'ASSOCIADA_DCEI_ETIOLOGIA_NATIVA_TRICUSPIDE_1', 'ENDOCARDITE_ETIOLOGIA_NATIVA_TRICUSPIDE_1'], 
    #'etio_estrutural_aortica_2': ['ETIOLOGIA_NAO_ESTRUTURAL_LEAK_VALVA_AORTICA_1', 'ETIOLOGIA_NAO_ESTRUTURAL_MISMATCH_VALVA_AORTICA_1', 'ETIOLOGIA_NAO_ESTRUTURAL_PANNUS_VALVA_AORTICA_1', 'ETIOLOGIA_NAO_ESTRUTURAL_ENDOCARDITE_VALVA_AORTICA_1', 'ETIOLOGIA_NAO_ESTRUTURAL_TROMBOSE_VALVA_AORTICA_1'],
    #'etio_estrutural_tricus_2': ['LEAK_ETIOLOGIA_VALVA_TRICUSPIDE_1', 'MISMATCH_ETIOLOGIA_VALVA_TRICUSPIDE_1', 'PANNUS_ETIOLOGIA_VALVA_TRICUSPIDE_1', 'ENDECARDITE_ETIOLOGIA_VALVA_TRICUSPIDE_1', 'TROMBOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1'],
    #'etio_estrutural_mitral_2': ['ETIOLOGIA_ESTRUTURAL_LEAK_1', 'ETIOLOGIA_ESTRUTURAL_MISMATCH_1', 'ETIOLOGIA_ESTRUTURAL_PANNUS_1', 'ETIOLOGIA_ESTRUTURAL_ENDOCARDITE_1', 'ETIOLOGIA_ESTRUTURAL_TROMBOSE_1'],
    #'tipo_valvula_prostetica': ['RB_MECANICA_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1', 'RB_BIOLOGICA_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1'],
}

#Atribuindo valores mapeados, binarios
preop_rn = {
    'RB_DM_NAO_1': 0, 'RB_DM_SIM_1': 1,
    'RB_HAS_NAO_1': 0, 'RB_HAS_SIM_1': 1,
	'RB_PS_UTI_ORIGEM_PACIENTE_1': 1, 'RB_ELETICO_ORIGEM_PACIENTE_1': 2,
	'RB_DIAS_AO_SINTOMAS_FICHA_PRE_1': 1, 'RB_MESES_SINTOMAS_FICHA_PRE_1': 2, 'RB_ANOS_SINTOMAS_FICHA_PRE_1': 3,
    'RB_HIPERTENSAO_PULMONAR_NAO_1': 0, 'RB_HIPERTENSAO_PULMONAR_SIM_1': 1, 'RB_HIPERTENSAO_ARTERIAL_PULMONAR_NAO_COMORBIDADES_PRINCIPAIS_1': 0,
    'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_COMORBIDADES_PRINCIPAIS_1': 1,
	'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_PSAP_ENTRE_31_55_MMHG_1': 2, 'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_PSAP_MENOR_31_MMHG_1': 1,
    'RB_HIPERTENSAO_ARTERIAL_PULMONAR_SIM_PSAP_MAIOR_55_MMHG_1': 3, 
	'RB_DISFUNCAO_VENTRICULAR_VE_NAO_1': 0, 'RB_DISFUNCAO_VENTRICULAR_VE_SIM_1': 1,
    'RB_DISFUNCAO_VENTRICULAR_VD_NAO_1': 0, 'RB_DISFUNCAO_VENTRICULAR_VD_SIM_1': 1,
    'RB_ACEITA_TRANSFUSAO_NAO_1': 0, 'RB_ACEITA_TRANSFUSAO_SIM_1': 1,
    'RB_ANTICOAGULACAO_VARFARINA_VALVOPATIAS_1': 1, 'RB_ANTICOAGULACAO_DOAC_VALVOPATIAS_1': 2, 
    'RB_ASSINTOMATICO_NAO_VALPATIAS_1': 0, 'RB_ASSINTOMATICO_SIM_VALPATIAS_1': 1,
    'RB_FADIGA_NAO_VALPATIAS_1': 0, 'RB_FADIGA_SIM_VALPATIAS_1': 1,
    'RB_PALPITACOES_NAO_VALVOPATIAS_1': 0, 'RB_PALPITACOES_SIM_VALVOPATIAS_1': 1, 
	'RB_SINCOPE_NAO_VALVOPATIAS_1': 0, 'RB_SINCOPE_SIM_VALVOPATIAS_1': 1,
    'RB_ANGINA_NAO_VALPATIAS_1': 0,  'RB_ANGINA_SIM_VALPATIAS_1': 1,
    'RB_ANGINA_SIM_CCS1_SINTOMAS_PRINCIPAL_1': 1, 'RB_ANGINA_SIM_CCS2_SINTOMAS_PRINCIPAL_1': 2, 'RB_ANGINA_SIM_CCS3_SINTOMAS_PRINCIPAL_1': 3, 
    'RB_ANGINA_SIM_CCS4_SINTOMAS_PRINCIPAL_1': 4,
    'RB_DISPNEIA_CF_NYHA_I_1': 1, 'RB_DISPNEIA_CF_NYHA_II_1': 2, 'RB_DISPNEIA_CF_NYHA_III_1': 3, 'RB_DISPNEIA_CF_NYHA_IV_1': 4,
    'RB_REOPERACAO_PRIMEIRA_VALPATIAS_1': 1, 'RB_REOPERACAO_SEGUNDA_VALPATIAS_1': 2, 'RB_REOPERACAO_TERCEIRO_VALPATIAS_1': 3,
    'RB_REOPERACAO_QUARTA_VALPATIAS_1': 4, 'RB_REOPERACAO_QUINTO_VALPATIAS_1': 5, 'RB_REOPERACAO_SEXTO_VALPATIAS_1': 6,
    'RB_ELETIVA_OPERACAO_1': 1, 'RB_URGENCIA_OPERACAO_1': 2, 'RB_EMERGENCIA_OPERACAO_1': 3, 'RB_SALVAMENTO_OPERACAO_1': 4,
	'RB_NAO_DISCUTIDO_HEART_TEAM_1': 0, 'RB_SIM_DISCUTIDO_HEART_TEAM_1': 1,
    'RB_DOENCA_RENAL_DIALISE_1': 4, 'RB_DOENCA_RENAL_MODERADA_CLCR_ENTRE_50_85_1': 2, 'RB_DOENCA_RENAL_NORMAL_CLCR_MAIOR_85_1': 1, 
    'RB_DOENCA_RENAL_SEVERA_CLCR_MENOR_50_1': 3,
    'RB_ESTADO_PRE_OPERATORIO_CRITICO_NAO_1': 0, 'RB_ESTADO_PRE_OPERATORIO_CRITICO_SIM_1': 1,
    'RB_MOBILIDADE_SEVERAMENTE_PREJUDICADA_NAO_1': 0, 'RB_MOBILIDADE_SEVERAMENTE_PREJUDICADA_SIM_1': 1,
    'RB_ARTERIOPATIA_EXTRACARDIACA_NAO_1': 0, 'RB_ARTERIOPATIA_EXTRACARDIACA_SIM_1': 1, 
    'RB_INFARTO_MIOCARDIO_MENOR_90_DIAS_NAO_1': 0, 'RB_INFARTO_MIOCARDIO_MENOR_90_DIAS_SIM_1': 1,
    'RB_USO_CORTICOIDE_BRONCODILATADOR_LONGA_DATA_NAO_1': 0, 'RB_USO_CORTICOIDE_BRONCODILATADOR_LONGA_DATA_SIM_1': 1,
    'RB_DOENCA_PULMONAR_CRONICA_NAO_COMORBIDADES_PRINCIPAIS_1': 0, 'RB_DOENCA_PULMONAR_CRONICA_SIM_COMORBIDADES_PRINCIPAIS_1': 1,
    'RB_DATA_EVENTO_MAIS_RECENTE_SEQUELA_NEUROLOGICA_NAO_1': 0, 'RB_DOENCA_NEUROLOGICA_PREVIA_NAO_1': 0, 'RB_DOENCA_NEUROLOGICA_PREVIA_SIM_1': 1,
    'RB_DATA_EVENTO_MAIS_RECENTE_SEQUELA_NEUROLOGICA_SIM_1': 1,
    'RB_EX_TABAGISTA_1': 1, 'RB_TABAGISMO_NAO_1': 0, 'RB_TABAGISMO_SIM_1': 2,
	'RB_DIETA_COMORBIDADES_PRINCIPAIS_1': 1, 'RB_HIPOGLICEMIANTE_ORAL_COMORBIDADES_PRINCIPAIS_1': 2, 'RB_INSULINA_COMORBIDADES_PRINCIPAIS_1': 3,
    'RB_ETILISTA_ATIVO_COMORBIDADES_PRINCIPAIS_1': 2, 'RB_ETILISTA_NAO_COMORBIDADES_PRINCIPAIS_1': 0, 'RB_EX_ETILISTA_COMORBIDADES_PRINCIPAIS_1': 1,
    'RB_MASSA_CORPORAL_CRITICA_CAQUEXIA_IMC_MENOR_18_1': 2, 'RB_MASSA_CORPORAL_CRITICA_NAO_1': 0, 'RB_MASSA_CORPORAL_CRITICA_SIM_1': 1,
    'RB_MASSA_CORPORAL_CRITICA_OBESIDADE_MORBIDA_IMC_MAIOR_30_1': 1,  
    'RB_ENDOCARDITE_ATIVA_NAO_1': 0, 'RB_ENDOCARDITE_ATIVA_SIM_1': 1,
    'RB_CLAUDICACAO_LIMITANTE_1': 1, 'RB_DOENCA_CAROTIDEA_MAIOR_IGUAL_50_1': 2, 'RB_AMPUTACAO_POR_DAOP_1': 3, 
    'RB_INTERVENCAO_PLANEJADA_PREVIA_AORTA_ABDOMINAL_MEMBROS_CAROTIDAS_1': 4,
    'RB_IMUNOSSUPRESSAO_NAO_COMORBIDADES_PRINCIPAIS_1': 0, 'RB_IMUNOSSUPRESSAO_SIM_COMORBIDADES_PRINCIPAIS_1': 1,
    'RB_DOENCA_COMORBIDADES_PRINCIPAIS_1': 1, 'RB_MEDICACAO_COMORBIDADES_PRINCIPAIS_1': 2,
    'RB_TIPO_CANCER_ATIVO_NAO_TRATADO_COMORBIDADES_PRINCIPAIS_1': 1, 'RB_TIPO_CANCER_TRATADO_MAIOR_5_COMORBIDADES_PRINCIPAIS_1': 3, 
    'RB_TIPO_CANCER_TRATADO_MENOR_5_COMORBIDADES_PRINCIPAIS_1': 2,
    'RB_CANCER_NAO_COMORBIDADES_PRINCIPAIS_1': 0, 'RB_CANCER_SIM_COMORBIDADES_PRINCIPAIS_1': 1,
    'RB_DOENCA_NAO_HEPATICA_COMORBIDADES_PRINCIPAIS_1': 0, 'RB_DOENCA_SIM_HEPATICA_COMORBIDADES_PRINCIPAIS_1': 1,
    'RB_CIRURGIA_CARDIACA_PREVIA_NAO_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1': 0, 'RB_CIRURGIA_CARDIACA_PREVIA_SIM_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1': 1,
    'RB_FIBRILACAO_ATRIAL_PAROXISTICA_RITMO_CARDIACO_ATUAL_1': 1, 'RB_FIBRILACAO_ATRIAL_PERMANENTE_RITMO_CARDIACO_ATUAL_1': 3, 
    'RB_FIBRILACAO_ATRIAL_PERSISTENTE_RITMO_CARDIACO_ATUAL_1': 2,
    'RB_MP_DEFINITIVO_RITMO_CARDIACO_ATUAL_1': 2, 'RB_MP_PROVISORIO_RITMO_CARDIACO_ATUAL_1': 1, 'RB_TRC_RITMO_CARDIACO_ATUAL_1': 3,
    'RB_CDI_RITMO_CARDIACO_ATUAL_1': 4,
    'RB_ANGIOTC_CORONARIAS_ESTUDO_CORONARIAS_1': 2, 'RB_CATETERISMO_CARDIACO_ESTUDO_CORONARIAS_1': 1, 'RB_NAO_REALIZADO_ESTUDO_CORONARIAS_1': 0,
    'RB_DOENCA_ARTERIAL_CORONARIANA_DAC_OBSTRUTIVA_NAO_ESTUDO_CORONARIAS_1': 0, 'RB_DOENCA_ARTERIAL_CORONARIANA_DAC_OBSTRUTIVA_SIM_ESTUDO_CORONARIAS_1': 1,
    'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_0_ESTUDO_CORONARIAS_1': 0, 
    'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_1_ESTUDO_CORONARIAS_1': 1, 
    'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_2_ESTUDO_CORONARIAS_1': 2, 
    'RB_SE_DAC_NUMERO_VASOS_PRINCIPAIS_ACOMETIDOS_MAIOR_70_ESTENOSE_3_ESTUDO_CORONARIAS_1': 3,
	'RB_TRONCO_CORONARIA_ESQUERDA_MAIOR_50_NAO_ESTUDO_CORONARIAS_1': 0, 'RB_TRONCO_CORONARIA_ESQUERDA_MAIOR_50_SIM_ESTUDO_CORONARIAS_1': 1,
    'RB_ODONTOLOGIA_LIBERADO_NAO_OUTROS_EXAMES_1': 0, 'RB_ODONTOLOGIA_LIBERADO_SIM_OUTROS_EXAMES_1': 1,
    
    
	




    #'RB_MECANICA_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1': 1, 'RB_BIOLOGICA_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1': 2,
    #'ETIOLOGIA_ESTRUTURAL_LEAK_1': 1, 'ETIOLOGIA_ESTRUTURAL_MISMATCH_1': 2, 'ETIOLOGIA_ESTRUTURAL_PANNUS_1': 3, 'ETIOLOGIA_ESTRUTURAL_ENDOCARDITE_1': 4,
    #'ETIOLOGIA_ESTRUTURAL_TROMBOSE_1': 5,
    #'LEAK_ETIOLOGIA_VALVA_TRICUSPIDE_1': 1, 'MISMATCH_ETIOLOGIA_VALVA_TRICUSPIDE_1': 2, 'PANNUS_ETIOLOGIA_VALVA_TRICUSPIDE_1': 3,
    #'ENDECARDITE_ETIOLOGIA_VALVA_TRICUSPIDE_1': 4, 'TROMBOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1': 5,
    #'ETIOLOGIA_NAO_ESTRUTURAL_LEAK_VALVA_AORTICA_1': 1, 'ETIOLOGIA_NAO_ESTRUTURAL_MISMATCH_VALVA_AORTICA_1': 2, 'ETIOLOGIA_NAO_ESTRUTURAL_PANNUS_VALVA_AORTICA_1': 3,
    #'ETIOLOGIA_NAO_ESTRUTURAL_ENDOCARDITE_VALVA_AORTICA_1': 4, 'ETIOLOGIA_NAO_ESTRUTURAL_TROMBOSE_VALVA_AORTICA_1': 5,
    #'PRIMARIA_ETIOLOGIA_NATIVA_TRICUSPIDE_1': 1, 'FUNCIONAL_ETIOLOGIA_NATIVA_TRICUSPIDE_1': 2, 'ASSOCIADA_DCEI_ETIOLOGIA_NATIVA_TRICUSPIDE_1': 3,
    #'ENDOCARDITE_ETIOLOGIA_NATIVA_TRICUSPIDE_1': 4,
    #'ETIOLOGIA_REUMATICA_1': 1, 'ETIOLOGIA_CALCIFICA_1': 2, 'ETIOLOGIA_FUNCIONAL_1': 3, 'ETIOLOGIA_FIBROELASTICA_PROLAPSO_BARLOW_1': 4,
    #'ETIOLOGIA_ENDOCARDITE_1': 5,
    #'ETIOLOGIA_ANEURISMA_DISSECCAO_VALVA_AORTICA_1': 1, 'ETIOLOGIA_BIVALVULAR_VALVA_AORTICA_1': 2, 'ETIOLOGIA_REUMATICA_VALVA_AORTICA_1': 3,
    #'ETIOLOGIA_CALCIFICA_VALVA_AORTICA_1': 4, 'ETIOLOGIA_ENDOCARDITE_VALVA_AORTICA_1': 5,
     
    
    
    
    
}

#Mapeamento direto, trocando nomes de colunas
preop_maps = {
    'CD_ATENDIMENTO': 						 'cd_atendimento',
    'CD_PACIENTE':							 'record_id',
    
	'DS_DURACAO_SINTOMAS_FICHA_PRE_1': 								'tempo_sintomas_duracao',
	'RB_HIPERTENSAO_PULMONAR_VALOR_PSAP_1': 						'resultado_eco_psap',
    'DS_AORTA_ASCENDENTE_PARAMETROS_ECOCARDIOGRAFICOS_1': 			'aorta_ascendente',
	'DS_RAIZ_AORTA_PARAMETRO_ECOCARDIOGRADICO_1': 					'raiz_aorta',
    'DT_DATA_ECO_1': 												'data_do_eco',
    'DS_ALTURA_MAPAR_1': 											'altura',
    'DS_PESO_MAPA_1': 												'peso',
    'DS_EUROSCORE_1': 												'euroscore_ii',
	'RB_DISFUNCAO_VENTRICULAR_FEVE_1':								'feve_encam',  
    'RB_DISFUNCAO_VENTRICULAR_TAPSE_1':								'tapse_encam',
    'RB_DISFUNCAO_VENTRICULAR_FAC_1': 								'fac_encam',
    'DS_CARGA_TABAGICA_COMORBIDADES_PRINCIPAIS_1': 					'carga_tabagica',
    'DT_DATA_EVENTO_MAIS_RECENTE_1': 								'data_avc_ait',
    'DT_DATA_DIAGNOSTICO_ENDOCARDITE_ATIVA_1': 						'data_endocardite_ativa',
	'RB_CANCER_TIPO_CANCER_COMORBIDADES_PRINCIPAIS_1':  			'tipo_de_cancer',
    'RB_CIRURGIA_AORTA_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1': 		'tipo_de_cirurgia_anterior___2',
    'RB_CIRURGIA_VALVAR_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1': 	'tipo_de_cirurgia_anterior___3',
    'RB_REVASCULARIZACAO_MIOCARDICA_SIM_CIRURGIA_ASSOCIAD_1': 		'tipo_de_cirurgia_anterior___1',
    'RB_SINUSAL_RITMO_CARDIACO_ATUAL_1':  							'tipo_arritmia___7',
    'RB_FLUTTER_ATRIAL_RITMO_CARDIACO_ATUAL_1':  					'tipo_arritmia___2',
    'RB_BLOQUEIO_ATRIOVENTRICULAR_BAV_ALTO_GRAU_RITMO_CARDIACO_ATUAL_1':  		'tipo_arritmia___6',
    'DT_DATA_ESTUDO_CORONARIAS_1': 									'data_coronaria_exame',
    'DT_DATA_ODONTOLOGIA_LIBERADO_OUTROS_EXAMES_1':  				'dt_libera_odonto',
	'RB_ANO_ULTIMA_CIRURGIA_CARDIACA_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1':	 'dt_prev_cir',
	'RB_EAO_VALPATIAS_1':											'diagnostico_principal___1', 
    'RB_IAO_VALVOPATIAS_1': 										'diagnostico_principal___2',
    'RB_EM_VALVOPATIAS_1':											'diagnostico_principal___3', 
	'RB_IM_VALVOPATIAS_1':											'diagnostico_principal___4', 
	'RB_DOENCA_TRICUSPIDE_VALVOPATIAS_1':							'diagnostico_principal___5', 
	'RB_ANEURISMA_AORTA_VALVOPATIAS_1':								'diagnostico_principal___6', 
	'RB_DISFUNCAO_PROTESE_VALVAR_VALVOPATIAS_1':					'diagnostico_principal___7', 
	'RB_ENDOCARDITE_VALVOPATIAS_1':									'diagnostico_principal___8', 
	'RB_TROMBOSE_DIAGNOSTICO_PRINCIPAL_1':							'diagnostico_principal___9',
	'RB_AORTICA_VALVOPATIAS_1':  									'valvula_afetada___1',
	'RB_MITRAL_VALVOPATIAS_1':  									'valvula_afetada___2',
	'RB_TRICUSPIDE_VALVOPATIAS_1':  								'valvula_afetada___3',

	# MITRAL - NATIVA
	'RB_NATIVA_VALVA_MITRAL_1':										'valvula_nativa___1',
	'RB_SEM_LESAO_DISCRETA_1':										'lesao_valvar_disc___1',
	'RB_DISFUNCAO_ESTENOSE_MODERADA_1':								'estenose_moderada___1',
	'RB_DISFUNCAO_ESTENOSE_IMPORTANTE_1':							'estenose_importante___1',
	'RB_DISFUNCAO_INSUFICIENCIA_MODERADA_1':						'insuficiencia_moderada___1',
	'RB_DISFUNCAO_INSUFICIENCIA_IMPORTANTE_1':						'insuficiencia_importante___1',
	'DS_GRADIENTES_MAX_VALVA_MITRAL_NATIVA_1':						'mi_nativa_gd_max',
	'DS_GRADIENTES_MEDIO_1':										'mi_nativa_gd_med',
	'DS_AREA_VALVAR_1':												'area_valvar',
	'RB_ETIOLOGIA_REUMATICA_1':										'etiologia_valva_mitral___1',
	'RB_ETIOLOGIA_CALCIFICA_1':										'etiologia_valva_mitral___2',
	'RB_ETIOLOGIA_FUNCIONAL_1':										'etiologia_valva_mitral___3',
	'RB_ETIOLOGIA_FIBROELASTICA_PROLAPSO_BARLOW_1':					'etiologia_valva_mitral___4',
	'RB_ETIOLOGIA_ENDOCARDITE_1':									'etiologia_valva_mitral___5',

	# MITRAL - PROTESE
	'RB_PROTESE_BIOLOGICA_1':										'protese_biologica___1',
	'RB_PROTESE_MECANICA_1':										'protese_mecanica___1',
	'RB_NORMOFUNCIONANTE_1':										'lesao_valvar_normo___1',
	'RB_DISFUNCAO_MODERADA_VALVA_MITRAL_1':							'disfuncao_moderada___1',
	'RB_DISFUNCAO_IMPORTANTE_VALVA_MITRAL_1':						'disfuncao_importante___1',
	'DS_GRADIENTES_MAX_VALVA_MITRAL_BIOLOGICA_MECANICA_1':			'mi_protese_gd_max',
	'DS_GRADIENTES_MEDIO_2':										'mi_protese_gd_med',
	'DS_AREA_VALVAR_2':												'area_valvar_2',
	'RB_ETIOLOGIA_ESTRUTURAL_ESTENOSE_1':							'etio_estrut_protese_mi___1',
	'RB_ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_1':						'etio_estrut_protese_mi___2',
	'RB_ETIOLOGIA_ESTRUTURAL_LEAK_1':								'etio_estrutural_mitral_2___1',
	'RB_ETIOLOGIA_ESTRUTURAL_MISMATCH_1':							'etio_estrutural_mitral_2___2',
	'RB_ETIOLOGIA_ESTRUTURAL_PANNUS_1':								'etio_estrutural_mitral_2___3',
	'RB_ETIOLOGIA_ESTRUTURAL_ENDOCARDITE_1':						'etio_estrutural_mitral_2___4',
	'RB_ETIOLOGIA_ESTRUTURAL_TROMBOSE_1':							'etio_estrutural_mitral_2___5',

	# AORTICA - NATIVA
	'RB_NATIVA_VALVA_AORTICA_1':										'valvula_nativa___2',
	'RB_SEM_LESAO_DISCRETA_MECANICA_VALVA_AORTICA_1':					'lesao_valvar_disc___2',
	'RB_DISFUNCAO_ESTENOSE_MODERADA_VALVA_AORTICA_1':					'estenose_moderada___2',
	'RB_DISFUNCAO_ESTENOSE_IMPORTANTE_VALVA_AORTICA_1':					'estenose_importante___2',
	'RB_DISFUNCAO_INSUFICIENCIA_MODERADA_VALVA_AORTICA_1':				'insuficiencia_moderada___2',
	'RB_DISFUNCAO_INSUFICIENCIA_IMPORTANTE_VALVA_AORTICA_1':			'insuficiencia_importante___2',
	'RB_GRADIENTES_MAX_VALVA_AORTICA_1':								'ao_nativa_gd_max',
	'RB_GRADIENTES_MEDIO_VALVA_AORTICA_1':								'ao_nativa_gd_med',
	'RB_AREA_VALVAR_VALVA_AORTICA_1':									'area_valvar_3',
	'RB_ETIOLOGIA_ANEURISMA_DISSECCAO_VALVA_AORTICA_1':					'etiologia_valva_aortica___1',
	'RB_ETIOLOGIA_BIVALVULAR_VALVA_AORTICA_1':							'etiologia_valva_aortica___2',
	'RB_ETIOLOGIA_REUMATICA_VALVA_AORTICA_1':							'etiologia_valva_aortica___3',
	'RB_ETIOLOGIA_CALCIFICA_VALVA_AORTICA_1':							'etiologia_valva_aortica___4',
	'RB_ETIOLOGIA_ENDOCARDITE_VALVA_AORTICA_1':							'etiologia_valva_aortica___5',

	# AORTICA - PROTESE
	'RB_PROTESE_BIOLOGICA_VALVA_AORTICA_1':								'protese_biologica___2',
	'RB_PROTESE_MECANICA_VALVA_AORTICA_1':								'protese_mecanica___2',
	'DS_NORMOFUNCIONANTE_VALVA_AORTICA_1':								'lesao_valvar_normo___2',
	'RB_DISFUNCAO_MODERADA_VALVA_AORTICA_1':							'disfuncao_moderada___2',
	'RB_DISFUNCAO_IMPORTANTE_VALVA_AORTICA_1':							'disfuncao_importante___2',
	'RB_GRADIENTES_MAX_VALVA_AORTICA_2':								'ao_protese_gd_max',
	'RB_GRADIENTES_MEDIO_VALVA_AORTICA_2':								'ao_protese_gd_med',
	'RB_AREA_VALVAR_VALVA_AORTICA_2':									'area_valvar_4',
    'RB_ETIOLOGIA_ESTRUTURAL_ESTONOSE_VALVA_AORTICA_1':					'etio_estrut_protese_ao___1', 
	'RB_ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_VALVA_AORTICA_1':			'etio_estrut_protese_ao___2', 
	'RB_ETIOLOGIA_NAO_ESTRUTURAL_LEAK_VALVA_AORTICA_1':					'etio_estrutural_aortica_2___1',
	'RB_ETIOLOGIA_NAO_ESTRUTURAL_MISMATCH_VALVA_AORTICA_1':				'etio_estrutural_aortica_2___2',
	'RB_ETIOLOGIA_NAO_ESTRUTURAL_PANNUS_VALVA_AORTICA_1':				'etio_estrutural_aortica_2___3',
	'RB_ETIOLOGIA_NAO_ESTRUTURAL_ENDOCARDITE_VALVA_AORTICA_1':			'etio_estrutural_aortica_2___4',
	'RB_ETIOLOGIA_NAO_ESTRUTURAL_TROMBOSE_VALVA_AORTICA_1':				'etio_estrutural_aortica_2___5',

	# TRICUSPIDE - NATIVA
	'RB_NATIVA_VALVA_TRICUSPIDE_1':											'valvula_nativa___3',
	'RB_SEM_LESAO_DISCRETA_VALVA_TRICUSPIDE_1':								'lesao_valvar_disc___3',
	'RB_ESTENOSE_MODERADA_VALVA_TRICUSPIDE_1':								'estenose_moderada___3',
	'RB_ESTENOSE_IMPORTANTE_1':												'estenose_importante___3',
	'RB_INSUFICIENCIA_MODERADA_VALVA_TRICUSPIDE_1':							'insuficiencia_moderada___3',
	'RB_INSUFICIENCIA_IMPORTANTE_1':										'insuficiencia_importante___3',
	'RB_ANEL_VALVA_TRICUSPIDE_1':											'anel_valva_tricuspide',
	'RB_PRIMARIA_ETIOLOGIA_NATIVA_TRICUSPIDE_1':							'etiologia_valva_tricuspide___1',
	'RB_FUNCIONAL_ETIOLOGIA_NATIVA_TRICUSPIDE_1':							'etiologia_valva_tricuspide___2',
	'RB_ASSOCIADA_DCEI_ETIOLOGIA_NATIVA_TRICUSPIDE_1':						'etiologia_valva_tricuspide___3',
	'RB_ENDOCARDITE_ETIOLOGIA_NATIVA_TRICUSPIDE_1':							'etiologia_valva_tricuspide___4',

	# TRICUSPIDE - PROTESE
	'RB_PROTESE_BIOLOGICA_VALVA_TRICUSPIDE_1':								'protese_biologica___3',
	'RB_PROTESE_MECANICA_VALVA_TRICUSPIDE_1':								'protese_mecanica___3',
	'RB_NORMOFUNCIONANTE_BIOLOGICA_MECANICA_VALVA_TRICUSPIDE_1':			'lesao_valvar_normo___3',
	'RB_DISFUNCAO_MODERADA_BIOLOGICA_MECANICA_VALVA_TRICUSPIDE_1':			'disfuncao_moderada___3',
	'RB_DISFUNCAO_IMPORTANTE_BIOLOGICA_MECANICA_VALVA_TRICUSPIDE_1':		'disfuncao_importante___3',
	'DS_AREA_VALVAR_3':														'area_valvar_5',
	'RB_ESTENOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1':								'etio_estrut_protese_tri___1',
	'RB_INSUFICIENCIA_ETIOLOGIA_VALVA_TRICUSPIDE_1':						'etio_estrut_protese_tri___2',
	'RB_LEAK_ETIOLOGIA_VALVA_TRICUSPIDE_1':									'etio_estrutural_tricus_2___1',
	'RB_MISMATCH_ETIOLOGIA_VALVA_TRICUSPIDE_1':								'etio_estrutural_tricus_2___2',
	'RB_PANNUS_ETIOLOGIA_VALVA_TRICUSPIDE_1':								'etio_estrutural_tricus_2___3',
	'RB_ENDECARDITE_ETIOLOGIA_VALVA_TRICUSPIDE_1':							'etio_estrutural_tricus_2___4',
	'RB_TROMBOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1':								'etio_estrutural_tricus_2___5',
	
	
}


preop_cols_excluir = {
    'RB_DISFUNCAO_VENTRICULAR_TAPSE_N_A_1', 'RB_DISFUNCAO_VENTRICULAR_FAC_N_A_1',
    'RB_ANTICOAGULACAO_NAO_VALVOPATIAS_1',
    'DS_RESIDENTE_1', 'DS_CASO_DISCUTIDO_STAFF_1', 'RB_PROTOCOLO_ESTUDO_SIM_DESCRICAO_1',
	'RB_PROTOCOLO_ESTUDO_SIM_1', 'RB_PROTOCOLO_ESTUDO_NAO_1',
    'RB_REVASCULARIZACAO_MIOCARDICA_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1',
    'RB_REVASCULARIZACAO_MIOCARDICA_NAO_CIRURGIA_ASSOCIAD_1',
    'RB_REVASCULARIZACAO_MIOCARDICA_SIM_CIRURGIA_ASSOCIAD_1',			
    'RB_CORRECAO_ANEURISMA_AORTA_SIM_CIRURGIA_ASSOCIADA_1',	
    'RB_CORRECAO_ANEURISMA_AORTA_NAO_CIRURGIA_ASSOCIADA_1',		
    'RB_ABLACAO_FA_FLUTTER_SIM_CIRURGIA_ASSOCIADA_1',
    'RB_ABLACAO_FA_FLUTTER_NAO_CIRURGIA_ASSOCIADA_1',					
    'RB_FECHAMENTO_APENDICE_ATRIAL_SIM_CIRURGIA_ASSOCIADA_1',
    'RB_FECHAMENTO_APENDICE_ATRIAL_NAO_CIRURGIA_ASSOCIADA_1',			
    'RB_FECHAMENTO_CIV_SIM_CIRURGIA_ASSOCIADA_1',
    'RB_FECHAMENTO_CIV_NAO_CIRURGIA_ASSOCIADA_1',						
    'RB_MIECTOMIA_SIM_CIRURGIA_ASSOCIADA_1',
    'RB_MIECTOMIA_NAO_CIRURGIA_ASSOCIADA_1',							
    'RB_TROMBECTOMIA_SIM_CIRURGIA_ASSOCIADA_1',
    'RB_TROMBECTOMIA_NAO_CIRURGIA_ASSOCIADA_1',
	'RB_MECANICA_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1',
    'RB_BIOLOGICA_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1',
    'DS_OBS_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1',
    'RB_TV_AORTICA_INTERVENCAO_CIRURGICA_1',
    'RB_TV_MITRAL_INTERVENCAO_CIRURGICA_1',							
    'RB_TV_TRICUSPIDE_INTERVENCAO_CIRURGICA_1',						
    'RB_PLASTIA_AORTICA_INTERVENCAO_CIRURGICA_1',						
    'RB_PLASTIA_MITRAL_INTERVENCAO_CIRURGICA_1',						
    'RB_PLASTIA_TRICUSPIDE_INTERVENCAO_CIRURGICA_1',
}


# 'cirurgia_proposta': ['PRIMEIRA_CIRURGIA_VALPATIAS_1',
# 'PRIMEIRA_CIRURGIA_VALPATIAS_1': 1, 
# 'RB_DOENCA_NEUROLOGICA_PREVIA_AVC_PREVIO_DEFICIT_1', 'RB_DOENCA_NEUROLOGICA_PREVIA_AIT_PREVIO_1',

In [57]:
# ----------------------------
# 1) Execução das consultas
# ----------------------------
txVALV = query_sql_to_dataframe(conn_string, sql_VALV_texto)  # use o SQL de docText acima
rbVALV = query_sql_to_dataframe(conn_string, sql_VALV_radio)  # use o SQL de docRadio acima

# ----------------------------
# 2) Limpeza inicial
# ----------------------------

# Garantir colunas esperadas existem (evita KeyError silencioso)
for df in (txVALV, rbVALV):
    for col in ['DS_LEITO', 'DH_DOCUMENTO', 'DS_IDENTIFICADOR', 'DS_RESPOSTA', 'CD_PACIENTE']:
        if col not in df.columns:
            df[col] = np.nan

# Corrige bug: aplicar na própria base
txVALV['DS_LEITO'] = txVALV['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)
rbVALV['DS_LEITO'] = rbVALV['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

# Concatena
dVal = pd.concat([txVALV, rbVALV], ignore_index=True)

# Remove pacientes de teste (função sua)
dVal = remove_test_patients(dVal, 'CD_PACIENTE')

# Datas em datetime (robustez)
dVal['DH_DOCUMENTO'] = pd.to_datetime(dVal['DH_DOCUMENTO'], errors='coerce')

# # ----------------------------
# # 3) Normalização de identificadores
# # ----------------------------
# def safe_trim_id(s: pd.Series, n=3) -> pd.Series:
#     s = s.astype(str).fillna('')
#     # Apenas trunca se a string tiver tamanho maior que n
#     return s.apply(lambda x: x[n:] if len(x) > n else x)

# # Se seus identificadores têm mesmo um prefixo fixo de 3 chars, mantenha:
# dVal['DS_IDENTIFICADOR'] = safe_trim_id(dVal['DS_IDENTIFICADOR'], 3).str.strip().str.upper()

# ----------------------------
# 4) Mapeamentos
# ----------------------------

# Inversão do dicionário preop_ls para mapeamento (identificador -> nome da coluna alvo)
map_ident_para_coluna = {item: col for col, lista in preop_ls.items() for item in lista}

# Flag dos itens mapeados (grupo A)
mask_mapeados = dVal['DS_IDENTIFICADOR'].isin(map_ident_para_coluna)

# Grupo A: itens com valores codificados (via preop_rn)
df_a = dVal[mask_mapeados].copy()
df_a['COLUNA'] = df_a['DS_IDENTIFICADOR'].map(map_ident_para_coluna)
df_a['VALOR'] = df_a['DS_IDENTIFICADOR'].map(preop_rn)

# Grupo B: genéricos (texto/booleanos)
df_b = dVal[~mask_mapeados].copy()
df_b['COLUNA'] = df_b['DS_IDENTIFICADOR']

map_gen = {'SIM': 1, 'S': 1, 'TRUE': 1, 'NAO': 0, 'N': 0, 'FALSE': 0}

def map_boolean_preservando_tipo(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().upper()
    if s in map_gen:
        return map_gen[s]
    # retorna original quando não é booleano; preserva numérico quando for
    return v

df_b['VALOR'] = df_b['DS_RESPOSTA'].apply(map_boolean_preservando_tipo)

# Junta tudo
df_unificado = pd.concat([df_a, df_b], ignore_index=True)

# ----------------------------
# 5) Pivot
# ----------------------------
# Para evitar conflitos e priorizar o valor mais recente em caso de duplicidade,
# vamos ordenar por data antes de pivotar e usar 'last' no aggfunc.
df_unificado = df_unificado.sort_values(['CD_PACIENTE', 'DH_DOCUMENTO'])

dVal1 = df_unificado.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'CD_DOCUMENTO'],
    columns='COLUNA',
    values='VALOR',
    aggfunc='last'  # prioriza o mais recente por ordem acima
).reset_index()

# Contagem de formulários por paciente (antes de agregar ciclos)
dVal1['qtd_repeticoes'] = dVal1.groupby('CD_PACIENTE')['CD_PACIENTE'].transform('count')

# ----------------------------
# 6) Renomeia colunas (mapeamento direto)
# ----------------------------
dVal2 = dVal1.rename(columns=preop_maps).copy()

# ----------------------------
# 7) Agrupamento por "ciclos" (180 dias)
# ----------------------------
GAP_LIMITE = 180

# Garante DH_DOCUMENTO com datetime (por segurança)
dVal2['DH_DOCUMENTO'] = pd.to_datetime(dVal2['DH_DOCUMENTO'], errors='coerce')

# Ordenação para cálculo do gap
dVal2 = dVal2.sort_values(['record_id', 'DH_DOCUMENTO'])

# Diferença de dias entre documentos do mesmo paciente
dVal2['diff'] = dVal2.groupby('record_id')['DH_DOCUMENTO'].diff().dt.days

# Marcação de início de novo ciclo quando gap > limite
dVal2['novo_ciclo'] = (dVal2['diff'] > GAP_LIMITE).fillna(True)  # primeira linha de cada paciente = novo ciclo

# Atribui ciclo_id cumulativo por paciente
dVal2['ciclo_id'] = dVal2.groupby('record_id')['novo_ciclo'].cumsum() - 1
# Agora ciclo_id começa em 0 para o primeiro ciclo de cada paciente

# ----------------------------
# 8) Consolidação por ciclo (último registro do ciclo)
# ----------------------------
# Se quiser o "estado mais recente do ciclo", usar last por data
dVal2 = dVal2.sort_values(['record_id', 'ciclo_id', 'DH_DOCUMENTO'])
dVal2 = dVal2.groupby(['record_id', 'ciclo_id'], as_index=False).last()

# (Opcional) Recontar quantos ciclos por paciente
dVal2['qtd_ciclos_paciente'] = dVal2.groupby('record_id')['ciclo_id'].transform('nunique')

dVal2 = dVal2.drop(columns=preop_cols_excluir, errors='ignore')


Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_16528\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_16528\303913626.py:128: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dVal2 = dVal2.groupby(['record_id', 'ciclo_id'], as_index=False).last()
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_16528\303913626.py:128: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dVal2 = dVal2.groupby(['record_id', 'ciclo_id'], as_index=False).last()


In [58]:
dVal2['record_id'] = dVal2['record_id'].astype('Int64')
print('Qtde de pctes únicos: ', dVal2['record_id'].nunique())
print('Qtde de colunas: ', len(dVal2.columns))
print('Colunas dVal2: ', dVal2.columns.tolist())


Qtde de pctes únicos:  224
Qtde de colunas:  184
Colunas dVal2:  ['record_id', 'ciclo_id', 'cd_atendimento', 'DH_DOCUMENTO', 'CD_DOCUMENTO', 'altura', 'aorta_ascendente', 'DS_AREA_VALVAR_1', 'DS_AREA_VALVAR_2', 'carga_tabagica', 'DS_CIRURGIA_VALVAR_ESPECIFICAR_ORDEM_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1', 'DS_DESCREVER_ACHADOS_RELEVANTES_OUTROS_EXAMES_1', 'tempo_sintomas_duracao', 'euroscore_ii', 'DS_GRADIENTES_MAX_VALVA_MITRAL_BIOLOGICA_MECANICA_1', 'DS_GRADIENTES_MAX_VALVA_MITRAL_NATIVA_1', 'DS_GRADIENTES_MEDIO_1', 'DS_GRADIENTES_MEDIO_2', 'DS_INTERVENCAO_CIRURGICA_OBSERVACAO_1', 'DS_NORMOFUNCIONANTE_VALVA_AORTICA_1', 'DS_OBS_DISCUTIDO_HEART_TEAM_1', 'DS_OUTRAS_COMORBIDADES_RELEVANTES_COMORBIDADES_PRINCIPAIS_1', 'DS_OUTROS_CIRURGIA_ASSOCIADA_1', 'DS_OUTRO_TIPO_CIRURGIA_CARDIACA_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1', 'peso', 'raiz_aorta', 'data_endocardite_ativa', 'DT_DATA_DISCUTIDO_HEART_TEAM_1', 'data_do_eco', 'data_coronaria_exame', 'data_avc_ait', 'dt_libera_odonto', 'RB_ANEL_VAL

In [ ]:
# 209 pctes ultima atualização
# 224 - 209 = 15 pacientes novos

In [59]:
# Função auxiliar para concatenar textos ignorando nulos e vazios
def concatenar_campos(row, campos_com_prefixo, separador=" -- "):
    partes = []
    for prefixo, coluna in campos_com_prefixo:
        valor = str(row[coluna]).strip() if pd.notnull(row[coluna]) else ""
        if valor and valor.lower() != "nan":
            # Se houver uma data associada (caso específico do Heart Team)
            if coluna == 'DS_OBS_DISCUTIDO_HEART_TEAM_1':
                data = str(row['DT_DATA_DISCUTIDO_HEART_TEAM_1']).strip() if pd.notnull(row['DT_DATA_DISCUTIDO_HEART_TEAM_1']) else ""
                if data and data.lower() != "nan":
                    partes.append(f"{prefixo}{valor} em {data}")
                else:
                    partes.append(f"{prefixo}{valor}")
            else:
                partes.append(f"{prefixo}{valor}")
    return separador.join(partes)

# 1. Criando 'tp_cir_prev'
dVal2['tp_cir_prev'] = dVal2.apply(
    lambda x: concatenar_campos(x, [("", 'DS_CIRURGIA_VALVAR_ESPECIFICAR_ORDEM_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1'), 
                                    ("", 'DS_OUTRO_TIPO_CIRURGIA_CARDIACA_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1')]), 
    axis=1
)

# 2. Criando 'observacao_encam'
# Definindo os campos e seus respectivos prefixos
campos_obs = [
    ("Achados em exames: ", 'DS_DESCREVER_ACHADOS_RELEVANTES_OUTROS_EXAMES_1'),
    ("Intervenção proposta: ", 'DS_INTERVENCAO_CIRURGICA_OBSERVACAO_1'),
    ("Cirurgia associada proposta: ", 'DS_OUTROS_CIRURGIA_ASSOCIADA_1'),
    ("Discussão Heart Team: ", 'DS_OBS_DISCUTIDO_HEART_TEAM_1')
]

dVal2['observacao_encam'] = dVal2.apply(lambda x: concatenar_campos(x, campos_obs), axis=1)

# 3. Criando 'outras_comorb_relevantes'
dVal2['outras_comorb_relevantes'] = dVal2['DS_OUTRAS_COMORBIDADES_RELEVANTES_COMORBIDADES_PRINCIPAIS_1']


# 4. Drop colunas
dVal2 = dVal2.drop(columns=['DS_CIRURGIA_VALVAR_ESPECIFICAR_ORDEM_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1', 
                            'DS_OUTRO_TIPO_CIRURGIA_CARDIACA_HISTORICO_CARDIACO_CIRURGICO_PREVIO_1',
                            'DS_DESCREVER_ACHADOS_RELEVANTES_OUTROS_EXAMES_1', 'DS_INTERVENCAO_CIRURGICA_OBSERVACAO_1',
                            'DS_OUTROS_CIRURGIA_ASSOCIADA_1', 'DS_OBS_DISCUTIDO_HEART_TEAM_1', 'DT_DATA_DISCUTIDO_HEART_TEAM_1',
                            'DS_OUTRAS_COMORBIDADES_RELEVANTES_COMORBIDADES_PRINCIPAIS_1'], errors='ignore')


In [60]:
dVal2.columns.tolist()

['record_id',
 'ciclo_id',
 'cd_atendimento',
 'DH_DOCUMENTO',
 'CD_DOCUMENTO',
 'altura',
 'aorta_ascendente',
 'DS_AREA_VALVAR_1',
 'DS_AREA_VALVAR_2',
 'carga_tabagica',
 'tempo_sintomas_duracao',
 'euroscore_ii',
 'DS_GRADIENTES_MAX_VALVA_MITRAL_BIOLOGICA_MECANICA_1',
 'DS_GRADIENTES_MAX_VALVA_MITRAL_NATIVA_1',
 'DS_GRADIENTES_MEDIO_1',
 'DS_GRADIENTES_MEDIO_2',
 'DS_NORMOFUNCIONANTE_VALVA_AORTICA_1',
 'peso',
 'raiz_aorta',
 'data_endocardite_ativa',
 'data_do_eco',
 'data_coronaria_exame',
 'data_avc_ait',
 'dt_libera_odonto',
 'RB_ANEL_VALVA_TRICUSPIDE_1',
 'diagnostico_principal___6',
 'dt_prev_cir',
 'valvula_afetada___1',
 'RB_AREA_VALVAR_VALVA_AORTICA_1',
 'RB_AREA_VALVAR_VALVA_AORTICA_2',
 'RB_ASSOCIADA_DCEI_ETIOLOGIA_NATIVA_TRICUSPIDE_1',
 'tipo_arritmia___6',
 'tipo_de_cancer',
 'tipo_de_cirurgia_anterior___2',
 'tipo_de_cirurgia_anterior___3',
 'RB_DISFUNCAO_ESTENOSE_IMPORTANTE_1',
 'RB_DISFUNCAO_ESTENOSE_IMPORTANTE_VALVA_AORTICA_1',
 'RB_DISFUNCAO_ESTENOSE_MODERADA_1',


In [ ]:
tsts = ['record_id',
'DH_DOCUMENTO',
'diagnostico_principal___1', 
'diagnostico_principal___2',
'diagnostico_principal___3', 
'diagnostico_principal___4', 
'diagnostico_principal___5', 
'diagnostico_principal___6', 
'diagnostico_principal___7', 
'diagnostico_principal___8', 
'diagnostico_principal___9',
'valvula_afetada___1',
'valvula_afetada___2',
'valvula_afetada___3',	



# MITRAL - NATIVA
'RB_NATIVA_VALVA_MITRAL_1',
'RB_SEM_LESAO_DISCRETA_1',
'RB_DISFUNCAO_ESTENOSE_MODERADA_1',
'RB_DISFUNCAO_ESTENOSE_IMPORTANTE_1',
'RB_DISFUNCAO_INSUFICIENCIA_MODERADA_1',
'RB_DISFUNCAO_INSUFICIENCIA_IMPORTANTE_1',
'DS_GRADIENTES_MAX_VALVA_MITRAL_NATIVA_1',
'DS_GRADIENTES_MEDIO_1',
'DS_AREA_VALVAR_1',
'RB_ETIOLOGIA_REUMATICA_1',
'RB_ETIOLOGIA_CALCIFICA_1',
'RB_ETIOLOGIA_FUNCIONAL_1',
'RB_ETIOLOGIA_FIBROELASTICA_PROLAPSO_BARLOW_1',
'RB_ETIOLOGIA_ENDOCARDITE_1',

# MITRAL - PROTESE
'RB_PROTESE_BIOLOGICA_1',
'RB_PROTESE_MECANICA_1',
'RB_NORMOFUNCIONANTE_1',
'RB_DISFUNCAO_MODERADA_VALVA_MITRAL_1',
'RB_DISFUNCAO_IMPORTANTE_VALVA_MITRAL_1',
'DS_GRADIENTES_MAX_VALVA_MITRAL_BIOLOGICA_MECANICA_1',
'DS_GRADIENTES_MEDIO_2',
'DS_AREA_VALVAR_2',
'RB_ETIOLOGIA_ESTRUTURAL_ESTENOSE_1',
'RB_ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_1',
'RB_ETIOLOGIA_ESTRUTURAL_LEAK_1',
'RB_ETIOLOGIA_ESTRUTURAL_MISMATCH_1',
'RB_ETIOLOGIA_ESTRUTURAL_PANNUS_1',
'RB_ETIOLOGIA_ESTRUTURAL_ENDOCARDITE_1',
'RB_ETIOLOGIA_ESTRUTURAL_TROMBOSE_1',


# AORTICA - NATIVA
'RB_NATIVA_VALVA_AORTICA_1',
'RB_SEM_LESAO_DISCRETA_MECANICA_VALVA_AORTICA_1',
'RB_DISFUNCAO_ESTENOSE_MODERADA_VALVA_AORTICA_1',
'RB_DISFUNCAO_ESTENOSE_IMPORTANTE_VALVA_AORTICA_1',
'RB_DISFUNCAO_INSUFICIENCIA_MODERADA_VALVA_AORTICA_1',
'RB_DISFUNCAO_INSUFICIENCIA_IMPORTANTE_VALVA_AORTICA_1',
'RB_GRADIENTES_MAX_VALVA_AORTICA_1',
'RB_GRADIENTES_MEDIO_VALVA_AORTICA_1',
'RB_AREA_VALVAR_VALVA_AORTICA_1',
'RB_ETIOLOGIA_ANEURISMA_DISSECCAO_VALVA_AORTICA_1',
'RB_ETIOLOGIA_BIVALVULAR_VALVA_AORTICA_1',
'RB_ETIOLOGIA_REUMATICA_VALVA_AORTICA_1',
'RB_ETIOLOGIA_CALCIFICA_VALVA_AORTICA_1',
'RB_ETIOLOGIA_ENDOCARDITE_VALVA_AORTICA_1',

# AORTICA - PROTESE
'RB_PROTESE_BIOLOGICA_VALVA_AORTICA_1',
'RB_PROTESE_MECANICA_VALVA_AORTICA_1',
'DS_NORMOFUNCIONANTE_VALVA_AORTICA_1',
'RB_DISFUNCAO_MODERADA_VALVA_AORTICA_1',
'RB_DISFUNCAO_IMPORTANTE_VALVA_AORTICA_1',
'RB_GRADIENTES_MAX_VALVA_AORTICA_2',
'RB_GRADIENTES_MEDIO_VALVA_AORTICA_2',
'RB_AREA_VALVAR_VALVA_AORTICA_2',
'RB_ETIOLOGIA_ESTRUTURAL_ESTONOSE_VALVA_AORTICA_1',
'RB_ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_VALVA_AORTICA_1',
'RB_ETIOLOGIA_NAO_ESTRUTURAL_LEAK_VALVA_AORTICA_1',
'RB_ETIOLOGIA_NAO_ESTRUTURAL_MISMATCH_VALVA_AORTICA_1',
'RB_ETIOLOGIA_NAO_ESTRUTURAL_PANNUS_VALVA_AORTICA_1',
'RB_ETIOLOGIA_NAO_ESTRUTURAL_ENDOCARDITE_VALVA_AORTICA_1',
'RB_ETIOLOGIA_NAO_ESTRUTURAL_TROMBOSE_VALVA_AORTICA_1',

# TRICUSPIDE - NATIVA
'RB_NATIVA_VALVA_TRICUSPIDE_1',
'RB_SEM_LESAO_DISCRETA_VALVA_TRICUSPIDE_1',
'RB_ESTENOSE_MODERADA_VALVA_TRICUSPIDE_1',
'RB_ESTENOSE_IMPORTANTE_1',
'RB_INSUFICIENCIA_MODERADA_VALVA_TRICUSPIDE_1',
'RB_INSUFICIENCIA_IMPORTANTE_1',
'RB_ANEL_VALVA_TRICUSPIDE_1',
'RB_PRIMARIA_ETIOLOGIA_NATIVA_TRICUSPIDE_1',
'RB_FUNCIONAL_ETIOLOGIA_NATIVA_TRICUSPIDE_1',
'RB_ASSOCIADA_DCEI_ETIOLOGIA_NATIVA_TRICUSPIDE_1',
'RB_ENDOCARDITE_ETIOLOGIA_NATIVA_TRICUSPIDE_1',


# TRICUSPIDE - PROTESE
'RB_PROTESE_BIOLOGICA_VALVA_TRICUSPIDE_1',
'RB_PROTESE_MECANICA_VALVA_TRICUSPIDE_1',
'RB_NORMOFUNCIONANTE_BIOLOGICA_MECANICA_VALVA_TRICUSPIDE_1',
'RB_DISFUNCAO_MODERADA_BIOLOGICA_MECANICA_VALVA_TRICUSPIDE_1',
'RB_DISFUNCAO_IMPORTANTE_BIOLOGICA_MECANICA_VALVA_TRICUSPIDE_1',
'DS_AREA_VALVAR_3',
'RB_ESTENOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1',
'RB_INSUFICIENCIA_ETIOLOGIA_VALVA_TRICUSPIDE_1',
'RB_LEAK_ETIOLOGIA_VALVA_TRICUSPIDE_1',
'RB_MISMATCH_ETIOLOGIA_VALVA_TRICUSPIDE_1',
'RB_PANNUS_ETIOLOGIA_VALVA_TRICUSPIDE_1',
'RB_ENDECARDITE_ETIOLOGIA_VALVA_TRICUSPIDE_1',
'RB_TROMBOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1',


]

# Filtra a lista 'tst', mantendo apenas as colunas que realmente existem no dVal2
colunas_existentes = [col for col in tsts if col in dVal2.columns]

# Cria o novo DataFrame de forma segura
tst = dVal2[colunas_existentes].copy()

In [35]:
def transformar_colunas_valvulopatias(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria novas colunas mapeadas e agregadas para dados de valvulopatias.
    - Faz cópia para evitar SettingWithCopyWarning;
    - Ajusta sufixos das válvulas conforme padrão ___1 (mitral), ___2 (aórtica), ___3 (tricúspide);
    - Trata nulos como '0' nas concatenações textuais;
    - Remove colunas originais (altas/observações e as 'antigas' do bloco de mapeamento).
    """
    df = df.copy()

    # --------------------------------------------------
    # 1️⃣ Renomeações diretas (1:1) — cria colunas novas com os valores das antigas
    #     e AO FINAL remove as "antigas" se existirem.
    # --------------------------------------------------
    mapeamentos = {
        'intervencoes_cardiacas_previas': 'cirurgia_anterior',
        'tipo_intervencao_previa___6': 'tipo_de_cirurgia_anterior___2',
        'tipo_intervencao_previa___3': 'tipo_de_cirurgia_anterior___3',
        'tipo_intervencao_previa___2': 'tipo_de_cirurgia_anterior___1',
        'tipo_intervencao_previa___5': 'tipo_de_cirurgia_anterior___4',
        'valvulopatia_aortica': 'valvula_afetada___1',
        'valvulopatia_mitral': 'valvula_afetada___2',
        'valvulopatia_tricuspide': 'valvula_afetada___3'
    }

    cols_antigas_existentes = []
    for nova, antiga in mapeamentos.items():
        if antiga in df.columns:
            df[nova] = df[antiga]
            cols_antigas_existentes.append(antiga)

    # --------------------------------------------------
    # 2️⃣ Colunas agregadas por lógica de Valvulopatias
    # --------------------------------------------------
    # Mapa de sufixos conforme padrão anterior: 1=mitral, 2=aórtica, 3=tricúspide
    SUFIXO = {'mi': '1', 'ao': '2', 'tr': '3'}

    def criar_tipo_valvulopatia(df_in: pd.DataFrame, valvula: str) -> None:
        """
        Cria df['tipo_valvulopatia_{valvula}'] com:
        1 = Estenose, 2 = Insuficiência, 3 = Dupla lesão
        Detecção por:
          - checkboxes (1) em estenose/insuficiência (moderada/importante)
          - OU presença de etiologia específica para estenose/insuficiência
        """
        idx = df_in.index
        sfx = SUFIXO[valvula]

        # Colunas de estenose para a válvula
        estenose_cols = [
            f'estenose_importante___{sfx}',
            f'estenose_moderada___{sfx}',
        ]
        # Campos de etiologia que indicam estenose por válvula
        etiologia_est_map = {
            'ao': 'ETIOLOGIA_ESTRUTURAL_ESTONOSE_VALVA_AORTICA_1',
            'mi': 'ETIOLOGIA_ESTRUTURAL_ESTENOSE_1',
            'tr': 'ESTENOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1',
        }
        if etiologia_est_map[valvula] in df_in.columns:
            estenose_cols.append(etiologia_est_map[valvula])

        estenose_cols = [c for c in estenose_cols if c in df_in.columns]
        if estenose_cols:
            cond_estenose = df_in[estenose_cols].apply(
                lambda x: x.eq(1) | x.notna(), axis=1
            ).any(axis=1)
        else:
            cond_estenose = pd.Series(False, index=idx)

        # Colunas de insuficiência para a válvula
        insuf_cols = [
            f'insuficiencia_importante___{sfx}',
            f'insuficiencia_moderada___{sfx}',
        ]
        etiologia_ins_map = {
            'ao': 'ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_VALVA_AORTICA_1',
            'mi': 'ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_1',
            'tr': 'INSUFICIENCIA_ETIOLOGIA_VALVA_TRICUSPIDE_1',
        }
        if etiologia_ins_map[valvula] in df_in.columns:
            insuf_cols.append(etiologia_ins_map[valvula])

        insuf_cols = [c for c in insuf_cols if c in df_in.columns]
        if insuf_cols:
            cond_insuf = df_in[insuf_cols].apply(
                lambda x: x.eq(1) | x.notna(), axis=1
            ).any(axis=1)
        else:
            cond_insuf = pd.Series(False, index=idx)

        # 1=Estenose, 2=Insuficiência, 3=Dupla Lesão
        cond_tipo1 = cond_estenose & ~cond_insuf
        cond_tipo2 = cond_insuf & ~cond_estenose
        cond_tipo3 = cond_estenose & cond_insuf

        tipo = np.select(
            [cond_tipo1, cond_tipo2, cond_tipo3], [1, 2, 3], default=np.nan
        )
        df_in[f'tipo_valvulopatia_{valvula}'] = pd.Series(tipo, index=idx).astype('Int64')

    for v in ['ao', 'mi', 'tr']:
        criar_tipo_valvulopatia(df, v)

    # --------------------------------------------------
    # 3️⃣ Concatenações com tratamento de Nulos ('0')
    # --------------------------------------------------
    def safe_str_col(df_in: pd.DataFrame, col: str) -> pd.Series:
        if col in df_in.columns:
            s = df_in[col].astype(str).str.strip()
            # Normaliza NaN/None/NaT e vazios para '0'
            return (
                s.replace(['nan', 'none', 'nat', 'NaN', 'None', 'NaT', ''], '0')
                 .fillna('0')
            )
        return pd.Series(['0'] * len(df_in), index=df_in.index)

    # Mitral (nativa/prótese)
    df['gradientes_max_medio'] = (
        'Nativa: ' + safe_str_col(df, 'GRADIENTES_MAX_VALVA_MITRAL_NATIVA_1') + ' / ' +
        safe_str_col(df, 'GRADIENTES_MEDIO_1') +
        '\nPrótese: ' + safe_str_col(df, 'GRADIENTES_MAX_VALVA_MITRAL_BIOLOGICA_MECANICA_1') + ' / ' +
        safe_str_col(df, 'GRADIENTES_MEDIO_2')
    )

    # Aórtica (nativa/prótese)
    df['gradientes_max_medio_2'] = (
        'Nativa: ' + safe_str_col(df, 'GRADIENTES_MAX_VALVA_AORTICA_1') + ' / ' +
        safe_str_col(df, 'GRADIENTES_MEDIO_VALVA_AORTICA_1') +
        '\nPrótese: ' + safe_str_col(df, 'GRADIENTES_MAX_VALVA_AORTICA_2') + ' / ' +
        safe_str_col(df, 'GRADIENTES_MEDIO_VALVA_AORTICA_2')
    )

    # Observação consolidada
    df['observacao_encam'] = (
        'Data Heart Team: ' + safe_str_col(df, 'DATA_DISCUTIDO_HEART_TEAM_1') + ' / ' +
        safe_str_col(df, 'OBS_DISCUTIDO_HEART_TEAM_1') + '\n' +
        safe_str_col(df, 'INTERVENCAO_CIRURGICA_OBSERVACAO_1') + ' / ' +
        safe_str_col(df, 'OBS_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1') + '\n' +
        safe_str_col(df, 'DESCREVER_ACHADOS_RELEVANTES_OUTROS_EXAMES_1') + ' / ' +
        safe_str_col(df, 'PROTOCOLO_ESTUDO_SIM_DESCRICAO_1')
    )

    # --------------------------------------------------
    # 4️⃣ Limpeza Final — remove originais
    # --------------------------------------------------
    cols_originais_altas = [
        'ETIOLOGIA_ESTRUTURAL_ESTONOSE_VALVA_AORTICA_1', 'ETIOLOGIA_ESTRUTURAL_ESTENOSE_1',
        'ESTENOSE_ETIOLOGIA_VALVA_TRICUSPIDE_1', 'ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_VALVA_AORTICA_1',
        'ETIOLOGIA_ESTRUTURAL_INSUFICIENCIA_1', 'INSUFICIENCIA_ETIOLOGIA_VALVA_TRICUSPIDE_1',
        'GRADIENTES_MAX_VALVA_MITRAL_NATIVA_1', 'GRADIENTES_MEDIO_1',
        'GRADIENTES_MAX_VALVA_MITRAL_BIOLOGICA_MECANICA_1', 'GRADIENTES_MEDIO_2',
        'GRADIENTES_MAX_VALVA_AORTICA_1', 'GRADIENTES_MEDIO_VALVA_AORTICA_1',
        'GRADIENTES_MAX_VALVA_AORTICA_2', 'GRADIENTES_MEDIO_VALVA_AORTICA_2',
        'DATA_DISCUTIDO_HEART_TEAM_1', 'OBS_DISCUTIDO_HEART_TEAM_1',
        'INTERVENCAO_CIRURGICA_OBSERVACAO_1', 'OBS_SE_PROTESE_QUAL_TIPO_INTERVENCAO_CIRURGICA_1',
        'DESCREVER_ACHADOS_RELEVANTES_OUTROS_EXAMES_1', 'PROTOCOLO_ESTUDO_SIM_DESCRICAO_1'
    ]
    cols_para_drop = [c for c in cols_originais_altas if c in df.columns] + cols_antigas_existentes

    return df.drop(columns=cols_para_drop, errors='ignore')

In [36]:
# Executa
dVal3 = transformar_colunas_valvulopatias(dVal2)

print("✅ Colunas criadas:",
      [c for c in dVal3.columns if 'valvulopatia' in c or 'gradientes' in c or 'observacao' in c])

# ----- 'tipo_arritmia___1' (FA: paroxística/persistente/permanente) -----
if 'tipo_de_fa' in dVal3.columns:
    dVal3['tipo_arritmia___1'] = np.where(dVal3['tipo_de_fa'].isin([1, 2, 3]), 1, 0).astype('int8')
else:
    dVal3['tipo_arritmia___1'] = 0

# ----- 'tipo_cirurgia___9' (se houve alguma cirurgia associada) -----
cols_alvo = [f'cirurgica_associada_encam___{i}' for i in range(1, 8)]
cols_presentes = [col for col in cols_alvo if col in dVal3.columns]
dVal3['tipo_cirurgia___9'] = 0
if cols_presentes:
    dVal3.loc[dVal3[cols_presentes].eq(1).any(axis=1), 'tipo_cirurgia___9'] = 1
else:
    print("Aviso: Nenhuma 'cirurgica_associada_encam___[1..7]' encontrada.")

✅ Colunas criadas: ['valvulopatia_aortica', 'valvulopatia_mitral', 'valvulopatia_tricuspide', 'tipo_valvulopatia_ao', 'tipo_valvulopatia_mi', 'tipo_valvulopatia_tr', 'gradientes_max_medio', 'gradientes_max_medio_2', 'observacao_encam']


In [37]:
# --- Normalização prévia do resultado_eco_psap ---
if 'resultado_eco_psap' in dVal3.columns:
    dVal3['_psap_norm'] = (
        dVal3['resultado_eco_psap']
        .astype(str)
        .str.upper()
        # normaliza codificações HTML de sinais
        .str.replace('&AMP;GT;', '>', regex=False)
        .str.replace('&AMP;LT;', '<', regex=False)
        .str.replace('&GT;', '>', regex=False)
        .str.replace('&LT;', '<', regex=False)
        # remove espaços, aspas, barras invertidas
        .str.replace(' ', '', regex=False)
        .str.replace('"', '', regex=False)
        .str.replace(r'\\', '', regex=True)
        # remove unidade mmHg e vírgulas decimais
        .str.replace('MMHG', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
else:
    dVal3['_psap_norm'] = ''

def mapear_psap_resultado(valor: str):
    """
    Mapeia texto do eco para ordinal:
      1: <31 mmHg
      2: 31-54 mmHg (inclui 'sinais indiretos')
      3: >=55 mmHg
    Entradas válidas: '45', '<31', '>55', 'SINAISINDIRETOS', 'NAO CALCULADA', etc.
    """
    if valor in ['', 'N/A', 'NAO', 'NAO_CALCULADA', 'SEMVALOR', '-', 'NAN', '<NA>']:
        return ''
    if valor.startswith('<'):
        return 1
    if valor.startswith('>'):
        num_str = valor[1:]
        try:
            num = float(num_str)
            return 3 if num >= 55 else 2
        except Exception:
            return 2
    if 'SINAISINDIRETOS' in valor:
        return 2
    try:
        num = float(valor)
        if num < 31:
            return 1
        elif num < 55:
            return 2
        else:
            return 3
    except Exception:
        return ''

def resolver_psap(row):
    # Se já existe valor explícito no formulário (1,2,3), preserva
    try:
        if pd.notna(row.get('valor_psap_encam')) and str(row['valor_psap_encam']).strip() != '':
            return int(row['valor_psap_encam'])
    except Exception:
        pass
    # Caso contrário, calcula pelo eco normalizado
    return mapear_psap_resultado(row.get('_psap_norm', ''))

dVal3['valor_psap_encam'] = dVal3.apply(resolver_psap, axis=1)
dVal3['valor_psap_encam'] = dVal3['valor_psap_encam'].replace('', pd.NA).astype('Int64')

# Remove coluna auxiliar
dVal3.drop(columns=['_psap_norm'], inplace=True, errors='ignore')


In [38]:
# 1) Instância por grupo temporal (ciclo)
dVal3['redcap_repeat_instance'] = dVal3.groupby(['record_id', 'ciclo_id']).cumcount() + 1

# 2) Ajuste de +1 para pacientes com cirurgia anterior (se essa lista existir)
if 'pct_cir_anterior' in globals():
    dVal3.loc[dVal3['record_id'].isin(pct_cir_anterior), 'redcap_repeat_instance'] += 1
else:
    # opcional: só para você saber em log
    print("Info: variável 'pct_cir_anterior' não definida; sem ajuste de instância.")

In [39]:
# Padronização de chaves antes do vínculo (boa prática)
dVal3['cd_atendimento'] = (
    dVal3['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
)
regvalv_cirurgia['cd_atendimento'] = (
    regvalv_cirurgia['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
)

# Vinculação híbrida:
dVal4 = vincular_instancias_temporal(
    df_principal=dVal3,
    df_referencia=regvalv_cirurgia,
    col_data_principal='DH_DOCUMENTO',
    col_data_referencia='data_cirurgia',
    tolerancia_dias=180,     # ✅ 6 meses
    col_atend_principal='cd_atendimento',
    col_atend_referencia='cd_atendimento'
)

In [40]:
# ========================
# Metadados REDCap
# ========================
# 1) Ajuste do nome do evento
dVal4['redcap_event_name'] = dVal4.get('redcap_event_name', pd.Series(index=dVal4.index, dtype='object'))
dVal4['redcap_event_name'] = dVal4['redcap_event_name'].replace({
    "fase3procedcirurgi_arm_1": "fase2preprocedimen_arm_1"
})
dVal4['redcap_event_name'] = dVal4['redcap_event_name'].fillna("fase2preprocedimen_arm_1")

# 2) Definição de defaults
dVal4['redcap_repeat_instance'] = (
    dVal4.get('redcap_repeat_instance', pd.Series(index=dVal4.index, dtype='object'))
         .fillna("1").astype(str).str.replace(r'\.0$', '', regex=True)
)
# Instrumento fixo para este dataset
dVal4['redcap_repeat_instrument'] = "pre_operatoria"

# Normaliza record_id e cd_atendimento (garante strings limpas)
for c in ['record_id', 'cd_atendimento']:
    if c in dVal4.columns:
        dVal4[c] = (dVal4[c].astype(str)
                              .str.replace(r'\.0$', '', regex=True)
                              .str.strip())

# ========================
# Listas de colunas (como você definiu)
# ========================
cols_drop_preop = [
    'ABLACAO_FA_FLUTTER_NAO_CIRURGIA_ASSOCIADA_1', 'ANTICOAGULACAO_NAO_VALVOPATIAS_1', 'CASO_DISCUTIDO_STAFF_1',
    'CORRECAO_ANEURISMA_AORTA_NAO_CIRURGIA_ASSOCIADA_1', 'DISFUNCAO_VENTRICULAR_FAC_N_A_1', 'DISFUNCAO_VENTRICULAR_TAPSE_N_A_1',
    'FECHAMENTO_APENDICE_ATRIAL_NAO_CIRURGIA_ASSOCIADA_1', 'FECHAMENTO_CIV_NAO_CIRURGIA_ASSOCIADA_1', 'MIECTOMIA_NAO_CIRURGIA_ASSOCIADA_1',
    'PROTOCOLO_ESTUDO_NAO_1', 'PROTOCOLO_ESTUDO_SIM_1', 'RESIDENTE_1', 'qtd_repeticoes', 'REVASCULARIZACAO_MIOCARDICA_NAO_CIRURGIA_ASSOCIAD_1',
    'TROMBECTOMIA_NAO_CIRURGIA_ASSOCIADA_1', 'ciclo_id', 'data_cirurgia', 'diff', 'tipo_cirurgia___1', 'tipo_cirurgia___2', 'tipo_cirurgia___3', 'tipo_cirurgia___4', 'tipo_cirurgia___10', 'tipo_cirurgia___5',
    'tipo_cirurgia___6', 'tipo_cirurgia___7', 'tipo_cirurgia___8', 'tipo_cirurgia___9', 'tipo_cirurgia_outros', 
    'cirurgica_associada_encam___1', 'cirurgica_associada_encam___2', 'cirurgica_associada_encam___3',
    'cirurgica_associada_encam___4', 'cirurgica_associada_encam___5', 'cirurgica_associada_encam___6',
    'cirurgica_associada_encam___7', 'tipo_valvula_prostetica',
]

cols_fatrisk = [
    'classe_nyha', 'angina', 'ccs', 'carga_sintomas_sincope', 'palpitacoes', 'fadiga', 'assintomatico', 'hipertensao',
    'diabetes', 'diabetes_controle', 'tabagismo', 'carga_tabagica', 'etilista', 'drc_clcr', 'dpoc', 'dpoc_medicamentos',
    'iam_menor_90dias', 'hipertensao_pulmonar_encam', 'valor_psap_encam', 'sequela_neurologica', 'data_avc_ait', 'avc_ait_previo',
    'endocardite_ativa', 'data_endocardite_ativa', 'massa_corporal_critica', 'avaliacao_corporal', 'arteriopatia_extracardiaca',
    'tipo_de_arteriopatia', 'mobilidade_severa_preju', 'estado_pr_operat_rio_cr_ti', 'imunossupressao', 'tipo_de_imunossupressao', 
    'cancer', 'tipo_de_cancer', 'tratamento_de_cancer', 'doenca_hepatica', 'intervencoes_cardiacas_previas',  
    'tipo_intervencao_previa___2', 'tipo_intervencao_previa___6', 'tipo_intervencao_previa___3', 'tipo_intervencao_previa___5', 'altura', 
    'peso', 'aceita_transfusao', 'anticoagulacao', 'sintomas_duracao', 'tempo_sintomas_duracao', 'outras_comorb_relevantes'
]

cols_eco = [
    'resultado_eco_psap', 'data_do_eco', 'fac_encam', 'feve_encam', 'tapse_encam', 'raiz_aorta', 'aorta_ascendente',
    'disfuncao_ventricular_encam', 'disfuncao_ventricular_encam_2', 
]

cols_coron = [
    'data_coronaria_exame', 'coronaria_exame', 'dac_exame', 'qtde_vasos_acometidos', 'tronco_de_coron_ria_esquer',
]

cols_odt = [
    'dt_libera_odonto', 'lib_cirurgia',
]

# cols_proc = [
#     'tipo_cirurgia___1', 'tipo_cirurgia___2', 'tipo_cirurgia___3', 'tipo_cirurgia___4', 'tipo_cirurgia___10', 'tipo_cirurgia___5',
#     'tipo_cirurgia___6', 'tipo_cirurgia___7', 'tipo_cirurgia___8', 'tipo_cirurgia___9', 'tipo_cirurgia_outros', 
#     'cirurgica_associada_encam___1', 'cirurgica_associada_encam___2', 'cirurgica_associada_encam___3',
#     'cirurgica_associada_encam___4', 'cirurgica_associada_encam___5', 'cirurgica_associada_encam___6',
#     'cirurgica_associada_encam___7', 'tipo_valvula_prostetica',
# ]

# ========================
# Conversões decimais unificadas
# ========================

colunas_ajuste = [
    'anel_valva_tricuspide', 'aorta_ascendente', 'area_valvar', 'area_valvar_3', 
    'area_valvar_2', 'area_valvar_4', 'euroscore_ii', 'area_valvar_5', 
    'peso', 'raiz_aorta', 'altura'
]

# Normalizador genérico de número: limpa lixos, troca vírgula por ponto e tenta converter
def _normalize_numeric_series(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    s = (s.str.replace('"', '', regex=False)
           .str.replace(r'\\', '', regex=True)
           .str.replace('%', '', regex=False)
           .str.replace('-', '', regex=False)
           .str.replace(' ', '', regex=False)
           .str.replace(',', '.', regex=False)
           .str.strip())
    # Remove strings que são só pontos: '.', '..', '...'
    s = s.apply(lambda x: '' if re.fullmatch(r'\.+', x or '') else x)
    # Zera tokens de nulo conhecidos
    null_tokens = {'nan', 'NaN', 'None', '<NA>', '&lt;NA&gt;', 'NAT', 'pd.NA', ''}
    s = s.apply(lambda x: '' if str(x).strip() in null_tokens else str(x).strip())
    # Converte para numérico onde possível
    out = pd.to_numeric(s, errors='coerce')
    return out

for col in colunas_ajuste:
    if col in dVal4.columns:
        dVal4[col] = _normalize_numeric_series(dVal4[col])

# ========================
# Limpeza de checkboxes e flags de valvulopatia
# ========================
# Mapeia qualquer verdade para '1', restante '0'
def _to_redcap_checkbox(s: pd.Series) -> pd.Series:
    if s.dtype != 'O':
        s = s.astype(object)
    s_str = s.astype(str).str.upper().str.strip()
    truthy = {'1', '1.0', 'TRUE', 'SIM', 'S', 'YES'}
    falsy  = {'0', '0.0', 'FALSE', 'NAO', 'N', 'NO', 'NÃO'}
    # default 0
    out = pd.Series('0', index=s.index, dtype=object)
    out[s_str.isin(truthy)] = '1'
    out[s_str.isin(falsy)]  = '0'
    # Onde for número real (1/0), preserva
    out[s_str == ''] = '0'
    return out

cols_checkbox = [c for c in dVal4.columns if '___' in c]
cols_valvulopatia = ['valvulopatia_aortica', 'valvulopatia_mitral', 'valvulopatia_tricuspide']
cols_para_limpar = [c for c in cols_checkbox + cols_valvulopatia if c in dVal4.columns]

for col in cols_para_limpar:
    dVal4[col] = _to_redcap_checkbox(dVal4[col])

# ========================
# Carga tabágica: remove '>' e extrai número
# ========================
if 'carga_tabagica' in dVal4.columns:
    s = dVal4['carga_tabagica'].astype(str).str.replace('&gt;', '>', regex=False).str.replace('>', '', regex=False)
    s = s.str.replace(',', '.', regex=False).str.strip()
    # tenta pegar o primeiro número na string
    s_num = s.str.extract(r'([0-9]+(?:\.[0-9]+)?)', expand=False)
    dVal4['carga_tabagica'] = s_num.fillna('')

# ========================
# Conversões finais de numéricos (onde fizer sentido para o REDCap)
# ========================
# Ex.: garantir numéricos verdadeiros (float) para estes campos
for col in ['euroscore_ii', 'peso', 'raiz_aorta', 'aorta_ascendente',
            'anel_valva_tricuspide', 'area_valvar', 'area_valvar_2', 'area_valvar_3', 'area_valvar_4', 'area_valvar_5']:
    if col in dVal4.columns:
        # Já está em float por _normalize_numeric_series, mas se tiver virado string em alguma etapa:
        dVal4[col] = pd.to_numeric(dVal4[col], errors='coerce')

# Garantir domínios de radio/ordinais
if 'valor_psap_encam' in dVal4.columns:
    dVal4['valor_psap_encam'] = pd.to_numeric(dVal4['valor_psap_encam'], errors='coerce').astype('Int64')
    dVal4.loc[~dVal4['valor_psap_encam'].isin([1, 2, 3]), 'valor_psap_encam'] = pd.NA


In [41]:
from pathlib import Path

# =========================
# 1) Leitura dos .parquet
# =========================
pasta_perfil = Path(r'C://Users/priscilla.sequetin/Documents/BasesDashs/perfil_pq')

dfs_perfil = []
for caminho in pasta_perfil.glob('atend_*.parquet'):
    try:
        df_tmp = pd.read_parquet(caminho)
        dfs_perfil.append(df_tmp)
        print(f"✅ Lido com sucesso: {caminho.name}")
    except Exception as e:
        print(f"❌ Erro ao ler {caminho.name}: {e}")

if dfs_perfil:
    df_perfil = pd.concat(dfs_perfil, ignore_index=True, sort=False)
    print(f"\n🚀 Total de registros concatenados: {len(df_perfil)}")
    print(f"Colunas encontradas: {df_perfil.columns.tolist()}")
else:
    raise RuntimeError("⚠️ Nenhum arquivo .parquet encontrado no padrão 'atend_*.parquet'.")

# =========================
# 2) Seleção de colunas (somente as existentes)
# =========================
cols_desejadas = [
    'ANO', 'CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'TABAGISMO', 'AVC', 'HAS', 
    'DM', 'DLP', 'DRC', 'DPOC', 'DAOP', 'DAC', 'DAC TIPOS', 'ICC', 'ICC TIPOS', 
    'ARRITMIA', 'ARRITMIA TIPOS', 'ANGINA', 'ANGINA TIPOS', 'EDEMA', 
    'DOR TORACICA', 'DOR TORACICA TIPOS', 'SINCOPE', 'DISPNEIA', 'DISPNEIA TIPOS', 
    'PALPITACAO', 'VALVOPATIA', 'VALVOPATIA TIPOS', 'VALVOPATIA PROTESE', 
    'TAVI PREVIA', 'IAO', 'EAO', 'IMI', 'EMI', 'ITRI', 'ETRI', 'TONTURA', 
    'CLASSIFICACAO PS'
]
cols_ok = [c for c in cols_desejadas if c in df_perfil.columns]
aCom1 = df_perfil.copy()
aCom2 = aCom1.loc[:, cols_ok].copy()

# =========================
# 3) Filtragem por listas (robusta em string)
# =========================
def _norm_id_list(seq):
    if seq is None:
        return []
    return [str(x).replace(r'\.0', '').strip() for x in seq]

lista_check_cir1 = _norm_id_list(regvalv_pcte.get('record_id') if isinstance(regvalv_pcte, pd.DataFrame) else regvalv_pcte)
lista_check_cir2 = _norm_id_list(dVal4.get('record_id') if isinstance(dVal4, pd.DataFrame) else dVal4)


lista_ftrisk = list(set(lista_check_cir1) | set(lista_check_cir2))
print(f"✅ Total record_id para filtrar: {len(lista_ftrisk)}")

# Filtra CD_PACIENTE como string normalizada
aCom2['CD_PACIENTE'] = aCom2['CD_PACIENTE'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
aCom2 = aCom2[aCom2['CD_PACIENTE'].isin(lista_ftrisk)].copy()

# =========================
# 4) Datas, ano e ordenação
# =========================
aCom2['DH_DOCUMENTO'] = pd.to_datetime(aCom2['DH_DOCUMENTO'], errors='coerce')
aCom2['ANO'] = pd.to_numeric(aCom2['ANO'], errors='coerce').astype('Int64')

# Ordena por paciente e data
aCom2 = aCom2.sort_values(['CD_PACIENTE', 'DH_DOCUMENTO'], ascending=True, kind='mergesort')

# Agrupa por PACIENTE e ANO e pega a última linha (mais recente no ano)
aCom3 = aCom2.groupby(['CD_PACIENTE', 'ANO'], as_index=False).last()

# =========================
# 5) Função de transformação p/ REDCap
# =========================
def criar_redcap_completo(df_original: pd.DataFrame) -> pd.DataFrame:
    df_temp = df_original.copy()

    # record_id
    if 'CD_PACIENTE' in df_temp.columns:
        df_temp['CD_PACIENTE'] = df_temp['CD_PACIENTE'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        record_id = df_temp['CD_PACIENTE']
        print(f'✅ record_id preservado: {len(record_id)} registros')
    else:
        record_id = pd.Series(range(len(df_temp)), index=df_temp.index)
        print('⚠️ CD_PACIENTE não encontrada, usando índice')

    # LIMPEZA de valores string (preserva colunas chaves)
    colunas_preservadas = ['CD_PACIENTE', 'ANO']
    for col in df_temp.select_dtypes('object').columns:
        if col not in colunas_preservadas:
            df_temp[col] = (
                df_temp[col]
                .str.strip()
                .str.upper()
                .str.replace('[-_/]', '', regex=True)
            )

    # -------- FATORES DE RISCO --------
    cols_ftrisk = {}
    cols_ftrisk['tabagismo'] = df_temp.get('TABAGISMO', pd.Series(index=df_temp.index)).map({
        'NAO': 0, 'EXTBG': 1, 'SIM': 2
    })
    cols_ftrisk['hipertensao'] = df_temp.get('HAS', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    cols_ftrisk['diabetes']    = df_temp.get('DM', pd.Series(index=df_temp.index)).map({'NAO': 0, 'DMID': 1, 'DMNID': 1})
    cols_ftrisk['tipo_diabetes'] = df_temp.get('DM', pd.Series(index=df_temp.index)).map({'DMID': 1, 'DMNID': 2})
    cols_ftrisk['diabetes_controle'] = df_temp.get('DM', pd.Series(index=df_temp.index)).map({'DMID': 3, 'DMNID': 2})
    cols_ftrisk['dislipidemia'] = df_temp.get('DLP', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    cols_ftrisk['doenca_renal_cronica'] = df_temp.get('DRC', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1, 'DIALITICO': 1})
    cols_ftrisk['drc_clcr'] = df_temp.get('DRC', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 2, 'DIALITICO': 4})
    cols_ftrisk['dpoc'] = df_temp.get('DPOC', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    cols_ftrisk['avc_ait_previo'] = df_temp.get('AVC', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    cols_ftrisk['angina'] = df_temp.get('ANGINA', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})

    def map_ccs(valor):
        if pd.isna(valor):
            return np.nan
        valor = str(valor)
        if 'CCS1' in valor: return 1
        if 'CCS2' in valor: return 2
        if 'CCS3' in valor: return 3
        if 'CCS4' in valor: return 4
        return np.nan

    cols_ftrisk['ccs'] = df_temp.get('ANGINA TIPOS', pd.Series(index=df_temp.index)).apply(map_ccs)

    nyha_map = {'DISPNEIAI': 1, 'DISPNEIAII': 2, 'DISPNEIAIII': 3, 'DISPNEIAIV': 4}
    cols_ftrisk['carga_sintomas_dispneia'] = df_temp.get('DISPNEIA TIPOS', pd.Series(index=df_temp.index)).map(nyha_map)
    cols_ftrisk['classe_nyha'] = cols_ftrisk['carga_sintomas_dispneia']

    cols_ftrisk['carga_sintomas_sincope'] = df_temp.get('SINCOPE', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    cols_ftrisk['palpitacoes'] = df_temp.get('PALPITACAO', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})

    dor_map = {'NAO': 0, 'TIPOC': 1, 'TIPOD': 1, 'TIPOB': 2, 'TIPOA': 3}
    cols_ftrisk['carga_sintomas_dor_toracica'] = df_temp.get('DOR TORACICA TIPOS', pd.Series(index=df_temp.index)).map(dor_map)
    cols_ftrisk['carga_sintomas_edema'] = df_temp.get('EDEMA', pd.Series(index=df_temp.index)).map({'SIM': 2, 'NAO': 0})

    # -------- PRÉ-OP --------
    cols_preop = {}
    cols_preop['arritmias'] = df_temp.get('ARRITMIA', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    # MP / TV baseados nos tipos de arritmia
    arr_tipos = df_temp.get('ARRITMIA TIPOS', pd.Series(index=df_temp.index))
    cols_preop['possui_mp_trc_cdi'] = (arr_tipos == 'MP').where(arr_tipos.notna()).astype('Int64')
    cols_preop['estado_pr_operat_rio_cr_ti'] = (arr_tipos == 'TV').where(arr_tipos.notna()).astype('Int64')

    cols_preop['valvulopatia_aortica'] = df_temp.get('IAO', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})
    cols_preop['valvulopatia_mitral']  = df_temp.get('IMI', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})

    # Radiobox mitrada aórtica
    vp_tipos = df_temp.get('VALVOPATIA TIPOS', pd.Series(index=df_temp.index))
    vp_prot  = df_temp.get('VALVOPATIA PROTESE', pd.Series(index=df_temp.index))

    cols_preop['valvula_nativa___2'] = (
        ((vp_tipos == 'NATIVA') | (df_temp.get('IAO', pd.Series(index=df_temp.index)) == 'SIM'))
        .where(vp_tipos.notna() | df_temp.get('IAO', pd.Series(index=df_temp.index)).notna())
        .astype('Int64')
    )
    cols_preop['protese_mecanica___2'] = (
        ((vp_prot == 'MECANICA') | (df_temp.get('IAO', pd.Series(index=df_temp.index)) == 'SIM'))
        .where(vp_prot.notna() | df_temp.get('IAO', pd.Series(index=df_temp.index)).notna())
        .astype('Int64')
    )
    cols_preop['protese_biologica___2'] = (
        ((vp_prot == 'BIOLOGICA') | (df_temp.get('IAO', pd.Series(index=df_temp.index)) == 'SIM'))
        .where(vp_prot.notna() | df_temp.get('IAO', pd.Series(index=df_temp.index)).notna())
        .astype('Int64')
    )

    # -------- CORONÁRIAS --------
    cols_coron = {}
    cols_coron['dac_exame'] = df_temp.get('DAC', pd.Series(index=df_temp.index)).map({'NAO': 0, 'SIM': 1})

    dac_tipos = df_temp.get('DAC TIPOS', pd.Series(index=df_temp.index))
    cols_ftrisk['intervencoes_cardiacas_previas'] = dac_tipos.isin(['ICPPREVIA', 'RMPREVIA']).where(dac_tipos.notna()).astype('Int64')
    cols_ftrisk['tipo_intervencao_previa___1'] = (dac_tipos == 'ICPPREVIA').where(dac_tipos.notna()).astype('Int64')
    cols_ftrisk['tipo_intervencao_previa___2'] = (dac_tipos == 'RMPREVIA').where(dac_tipos.notna()).astype('Int64')

    # -------- DF FINAL --------
    df_redcap = pd.concat(
        [
            pd.Series(record_id, name='record_id', index=df_temp.index),
            df_temp[['ANO']].astype('Int64') if 'ANO' in df_temp.columns else pd.DataFrame(index=df_temp.index),
            pd.DataFrame(cols_ftrisk, index=df_temp.index),
            pd.DataFrame(cols_preop, index=df_temp.index),
            pd.DataFrame(cols_coron, index=df_temp.index),
        ],
        axis=1
    )

    # Tipos (mantém NaN com Int64)
    cols_ignorar = ['record_id', 'ANO']
    for col in df_redcap.columns:
        if col not in cols_ignorar and pd.api.types.is_numeric_dtype(df_redcap[col]):
            df_redcap[col] = df_redcap[col].astype('Int64')

    print('✅ DF REDCap FINAL CRIADO!')
    print(f'📊 Shape: {df_redcap.shape}')
    print(f'🎯 record_id únicos: {df_redcap["record_id"].nunique()}')
    if 'ANO' in df_redcap.columns:
        print(f'📅 Anos processados: {sorted([a for a in df_redcap["ANO"].dropna().unique().tolist()])}')
    return df_redcap

# EXECUTA
aCom4 = criar_redcap_completo(aCom3)

# =========================
# 6) Vincular por record_id + ANO com regvalv_cirurgia
# =========================
aCom4['record_id'] = aCom4['record_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

regvalv_cirurgia['record_id'] = (
    regvalv_cirurgia['record_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
)
regvalv_cirurgia['data_cirurgia'] = pd.to_datetime(regvalv_cirurgia['data_cirurgia'], errors='coerce')
regvalv_cirurgia['ANO'] = regvalv_cirurgia['data_cirurgia'].dt.year.astype('Int64')

cols_ref = ['record_id', 'ANO', 'redcap_event_name', 'redcap_repeat_instance']
for c in cols_ref:
    if c not in regvalv_cirurgia.columns:
        regvalv_cirurgia[c] = pd.NA

aCom5 = regvalv_cirurgia[cols_ref].merge(
    aCom4,
    on=['record_id', 'ANO'],
    how='left'
)

# Instância (string)
aCom5['redcap_repeat_instance'] = (
    aCom5['redcap_repeat_instance']
        .fillna(1)
        .apply(lambda x: str(int(float(x))) if pd.notna(x) else '1')
)




✅ Lido com sucesso: atend_2022_13.03.2025.parquet
✅ Lido com sucesso: atend_2023_13.03.2025.parquet
✅ Lido com sucesso: atend_2024_13.03.2025.parquet
✅ Lido com sucesso: atend_2025_05.01.2026.parquet
✅ Lido com sucesso: atend_2026_19.02.2026.parquet

🚀 Total de registros concatenados: 2062739
Colunas encontradas: ['ANO', 'MES', 'CD_ATENDIMENTO', 'CD_PACIENTE', 'CD_CID', 'GRUPO_CID', 'SEXO', 'ESTADO_CIVIL', 'COR', 'CIDADANIA', 'RELIGIAO', 'ESCOLARIDADE', 'PROFISSAO', 'CD_UF', 'CIDADE', 'NR_CEP', 'NM_BAIRRO', 'DT_NASCIMENTO', 'SETOR', 'UNI_HOSP', 'UNI_LOCAL', 'UNIDADE_ENTRADA', 'UNIDADE_ALTA', 'HR_MOV_INT', 'DS_TIPO_INTERNACAO', 'CD_PROCEDIMENTO', 'DS_PROCEDIMENTO', 'TIPO_ALTA', 'MOTIVO_ALTA', 'HR_ALTA_MEDICA', 'HR_ATENDIMENTO', 'TP_PERM_DIAS', 'TP_PERM_HORAS', 'TP_PERM_MIN', 'TP_ATENDIMENTO', 'TP_ORIGEM', 'DS_SENHA', 'CD_COR_REFERENCIA', 'NM_COR', 'CD_SINTOMA_AVALIACAO', 'DS_SINTOMA', 'CD_CLASSIFICACAO', 'DS_TIPO_RISCO', 'DS_QUEIXA_PRINCIPAL', 'CD_FILA_SENHA', 'DS_FILA', 'DS_ALERGIA', '

C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_18248\1089264040.py:229: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(1)


In [142]:

# ================================================
# HARMONIZAÇÃO dVal4 (base) COM aCom5 (complemento)
# Executar ANTES do fracionamento em sub-DFs
# Saída: dVal5  (harmonizado)
# ================================================

def _norm_id_series(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.replace(r'\.0$', '', regex=True)
         .str.strip()
    )

def _to_checkbox_str01(s: pd.Series) -> pd.Series:
    """
    Converte para strings '0'/'1' (padrão REDCap) com defaults estáveis.
    """
    if s is None:
        return s
    s = s.astype(str).str.upper().str.strip()
    truthy = {'1', '1.0', 'TRUE', 'SIM', 'S', 'YES'}
    falsy  = {'0', '0.0', 'FALSE', 'NAO', 'N', 'NO', 'NÃO'}

    out = pd.Series('', index=s.index, dtype=object)
    out[s.isin(truthy)] = '1'
    out[s.isin(falsy)]  = '0'

    # Onde não bate, tenta numérico
    mask_rest = ~(s.isin(truthy | falsy))
    if mask_rest.any():
        numeric = pd.to_numeric(s[mask_rest], errors='coerce')
        out.loc[mask_rest & numeric.eq(1, fill_value=False)] = '1'
        out.loc[mask_rest & numeric.eq(0, fill_value=False)] = '0'

    return out

def _is_binary_like(col_s: pd.Series) -> bool:
    """
    Heurística: depois de coercer para numérico, só aparecem 0/1/NaN.
    """
    if col_s is None:
        return False
    s = pd.to_numeric(col_s, errors='coerce')
    uniq = set(s.dropna().unique().tolist())
    return uniq.issubset({0, 1})

def _dedup_columns(df: pd.DataFrame, name: str = 'df') -> pd.DataFrame:
    """
    Remove colunas duplicadas preservando a primeira ocorrência.
    """
    df = df.copy()
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].tolist()
        print(f"⚠️ {name}: removendo colunas duplicadas: {sorted(set(dupes))}")
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

# --- 1) Preparação de chaves, ANO e deduplicação de colunas
dVal4 = _dedup_columns(dVal4, 'dVal4')
aCom5 = _dedup_columns(aCom5, 'aCom5')

dVal4 = dVal4.copy()
aCom5 = aCom5.copy()

if 'record_id' in dVal4.columns:
    dVal4['record_id'] = _norm_id_series(dVal4['record_id'])
if 'record_id' in aCom5.columns:
    aCom5['record_id'] = _norm_id_series(aCom5['record_id'])

# ANO: derivado de DH_DOCUMENTO no dVal4
if 'DH_DOCUMENTO' in dVal4.columns:
    dVal4['DH_DOCUMENTO'] = pd.to_datetime(dVal4['DH_DOCUMENTO'], errors='coerce')
    dVal4['ANO'] = dVal4['DH_DOCUMENTO'].dt.year.astype('Int64')
else:
    if 'ANO' not in dVal4.columns:
        dVal4['ANO'] = pd.NA

# ANO no aCom5 (já existe; apenas tipar)
aCom5['ANO'] = pd.to_numeric(aCom5.get('ANO', pd.Series(index=aCom5.index)), errors='coerce').astype('Int64')

# --- 2) Descobrir colunas sobrepostas que serão harmonizadas
#    ⚠️ Excluímos as chaves ('record_id', 'ANO') e metadados REDCap do overlap
meta_cols_exclude = {
    'record_id', 'ANO',  # chaves
    'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance',
    'DH_DOCUMENTO', 'cd_atendimento'
}
overlap_cols = [c for c in dVal4.columns.intersection(aCom5.columns) if c not in meta_cols_exclude]

# Classificar sobrepostas
checkbox_cols = [c for c in overlap_cols if '___' in c]

bin_like_cols = []
for c in [c for c in overlap_cols if c not in checkbox_cols]:
    if _is_binary_like(dVal4[c]) or _is_binary_like(aCom5[c]):
        bin_like_cols.append(c)

other_cols = [c for c in overlap_cols if c not in set(checkbox_cols + bin_like_cols)]

print(
    f"🔎 Overlap total: {len(overlap_cols)} | "
    f"checkboxes: {len(checkbox_cols)} | "
    f"binários: {len(bin_like_cols)} | "
    f"outros: {len(other_cols)}"
)

# --- 3) Alinhar aCom5 por (record_id, ANO) e mesclar no dVal4
#     ⚠️ Evite repetir 'record_id'/'ANO' dentro do 'overlap_cols'
aCom5_slim = aCom5[['record_id', 'ANO'] + overlap_cols].copy()
aCom5_slim = (
    aCom5_slim
    .sort_values(['record_id', 'ANO'])
    .drop_duplicates(subset=['record_id', 'ANO'], keep='last')
)

dVal4m = dVal4.merge(
    aCom5_slim,
    on=['record_id', 'ANO'],
    how='left',
    suffixes=('', '_ac5')
)

# --- 4) Regras de harmonização (precedência)
update_report = {'checkbox_updated': [], 'binary_updated': [], 'filled_from_ac6': []}

# 4.1) Checkboxes: '0'/'1' → regra do máximo + preencher nulos
for c in checkbox_cols:
    c_ac5 = f'{c}_ac5'
    if c_ac5 not in dVal4m.columns:
        continue

    left = _to_checkbox_str01(dVal4m[c])
    right = _to_checkbox_str01(dVal4m[c_ac5])

    left_num = pd.to_numeric(left.replace('', np.nan), errors='coerce')
    right_num = pd.to_numeric(right.replace('', np.nan), errors='coerce')

    final = left_num.copy()

    # ambos presentes → máximo
    mask_both = left_num.notna() & right_num.notna()
    final.loc[mask_both] = np.fmax(left_num[mask_both], right_num[mask_both])

    # left nulo e right válido → right
    mask_left_na = left_num.isna() & right_num.notna()
    final.loc[mask_left_na] = right_num[mask_left_na]

    # reconverte para '0'/'1' ou '' se NaN
    final_str = final.apply(lambda x: '' if pd.isna(x) else ('1' if int(x) == 1 else '0')).astype(object)
    changed = (final_str != left.fillna('')).sum()
    if changed > 0:
        update_report['checkbox_updated'].append((c, int(changed)))

    dVal4m[c] = final_str

# 4.2) Binários (sem ___): usar máximo e preencher nulos
for c in bin_like_cols:
    c_ac5 = f'{c}_ac5'
    if c_ac5 not in dVal4m.columns:
        continue

    left = pd.to_numeric(dVal4m[c], errors='coerce').astype('Int64')
    right = pd.to_numeric(dVal4m[c_ac5], errors='coerce').astype('Int64')

    final = left.copy()

    mask_both = left.notna() & right.notna()
    final.loc[mask_both] = final.loc[mask_both].where(
        final.loc[mask_both] >= right.loc[mask_both],
        right.loc[mask_both]
    )

    mask_left_na = left.isna() & right.notna()
    final.loc[mask_left_na] = right.loc[mask_left_na]

    changed = (final != left).sum()
    if changed > 0:
        update_report['binary_updated'].append((c, int(changed)))

    dVal4m[c] = final  # Int64 mantém NaN

# 4.3) Outros campos (numérico/radio/texto): preencher só nulos do dVal4 com aCom5
for c in other_cols:
    c_ac5 = f'{c}_ac5'
    if c_ac5 not in dVal4m.columns:
        continue

    left = dVal4m[c]
    right = dVal4m[c_ac5]

    if pd.api.types.is_numeric_dtype(left) or pd.api.types.is_numeric_dtype(right):
        left_num = pd.to_numeric(left, errors='coerce')
        right_num = pd.to_numeric(right, errors='coerce')
        filled = left_num.fillna(right_num)
        changed = (filled != left_num).sum()
        if changed > 0:
            update_report['filled_from_ac6'].append((c, int(changed)))
        # mantém dtype original quando possível
        dVal4m[c] = filled.astype(left.dtype) if left.dtype != object else filled
    else:
        # texto/categorias
        filled = left.where(left.notna() & (left.astype(str).str.strip() != ''), right)
        changed = (filled != left).sum()
        if changed > 0:
            update_report['filled_from_ac6'].append((c, int(changed)))
        dVal4m[c] = filled

# --- 5) Limpeza de colunas auxiliares (_ac5) e saída
aux_cols = [f'{c}_ac5' for c in overlap_cols if f'{c}_ac5' in dVal4m.columns]
dVal5 = dVal4m.drop(columns=aux_cols, errors='ignore')  # <- Saída final harmonizada

print("🧾 Harmonização concluída.")
print(" • Checkboxes atualizadas:", update_report['checkbox_updated'])
print(" • Binários atualizados :", update_report['binary_updated'])
print(" • Campos preenchidos (nulos -> aCom5):", update_report['filled_from_ac6'])

🔎 Overlap total: 21 | checkboxes: 4 | binários: 13 | outros: 4
🧾 Harmonização concluída.
 • Checkboxes atualizadas: [('valvula_nativa___2', 8), ('protese_biologica___2', 42), ('protese_mecanica___2', 37)]
 • Binários atualizados : [('angina', 1), ('avc_ait_previo', 2), ('carga_sintomas_sincope', 1), ('dac_exame', 6), ('dpoc', 2), ('hipertensao', 4), ('palpitacoes', 2), ('valvulopatia_aortica', 2), ('valvulopatia_mitral', 5)]
 • Campos preenchidos (nulos -> aCom5): []


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1524377433.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out.loc[mask_rest & numeric.eq(1, fill_value=False)] = '1'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1524377433.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out.loc[mask_rest & numeric.eq(0, fill_value=False)] = '0'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1524377433.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in 

In [46]:
dVal5 = dVal4[dVal4['record_id']=='752955']

In [44]:
# =====================================================
# --- 1. PREPARAÇÃO DOS DADOS DO REDCAP (EXISTENTES) ---

# Garantir que record_id seja string em ambos os lados para evitar erros de comparação ( "100" != 100 )
regvalv_preop['record_id'] = regvalv_preop['record_id'].astype(str)

# Criamos um "Set" de tuplas com (ID, EVENTO). 
# Isso funciona como uma lista de presença: "O paciente X já tem ficha no evento Y"
preop_existentes = set(zip(
    regvalv_preop['record_id'].astype(str), 
    regvalv_preop['redcap_event_name'].astype(str),
    regvalv_preop['redcap_repeat_instance'].astype(str)
    ))

print(f"Total de registros já existentes no Redcap: {len(preop_existentes)}")


# --- 2. FILTRANDO OS DADOS LOCAIS ---

# Filtra cada fase
dVal5 = filtrar_novos_registros(dVal5, preop_existentes)


# Exibe o resultado
print('Qtde de pctes únicos: ', dVal5['record_id'].nunique())
print('Qtde de colunas: ', len(dVal5.columns))
print('Colunas dVal5: ', dVal5.columns.tolist())
print('Registros: ', dVal5['record_id'].unique())

Total de registros já existentes no Redcap: 552
📊 Filtro aplicado:
   - Registros originais no DataFrame: 220
   - Registros ignorados (já existem no REDCap): 210
   - Novos registros para importar: 10
Qtde de pctes únicos:  10
Qtde de colunas:  183
Colunas dVal5:  ['record_id', 'ciclo_id', 'cd_atendimento', 'DH_DOCUMENTO', 'CD_DOCUMENTO', 'ABLACAO_FA_FLUTTER_NAO_CIRURGIA_ASSOCIADA_1', 'cirurgica_associada_encam___3', 'altura', 'anel_valva_tricuspide', 'diagnostico_principal___6', 'dt_prev_cir', 'ANTICOAGULACAO_NAO_VALVOPATIAS_1', 'aorta_ascendente', 'area_valvar', 'area_valvar_3', 'area_valvar_2', 'area_valvar_4', 'tipo_de_cancer', 'carga_tabagica', 'CASO_DISCUTIDO_STAFF_1', 'tp_cir_prev', 'CORRECAO_ANEURISMA_AORTA_NAO_CIRURGIA_ASSOCIADA_1', 'cirurgica_associada_encam___2', 'data_endocardite_ativa', 'data_do_eco', 'data_coronaria_exame', 'data_avc_ait', 'dt_libera_odonto', 'estenose_importante___1', 'estenose_importante___2', 'estenose_moderada___1', 'estenose_moderada___2', 'disfunca

In [47]:

# =====================================================
# 0) FUNÇÕES UTILITárias
# =====================================================

def _to_redcap_checkbox(s: pd.Series) -> pd.Series:
    """Converte para strings '0'/'1' (padrão REDCap) com defaults estáveis."""
    if s is None:
        return s
    
    s_str = s.astype(str).str.upper().str.strip()
    truthy = {'1', '1.0', 'TRUE', 'SIM', 'S', 'YES'}
    falsy = {'0', '0.0', 'FALSE', 'NAO', 'N', 'NO', 'NÃO'}

    out = pd.Series('0', index=s.index, dtype=object)
    out[s_str.isin(truthy)] = '1'
    out[s_str.isin(falsy)] = '0'

    # Onde não casar, tenta numérico (1/0)
    mask_rest = ~(s_str.isin(truthy | falsy))
    if mask_rest.any():
        num = pd.to_numeric(s_str[mask_rest], errors='coerce')
        out.loc[mask_rest & num.eq(1)] = '1'
        out.loc[mask_rest & num.eq(0)] = '0'
    
    return out


def dedup_redcap(df: pd.DataFrame) -> pd.DataFrame:
    """Deduplica por chave REDCap. Mantém o mais recente por DH_DOCUMENTO, se existir."""
    key = ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance']
    keep_cols = [c for c in key if c in df.columns]
    
    if not keep_cols:
        return df
        
    if 'DH_DOCUMENTO' in df.columns:
        return (df.sort_values('DH_DOCUMENTO')
                   .drop_duplicates(subset=keep_cols, keep='last'))
    return df.drop_duplicates(subset=keep_cols, keep='last')


def extrair_instrumento(df: pd.DataFrame, cols_especificas, instrumento, evento=None, drop_all_na=True):
    """
    Extrai colunas para um instrumento específico, adiciona metadados REDCap,
    remove linhas completamente vazias nas colunas específicas.
    """
    base = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'redcap_repeat_instrument']
    use_cols = [c for c in (base + cols_especificas) if c in df.columns]
    
    if not use_cols:
        return pd.DataFrame(columns=base + [f'{instrumento}_complete'])

    out = df.loc[:, use_cols].copy()

    # Evento/instrumento
    if evento is not None and 'redcap_event_name' in out.columns:
        out['redcap_event_name'] = evento
    out['redcap_repeat_instrument'] = instrumento
    out[f'{instrumento}_complete'] = 0  # altere para 2 se quiser "finalizado"

    # Checkboxes: normaliza '___' para '0'/'1'
    check_cols = [c for c in out.columns if '___' in c]
    for c in check_cols:
        out[c] = _to_redcap_checkbox(out[c])

    # Remove linhas "todas vazias" no conjunto das colunas específicas
    if drop_all_na and cols_especificas:
        exist = [c for c in cols_especificas if c in out.columns]
        if exist:
            mask_all_na = out[exist].isna().all(axis=1)
            mask_all_empty = out[exist].apply(lambda col: col.astype(str).str.strip().eq('').fillna(True)).all(axis=1)
            out = out.loc[~(mask_all_na | mask_all_empty)].copy()

    # Dedup por chave REDCap do instrumento
    return dedup_redcap(out)


def _to_float_series(s: pd.Series, preserve_precision: bool = True) -> pd.Series:
    """
    Normaliza strings numéricas para float com limpeza robusta.
    
    Args:
        preserve_precision: Se True, mantém precisão original (sem arredondamento automático)
    """
    if s is None:
        return s
    
    ss = (s.astype(str)
           .str.upper()
           .str.replace('&GT;', '>', regex=False)
           .str.replace('&LT;', '<', regex=False)
           .str.replace('<', '', regex=False)
           .str.replace('>', '', regex=False)
           .str.replace('"', '', regex=False)
           .str.replace('%', '', regex=False)
           .str.replace(' ', '', regex=False)
           .str.replace(',', '.', regex=False))
    
    # Remove caracteres não numéricos (exceto ponto e sinal)
    ss = ss.str.replace(r'[^0-9\.\-]', '', regex=True)

    # Normaliza múltiplos pontos — mantém apenas o primeiro
    def _fix_multi_dots(x: str) -> str:
        if pd.isna(x) or x.count('.') <= 1:
            return x
        head, *tails = x.split('.')
        return head + '.' + ''.join(tails)

    ss = ss.apply(lambda x: _fix_multi_dots(x) if isinstance(x, str) and pd.notna(x) else x)
    
    # Converte para float preservando precisão
    result = pd.to_numeric(ss, errors='coerce')
    
    if preserve_precision:
        # Evita arredondamento automático mantendo como float64
        # (não força decimal places)
        pass
    
    return result


def _normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Padroniza chaves e instâncias como string (sem '.0')."""
    if df is None or df.empty:
        return df
    
    if 'record_id' in df.columns:
        df['record_id'] = (df['record_id'].astype(str)
                                 .str.replace(r'\.0$', '', regex=True)
                                 .str.strip())
    
    if 'redcap_repeat_instance' in df.columns:
        df['redcap_repeat_instance'] = (df['redcap_repeat_instance'].astype(str)
                                              .str.replace(r'\.0$', '', regex=True)
                                              .str.strip()
                                              .replace({'nan': '1', 'None': '1', '' : '1'}))
    return df


def _peek_cols(df, label, cols):
    """Sanity-check rápido das colunas formatadas."""
    if df is None or df.empty:
        print(f"{label}: vazio")
        return
    print(f"\n{label} - amostra pós-formatação:")
    for c in cols:
        if c in df.columns:
            print(f"  {c}: {df[c].dropna().astype(str).head(5).tolist()}")

# =====================================================
# 1) CONFIGURAÇÕES GLOBAIS
# =====================================================

# Colunas base obrigatórias
COLS_BASE = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'redcap_repeat_instrument']

# Configuração dos instrumentos (assumindo que as variáveis cols_* existem)
CONFIG_INSTRUMENTOS = {
    'fatores_risco': (cols_fatrisk, 'fase1preoperatorio_arm_1'),
    'ecocardiograma': (cols_eco, 'fase1preoperatorio_arm_1'),
    'coronarias': (cols_coron, 'fase1preoperatorio_arm_1'),
    'odontologia': (cols_odt, 'fase1preoperatorio_arm_1'),
    #'procedimento': (cols_proc, 'fase3procedcirurgi_arm_1')
}

# Colunas auxiliares para remoção
COLS_AUX = [
    'cd_atendimento','DH_DOCUMENTO','CD_DOCUMENTO','data_cirurgia',
    'ciclo_id','diff','novo_ciclo','qtd_ciclos_paciente','qtd_repeticoes',
    'redcap_repeat_instance__ref','record_id_ref', 'ANO'
]

# Colunas numéricas para formatação
COLS_NUMERICAS = [
    'area_valvar', 'area_valvar_2', 'area_valvar_3', 'area_valvar_4',
    'euroscore_ii', 'peso', 'fac_encam', 'feve_encam', 'tapse_encam'
]

# =====================================================
# 2) PROCESSAMENTO PRINCIPAL
# =====================================================

def processar_dados_redcap(dVal5: pd.DataFrame, cols_drop_preop: list) -> dict:
    """
    Pipeline completo de processamento REDCap.
    Retorna dicionário com DataFrames processados.
    """
    
    # 2.1) Extração dos sub-instrumentos
    sub_dfs = {}
    todas_cols_mapeadas = []
    
    for inst_nome, (cols_especificas, evento_nome) in CONFIG_INSTRUMENTOS.items():
        df_temp = extrair_instrumento(dVal5, cols_especificas, inst_nome, evento=evento_nome)
        sub_dfs[inst_nome] = df_temp
        todas_cols_mapeadas.extend([c for c in cols_especificas if c in dVal5.columns])
    
    # Atribuições individuais para compatibilidade
    aFtr = sub_dfs.get('fatores_risco', pd.DataFrame())
    aEco = sub_dfs.get('ecocardiograma', pd.DataFrame())
    aCor = sub_dfs.get('coronarias', pd.DataFrame())
    aOdt = sub_dfs.get('odontologia', pd.DataFrame())
    #aPrc = sub_dfs.get('procedimento', pd.DataFrame())
    
    # 2.2) Limpeza do DataFrame principal (pré_operatoria)
    cols_drop_preop_extra = list(set(COLS_AUX + cols_drop_preop))
    cols_remover = list(set(todas_cols_mapeadas)) + cols_drop_preop_extra
    
    dVal5_limpa = dVal5.drop(columns=[c for c in cols_remover if c in dVal5.columns], errors='ignore').copy()
    
    # Metadados do formulário principal
    if 'redcap_repeat_instrument' in dVal5_limpa.columns:
        dVal5_limpa['redcap_repeat_instrument'] = dVal5_limpa['redcap_repeat_instrument'].replace('', 'pre_operatoria')
    
    dVal5_limpa['pre_operatoria_complete'] = 0
    
    # Normaliza checkboxes remanescentes
    check_cols_main = [c for c in dVal5_limpa.columns if '___' in c]
    for c in check_cols_main:
        dVal5_limpa[c] = _to_redcap_checkbox(dVal5_limpa[c])
    
    dVal5_limpa = dedup_redcap(dVal5_limpa)
    
    # 2.3) Tratamentos específicos
    if not aFtr.empty and 'altura' in aFtr.columns and 'peso' in aFtr.columns:
        # Normaliza altura (3 dígitos, cm)
        aFtr['altura'] = (
            aFtr['altura']
            .astype(str).str.replace('.', '', regex=False)
            .replace({'nan': '', 'None': ''})
            .apply(lambda x: x.ljust(3, '0')[:3] if x not in ['', 'None'] else '')
        )
        
        # Calcula IMC
        peso_val = pd.to_numeric(aFtr['peso'], errors='coerce')
        altura_m = pd.to_numeric(aFtr['altura'], errors='coerce') / 100.0
        aFtr['imc'] = np.where(altura_m > 0, peso_val / (altura_m ** 2), np.nan)
        
        def categorizar_imc(imc):
            if pd.isna(imc) or np.isinf(imc) or imc <= 0:
                return ""
            if imc < 18.5: return "Magreza grau I"
            if imc < 25.0: return "Eutrofia"
            if imc < 30.0: return "Pré-obesidade"
            if imc < 35.0: return "Obesidade moderada (grau I)"
            if imc < 40.0: return "Obesidade severa (grau II)"
            return "Obesidade muito severa (grau III)"
        
        aFtr['imc_classificacao'] = aFtr['imc'].apply(categorizar_imc)
        aFtr = aFtr.drop(columns=['imc'], errors='ignore')
        aFtr = dedup_redcap(aFtr)
    
    # # Remove campo duplicado do procedimento
    # if not aPrc.empty:
    #     aPrc = aPrc.drop(columns=['procedimento_complete'], errors='ignore')
    #     aPrc = dedup_redcap(aPrc)
    
    # 2.4) Formatação final numérica
    for df_temp in [dVal5_limpa, aFtr, aEco, aCor, aOdt]:
        for col in COLS_NUMERICAS:
            if col in df_temp.columns:
                df_temp[col] = _to_float_series(df_temp[col])
    
    # 2.5) Normalização de chaves
    for df_temp in [dVal5_limpa, aFtr, aEco, aCor, aOdt]:
        _normalize_keys(df_temp)
    
    # 2.6) Consolidado (opcional)
    dExport = pd.concat([dVal5_limpa, aFtr, aEco, aCor, aOdt], 
                       ignore_index=True, sort=False)
    dExport = dedup_redcap(dExport)
    
    return {
        'consolidado': dExport,
        'pre_operatoria': dVal5_limpa,
        'fatores_risco': aFtr,
        'ecocardiograma': aEco,
        'coronarias': aCor,
        'odontologia': aOdt,
        #'procedimento': aPrc
    }


# =====================================================
# 3) EXECUÇÃO
# =====================================================

# Executa o pipeline
resultados = processar_dados_redcap(dVal5, cols_drop_preop)

# Atribuições individuais
dVal5 = resultados['pre_operatoria']
aFtr = resultados['fatores_risco']
aEco = resultados['ecocardiograma']
aCor = resultados['coronarias']
aOdt = resultados['odontologia']
#aPrc = resultados['procedimento']
dExport = resultados['consolidado']

# Relatório final
print("✅ Pronto para exportar.")
print("Shapes:", {k: v.shape for k, v in resultados.items()})

# Sanity-check das colunas formatadas
_peek_cols(dVal5, "PréOp", COLS_NUMERICAS)
_peek_cols(aFtr, "FatoresRisco", COLS_NUMERICAS)
_peek_cols(aEco, "Eco", COLS_NUMERICAS)
_peek_cols(aCor, "Coronárias", COLS_NUMERICAS)
_peek_cols(aOdt, "Odontologia", COLS_NUMERICAS)
#_peek_cols(aPrc, "Procedimento", COLS_NUMERICAS)


✅ Pronto para exportar.
Shapes: {'consolidado': (5, 148), 'pre_operatoria': (1, 79), 'fatores_risco': (1, 54), 'ecocardiograma': (1, 14), 'coronarias': (1, 10), 'odontologia': (1, 7)}

PréOp - amostra pós-formatação:
  area_valvar: []
  area_valvar_2: ['1.8']
  area_valvar_3: []
  area_valvar_4: []
  euroscore_ii: ['25.5']

FatoresRisco - amostra pós-formatação:
  peso: ['66.0']

Eco - amostra pós-formatação:
  fac_encam: ['34']
  feve_encam: ['48']
  tapse_encam: []

Coronárias - amostra pós-formatação:

Odontologia - amostra pós-formatação:


In [48]:
print("✅ Instrumentos REDCap prontos:")
print(f"PréOp: {len(dVal5)} registros")
print(f"Fatores Risco: {len(aFtr)} registros")
print(f"Eco: {len(aEco)} registros")
print(f"Coronárias: {len(aCor)} registros")
print(f"Odontologia: {len(aOdt)} registros")
#print(f"Procedimento: {len(aPrc)} registros")

✅ Instrumentos REDCap prontos:
PréOp: 1 registros
Fatores Risco: 1 registros
Eco: 1 registros
Coronárias: 1 registros
Odontologia: 1 registros


In [ ]:
#aOdt.to_excel(f"C://Users/priscilla.sequetin/Downloads/aOdt_1.xlsx", index=False)

In [156]:
# --- Configuração comum
COLS_CALC_REDCAP = ['imc', 'idade_calc']
META_COLS_EXCLUDE = {
    'record_id',
    'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance'
}
# Também removeremos *_complete dinamicamente


def _try_norm_id_series(s: pd.Series) -> pd.Series:
    """
    Usa _norm_id_series se existir; caso contrário, normaliza de forma genérica.
    """
    try:
        return _norm_id_series(s)
    except NameError:
        return s.astype(str).str.strip().str.lower()


def _series_is_textual(s: pd.Series) -> bool:
    """
    Heurística simples para considerar uma série como textual livre
    (dtype object e não binária).
    """
    if s.dtype == object:
        # Se for object mas se comporta como binária, não é "texto livre"
        try:
            if _is_binary_like(s):
                return False
        except Exception:
            pass
        return True
    return False


def _as_stripped_obj(s: pd.Series) -> pd.Series:
    """Converte para object/string e faz strip seguro (preserva NaN)."""
    s2 = s.astype('object')
    return s2.where(s2.isna(), s2.astype(str).str.strip())


def _pick_keys(reg_df: pd.DataFrame, src_df: pd.DataFrame) -> list[str]:
    """
    Chaves de junção:
      - usa ['record_id', 'ANO'] se ambas existirem nas duas DFs
      - senão, usa apenas ['record_id'].
    """
    keys = ['record_id']
    if 'record_id' in reg_df.columns and 'record_id' in src_df.columns:
        if 'ANO' in reg_df.columns and 'ANO' in src_df.columns:
            keys = ['record_id', 'ANO']
    return keys


def _prep_src_one_row_per_key(src_df: pd.DataFrame, keys: list[str], keep_cols: list[str]) -> pd.DataFrame:
    """
    Reduz o DF de origem a uma linha por chave, mantendo a última ocorrência,
    ordenando por keys + redcap_repeat_instance (quando existir).
    """
    cols = list(dict.fromkeys(keys + keep_cols))  # dedup preservando ordem
    slim = src_df.loc[:, [c for c in cols if c in src_df.columns]].copy()

    sort_cols = [c for c in keys if c in slim.columns]
    if 'redcap_repeat_instance' in slim.columns:
        sort_cols.append('redcap_repeat_instance')

    if sort_cols:
        slim = slim.sort_values(sort_cols)

    slim = slim.drop_duplicates(subset=[c for c in keys if c in slim.columns], keep='last')
    return slim


def _classify_overlap_cols(reg_df: pd.DataFrame, src_df: pd.DataFrame, overlap_cols: list[str]):
    """
    Classifica colunas sobrepostas em: checkboxes, bin_like e outras (numéricas ou texto).
    """
    checkbox_cols = [c for c in overlap_cols if '___' in c]

    bin_like_cols = []
    for c in [c for c in overlap_cols if c not in checkbox_cols]:
        reg_c = reg_df[c] if c in reg_df.columns else pd.Series(dtype=object)
        src_c = src_df[c] if c in src_df.columns else pd.Series(dtype=object)
        try:
            if _is_binary_like(reg_c) or _is_binary_like(src_c):
                bin_like_cols.append(c)
        except Exception:
            # Se der erro na heurística, deixa para "outras"
            pass

    other_cols = [c for c in overlap_cols if c not in set(checkbox_cols + bin_like_cols)]
    return checkbox_cols, bin_like_cols, other_cols


def harmonize_reg_with_src(
    reg_df: pd.DataFrame,
    src_df: pd.DataFrame,
    name: str,
    cols_calc_redcap: list[str] = COLS_CALC_REDCAP
) -> tuple[pd.DataFrame, dict]:
    """
    Harmoniza reg_df (base) com src_df (export REDCap),
    respeitando as regras solicitadas.
    Retorna (reg_df_harmonizado, update_report).
    """

    # 0) Cópias e deduplicação de colunas
    reg = _dedup_columns(reg_df, f'{name} (reg)')
    src = _dedup_columns(src_df, f'{name} (src)')

    # 1) Normalização de chaves
    if 'record_id' in reg.columns:
        reg['record_id'] = _try_norm_id_series(reg['record_id'])
    if 'record_id' in src.columns:
        src['record_id'] = _try_norm_id_series(src['record_id'])

    # 2) Remover cols calculadas que vieram do REDCap apenas no reg
    drop_calc = [c for c in cols_calc_redcap if c in reg.columns]
    if drop_calc:
        print(f"🧮 {name}: removendo colunas calculadas do reg: {drop_calc}")
        reg = reg.drop(columns=drop_calc)

    # 3) Excluir metadados e *_complete da interseção
    complete_cols_reg = [c for c in reg.columns if c.endswith('_complete')]
    complete_cols_src = [c for c in src.columns if c.endswith('_complete')]
    meta_exclude = set(META_COLS_EXCLUDE) | set(complete_cols_reg) | set(complete_cols_src)

    # 4) Descobrir chaves e overlap
    keys = _pick_keys(reg, src)
    overlap_cols = [c for c in reg.columns.intersection(src.columns) if c not in meta_exclude and c not in keys]

    checkbox_cols, bin_like_cols, other_cols = _classify_overlap_cols(reg, src, overlap_cols)

    print(
        f"🔎 {name}: Overlap total={len(overlap_cols)} | "
        f"checkboxes={len(checkbox_cols)} | binários={len(bin_like_cols)} | outros={len(other_cols)} | keys={keys}"
    )

    # 5) Garantir src com 1 linha por chave e mesclar
    src_slim = _prep_src_one_row_per_key(src, keys, overlap_cols)

    merged = reg.merge(src_slim, on=keys, how='left', suffixes=('', '_src'))

    report = {
        'checkbox_updated': [],
        'binary_updated': [],
        'filled_from_src': [],
        'text_concatenated': []
    }

    # -------------------------
    # 6.1) Checkboxes ('___'): trabalhar como '0'/'1' com a regra do máximo + preencher nulos
    for c in checkbox_cols:
        c_src = f'{c}_src'
        if c_src not in merged.columns:
            continue

        left = _to_checkbox_str01(merged[c])
        right = _to_checkbox_str01(merged[c_src])

        # Converter '' -> NaN, depois para numérico
        left_num = pd.to_numeric(left.replace('', np.nan), errors='coerce')
        right_num = pd.to_numeric(right.replace('', np.nan), errors='coerce')

        final = left_num.copy()

        both = left_num.notna() & right_num.notna()
        final.loc[both] = np.fmax(left_num[both], right_num[both])

        left_na = left_num.isna() & right_num.notna()
        final.loc[left_na] = right_num[left_na]

        # Reconverte para '0'/'1' (string) ou '' se NaN
        final_str = final.apply(lambda x: '' if pd.isna(x) else ('1' if int(x) == 1 else '0')).astype(object)
        changed = (final_str != left.fillna('')).sum()
        if changed > 0:
            report['checkbox_updated'].append((c, int(changed)))

        merged[c] = final_str

    # -------------------------
    # 6.2) Binários (sem ___): máximo + preencher nulos
    for c in bin_like_cols:
        c_src = f'{c}_src'
        if c_src not in merged.columns:
            continue

        left = pd.to_numeric(merged[c], errors='coerce').astype('Int64')
        right = pd.to_numeric(merged[c_src], errors='coerce').astype('Int64')

        final = left.copy()

        both = left.notna() & right.notna()
        final.loc[both] = final.loc[both].where(
            final.loc[both] >= right.loc[both],
            right.loc[both]
        )

        left_na = left.isna() & right.notna()
        final.loc[left_na] = right.loc[left_na]

        changed = (final != left).sum()
        if changed > 0:
            report['binary_updated'].append((c, int(changed)))

        merged[c] = final  # mantém Int64 (com NaN)

    # -------------------------
    # 6.3) Outros campos:
    #   - Se ambos são textos (object não binário) e ambos possuem conteúdo -> concatena "reg // src" APENAS se diferentes
    #   - Caso contrário: preenche somente nulos/vazios do reg com src (mantém prioridade do reg)
    import unicodedata
    import re

    def _canon_series(s: pd.Series) -> pd.Series:
        """
        Canoniza string: strip, colapsa espaços, remove acentos e coloca em minúsculas.
        Retorna '' para nulos/vazios, facilitando comparações.
        """
        s2 = s.astype('object')
        s2 = s2.where(s2.isna(), s2.astype(str).str.strip())
        s2 = s2.fillna('')
        # colapsa espaços internos
        s2 = s2.str.replace(r'\s+', ' ', regex=True)
        # normaliza/remover acentos
        s2 = s2.apply(lambda x: ''.join(
            ch for ch in unicodedata.normalize('NFKD', x)
            if not unicodedata.combining(ch)
        ).lower())
        return s2

    # --- dentro do loop "outros" (seção 6.3):
    for c in other_cols:
        c_src = f'{c}_src'
        if c_src not in merged.columns:
            continue

        left = merged[c]
        right = merged[c_src]

        left_txt = _series_is_textual(left) or _series_is_textual(right)

        if left_txt:
            # Normaliza para texto com strip
            l = _as_stripped_obj(left)
            r = _as_stripped_obj(right)

            # Máscaras de existência de texto
            has_l = l.notna() & (l.astype(str).str.strip() != '')
            has_r = r.notna() & (r.astype(str).str.strip() != '')

            # Canoniza para comparação robusta
            lc = _canon_series(l)
            rc = _canon_series(r)

            # Ambos com texto e diferentes (após canonização) --> CONCATENAR
            both_text = has_l & has_r
            diff_text = both_text & (lc != rc)

            # Resultado base: começa com l
            final = l.copy()

            # 1) Concatena apenas quando forem diferentes
            final.loc[diff_text] = (
                l[diff_text].astype(str) + " // " + r[diff_text].astype(str)
            )

            # 2) Preenche quando l é vazio/nulo e r tem texto
            fill_mask = (~has_l) & has_r
            final.loc[fill_mask] = r[fill_mask]

            # Métricas de relatório
            concat_count = int(diff_text.sum())
            fill_count = int(fill_mask.sum())
            changed = ((final != l).fillna(False)).sum()
            if concat_count > 0:
                report['text_concatenated'].append((c, concat_count))
            if fill_count > 0:
                report['filled_from_src'].append((c, fill_count))

            merged[c] = final.astype(object)

        else:
            # (mantém a lógica anterior para numéricos/categorias)
            if pd.api.types.is_numeric_dtype(left) or pd.api.types.is_numeric_dtype(right):
                left_num = pd.to_numeric(left, errors='coerce')
                right_num = pd.to_numeric(right, errors='coerce')
                filled = left_num.fillna(right_num)
                changed = (filled != left_num).sum()
                if changed > 0:
                    report['filled_from_src'].append((c, int(changed)))
                merged[c] = filled.astype(left.dtype) if left.dtype != object else filled
            else:
                l = _as_stripped_obj(left)
                r = _as_stripped_obj(right)
                filled = l.where(l.notna() & (l.astype(str).str.strip() != ''), r)
                changed = (filled != l).sum()
                if changed > 0:
                    report['filled_from_src'].append((c, int(changed)))
                merged[c] = filled

    # 7) Remover colunas auxiliares _src e devolver somente colunas do reg (mesma estrutura)
    aux_cols = [f'{c}_src' for c in overlap_cols if f'{c}_src' in merged.columns]
    out = merged.drop(columns=aux_cols, errors='ignore')

    print(f"✅ {name}: harmonização concluída.")
    print(" • Checkboxes atualizadas:", report['checkbox_updated'])
    print(" • Binários atualizados :", report['binary_updated'])
    print(" • Campos preenchidos    :", report['filled_from_src'])
    print(" • Textos concatenados   :", report['text_concatenated'])


    return out, report

In [157]:
# 1) Pré-op (regvalv_preop  x  dVal5)
regvalv_preop_h, rep_preop = harmonize_reg_with_src(
    regvalv_preop, dVal5, name='regvalv_preop x dVal5'
)

# 2) Fatores de risco (regvalv_ftrisk  x  aFtr)
regvalv_ftrisk_h, rep_ftr = harmonize_reg_with_src(
    regvalv_ftrisk, aFtr, name='regvalv_ftrisk x aFtr'
)

# 3) Ecocardiograma (regvalv_eco  x  aEco)
regvalv_eco_h, rep_eco = harmonize_reg_with_src(
    regvalv_eco, aEco, name='regvalv_eco x aEco'
)

# 4) Coronária (regvalv_coron  x  aCor)
regvalv_coron_h, rep_cor = harmonize_reg_with_src(
    regvalv_coron, aCor, name='regvalv_coron x aCor'
)

# 5) Odonto (regvalv_odonto  x  aOdt)
regvalv_odonto_h, rep_odt = harmonize_reg_with_src(
    regvalv_odonto, aOdt, name='regvalv_odonto x aOdt'
)

# # 6) Cirurgia (regvalv_cirurgia  x  aPrc)
# regvalv_cirurgia_h, rep_prc = harmonize_reg_with_src(
#     regvalv_cirurgia, aPrc, name='regvalv_cirurgia x aPrc'
# )

🔎 regvalv_preop x dVal5: Overlap total=74 | checkboxes=44 | binários=23 | outros=7 | keys=['record_id']


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1524377433.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out.loc[mask_rest & numeric.eq(1, fill_value=False)] = '1'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1524377433.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out.loc[mask_rest & numeric.eq(0, fill_value=False)] = '0'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1524377433.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in 

TypeError: cannot safely cast non-equivalent float64 to int64

In [ ]:
#dVal5.to_excel(f"C://Users/priscilla.sequetin/Downloads/dVal5_ajuste.xlsx", index=False)

In [49]:
# =====================================================
# Importando todos os instrumentos
# =====================================================

dfs_teste = {
    "PréOp": dVal5,
    "FatoresRisco": aFtr,
    "Eco": aEco,
    "Coronarias": aCor,
    "Odontologia": aOdt,
    #"Procedimento": regvalv_cirurgia_h
}

# Dicionário para armazenar IDs bloqueados por instrumento, caso precise consultar depois
ids_bloqueados_total = {}

for nome, df in dfs_teste.items():
    if df is None or df.empty:
        print(f"⏩ Pulando {nome}: DataFrame vazio ou inexistente.")
        continue

    print(f"🚀 Importando {nome} ({len(df)} registros)")
    
    try:
        # A função é chamada uma única vez e o resultado é guardado em 'batch_error'
        batch_error = import_redcap(
            df, 
            api_url=api_url,
            api_key=api_key,
            batch_size=30,
            overwrite=False
        )
        
        # Se houver retorno de erro da API (IDs travados/locked)
        if batch_error:
            ids = extract_locked_ids_only(batch_error)
            if ids:
                print(f"⚠️ {nome}: IDs bloqueados no REDCap: {ids}")
                ids_bloqueados_total[nome] = ids
        
        print(f"✅ Processamento de {nome} finalizado!")

    except Exception as e:
        print(f"❌ Erro crítico de dtypes ou conexão em {nome}: {e}")

print("\n--- Relatório Final de Bloqueios ---")
print(ids_bloqueados_total)

🚀 Importando PréOp (1 registros)
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Processamento de PréOp finalizado!
🚀 Importando FatoresRisco (1 registros)
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Processamento de FatoresRisco finalizado!
🚀 Importando Eco (1 registros)
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Processamento de Eco finalizado!
🚀 Importando Coronarias (1 registros)
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Processamento de Coronarias finalizado!
🚀 Importando Odontologia (1 registros)
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Processamento de Odontologia finalizado!

--- Relatório Final de Bloqueios ---
{}


# EUROSCORE II

In [123]:
import re
import unicodedata

# ===== Helpers robustos =====
def select_if_exists(df, cols):
    keep = [c for c in cols if c in df.columns]
    return df[keep].copy()

def col_to_int_flag(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    else:
        return pd.Series(0, index=df.index, dtype=int)

def col_to_num(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce')
    else:
        return pd.Series(np.nan, index=df.index)

def col_text(df, col):
    if col in df.columns:
        return df[col]
    else:
        return pd.Series([""] * len(df), index=df.index)

def cockcroft_gault(creatinine_umol_l, age, weight_kg, female=False):
    """Calcula Creatinine Clearance via Cockcroft-Gault (ml/min).
       Espera creatinina em µmol/L (converte internamente para mg/dL)."""
    if pd.isna(creatinine_umol_l) or pd.isna(weight_kg) or pd.isna(age):
        return np.nan
    creatinine_mg_dl = creatinine_umol_l / 88.4
    factor = 0.85 if female else 1.0
    cc = ((140 - age) * weight_kg * factor) / (72 * creatinine_mg_dl)
    return cc

def _contar_itens_texto_outros(txt):
    """Conta nº de cirurgias em texto (separadores: , ; / | + & e conectores e/ou)."""
    if pd.isna(txt):
        return 0
    s = str(txt).strip()
    if s == "":
        return 0
    s = unicodedata.normalize("NFKD", s).encode("ASCII", "ignore").decode("ASCII").lower()
    s = re.sub(r'\s+(e/ou|e|ou)\s+', ',', s)      # conectores -> vírgula
    s = re.sub(r'\s*[;,/\\\|\+&]\s*', ',', s)     # separadores
    s = re.sub(r'[\r\n]+', ',', s)                # quebras de linha
    partes = [p.strip() for p in s.split(",") if p.strip() != ""]
    if not partes:
        return 0
    partes_unicas = list(dict.fromkeys(partes))
    return len(partes_unicas)

def calcular_euroscore_ii_multi_df(
    regvalv_cirurgia, regvalv_pcte, regvalv_ftrisk, regvalv_geronto, 
    regvalv_preop, regvalv_labs, regvalv_posop, regvalv_eco, regvalv_obito,
    return_audit: bool = True,
    include_ccs4_term: bool = True  # se True, adiciona coef 0.2226147 para CCS IV
):
    """
    Calcula EuroSCORE II consolidando múltiplos DataFrames REDCap.
    Retorna (df_final, df_auditoria) se return_audit=True; senão, df_final.
    """
    # Padroniza chaves
    dfs = [regvalv_cirurgia, regvalv_pcte, regvalv_ftrisk, regvalv_geronto, 
           regvalv_preop, regvalv_labs, regvalv_posop, regvalv_eco, regvalv_obito]
    for df in dfs:
        df['record_id'] = df['record_id'].astype(str)
        if 'redcap_repeat_instance' in df.columns:
            df['redcap_repeat_instance'] = df['redcap_repeat_instance'].astype(str).replace('nan', '0')
    
    # Coeficientes oficiais EuroSCORE II (+ ccs4 opcional)
    coeffs = {
        'intercept': -5.324537,
        'age': 0.0285181,
        'female': 0.2196434,
        'iddm': 0.3542749,
        'eca': 0.5360268,
        'cpd': 0.1886564,
        'mob': 0.2407181,
        'redo': 1.118599,
        'dialysis': 0.6421508,
        'cc_le50': 0.8592256,
        'cc_50_85': 0.303553,
        'ae': 0.6194522,
        'critical': 1.086517,
        # lv_ef idx: 0=normal, 1=<=50, 2=<=30, 3=<=20
        'lv_ef': [0, 0.3150652, 0.8084096, 0.9346919],
        'recent_mi': 0.1528943,
        # pa_pressure idx: 0=<31, 1=31-55, 2=>=55
        'pa_pressure': [0, 0.1788899, 0.3491475],
        # urgency idx: 0=Eletiva, 1=Urgência, 2=Emergência, 3=Salvamento
        'urgency': [0, 0.3174673, 0.7039121, 1.362947],
        # n_procedures idx: 0=1 proc, 1=2 proc, 2=3 proc, 3=>=4 proc
        'n_procedures': [0, 0.0062118, 0.5521478, 0.9724533],
        'thoracic_aorta': 0.6527205,
        'ccs4': 0.2226147
    }
    
    # Base
    df_base = regvalv_cirurgia[['record_id', 'redcap_repeat_instance', 'idade_calc']].copy()
    df_base = df_base.rename(columns={'idade_calc': 'age'})
    df_base['age'] = pd.to_numeric(df_base['age'], errors='coerce').fillna(60)
    
    # FEMALE
    pcte_merge = select_if_exists(regvalv_pcte, ['record_id', 'genero']).drop_duplicates()
    df_base = df_base.merge(pcte_merge, on='record_id', how='left')
    df_base['female'] = (pd.to_numeric(df_base.get('genero'), errors='coerce') == 2).astype(int)
    
    # NYHA (classe -> coeficiente direto, incluindo classe II)
    ftrisk_nyha = select_if_exists(regvalv_ftrisk, ['record_id', 'redcap_repeat_instance', 'classe_nyha'])
    df_base = df_base.merge(ftrisk_nyha, on=['record_id', 'redcap_repeat_instance'], how='left')
    nyha_class = pd.to_numeric(df_base.get('classe_nyha'), errors='coerce')
    df_base['nyha_coef'] = np.select(
        [nyha_class == 2, nyha_class == 3, nyha_class == 4],
        [0.1070545,      0.2958358,      0.5597929],
        default=0.0
    )
    
    # MOBILIDADE + SEQUELA NEUROLÓGICA
    ftrisk_mob = select_if_exists(regvalv_ftrisk, ['record_id', 'redcap_repeat_instance', 'mobilidade_severa_preju', 'sequela_neurologia'])
    geronto_mob = select_if_exists(regvalv_geronto, ['record_id', 'redcap_repeat_instance', 'eq5d_mobilidade', 'barthel_mobilidade'])
    df_base = df_base.merge(ftrisk_mob, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base = df_base.merge(geronto_mob, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base['mob'] = (
        (col_to_int_flag(df_base, 'mobilidade_severa_preju') == 1) |
        (pd.to_numeric(df_base.get('eq5d_mobilidade'), errors='coerce') == 3) |
        (pd.to_numeric(df_base.get('barthel_mobilidade'), errors='coerce') == 0) |
        (col_to_int_flag(df_base, 'sequela_neurologia') == 1)
    ).astype(int)
    
    # REDO
    ftrisk_redo = select_if_exists(
        regvalv_ftrisk,
        ['record_id', 'redcap_repeat_instance', 'tipo_intervencao_previa___2', 'tipo_intervencao_previa___3', 'tipo_intervencao_previa___6']
    )
    preop_redo = select_if_exists(regvalv_preop, ['record_id', 'redcap_repeat_instance', 'cirurgia_anterior'])
    df_base = df_base.merge(ftrisk_redo, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base = df_base.merge(preop_redo, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base['redo'] = (
        (pd.to_numeric(df_base.get('cirurgia_anterior'), errors='coerce') == 1) |
        (pd.to_numeric(df_base.get('tipo_intervencao_previa___2'), errors='coerce') == 1) |
        (pd.to_numeric(df_base.get('tipo_intervencao_previa___3'), errors='coerce') == 1) |
        (pd.to_numeric(df_base.get('tipo_intervencao_previa___6'), errors='coerce') == 1)
    ).astype(int)
    
    # --- FUNÇÃO RENAL + DIÁLISE (creatinina mg/dL; diálise via estagio_drc) ---
    labs_cc = select_if_exists(regvalv_labs, ['record_id', 'redcap_repeat_instance', 'creatinina_pre'])
    ftrisk_cc = select_if_exists(regvalv_ftrisk, ['record_id', 'redcap_repeat_instance', 'peso', 'drc_clcr', 'estagio_drc'])
    df_base = df_base.merge(labs_cc, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base = df_base.merge(ftrisk_cc, on=['record_id', 'redcap_repeat_instance'], how='left')

    # Peso
    df_base['weight_kg'] = col_to_num(df_base, 'peso')

    # Creatinina (mg/dL) - trata vírgula decimal e converte para µmol/L p/ Cockcroft
    creat_raw = df_base.get('creatinina_pre')
    if creat_raw is not None:
        creat_mgdl = pd.to_numeric(
            creat_raw.astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )
    else:
        creat_mgdl = pd.Series(np.nan, index=df_base.index)
    df_base['creatinina_pre_mgdl'] = creat_mgdl
    df_base['creatinine_umol_l'] = creat_mgdl * 88.4

    # Diálise por estagio_drc (radio: 6 = Estágio 5)
    df_base['dialysis'] = (col_to_int_flag(df_base, 'estagio_drc') == 6).astype(int)

    # Cockcroft-Gault
    df_base['cc_ml_min'] = [
        cockcroft_gault(row['creatinine_umol_l'], row['age'], row['weight_kg'], row['female'])
        for _, row in df_base.iterrows()
    ]

    # Fallback de faixas via drc_clcr quando cc_ml_min faltar (sem confundir com diálise)
    drc = col_to_int_flag(df_base, 'drc_clcr')
    cc_proxy = df_base['cc_ml_min'].copy()
    cc_proxy.loc[cc_proxy.isna() & (drc == 4)] = 50   # força <=50
    cc_proxy.loc[cc_proxy.isna() & (drc == 3)] = 85   # força 51–85
    df_base['cc_ml_min_proxy'] = cc_proxy
    
    # FATORES SIMPLES (mantém IDDM via diabetes_controle == 3)
    ftrisk_simple = select_if_exists(
        regvalv_ftrisk,
        ['record_id', 'redcap_repeat_instance', 'diabetes_controle', 'diabetes', 'arteriopatia_extracardiaca',
         'dpoc', 'endocardite_ativa', 'estado_pr_operat_rio_cr_ti', 'iam_menor_90dias', 'valor_psap_encam']
    )
    df_base = df_base.merge(ftrisk_simple, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base['iddm'] = (col_to_int_flag(df_base, 'diabetes_controle') == 3).astype(int)
    df_base['eca'] = (col_to_int_flag(df_base, 'arteriopatia_extracardiaca') == 1).astype(int)
    df_base['cpd'] = (col_to_int_flag(df_base, 'dpoc') == 1).astype(int)
    df_base['ae'] = (col_to_int_flag(df_base, 'endocardite_ativa') == 1).astype(int)
    df_base['critical'] = (col_to_int_flag(df_base, 'estado_pr_operat_rio_cr_ti') == 1).astype(int)
    df_base['recent_mi'] = (col_to_int_flag(df_base, 'iam_menor_90dias') == 1).astype(int)

    # PSAP categórico (1=<31, 2=31-55, 3=>=55) -> índices 0/1/2
    psap_cat = pd.to_numeric(df_base.get('valor_psap_encam'), errors='coerce').fillna(1).astype(int)
    df_base['pa_pressure'] = (psap_cat - 1).clip(0, 2)
    
    # LV_EF (limites inclusivos + vírgula decimal)
    eco_ef = select_if_exists(regvalv_eco, ['record_id', 'redcap_repeat_instance', 'feve_encam'])
    df_base = df_base.merge(eco_ef, on=['record_id', 'redcap_repeat_instance'], how='left')
    feve_raw = df_base.get('feve_encam')
    if feve_raw is not None:
        feve_num = pd.to_numeric(
            feve_raw.astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )
    else:
        feve_num = pd.Series(np.nan, index=df_base.index)

    def map_lv_ef_idx(x):
        if pd.isna(x):
            return 0
        x = float(x)
        if x <= 20:
            return 3
        if x <= 30:
            return 2
        if x <= 50:
            return 1
        return 0

    df_base['lv_ef'] = feve_num.apply(map_lv_ef_idx).astype(int)
    
    # URGENCY (pré-op)
    preop_urg = select_if_exists(
        regvalv_preop,
        ['record_id', 'redcap_repeat_instance', 'tipo_operacao',
         'diagnostico_principal___8', 'diagnostico_principal___6', 'tipo_de_cirurgia_anterior___2']
    )
    df_base = df_base.merge(preop_urg, on=['record_id', 'redcap_repeat_instance'], how='left')

    # tipo_operacao: 1=Eletiva, 2=Urgência, 3=Emergência, 4=Salvamento -> índices 0..3
    urg_raw = pd.to_numeric(df_base.get('tipo_operacao'), errors='coerce')
    df_base['urgency'] = (urg_raw.fillna(1).astype(int) - 1).clip(0, 3)

    # CCS (termo separado e opcional)
    ccs_merge = select_if_exists(regvalv_ftrisk, ['record_id', 'redcap_repeat_instance', 'ccs'])
    df_base = df_base.merge(ccs_merge, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base['ccs'] = pd.to_numeric(df_base.get('ccs'), errors='coerce')
    df_base['ccs4'] = (df_base['ccs'] == 4).astype(int)

    # N_PROCEDURES
    preop_proc = select_if_exists(
        regvalv_preop,
        ['record_id', 'redcap_repeat_instance',
         'outro_tipo_cirurgia', 'tp_cir_prev',
         'tipo_de_cirurgia_anterior___1',
         'tipo_de_cirurgia_anterior___2',
         'tipo_de_cirurgia_anterior___3',
         'tipo_de_cirurgia_anterior___4']
    )
    df_base = df_base.merge(preop_proc, on=['record_id', 'redcap_repeat_instance'], how='left')

    rev_flag = (
        (col_to_int_flag(df_base, 'tipo_de_cirurgia_anterior___1') == 1) |
        (col_to_int_flag(df_base, 'tipo_intervencao_previa___2') == 1)
    ).astype(int)
    aorta_flag = (
        (col_to_int_flag(df_base, 'tipo_de_cirurgia_anterior___2') == 1) |
        (col_to_int_flag(df_base, 'tipo_intervencao_previa___6') == 1)
    ).astype(int)
    valvar_flag = (
        (col_to_int_flag(df_base, 'tipo_de_cirurgia_anterior___3') == 1) |
        (col_to_int_flag(df_base, 'tipo_intervencao_previa___3') == 1)
    ).astype(int)
    outra_checkbox = (col_to_int_flag(df_base, 'tipo_de_cirurgia_anterior___4') == 1).astype(int)

    # 1. Textos disponíveis
    txt_outros = col_text(df_base, 'outro_tipo_cirurgia')
    txt_prev = col_text(df_base, 'tp_cir_prev')
    
    # 2. Contagem apenas do campo "Outros" (usado se checkboxes estiverem preenchidos)
    n_outros_text_only = txt_outros.fillna('').apply(_contar_itens_texto_outros).astype(int)
    n_outros = np.where(n_outros_text_only > 0, n_outros_text_only, outra_checkbox)

    # 3. Contagem total de texto (usado como Fallback caso TUDO esteja vazio)
    txt_comb_total = (txt_outros.fillna('') + ',' + txt_prev.fillna('')).str.strip(',')
    n_text_total = txt_comb_total.apply(_contar_itens_texto_outros).astype(int)

    # Verifica se preencheram ALGUM checkbox
    soma_checkboxes = rev_flag + aorta_flag + valvar_flag + outra_checkbox

    # REGRA: Se nenhum checkbox foi marcado, conta pelo texto total. 
    # Se algum foi marcado, usa a contagem estruturada (flags + campo outros específico).
    n_total_raw = np.where(
        soma_checkboxes == 0,
        n_text_total,
        (rev_flag + aorta_flag + valvar_flag + n_outros)
    ).astype(int)

    # Mapeamento para índice dos coeficientes
    n_total_display = np.maximum(n_total_raw, 1)  # baseline para índice
    df_base['n_procedures_calc'] = n_total_display

    def _map_nproc_to_idx(n):
        if n <= 1:
            return 0
        elif n == 2:
            return 1
        elif n == 3:
            return 2
        else:
            return 3

    df_base['n_procedures_idx'] = df_base['n_procedures_calc'].apply(_map_nproc_to_idx).astype(int)

    # TÓRAX/AORTA & ENDOCARDITE
    obito_ae = select_if_exists(regvalv_obito, ['record_id', 'redcap_repeat_instance', 'endocardite'])
    df_base = df_base.merge(obito_ae, on=['record_id', 'redcap_repeat_instance'], how='left')
    df_base['ae'] = (
        (col_to_int_flag(df_base, 'endocardite_ativa') == 1) |
        (col_to_int_flag(df_base, 'diagnostico_principal___8') == 1) |
        (col_to_int_flag(df_base, 'endocardite') == 1)
    ).astype(int)
    df_base['thoracic_aorta'] = (
        (col_to_int_flag(df_base, 'tipo_intervencao_previa___6') == 1) |
        (col_to_int_flag(df_base, 'diagnostico_principal___6') == 1) |
        (col_to_int_flag(df_base, 'tipo_de_cirurgia_anterior___2') == 1)
    ).astype(int)
    
    # ===== CÁLCULO FINAL EUROSCORE II =====
    logit = coeffs['intercept'] * np.ones(len(df_base))
    age_term = np.maximum(df_base['age'] - 59, 1)
    df_base['age_term'] = age_term

    logit += coeffs['age'] * age_term
    logit += coeffs['female'] * df_base['female']

    # NYHA (coef direto corrigido)
    logit += df_base['nyha_coef']

    # Variáveis binárias
    for var in ['iddm', 'eca', 'cpd', 'mob', 'redo', 'dialysis', 'ae', 'critical', 'recent_mi', 'thoracic_aorta']:
        logit += coeffs[var] * df_base[var]

    # CCS IV (separado, opcional no logit)
    if include_ccs4_term:
        logit += coeffs['ccs4'] * df_base['ccs4']

    # Função renal por categorias (usa proxy com fallback)
    logit += np.where(df_base['cc_ml_min_proxy'] <= 50, coeffs['cc_le50'], 0)
    logit += np.where((df_base['cc_ml_min_proxy'] > 50) & (df_base['cc_ml_min_proxy'] <= 85), coeffs['cc_50_85'], 0)

    # Categóricas por índice
    logit += np.array([coeffs['lv_ef'][i] for i in df_base['lv_ef']])
    logit += np.array([coeffs['pa_pressure'][i] for i in df_base['pa_pressure']])
    logit += np.array([coeffs['urgency'][i] for i in df_base['urgency']])
    logit += np.array([coeffs['n_procedures'][i] for i in df_base['n_procedures_idx']])

    df_base['euroscore_ii_python'] = 100 * (1 / (1 + np.exp(-logit)))
    df_base['euroscore_ii_python'] = df_base['euroscore_ii_python'].round(2)
    
    # ===== Saídas =====
    df_final = df_base[['record_id', 'redcap_repeat_instance', 'euroscore_ii_python']].copy()
    df_final['redcap_event_name'] = 'fase2preprocedimen_arm_1'
    df_final['redcap_repeat_instrument'] = 'pre_operatoria'
    
    if not return_audit:
        return df_final

    cols_audit = [
        'record_id', 'redcap_repeat_instance',
        # Demografia
        'age', 'female', 'age_term',
        # NYHA
        'classe_nyha', 'nyha_coef',
        # Mobilidade + neuro
        'mobilidade_severa_preju', 'eq5d_mobilidade', 'barthel_mobilidade', 'sequela_neurologia', 'mob',
        # Reoperação
        'cirurgia_anterior', 'tipo_intervencao_previa___2', 'tipo_intervencao_previa___3', 'tipo_intervencao_previa___6', 'redo',
        # Renal
        'peso', 'creatinina_pre', 'creatinina_pre_mgdl', 'cc_ml_min', 'cc_ml_min_proxy', 'drc_clcr', 'estagio_drc', 'dialysis',
        # Simples
        'diabetes', 'diabetes_controle', 'iddm', 'arteriopatia_extracardiaca', 'eca', 'dpoc', 'cpd',
        'estado_pr_operat_rio_cr_ti', 'critical', 'iam_menor_90dias', 'recent_mi',
        # Endocardite / TAA
        'endocardite_ativa', 'diagnostico_principal___8', 'endocardite', 'ae',
        'tipo_intervencao_previa___6', 'diagnostico_principal___6', 'tipo_de_cirurgia_anterior___2', 'thoracic_aorta',
        # PSAP
        'valor_psap_encam', 'pa_pressure',
        # FE
        'feve_encam', 'lv_ef',
        # CCS
        'ccs', 'ccs4',
        # n_procedures
        'tipo_de_cirurgia_anterior___1', 'tipo_de_cirurgia_anterior___2', 'tipo_de_cirurgia_anterior___3',
        'tipo_de_cirurgia_anterior___4', 'outro_tipo_cirurgia', 'tp_cir_prev', 'n_procedures_calc', 'n_procedures_idx',
        # Resultado
        'euroscore_ii_python'
    ]
    cols_audit = [c for c in cols_audit if c in df_base.columns]
    df_auditoria = df_base[cols_audit].copy()

    return df_final, df_auditoria

In [124]:
df_euroscore, df_auditoria = calcular_euroscore_ii_multi_df(
    regvalv_cirurgia, regvalv_pcte, regvalv_ftrisk, regvalv_geronto, 
    regvalv_preop, regvalv_labs, regvalv_posop, regvalv_eco, regvalv_obito,
    return_audit=True
)

df_euroscore['euroscore_ii_python'] = df_euroscore['euroscore_ii_python'].replace(',', '.', regex=False)

#print(df_euroscore.head())
print(f"Registros únicos: {df_euroscore['record_id'].nunique()}")
print(f"EuroSCORE médio: {df_euroscore['euroscore_ii_python'].mean():.1f}%")



Registros únicos: 537
EuroSCORE médio: 2.0%


In [125]:
tsts = df_euroscore[df_euroscore['record_id']=='310758'].copy()
tsts

,record_id,redcap_repeat_instance,euroscore_ii_python,redcap_event_name,redcap_repeat_instrument
0,310758,1,14.6,fase2preprocedimen_arm_1,pre_operatoria
1,310758,1,14.6,fase2preprocedimen_arm_1,pre_operatoria


In [117]:
tsts2 = df_auditoria[df_auditoria['record_id']=='310758'].copy()
tsts2

,record_id,redcap_repeat_instance,age,female,age_term,classe_nyha,nyha_coef,mobilidade_severa_preju,eq5d_mobilidade,barthel_mobilidade,mob,cirurgia_anterior,tipo_intervencao_previa___2,tipo_intervencao_previa___3,tipo_intervencao_previa___6,redo,peso,creatinina_pre,creatinina_pre_mgdl,cc_ml_min,cc_ml_min_proxy,drc_clcr,estagio_drc,dialysis,diabetes,diabetes_controle,iddm,arteriopatia_extracardiaca,eca,dpoc,cpd,estado_pr_operat_rio_cr_ti,critical,iam_menor_90dias,recent_mi,endocardite_ativa,diagnostico_principal___8,endocardite,ae,tipo_intervencao_previa___6,diagnostico_principal___6,thoracic_aorta,valor_psap_encam,pa_pressure,feve_encam,lv_ef,ccs,ccs4,tipo_de_cirurgia_anterior___1,tipo_de_cirurgia_anterior___3,tipo_de_cirurgia_anterior___4,outro_tipo_cirurgia,tp_cir_prev,n_procedures_calc,n_procedures_idx,euroscore_ii_python
0,310758,1,74,0,15,1,0.0,0,1,15,0,1,0,0,1,1,58.0,1.20,1.2,44.305556,44.305556,2,,0,0,,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,2,1,58,0,0.0,0,0,0,1,,CORREÇAO COARCTAÇAO DE AORTA 1972 \r\n\r\nAORT...,2,1,11.09
1,310758,1,74,0,15,1,0.0,0,1,15,0,1,0,0,1,1,58.0,1.20,1.2,44.305556,44.305556,2,,0,0,,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,2,1,58,0,0.0,0,0,0,1,,CORREÇAO COARCTAÇAO DE AORTA 1972 \r\n\r\nAORT...,2,1,11.09


In [ ]:
#df_euroscore.to_excel(f"C://Users/priscilla.sequetin/Downloads/euroscore2.xlsx", index=False)

In [126]:
euroscore_results = import_redcap(
            df_euroscore, 
            api_url=api_url,
            api_key=api_key,
            batch_size=30,
            overwrite=False
        )

Batch 1/27 imported successfully. Records in this batch: 22. Total records imported: 22
Batch 2/27 imported successfully. Records in this batch: 24. Total records imported: 46
Batch 3/27 imported successfully. Records in this batch: 21. Total records imported: 67
Batch 4/27 imported successfully. Records in this batch: 21. Total records imported: 88
Batch 5/27 imported successfully. Records in this batch: 20. Total records imported: 108
Batch 6/27 imported successfully. Records in this batch: 19. Total records imported: 127
Batch 7/27 imported successfully. Records in this batch: 22. Total records imported: 149
Batch 8/27 imported successfully. Records in this batch: 22. Total records imported: 171
Batch 9/27 imported successfully. Records in this batch: 23. Total records imported: 194
Batch 10/27 imported successfully. Records in this batch: 21. Total records imported: 215
Batch 11/27 imported successfully. Records in this batch: 18. Total records imported: 233
Batch 12/27 imported su

# DEMOGRAFICOS

In [159]:
# Buscando rbText PCTE

sql_PCTE = f"""
SET DATEFORMAT ymd;

WITH dip AS (
    SELECT 
        NK_CD_PACIENTE,
        NM_PACIENTE,
        TP_SITUACAO,
        SEXO,
        COR,
        DT_NASCIMENTO,
        NR_DDD_FONE,
        NR_FONE_1,
        NR_DDD_CELULAR,
        NR_CELULAR,
        NR_DDD_FONE_COMERCIAL,
        NR_FONE_COMERCIAL,
        -- Usar ROW_NUMBER é mais performático que Subquery para grandes bases
        ROW_NUMBER() OVER (PARTITION BY NK_CD_PACIENTE ORDER BY DT_ATUAL DESC) as rn
    FROM dim_paciente
)

SELECT 
    NK_CD_PACIENTE as record_id, -- Renomeado para facilitar o merge no REDCap
    NM_PACIENTE as nome_paciente,
    TP_SITUACAO as tp_situacao,
    SEXO as genero,
    COR as etnia,
    DT_NASCIMENTO as data_nascimento,
    NR_DDD_FONE as nr_ddd_fone,
    NR_FONE_1 as nr_fone_1,
    NR_DDD_CELULAR as nr_ddd_celular,
    NR_CELULAR as nr_celular,
    NR_DDD_FONE_COMERCIAL as nr_ddd_fone_comercial,
    NR_FONE_COMERCIAL as nr_fone_comercial
FROM dip 
WHERE rn = 1 -- Garante apenas a última atualização de cada paciente

""" 

# Executa a função para buscar os dados
dPCTE = query_sql_to_dataframe(conn_string, sql_PCTE)

dPCTE['record_id'] = dPCTE['record_id'].astype('Int64')

Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [177]:
dPct = dPCTE[dPCTE['record_id'].astype('Int64').isin(dVal['CD_PACIENTE'].astype('Int64'))]

In [178]:
# 1. Definição do Evento
dPct['redcap_event_name'] = 'fase1preoperatorio_arm_1'
dPct['dados_demograficos_complete'] = 0  # Incompleto

# 2. Mapeamento de Gênero e Etnia
mapa = {'MASCULINO': 1, 'FEMININO': 2, 'AMARELA': 4, 'BRANCA': 1, 'NEGRA': 2, 'PARDA': 3}

# Dica: Verifique se 'genero' e 'etnia' não estão nulos antes do map, 
# senão fillna(3) vai preencher tudo que não casou com 3.
dPct['genero'] = dPct['genero'].map(mapa).fillna('').astype(int)
dPct['etnia'] = dPct['etnia'].map(mapa).fillna('').astype(int)

# 3. Conversão Segura dos Telefones
cols_telefones = [
    'nr_ddd_fone', 
    'nr_fone_1', 
    'nr_ddd_celular', 
    'nr_celular', 
    'nr_ddd_fone_comercial', 
    'nr_fone_comercial'
]

for col in cols_telefones:
    if col in dPct.columns:
        # A. (Opcional) Remove caracteres comuns de telefone se existirem, como (-) ou ( )
        # Se os dados já vierem limpos (apenas dígitos), pode remover essa linha 'A'
        dPct[col] = dPct[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        
        # B. Converte para Numérico
        # 'coerce' transforma vazios, erros e textos em NaN
        dPct[col] = pd.to_numeric(dPct[col], errors='coerce')
        
        # C. Arredonda e Converte para Int64
        # .round() garante que 9999.0 vire 9999 antes de virar inteiro
        # 'Int64' (com maiúscula) aceita NaN (o 'int' normal quebraria com NaN)
        dPct[col] = dPct[col].round().astype('Int64')

# Cria a data e já formata como texto (dia/mes/ano)
dPct['data_inclusao'] = pd.to_datetime('today').strftime('%d/%m/%Y')


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\4014360764.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dPct['redcap_event_name'] = 'fase1preoperatorio_arm_1'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\4014360764.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dPct['dados_demograficos_complete'] = 0  # Incompleto
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\4014360764.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [179]:
dPct['genero'].value_counts()

genero
2    116
1     93
Name: count, dtype: int64

In [180]:
# VERIFICAR ÓBITO
# 1. Pacientes que constam como ÓBITO no sistema (dPct)
check_obito_dim = dPct[dPct['tp_situacao'] == 'ÓBITO']
ids_check = set(check_obito_dim['record_id'])

# 2. Pacientes que constam como VIVOS (ou não obito) no REDCap
check_obito_reg = regvalv_obito[regvalv_obito['mortalidade_hospitalar'] == '0']
ids_regvalv = set(check_obito_reg['record_id'])

# 3. CORREÇÃO: Queremos quem morreu no sistema E está vivo no REDCap
obito_a_verificar = ids_check.intersection(ids_regvalv)

print(f"IDs que constam como ÓBITO no sistema, mas VIVOS no REDCap: {obito_a_verificar}")
print(f"Total a verificar: {len(obito_a_verificar)}")

# Filtra apenas os registros marcados como óbito no REDCap
obitos_atuais_redcap = regvalv_obito[regvalv_obito['mortalidade_hospitalar'] == '1']

# Calcula a quantidade
qtd_obitos_redcap = len(obitos_atuais_redcap)

# Pega a lista com quem são esses pacientes (os IDs)
lista_ids_redcap = obitos_atuais_redcap['record_id'].unique().tolist()

print(f"--- ÓBITOS ATUAIS NO REDCAP ---")
print(f"Quantidade total: {qtd_obitos_redcap}")
print(f"Quem são (IDs): {lista_ids_redcap}\n")

IDs que constam como ÓBITO no sistema, mas VIVOS no REDCap: set()
Total a verificar: 0
--- ÓBITOS ATUAIS NO REDCAP ---
Quantidade total: 41
Quem são (IDs): ['363345', '434034', '531219', '604177', '607952', '689671', '782398', '791370', '823441', '863612', '938088', '995749', '1017483', '1022380', '1034106', '1051655', '1082666', '1089242', '1106191', '1120863', '1124671', '1127821', '1131213', '1133255', '1133627', '1135603', '1135780', '1138903', '1143224', '1143859', '1145024', '1145472', '1146325', '1147779', '1148518', '1149738', '1151686', '1152714', '1156364', '1157706', '1161139']



In [181]:
# =====================================================
# --- 1. PREPARAÇÃO DOS DADOS DO REDCAP (EXISTENTES) ---

# Garantir tipagem string
regvalv_pcte['record_id'] = regvalv_pcte['record_id'].astype(str)

# Criamos um Set apenas com os IDs, já que não há instâncias aqui
pcte_existentes = set(regvalv_pcte['record_id'].unique())

print(f"Total de IDs de pacientes já existentes no Redcap: {len(pcte_existentes)}")

def filtrar_pacientes_novos(df_novo, ids_proibidos):
    """
    Versão adaptada para instrumentos não-repetitivos (apenas record_id).
    """
    if df_novo.empty:
        return df_novo
    
    df_novo = df_novo.copy()
    df_novo['record_id'] = df_novo['record_id'].astype(str)
    
    # Filtra mantendo apenas quem NÃO está no set de existentes
    df_filtrado = df_novo[~df_novo['record_id'].isin(ids_proibidos)].copy()
    
    ignorado_count = len(df_novo) - len(df_filtrado)
    
    print(f"📊 Filtro de Pacientes Aplicado:")
    print(f"   - Registros originais: {len(df_novo)}")
    print(f"   - IDs já cadastrados (ignorados): {ignorado_count}")
    print(f"   - Novos pacientes para importar: {len(df_filtrado)}")
    print(f"   - Registros: {df_filtrado['record_id'].unique()}")
    
    return df_filtrado

# --- 2. FILTRANDO OS DADOS LOCAIS ---
aPct1 = filtrar_pacientes_novos(dPct, pcte_existentes)

aPct1 = aPct1.drop(columns='tp_situacao', errors='ignore')

Total de IDs de pacientes já existentes no Redcap: 545
📊 Filtro de Pacientes Aplicado:
   - Registros originais: 209
   - IDs já cadastrados (ignorados): 206
   - Novos pacientes para importar: 3
   - Registros: ['1130237' '1128722' '1154426']


In [ ]:
#aPct1.to_excel(f"C://Users/priscilla.sequetin/Downloads/demo1.xlsx", index=False)

In [182]:
if not aPct1.empty:
    # 1. Criar o DataFrame de sinalização (Gatilho)
    espera_cir = pd.DataFrame({
        'record_id': aPct1['record_id'].unique(),
        'redcap_event_name': 'fase3procedcirurgi_arm_1',
        'redcap_repeat_instance': 1,
        'redcap_repeat_instrument': 'procedimento',
        'procedimento_complete': '1'
    })

    # 2. Concatenar
    aPct2 = pd.concat([aPct1, espera_cir], ignore_index=True)

    # --- TRATAMENTO DEFinitivo PARA .0 EM COLUNAS DE CATEGORIA ---
    
    # Convertemos tudo para object para aceitar strings vazias sem erro de dtype
    aPct2 = aPct2.astype(object)

    # Lista ampliada de colunas que não podem ter decimais
    colunas_para_limpar = [
        'record_id', 'genero', 'etnia', 
        'dados_demograficos_complete', 'procedimento_complete',
        'redcap_repeat_instance'
    ]

    for col in colunas_para_limpar:
        if col in aPct2.columns:
            aPct2[col] = (
                aPct2[col]
                .astype(str)
                .str.replace(r'\.0$', '', regex=True)
                .replace(['nan', 'None', '<NA>', 'NaN'], '')
            )

    # Garante que qualquer outro nulo no DF vire string vazia
    aPct2 = aPct2.fillna("")

    print(f"🚀 Importando com colunas limpas: {aPct2.columns.tolist()}")
    
    try:
        # 3. Importação
        batch_error = import_redcap(
            aPct2,
            api_url=api_url,
            api_key=api_key,
            batch_size=10,
            overwrite=False
        )
        
        if not batch_error:
            print("✅ Importação bem-sucedida! Verifique o REDCap.")
        else:
            print(f"⚠️ Erro reportado pela API: {batch_error}")

    except Exception as e:
        print(f"❌ Erro crítico: {e}")

print(f"Pacientes importados: ", aPct2['record_id'].unique())

🚀 Importando com colunas limpas: ['record_id', 'nome_paciente', 'genero', 'etnia', 'data_nascimento', 'nr_ddd_fone', 'nr_fone_1', 'nr_ddd_celular', 'nr_celular', 'nr_ddd_fone_comercial', 'nr_fone_comercial', 'redcap_event_name', 'dados_demograficos_complete', 'data_inclusao', 'redcap_repeat_instance', 'redcap_repeat_instrument', 'procedimento_complete']
Batch 1/1 imported successfully. Records in this batch: 3. Total records imported: 3
✅ Importação bem-sucedida! Verifique o REDCap.
Pacientes importados:  ['1130237' '1128722' '1154426']


# OBITOS/ALTAS

In [197]:
# Query atualizada conforme o novo select do Power Query
sql_query = """
SELECT 
    fde.CD_ATENDIMENTO,
    fde.CD_PACIENTE,
    fde.DH_DOCUMENTO,
    diu.NM_SETOR,
    MAX(CASE WHEN fde.DS_IDENTIFICADOR = 'DS_TEMPO_PERMANENCIA_1' THEN fde.DS_RESPOSTA END) AS DS_TEMPO_PERMANENCIA_1,
    MAX(CASE WHEN fde.DS_IDENTIFICADOR = 'DT_ADMISSAO_1' THEN fde.DS_RESPOSTA END) AS DT_ADMISSAO_1,
    MAX(CASE WHEN fde.DS_IDENTIFICADOR = 'DT_ALTA_1' THEN fde.DS_RESPOSTA END) AS DT_ALTA_1,
    MAX(CASE WHEN fde.DS_IDENTIFICADOR = 'DS_CAUSA_DA_MORTE_1' THEN fde.DS_RESPOSTA END) AS DS_CAUSA_DA_MORTE_1
FROM [DATADANTE].[dbo].[ft_doc_eletronico_texto] fde
JOIN dim_unidade_internacao diu ON diu.NK_CD_UNID_INT = fde.CD_UNIDADE_INTERNACAO
WHERE fde.CD_DOCUMENTO IN (1043, 1037)  -- 1043: RESUMO DE OBITO | 1037: RESUMO DE ALTA
GROUP BY fde.CD_ATENDIMENTO, fde.CD_PACIENTE, fde.DH_DOCUMENTO, diu.NM_SETOR;
"""

# Executa a função para buscar os dados
dfDESFECHO = query_sql_to_dataframe(conn_string, sql_query)


Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [198]:
# Query atualizada conforme o novo select do Power Query
sql_query = """
WITH Internacoes AS (
    SELECT 
        fiu.CD_PACIENTE,
        fiu.CD_ATENDIMENTO,
        MIN(fiu.HR_ATENDIMENTO) AS DT_INTERNADO,
        MAX(COALESCE(fiu.HR_ALTA, fat.HR_ALTA_MEDICA)) AS DT_FIM,
        dma.DS_MOT_ALT,
        SUM(CASE WHEN fiu.UNIDADE_PASSAGEM like '%UTI%' THEN 1 ELSE 0 END) AS CNT_UTI,
        MIN(CASE WHEN fiu.UNIDADE_PASSAGEM like '%UTI%' THEN fiu.HR_MOV_INT END) AS DT_ENT_UTI,
        MAX(CASE WHEN fiu.UNIDADE_PASSAGEM like '%UTI%' THEN fiu.HR_SAIDA_MOV_INT END) AS DT_SAIDA_UTI
    FROM ft_internacao_unidade fiu
    LEFT JOIN ft_atendimento fat ON fat.NK_CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
    LEFT JOIN dim_motivo_alta dma ON dma.NK_CD_MOT_ALT = fat.CD_MOT_ALT
    GROUP BY fiu.CD_PACIENTE, fiu.CD_ATENDIMENTO, dma.DS_MOT_ALT
)
SELECT 
    CD_PACIENTE,
    CD_ATENDIMENTO,
    DT_INTERNADO,
    CASE WHEN DS_MOT_ALT LIKE '%ALTA%' THEN DT_FIM END AS DT_ALTA,
    CASE WHEN DS_MOT_ALT LIKE '%ÓBITO%' OR DS_MOT_ALT LIKE '%OBITO%' THEN DT_FIM END AS DT_OBITO,
    DT_ENT_UTI,
    DT_SAIDA_UTI,
    IIF(CNT_UTI > 1, 1, 0) AS READMISSAO_UTI,  -- 1=sim, 0=não
    DATEDIFF(HOUR, DT_ENT_UTI, DT_SAIDA_UTI) AS TPO_HORAS_UTI,
    DATEDIFF(DAY, DT_INTERNADO, DT_FIM) AS TPO_DIAS_INTERNACAO
FROM Internacoes
WHERE DT_INTERNADO IS NOT NULL
"""

# Executa a função para buscar os dados
dfDESFECHO2 = query_sql_to_dataframe(conn_string, sql_query)


Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [202]:
len(lista_a_processar)

87

In [209]:
# ==============================================================================
# 1. CARREGAMENTO E LIMPEZA INICIAL
# ==============================================================================

# Bases originais
df_doc = dfDESFECHO.copy()   # Query 1: Resumo de Óbito e Alta
df_uti = dfDESFECHO2.copy()  # Query 2: Passagem pela UTI e Enfermaria

# Removendo pacientes de teste
df_doc = remove_test_patients(df_doc, 'CD_PACIENTE')
df_uti = remove_test_patients(df_uti, 'CD_PACIENTE')

# Filtrando os pacientes autorizados (lista_a_processar)
df_doc = df_doc[df_doc['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))]
df_doc = df_doc.drop_duplicates(subset=['CD_PACIENTE', 'CD_ATENDIMENTO'])

df_uti = df_uti[df_uti['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))]
df_uti = df_uti.drop_duplicates(subset=['CD_PACIENTE', 'CD_ATENDIMENTO'])

# Padronizando CD_ATENDIMENTO na cirurgia
regvalv_cirurgia['cd_atendimento'] = regvalv_cirurgia['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

In [211]:


# ==============================================================================
# 2. TRATAMENTO: BASE DOCUMENTOS (Óbito e Alta Hospitalar)
# ==============================================================================

df_doc = df_doc.rename(columns={
    'CD_PACIENTE': 'record_id',
    'CD_ATENDIMENTO': 'cd_atendimento',
    'DS_TEMPO_PERMANENCIA_1': 'tempo_permanencia_hosp', 
    'DT_ADMISSAO_1': 'admissao_enf', 
    'DT_ALTA_1': 'data_alta',
    'DS_CAUSA_DA_MORTE_1': 'causa_obito'
})
# 1. Identificar quem realmente foi a Óbito 
# (causa_obito não nula E não composta apenas por espaços vazios)
is_obito = df_doc['causa_obito'].notna() & (df_doc['causa_obito'].str.strip() != '')

# 2. Inicializar as colunas vazias para todo mundo
df_doc['data_obito'] = ''
df_doc['mortalidade_hospitalar'] = 0
df_doc['local_obito'] = ''

# 3. Aplicar as Regras de Óbito APENAS onde a máscara 'is_obito' for Verdadeira (True)
df_doc.loc[is_obito, 'data_obito'] = pd.to_datetime(df_doc.loc[is_obito, 'DH_DOCUMENTO'], errors='coerce').dt.strftime('%Y-%m-%d')
df_doc.loc[is_obito, 'mortalidade_hospitalar'] = 1

# O np.select também ganha a condição 'is_obito &' para garantir que não vai gerar local_obito para pacientes de Alta
condicoes_setor = [
    is_obito & df_doc['NM_SETOR'].str.contains('CENTRO', na=False, case=False),
    is_obito & df_doc['NM_SETOR'].str.contains('UTI|UCO', na=False, case=False),
    is_obito & df_doc['NM_SETOR'].str.contains('ENF', na=False, case=False)
]
df_doc['local_obito'] = np.select(condicoes_setor, [1, 2, 3], default='')



# Padronizando cd_atendimento e Vinculando as instâncias temporalmente (permanece igual)
df_doc['cd_atendimento'] = df_doc['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_doc_vinculado = vincular_instancias_temporal(
    df_principal=df_doc,
    df_referencia=regvalv_cirurgia,
    col_data_principal='DH_DOCUMENTO',
    col_data_referencia='data_cirurgia',
    tolerancia_dias=180,
    col_atend_principal='cd_atendimento',
    col_atend_referencia='cd_atendimento'
)

# ==============================================================================
# 3. TRATAMENTO: BASE UTI (Pós Operatório)
# ==============================================================================

df_uti = df_uti.rename(columns={
    'CD_PACIENTE': 'record_id', 
    'CD_ATENDIMENTO': 'cd_atendimento', 
    'DT_ENT_UTI': 'admissao_uti', 
    'DT_SAIDA_UTI': 'saida_uti', 
    'READMISSAO_UTI': 'readmissao_em_uti', 
    'TPO_HORAS_UTI': 'tempo_uti'
})

# Padronizando cd_atendimento
df_uti['cd_atendimento'] = df_uti['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Vinculando instâncias das Cirurgias (Usando a data de internação como âncora)
df_uti_vinculado = vincular_instancias_temporal(
    df_principal=df_uti,
    df_referencia=regvalv_cirurgia,
    col_data_principal='DT_INTERNADO', 
    col_data_referencia='data_cirurgia',
    tolerancia_dias=180,
    col_atend_principal='cd_atendimento',
    col_atend_referencia='cd_atendimento'
)

# ==============================================================================
# 4. GERAÇÃO DAS TRÊS IMPORTAÇÕES FINAIS (REDCAP)
# ==============================================================================

# --- Mapear Óbitos Existentes ---
regvalv_obito['record_id'] = regvalv_obito['record_id'].astype(str)
obito_existentes = set(zip(
    regvalv_obito['record_id'].astype(str), 
    regvalv_obito['redcap_event_name'].astype(str),
    regvalv_obito['redcap_repeat_instance'].astype(str)
))

# ---------------------------------------------------------
# IMPORTAÇÃO 1: Ficha de Complicações (Óbito)
# ---------------------------------------------------------
df_ficha_complicacoes = filtrar_novos_registros(df_doc_vinculado, obito_existentes)
cols_obito = ['record_id', 'causa_obito', 'data_obito', 'mortalidade_hospitalar', 'local_obito', 'redcap_repeat_instance']

df_ficha_complicacoes = df_ficha_complicacoes[[c for c in cols_obito if c in df_ficha_complicacoes.columns]].copy()
if not df_ficha_complicacoes.empty:
    df_ficha_complicacoes['redcap_event_name'] = 'encerramento_arm_1'
    df_ficha_complicacoes['redcap_repeat_instrument'] = 'ficha_complicacoes'
    df_ficha_complicacoes = df_ficha_complicacoes.replace(['nan', 'NaN', 'None', np.nan], "")


# ---------------------------------------------------------
# IMPORTAÇÃO 2: Alta Hospitalar
# ---------------------------------------------------------
cols_alta = ['record_id', 'tempo_permanencia_hosp', 'admissao_enf', 'data_alta', 'redcap_repeat_instance']

df_alta_hospitalar = df_doc_vinculado[[c for c in cols_alta if c in df_doc_vinculado.columns]].copy()
if not df_alta_hospitalar.empty:
    df_alta_hospitalar['redcap_event_name'] = 'fase5recuperacaoin_arm_1'
    df_alta_hospitalar['redcap_repeat_instrument'] = 'alta_hospitalar'
    df_alta_hospitalar = df_alta_hospitalar.replace(['nan', 'NaN', 'None', np.nan], "")


# ---------------------------------------------------------
# IMPORTAÇÃO 3: Pós Operatório (UTI)
# ---------------------------------------------------------
cols_uti = ['record_id', 'admissao_uti', 'saida_uti', 'readmissao_em_uti', 'tempo_uti', 'redcap_repeat_instance']

df_pos_operatorio = df_uti_vinculado[[c for c in cols_uti if c in df_uti_vinculado.columns]].copy()
if not df_pos_operatorio.empty:
    df_pos_operatorio['redcap_event_name'] = 'fase4posprocedimed_arm_1'
    df_pos_operatorio['redcap_repeat_instrument'] = 'pos_operatorio'
    df_pos_operatorio['redcap_repeat_instance'] = df_pos_operatorio['redcap_repeat_instance'].astype('Int64')
    df_pos_operatorio = df_pos_operatorio.replace(['nan', 'NaN', 'None', np.nan], "")

print("Processamento concluído: df_ficha_complicacoes, df_alta_hospitalar e df_pos_operatorio prontos.")

📊 Filtro aplicado:
   - Registros originais no DataFrame: 514
   - Registros ignorados (já existem no REDCap): 0
   - Novos registros para importar: 514
Processamento concluído: df_ficha_complicacoes, df_alta_hospitalar e df_pos_operatorio prontos.


In [ ]:
df_ficha_complicacoes.to_excel('C://Users/priscilla.sequetin/Downloads/obito_1.xlsx', index=False)

In [214]:
regvalv_obito

,record_id,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,mortalidade_hospitalar,local_obito,data_obito,causa_obito,obito_fora_local,mortalidade_classificacao,tipo_mortalidade_cv,endocardite,data_endocardite,agente_endocardite,tratamento_endocardite,desfecho_endocardite,drenagem_marfan,data_drenagem_marfan,volume_drenado,desfecho_drenagem,evento_tromboembolico_tardio,data_evento_tromboembolico,tipo_evento_tromboemb_tardio,anticoagulacao_uso,tipo_anticoagulacao,inr_evento,degeneracao_prostetica,data_degeneracao,tipo_degeneracao,conduta_degeneracao,disfuncao_valvar_estrutural,tipo_disfuncao_estrutural___1,tipo_disfuncao_estrutural___2,tipo_disfuncao_estrutural___3,tipo_disfuncao_estrutural___4,tipo_disfuncao_estrutural___5,tipo_disfuncao_estrutural___6,disfuncao_valvar_nao_estrutural,tipo_disfuncao_nao_estrutural___1,tipo_disfuncao_nao_estrutural___2,tipo_disfuncao_nao_estrutural___3,tipo_disfuncao_nao_estrutural___4,tipo_disfuncao_nao_estrutural___5,tipo_disfuncao_nao_estrutural___6,tipo_disfuncao_nao_estrutural___7,gradiente_valvar_medio,area_valvar_efetiva,mismatch_paciente_protese,regurgitacao_paravalvar_grau,avaliacao_imagem_valvar___1,avaliacao_imagem_valvar___2,avaliacao_imagem_valvar___3,avaliacao_imagem_valvar___4,avaliacao_imagem_valvar___5,avaliacao_imagem_valvar___6,data_diagnostico_disfuncao,trombose_valvar,classificacao_trombose_valvar,tratamento_trombose_valvar___1,tratamento_trombose_valvar___2,tratamento_trombose_valvar___3,tratamento_trombose_valvar___4,tratamento_trombose_valvar___5,insuficiencia_cardiaca,tipo_insuficiencia_cardiaca,hospitalizacao_ic,data_hospitalizacao_ic,duracao_hospitalizacao_ic,ic_classificacao_nyha,biomarcadores_ic___1,biomarcadores_ic___2,biomarcadores_ic___3,biomarcadores_ic___4,valor_bnp,valor_nt_probnp,valor_troponina,desfecho_eficacia_dispositivo,componentes_eficacia___1,componentes_eficacia___2,componentes_eficacia___3,componentes_eficacia___4,componentes_eficacia___5,componentes_eficacia___6,componentes_eficacia___7,componentes_eficacia___8,desfecho_seguranca_dispositivo,componentes_seguranca___1,componentes_seguranca___2,componentes_seguranca___3,componentes_seguranca___4,componentes_seguranca___5,componentes_seguranca___6,componentes_seguranca___7,componentes_seguranca___8,ficha_complicacoes_complete
2,302076,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,,,,,,,,,,,,,,,,,,,,,0,0,0,0,0,0,,0,0,0,0,0,0,0,,,,,0,0,0,0,0,0,,,,0,0,0,0,0,,,,,,,0,0,0,0,,,,,0,0,0,0,0,0,0,0,,0,0,0,0,0,0,0,0,0
4,310758,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,0,,,,,0,,,,0,,,,,,0,,,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,,,,0,0,0,0,0,0,,0,,0,0,0,0,0,1,1,0,,,1,0,0,0,1,,,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
6,319436,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,0,,,,,0,,,,0,,,,,,0,,,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,2.1,,,0,0,0,0,0,0,,0,,0,0,0,0,0,0,,,,,,0,0,0,0,,,,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,2
9,328265,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,0,,,,,0,,,,0,,,,,,0,,,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,19,,,,0,0,0,0,0,0,,0,,0,0,0,0,0,0,,,,,,0,0,0,0,,,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
11,328585,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,,,,,,,,,,,,,,,,,,,,,0,0,0,0,0,0,,0,0,0,0,0,0,0,,,,,0,0,0,0,0,0,,,,0,0,0,0,0,,,,,,,0,0,0,0,,,,,0,0,0,0,0,0,0,0,,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1006,1162610,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,,,,,,,,,,,,,,,,,,,,,0,0,0,0,0,0,,0,0,0,0,0,0,0,,,,,0,0,0,0,0,0,,,,0,0,0,0,0,,,,,,,0,0,0,0,,,,,0,0,0,0,0,0,0,0,,0,0,0,0,0,0,0,0,0
1008,1162983,encerramento_arm_1,ficha_complicacoes,1,0,,,,,,,,,,,,,,,,,,,,,,,,,,,0,0,0,0,0,0,,0,0,0,0,0,0,0,,,,,0,0,0,0,0,0,,,,0,0,0,0,0,,,,,,,0,0,

In [222]:
# 1. Pegar todos os IDs únicos da base de pacientes
ids_pcte = set(regvalv_pcte['record_id'].astype(str))

# 2. Pegar todos os IDs únicos da base de óbitos
ids_obito = set(regvalv_obito['record_id'].astype(str))

# 3. Subtrair: IDs dos pacientes MENOS os IDs que já estão na tabela de óbitos
pacientes_sem_obito = ids_pcte - ids_obito

# Mostrar os resultados
print(f"Total de pacientes sem registro na tabela de óbito: {len(pacientes_sem_obito)}")
print(f"IDs dos pacientes: {pacientes_sem_obito}")

Total de pacientes sem registro na tabela de óbito: 3
IDs dos pacientes: {'1130237', '1154426', '1128722'}


In [ ]:
lista_pacientes_sem_obito = list(pacientes_sem_obito)

# 1. Cria o DataFrame baseando-se na lista de IDs
df_novos_obitos = pd.DataFrame({'record_id': lista_pacientes_sem_obito})

# 2. Adiciona as colunas com os valores fixos solicitados
df_novos_obitos['mortalidade_hospitalar'] = 1
df_novos_obitos['redcap_event_name'] = 'encerramento_arm_1'  # Corrigido para o padrão do braço
df_novos_obitos['redcap_repeat_instrument'] = 'ficha_complicacoes'

# Opcional: Se o seu projeto exigir a instância para formulários repetitivos, adicione:
df_novos_obitos['redcap_repeat_instance'] = 1

In [ ]:
# # Agora você importa o aCir6
# import_redcap(
#     df_sorted=aEnd4,
#     api_url=api_url,
#     api_key=api_key,
#     batch_size=10,
#     overwrite=False
# )

In [ ]:
# # Agora você importa o aCir6
# import_redcap(
#     df_sorted=aEnd5,
#     api_url=api_url,
#     api_key=api_key,
#     batch_size=10,
#     overwrite=False
# )

In [ ]:
# # Agora você importa o aCir6
# import_redcap(
#     df_sorted=aEnd6,
#     api_url=api_url,
#     api_key=api_key,
#     batch_size=10,
#     overwrite=False
# )

# GERONTO


In [223]:
# Buscando fichas TEXTO

sql_GERON = """
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
),
dip AS (
	SELECT 
		NK_CD_PACIENTE,
		NM_PACIENTE,
		TP_SITUACAO,
		SEXO,
		COR,
		DT_NASCIMENTO,
		NR_DDD_FONE,
		NR_FONE_1,
		NR_DDD_CELULAR,
		NR_CELULAR,
		NR_DDD_FONE_COMERCIAL,
		NR_FONE_COMERCIAL,
		DT_ATUAL
	FROM dim_paciente dip1
	WHERE DT_ATUAL = (
		SELECT MAX(DT_ATUAL) 
		FROM dim_paciente dip2 
		WHERE dip2.NK_CD_PACIENTE = dip1.NK_CD_PACIENTE
	)
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    --dcd.CD_METADADO,
    fde.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,
    dip.NM_PACIENTE as nome_paciente,
    dip.TP_SITUACAO as tp_situacao,
    dip.SEXO as genero,
    dip.COR as etnia,
    dip.DT_NASCIMENTO as data_nascimento,
    dip.NR_DDD_FONE as nr_ddd_fone,
    dip.NR_FONE_1 as nr_fone_1,
    dip.NR_DDD_CELULAR as nr_ddd_celular,
    dip.NR_CELULAR as nr_celular,
    dip.NR_DDD_FONE_COMERCIAL as nr_ddd_fone_comercial,
    dip.NR_FONE_COMERCIAL as nr_fone_comercial,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM ft_doc_eletronico_texto fde 
JOIN dip on dip.NK_CD_PACIENTE = fde.CD_PACIENTE
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros
WHERE fde.CD_DOCUMENTO IN (1172)
	AND fde.DS_CAMPO not in ( 'Chave do Documento', 'Código da Empresa', 'Código do Atendimento', 'Código do Documento', 'Data de Registro', 
    'Registro de Documento', 'Registro do editor', 'Identificador', 'Código do Item na Prescrição', 'Código do Paciente', 'Código do Usuário', 
    'Sistema responsável', 'Último Resultado') 
    AND DS_RESPOSTA NOT IN ('false')
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
txGERON = query_sql_to_dataframe(conn_string, sql_GERON)

Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [224]:
# Buscando fichas BOTOES

sql_GERON = """
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
),
dip AS (
	SELECT 
		NK_CD_PACIENTE,
		NM_PACIENTE,
		TP_SITUACAO,
		SEXO,
		COR,
		DT_NASCIMENTO,
		NR_DDD_FONE,
		NR_FONE_1,
		NR_DDD_CELULAR,
		NR_CELULAR,
		NR_DDD_FONE_COMERCIAL,
		NR_FONE_COMERCIAL,
		DT_ATUAL
	FROM dim_paciente dip1
	WHERE DT_ATUAL = (
		SELECT MAX(DT_ATUAL) 
		FROM dim_paciente dip2 
		WHERE dip2.NK_CD_PACIENTE = dip1.NK_CD_PACIENTE
	)
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    --dcd.CD_METADADO,
    fde.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.resposta as DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,
    dip.NM_PACIENTE as nome_paciente,
    dip.TP_SITUACAO as tp_situacao,
    dip.SEXO as genero,
    dip.COR as etnia,
    dip.DT_NASCIMENTO as data_nascimento,
    dip.NR_DDD_FONE as nr_ddd_fone,
    dip.NR_FONE_1 as nr_fone_1,
    dip.NR_DDD_CELULAR as nr_ddd_celular,
    dip.NR_CELULAR as nr_celular,
    dip.NR_DDD_FONE_COMERCIAL as nr_ddd_fone_comercial,
    dip.NR_FONE_COMERCIAL as nr_fone_comercial,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM FT_DOC_ELETRONICO fde 
JOIN dip on dip.NK_CD_PACIENTE = fde.CD_PACIENTE
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros 
WHERE fde.CD_DOCUMENTO IN (1172)
	AND fde.DS_CAMPO not in ( 'Chave do Documento', 'Código da Empresa', 'Código do Atendimento', 'Código do Documento', 'Data de Registro', 
    'Registro de Documento', 'Registro do editor', 'Identificador', 'Código do Item na Prescrição', 'Código do Paciente', 'Código do Usuário', 
    'Sistema responsável', 'Último Resultado') 
    AND resposta = 'SIM'
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
rbGERON = query_sql_to_dataframe(conn_string, sql_GERON)


Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [225]:
tDocG = txGERON.copy()

rDocG = rbGERON.copy()


In [226]:
# 1. Concatenar e pré-processar
dGer = pd.concat([tDocG, rDocG], ignore_index=True)
dGer = remove_test_patients(dGer, 'CD_PACIENTE')
dGer["DS_IDENTIFICADOR"] = dGer["DS_IDENTIFICADOR"].str[3:]

# 2. Ordenar e converter tipos (uma vez só)
dGer = dGer.sort_values('DH_DOCUMENTO', ascending=False).reset_index(drop=True)
dGer['CD_PACIENTE'] = dGer['CD_PACIENTE'].astype('Int64')
regvalv_pcte['record_id'] = regvalv_pcte['record_id'].astype('Int64')

# 3. Identificar novos pacientes (otimizado)
set_pcte_redcap = set(regvalv_pcte['record_id'].dropna().unique())
set_doc_1172 = set(dGer['CD_PACIENTE'].dropna().unique())
geronto_a_processar = sorted(list(set_doc_1172 | set_pcte_redcap))

# Logs limpos
print(f"REDCap: {len(set_pcte_redcap)} IDs")
print(f"Doc1172: {len(set_doc_1172)} IDs") 
print(f"A processar: {len(geronto_a_processar)} IDs - {geronto_a_processar}")

# 4. Filtrar final (único filtro vetorizado)
dGer = dGer[dGer['CD_PACIENTE'].isin(geronto_a_processar)].copy()

REDCap: 548 IDs
Doc1172: 155 IDs
A processar: 548 IDs - [302076, 310758, 319436, 324016, 328265, 328585, 331494, 335458, 336893, 340927, 341220, 349574, 363345, 379425, 391604, 392339, 397261, 398466, 401562, 401747, 401775, 413272, 428585, 431697, 432670, 433309, 434034, 435283, 440067, 440359, 440894, 442498, 443558, 446949, 454594, 454932, 458634, 472053, 472525, 474291, 475623, 478110, 479406, 484988, 486660, 488589, 488877, 496131, 510205, 518709, 530667, 531219, 536012, 539795, 550953, 554132, 556037, 558067, 562568, 568122, 570632, 571572, 573801, 576016, 584721, 584768, 587546, 593227, 600475, 600762, 603505, 604177, 607952, 612689, 618437, 626508, 633206, 650654, 653951, 661824, 661977, 677960, 678566, 682776, 686028, 689671, 699640, 706309, 707603, 708834, 712755, 721749, 736562, 741733, 743944, 747345, 751740, 757153, 763569, 766984, 773079, 777948, 782398, 788698, 790044, 791370, 793042, 793706, 797757, 802117, 804401, 806530, 815158, 822752, 823441, 834779, 840112, 845113,

In [227]:
geronto_ls = {
 
# --- LF ---
'lf_perda_peso':['PERDA_PESO_NAO_INTENCIONAL_NAO_1', 'PERDA_PESO_NAO_INTENCIONAL_SIM_1',],
'lf_fadiga':[ 'FADIGA_EXAUSTAO_AUTORREFERIDA_NAO_1', 'FADIGA_EXAUSTAO_AUTORREFERIDA_SIM_1',],
'lf_dim_forca':['FORCA_MUSCULAR_REDUZIDA_NAO_1', 'FORCA_MUSCULAR_REDUZIDA_SIM_1',],
'lf_lentidao_marcha':['VELOCIDADE_MARCHA_LENTA_NAO_1', 'VELOCIDADE_MARCHA_LENTA_SIM_1',],
'lf_baixa_ativ_fisica':['BAIXO_NIVEL_ATIVIDADE_FISICA_NAO_1', 'BAIXO_NIVEL_ATIVIDADE_FISICA_SIM_1',],

# --- BARTHEL ---
'barthel_alimentacao':['INDEPENDENTE_ALIMENTACAO_1', 'PRECISA_AJUDA_ALIMENTACAO_1', 'INCAPACITADO_ALIMENTACAO_1',],
'barthel_banho':['INDEPENDENTE_BANHO_1', 'DEPENDENTE_BANHO_1',],
'barthel_hig_pessoal':['INDEPENDENTE_ATIVIDADES_ROTINEIRAS_1', 'PRECISA_AJUDA_HIGIENE_PESSOAL_ATIVIDADE_ROTINEIRAS_1',],
'barthel_vestir':['INDEPENDENTE_VESTIR_SE_1', 'PRECISA_AJUDA_VESTIR_SE_1', 'DEPENDENTE_VESTIR_SE_1',],
'barthel_intestinos':['INCONTINENTE_INTESTINO_1', 'ACIDENTE_OCASIONAL_INTESTINO_1', 'CONTINENTE_INTESTINO_1',],
'barthel_bexiga':['INCONTINENTE_CATETERIZADIZADO_INCAPAZ_MANEJO_SISTEMA_URINARIO_1', 'ACIDENTE_OCASIONAL_SISTEMA_URINARIO_1', 'CONTINENTE_SISTEMA_URINARIO_1',],
'barthel_uso_vaso':['INDEPENDENTE_USO_BANHEIRO_1', 'PRECISA_ALGUMA_AJUDA_PARCIAL_USO_BANHEIRO_1', 'DEPENDENTE_USO_BANHEIRO_1',],
'barthel_transfer':['INDEPENDENTE_TRANSFERENCIA_1', 'POUCA_AJUDA_TRANSFERENCIA_1', 'MUITA_AJUDA_TRANSFERENCIA_1', 'INCAPACITADO_TRANSFERENCIA_1',],
'barthel_mobilidade':['IMOVEL_MOBILIDADE_1', 'CADEIRAS_RODAS_INDEPENDENTE_MOBILIDADE_1', 'CAMINHA_AJUDA_PESSOA_MOBILIDADE_1', 'INDEPENDENTE_MOBILIDADE_1',],
'barthel_escadas':['INDEPENDENTE_ESCADAS_1', 'PRECISA_AJUDA_ESCADAS_1', 'INCAPACITADO_ESCADAS_1',],



# --- LAWTON ---
'lawton_telefone':['USAR_TELEFONE_NAO_AIVD_1',  'USAR_TELEFONE_SIM_AIVD_1',],
'lawton_compras':['FAZER_COMPRAS_NAO_AIVD_1',  'FAZER_COMPRAS_SIM_AIVD_1',],
'lawton_refeicoes':['PREPARAR_REFEICOES_NAO_AIVD_1',  'PREPARAR_REFEICOES_SIM_AIVD_1',],
'lawton_casa':['CUIDAR_CASA_NAO_AIVD_1',  'CUIDAR_CASA_SIM_AIVD_1',],
'lawton_lavar_roupa':['LAVAR_ROUPA_NAO_AIVD_1',  'LAVAR_ROUPA_SIM_AIVD_1',],
'lawton_transporte':['USAR_MEIOS_TRANSPORTE_NAO_AIVD_1',  'USAR_MEIOS_TRANSPORTE_SIM_AIVD_1',],
'lawton_medicamentos':['RESPONSABILIDADE_MEDICAMENTOS_NAO_AIVD_1',  'RESPONSABILIDADE_MEDICAMENTOS_SIM_AIVD_1',],
'lawton_financas':['GERENCIAR_FINANCAS_NAO_AIVD_1',  'GERENCIAR_FINANCAS_SIM_AIVD_1',],


# --- SARCF ---
'sarcf_forca':['DIFICULDADE_LEVANTAR_CARREGAR_ALGUMA_SARC_F_1',  'DIFICULDADE_LEVANTAR_CARREGAR_MUITA_SARC_F_1',
            'DIFICULDADE_LEVANTAR_CARREGAR_NENHUMA_SARC_F_1',],
'sarcf_caminho':['CAMINHAR_DENTRO_COMODO_ALGUMA_SARC_F_1',  'CAMINHAR_DENTRO_COMODO_MUITA_SARC_F_1',
            'CAMINHAR_DENTRO_COMODO_NENHUMA_SARC_F_1',],
'sarcf_levanta_cadeira':['LEVANTAR_DE_UMA_CADEIRA_ALGUMA_SARC_F_1',  'LEVANTAR_DE_UMA_CADEIRA_MUITA_SARC_F_1',
            'LEVANTAR_DE_UMA_CADEIRA_NENHUMA_SARC_F_1',],
'sarcf_escadas':['SUBIR_LANCE_10_DEGRAUS_ALGUMA_SARC_F_1',  'SUBIR_LANCE_10_DEGRAUS_MUITA_SARC_F_1',
			'SUBIR_LANCE_10_DEGRAUS_NENHUMA_SARC_F_1',],
'sarcf_queda':['QUANTAS_VEZES_CAIU_ULTIMO_ANO_ALGUMA_SARC_F_1',  'QUANTAS_VEZES_CAIU_ULTIMO_ANO_MUITA_SARC_F_1',
			'QUANTAS_VEZES_CAIU_ULTIMO_ANO_NENHUMA_SARC_F_1',],


# --- APGAR FAMILIAR ---
'apgar_ajuda':['SATIFEITO_AJUDA_RECEBE_SUA_FAMILIA_QUANDO_ALGUM_PROBLEMA_ALGUMAS_VEZES_APGAR_FAMILIAR_1',
 	'SATIFEITO_AJUDA_RECEBE_SUA_FAMILIA_QUANDO_ALGUM_PROBLEMA_QUASE_SEMPRE_APGAR_FAMILIAR_1',
 	'SATIFEITO_AJUDA_RECEBE_SUA_FAMILIA_QUANDO_ALGUM_PROBLEMA_RARAMENTE_APGAR_FAMILIAR_1',],
'apgar_discute_problemas':['SATIFEITO_FORMA_COMO_SUA_FAMILIA_CONVERSA_SOBRE_PROBLEMAS_ALGUMAS_VEZES_APGAR_FAMILIAR_1',
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_CONVERSA_SOBRE_PROBLEMAS_QUASE_SEMPRE_APGAR_FAMILIAR_1',
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_CONVERSA_SOBRE_PROBLEMAS_RARAMENTE_APGAR_FAMILIAR_1',],
'apgar_apoia_desejos':['SATIFEITO_FORMA_COMO_SUA_FAMILIA_ACEITA_SUES_DESEJOS_ALGUMAS_VEZES_APGAR_FAMILIAR_1',
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_ACEITA_SUES_DESEJOS_QUASE_SEMPRE_APGAR_FAMILIAR_1',
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_ACEITA_SUES_DESEJOS_RARAMENTE_APGAR_FAMILIAR_1',],
'apgar_responde_emocoes':['SATIFEITO_FORMA_COMO_PODE_CONTAR_FAMILIA_SITUACOES_DIFICEIS_ALGUMAS_VEZES_APGAR_FAMILIAR_1',
 	'SATIFEITO_FORMA_COMO_PODE_CONTAR_FAMILIA_SITUACOES_DIFICEIS_QUASE_SEMPRE_APGAR_FAMILIAR_1',
 	'SATIFEITO_FORMA_COMO_PODE_CONTAR_FAMILIA_SITUACOES_DIFICEIS_RARAMENTE_APGAR_FAMILIAR_1',],
'apgar_tempo_juntos':['SATIFEITO_COM_TEMPO_FAMILIA_JUNTOS_ALGUMAS_VEZES_APGAR_FAMILIAR_1',
 	'SATIFEITO_COM_TEMPO_FAMILIA_JUNTOS_QUASE_SEMPRE_APGAR_FAMILIAR_1',
 	'SATIFEITO_COM_TEMPO_FAMILIA_JUNTOS_RARAMENTE_APGAR_FAMILIAR_1',],


# --- GDS-15 ---
'gds15_1_satisfeito_vida':['ESTA_SATISFEIRO_COM_SUA_VIDA_NAO_GDS_15_1',  'ESTA_SATISFEIRO_COM_SUA_VIDA_SIM_GDS_15_1',],
'gds15_2_abandono_atividade':['INTERROMPEU_MUITA_ATIVIDADES_NAO_GDS_15_1',  'INTERROMPEU_MUITA_ATIVIDADES_SIM_GDS_15_1',],
'gds15_3_vida_vazia':['ACHA_SUA_VIDA_VAZIA_NAO_GDS_15_1',  'ACHA_SUA_VIDA_VAZIA_SIM_GDS_15_1',],
'gds15_4_aborrecido_freq':['ABORRECE_SE_COM_FREQUENCIA_NAO_GDS_15_1',  'ABORRECE_SE_COM_FREQUENCIA_SIM_GDS_15_1',],
'gds15_5_animado_humor':['SENTE_SE_BEM_COM_VIDA_MAIOR_PARTE_TEMPO_NAO_GDS_15_1',  'SENTE_SE_BEM_COM_VIDA_MAIOR_PARTE_TEMPO_SIM_GDS_15_1',],
'gds15_6_medo_ruim':['TEME_QUE_ALGO_RUIM_LHE_ACONTECA_NAO_GDS_15_1',  'TEME_QUE_ALGO_RUIM_LHE_ACONTECA_SIM_GDS_15_1',],
'gds15_7_feliz_todo':['SENTE_SE_ALEGRE_MAIOR_PARTE_TEMPO_NAO_GDS_15_1',  'SENTE_SE_ALEGRE_MAIOR_PARTE_TEMPO_SIM_GDS_15_1',],
'gds15_8_desamparado_freq':['SENTE_SE_DESAMPARADO_COM_FREQUENCIA_NAO_GDS_15_1',  'SENTE_SE_DESAMPARADO_COM_FREQUENCIA_SIM_GDS_15_1',],
'gds15_9_prefere_casa':['PREFERE_FICAR_CASA_SAIR_FAZER_COISAS_NOVAS_NAO_GDS_15_1',  'PREFERE_FICAR_CASA_SAIR_FAZER_COISAS_NOVAS_SIM_GDS_15_1',],
'gds15_10_problema_memoria':['ACHA_MAIS_PROBLEMAS_MEMORIA_OUTRAS_PESSOAS_NAO_GDS_15_1',  'ACHA_MAIS_PROBLEMAS_MEMORIA_OUTRAS_PESSOAS_SIM_GDS_15_1',],
'gds15_11_maravilha_vivo':['ACHA_MARAVILHOSO_VIVO_NAO_GDS_15_1',  'ACHA_MARAVILHOSO_VIVO_SIM_GDS_15_1',],
'gds15_12_inutil_semvalor':['SENTE_SE_INUTIL_NAO_GDS_15_1',  'SENTE_SE_INUTIL_SIM_GDS_15_1',],
'gds15_13_energizado':['SENTE_SE_CHEIO_DE_ENERGIA_NAO_GDS_15_1',  'SENTE_SE_CHEIO_DE_ENERGIA_SIM_GDS_15_1',],
'gds15_14_sem_esperanca':['SENTE_SE_SEM_ESPERANCA_NAO_GDS_15_1',  'SENTE_SE_SEM_ESPERANCA_SIM_GDS_15_1',],
'gds15_15_pessoas_melhores':['ACHA_OUTROS_MAIS_SORTE_VOCE_NAO_GDS_15_1',  'ACHA_OUTROS_MAIS_SORTE_VOCE_SIM_GDS_15_1',],


# --- KCCQ-12 ---
'kccq_1a_banho':['TOMAR_BANHO_FUI_LIMITADO_OUTRAS_RAZOES_1',  'TOMAR_BANHO_LIMITOU_BASTANTE_1',  'TOMAR_BANHO_LIMITOU_MUITISSIMO_1',
        'TOMAR_BANHO_LIMITOU_MUITO_POUCO_1', 'TOMAR_BANHO_LIMITOU_UM_POUCO_1',  'TOMAR_BANHO_NAO_LIMITOU_2'],
'kccq_1b_caminho':['CAMINHAR_QUARTEIRAO_TERRENO_PLANO_FUI_LIMITADO_OUTRAS_RAZOES_NAO_FIZ_TAL_ATIVIDADE_1',  'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_BASTANTE_1',
 		'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_MUITISSIMO_1',  'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_MUITO_POUCO_1',
 		'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_UM_POUCO_1',  'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_NAO_LIMITOU_1',],
'kcc1_1c_correr':['CORRER_ANDAR_APRESSADAMENTE_FUI_LIMITADO_POR_OUTRAS_RAZOES_1',  'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_BASTANTE_1',
 		'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_MUITISSIMO_1',  'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_MUITO_POUCO_1',
 		'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_UM_POUCO_1',  'CORRER_ANDAR_APRESSADAMENTE_NAO_LIMITOU_1',],
'tb_kccq_2':['1_2_VEZES_POR_SEMANA_QUESTAO_2_1', '3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODOS_OS_DIAS_QUESTAO_2_1', 'MENOS_DE_1_VEZ_POR_SEMANA_QUASTAO_2_1',
        'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_2_1', 'TODAS_AS_MANHAS_QUESTAO_2_1',],
'tb_kccq_3':[ '1_2_VEZES_POR_SEMANA_QUESTAO_3_1', '3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODOS_OS_DIAS_QUESTAO_3_1', 'MENOS_DE_1_POR_SEMANA_QUESTAO_3_1',
        'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_3_1', 'O_TEMPO_TODO_QUESTAO_3_1', 'PELO_MENOS_1_VEZES_POR_DIA_QUESTAO_3_1', 'VARIA_VEZES_POR_DIA_QUESTAO_3_1',],
'tb_kccq_4':['1_2_VEZES_POR_SEMANA_QUESTAO_4_1', '3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODOS_OS_DIAS_QUESTAO_4_1', 'MENOS_DE_1_POR_SEMANA_QUESTAO_4_1', 
        'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_4_1', 'O_TEMPO_TODO_QUESTAO_4_1', 'PELO_MENOS_1_VEZES_POR_DIA_QUESTAO_4_1', 'VARIA_VEZES_POR_DIA_QUESTAO_4_1',],
'tb_kccq_5':[ '1_2_VEZES_POR_SEMANA_QUESTAO_5_1',  '3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODAS_AS_NOITES_QUESTAO_5_1', 'MENOS_DE_1_POR_SEMANA_QUESTAO_5_1',
        'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_5_1', 'O_TEMPO_TODOS_QUESTAO_5_1',],
'tb_kccq_6':['IMPEDIU_BASTANTE_QUESTAO_6_1',  'IMPEDIU_MUITISSIMO_QUESTAO_6_1',  'IMPEDIU_MUITO_POUCO_QUESTAO_6_1',  'IMPEDIU_UM_POUCO_QUESTAO_6_1', 
        'NAO_IMPEDIU_EM_NADA_QUESTAO_6_1',],
'tb_kccq_7':['BASTANTE_INSATISFEITO_QUESTAO_7_1',  'BASTANTE_SATISFEITO_QUESTAO_7_1', 'TOTALMENTE_INSATISFEITO_QUESTAO_7_1',  'TOTALMENTE_SATISFEITO_QUESTAO_7_1',
        'UM_POUCO_SATISFEITO_QUESTAO_7_1',],
'kccq_8a_passatempos':['PASSATEMPOS_ATIVIDADES_RECREATIVAS_FUI_LIMITADO_POR_OUTROS_RAZOES_1',  'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_BASTANTE_1',
        'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_MUITISSIMO_1',  'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_MUITO_POUCO_1',
        'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_UM_POUCO_1',  'PASSATEMPOS_ATIVIDADES_RECREATIVAS_NAO_LIMITOU_1',],
'kccq_8b_tarefas':['TRABALHAR_FAZER_TAREFAS_DOMESTICAS_FUI_LIMITADO_POR_OUTRAS_RAZOES_1',  'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_BASTANTE_1',
        'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_MUITISSIMO_1',  'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_MUITO_POUCO_1',
        'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_UM_POUCO_1',  'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_NAO_LIMITOU_1',],
'kccq_8c_visitas':['VISITAR_PARENTES_OU_AMIGOS_FUI_LIMITADO_POR_OUTRAS_RAZOES_1',  'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_BASTANTE_1',
        'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_MUITISSIMO_1',  'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_MUITO_POUCO_1',  'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_UM_POUCO_1',
        'VISITAR_PARENTES_OU_AMIGOS_NAO_LIMITOU_1'],


# --- EQ5D ---
'eq5d_mobilidade':['NAO_TENHO_PROBLEMAS_ANDAR_MOBILIDADE_EQ_5D_1', 'TENHO_ALGUNS_PROBLEMAS_ANDAR_MOBILIDADE_EQ_5D_1', 'TENHO_DE_ESTAR_NA_CAMA_MOBILIDADE_EQ_5D_1',],
'eq5d_cuidados_pessoais':['NAO_TENHO_PROBLEMAS_EM_CUIDAR_DE_MIM_CUIDADOS_PESSOAIS_EQ_5D_1', 'TENHO_ALGUNS_PROBLEMAS_A_LAVAR_ME_OU_VESTIR_ME_CUIDADOS_PESSOAIS_EQ_5D_1',
        'SOU_INCAPAZ_DE_ME_LAVAR_OU_VESTIR_SOZINHO_CUIDADOS_PESSOAIS_EQ_5D_1',],
'eq5d_atividades_habituais':['NAO_TENHO_PROBLEMAS_EM_DESEMPENHAR_AS_MINHAS_ATIVIDADES_HABITUAIS_ATIVIDADES_HABITUAIS_5Q_5D_1', 
        'TENHO_ALGUNS_PROBLEMAS_EM_DESEMPENHAR_AS_MINHAS_ATIVIDADES_HABITUAIS_ATIVIDADES_HABITUAIS_5Q_5D_1',
  		'SOU_INCAPAZ_DE_DESEMPENHAR_AS_MINHAS_ATIVIDADES_HABITUAIS_ATIVIDADES_HABITUAIS_5Q_5D_1',],
'eq5d_dor': ['NAO_TENHO_DORES_OU_MAL_ESTAR_DOR_MAL_ESTAR_EQ_5D_1', 'TENHO_DORES_OU_MAL_ESTAR_EXTREMOS_DOR_MAL_ESTAR__1',
 		'TENHO_DORES_OU_MAL_ESTAR_MODERADOS_DOR_MAL_ESTAR_5Q_5D_1',],
'eq5d_ansiedade':['NAO_ESTOU_ANSIOSO_OU_DEPRIMIDO_ANSIEDADE_DEPRESSAO_EQ_5D_1', 'ESTOU_EXTREMAMENTE_ANSIOSO_OU_DEPRIMIDO_ANSIEDADE_DEPRESSAO_EQ_5D_1',
 		'ESTOU_MODERAMENTE_ANSIOSO_OU_DEPRIMIDO_ANSIEDADE_DEPRESSAO_EQ_5D_1',],
'eq5d_comparacao_12meses':['MELHOR_COMPARACAO_COM_O_ULTIMO_ANO_1', 'O_MESMO_COMPARACAO_COM_O_ULTIMO_ANO_1', 'PIOR_COMPARACAO_COM_O_ULTIMO_ANO_1',],


}


In [228]:
geronto_rn = {
    # --- LINDA FRIED (0=Não, 1=Sim) ---
    'PERDA_PESO_NAO_INTENCIONAL_NAO_1': 0,
    'PERDA_PESO_NAO_INTENCIONAL_SIM_1': 1,
    'FADIGA_EXAUSTAO_AUTORREFERIDA_NAO_1': 0,
    'FADIGA_EXAUSTAO_AUTORREFERIDA_SIM_1': 1,
    'FORCA_MUSCULAR_REDUZIDA_NAO_1': 0,
    'FORCA_MUSCULAR_REDUZIDA_SIM_1': 1,
    'VELOCIDADE_MARCHA_LENTA_NAO_1': 0,
    'VELOCIDADE_MARCHA_LENTA_SIM_1': 1,
    'BAIXO_NIVEL_ATIVIDADE_FISICA_NAO_1': 0,
    'BAIXO_NIVEL_ATIVIDADE_FISICA_SIM_1': 1,

    # --- BARTHEL (Scores variados) ---
    # Alimentação
    'INDEPENDENTE_ALIMENTACAO_1': 10,
    'PRECISA_AJUDA_ALIMENTACAO_1': 5,
    'INCAPACITADO_ALIMENTACAO_1': 0,
    # Banho
    'INDEPENDENTE_BANHO_1': 5,
    'DEPENDENTE_BANHO_1': 0,
    # Higiene
    'INDEPENDENTE_ATIVIDADES_ROTINEIRAS_1': 5,
    'PRECISA_AJUDA_HIGIENE_PESSOAL_ATIVIDADE_ROTINEIRAS_1': 0,
    # Vestir
    'INDEPENDENTE_VESTIR_SE_1': 10,
    'PRECISA_AJUDA_VESTIR_SE_1': 5,
    'DEPENDENTE_VESTIR_SE_1': 0,
    # Intestino
    'CONTINENTE_INTESTINO_1': 10,
    'ACIDENTE_OCASIONAL_INTESTINO_1': 5,
    'INCONTINENTE_INTESTINO_1': 0,
    # Bexiga
    'CONTINENTE_SISTEMA_URINARIO_1': 10,
    'ACIDENTE_OCASIONAL_SISTEMA_URINARIO_1': 5,
    'INCONTINENTE_CATETERIZADIZADO_INCAPAZ_MANEJO_SISTEMA_URINARIO_1': 0,
    # Vaso
    'INDEPENDENTE_USO_BANHEIRO_1': 10,
    'PRECISA_ALGUMA_AJUDA_PARCIAL_USO_BANHEIRO_1': 5,
    'DEPENDENTE_USO_BANHEIRO_1': 0,
    # Transferência
    'INDEPENDENTE_TRANSFERENCIA_1': 15,
    'POUCA_AJUDA_TRANSFERENCIA_1': 10,
    'MUITA_AJUDA_TRANSFERENCIA_1': 5,
    'INCAPACITADO_TRANSFERENCIA_1': 0,
    # Mobilidade
    'INDEPENDENTE_MOBILIDADE_1': 15,
    'CAMINHA_AJUDA_PESSOA_MOBILIDADE_1': 10,
    'CADEIRAS_RODAS_INDEPENDENTE_MOBILIDADE_1': 5,
    'IMOVEL_MOBILIDADE_1': 0,
    # Escadas
    'INDEPENDENTE_ESCADAS_1': 10,
    'PRECISA_AJUDA_ESCADAS_1': 5,
    'INCAPACITADO_ESCADAS_1': 0,

    # --- LAWTON ---
    'USAR_TELEFONE_NAO_AIVD_1': 0,
    'USAR_TELEFONE_SIM_AIVD_1': 1,
    'FAZER_COMPRAS_NAO_AIVD_1': 0,  
    'FAZER_COMPRAS_SIM_AIVD_1': 1,
	'PREPARAR_REFEICOES_NAO_AIVD_1': 0,  
	'PREPARAR_REFEICOES_SIM_AIVD_1': 1,
	'CUIDAR_CASA_NAO_AIVD_1': 0,  
	'CUIDAR_CASA_SIM_AIVD_1': 1,
	'LAVAR_ROUPA_NAO_AIVD_1': 0,  
	'LAVAR_ROUPA_SIM_AIVD_1': 1,
	'USAR_MEIOS_TRANSPORTE_NAO_AIVD_1': 0,  
	'USAR_MEIOS_TRANSPORTE_SIM_AIVD_1': 1,
	'RESPONSABILIDADE_MEDICAMENTOS_NAO_AIVD_1': 0,  
	'RESPONSABILIDADE_MEDICAMENTOS_SIM_AIVD_1': 1,
	'GERENCIAR_FINANCAS_NAO_AIVD_1': 0,  
	'GERENCIAR_FINANCAS_SIM_AIVD_1': 1,
    

    # --- SARC-F (0=Nenhuma, 1=Alguma, 2=Muita) ---
    # Força
    'DIFICULDADE_LEVANTAR_CARREGAR_NENHUMA_SARC_F_1': 0,
    'DIFICULDADE_LEVANTAR_CARREGAR_ALGUMA_SARC_F_1': 1,
    'DIFICULDADE_LEVANTAR_CARREGAR_MUITA_SARC_F_1': 2,
    # Caminhada
    'CAMINHAR_DENTRO_COMODO_NENHUMA_SARC_F_1': 0,
    'CAMINHAR_DENTRO_COMODO_ALGUMA_SARC_F_1': 1,
    'CAMINHAR_DENTRO_COMODO_MUITA_SARC_F_1': 2,
    # Levantar Cadeira
    'LEVANTAR_DE_UMA_CADEIRA_NENHUMA_SARC_F_1': 0,
    'LEVANTAR_DE_UMA_CADEIRA_ALGUMA_SARC_F_1': 1,
    'LEVANTAR_DE_UMA_CADEIRA_MUITA_SARC_F_1': 2,
    # Escadas
    'SUBIR_LANCE_10_DEGRAUS_NENHUMA_SARC_F_1': 0,
    'SUBIR_LANCE_10_DEGRAUS_ALGUMA_SARC_F_1': 1,
    'SUBIR_LANCE_10_DEGRAUS_MUITA_SARC_F_1': 2,
    # Quedas
    'QUANTAS_VEZES_CAIU_ULTIMO_ANO_NENHUMA_SARC_F_1': 0,
    'QUANTAS_VEZES_CAIU_ULTIMO_ANO_ALGUMA_SARC_F_1': 1,
    'QUANTAS_VEZES_CAIU_ULTIMO_ANO_MUITA_SARC_F_1': 2,

    # --- APGAR FAMILIAR (0=Raramente, 1=Algumas vezes, 2=Quase sempre) ---
    'SATIFEITO_AJUDA_RECEBE_SUA_FAMILIA_QUANDO_ALGUM_PROBLEMA_RARAMENTE_APGAR_FAMILIAR_1': 0,
    'SATIFEITO_AJUDA_RECEBE_SUA_FAMILIA_QUANDO_ALGUM_PROBLEMA_ALGUMAS_VEZES_APGAR_FAMILIAR_1': 1,
    'SATIFEITO_AJUDA_RECEBE_SUA_FAMILIA_QUANDO_ALGUM_PROBLEMA_QUASE_SEMPRE_APGAR_FAMILIAR_1': 2,
    'SATIFEITO_FORMA_COMO_SUA_FAMILIA_CONVERSA_SOBRE_PROBLEMAS_ALGUMAS_VEZES_APGAR_FAMILIAR_1': 1,
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_CONVERSA_SOBRE_PROBLEMAS_QUASE_SEMPRE_APGAR_FAMILIAR_1': 2,
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_CONVERSA_SOBRE_PROBLEMAS_RARAMENTE_APGAR_FAMILIAR_1': 0,
	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_ACEITA_SUES_DESEJOS_ALGUMAS_VEZES_APGAR_FAMILIAR_1': 1,
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_ACEITA_SUES_DESEJOS_QUASE_SEMPRE_APGAR_FAMILIAR_1': 2,
 	'SATIFEITO_FORMA_COMO_SUA_FAMILIA_ACEITA_SUES_DESEJOS_RARAMENTE_APGAR_FAMILIAR_1': 0,
	'SATIFEITO_FORMA_COMO_PODE_CONTAR_FAMILIA_SITUACOES_DIFICEIS_ALGUMAS_VEZES_APGAR_FAMILIAR_1': 1,
 	'SATIFEITO_FORMA_COMO_PODE_CONTAR_FAMILIA_SITUACOES_DIFICEIS_QUASE_SEMPRE_APGAR_FAMILIAR_1': 2,
 	'SATIFEITO_FORMA_COMO_PODE_CONTAR_FAMILIA_SITUACOES_DIFICEIS_RARAMENTE_APGAR_FAMILIAR_1': 0,
	'SATIFEITO_COM_TEMPO_FAMILIA_JUNTOS_ALGUMAS_VEZES_APGAR_FAMILIAR_1': 1,
 	'SATIFEITO_COM_TEMPO_FAMILIA_JUNTOS_QUASE_SEMPRE_APGAR_FAMILIAR_1': 2,
 	'SATIFEITO_COM_TEMPO_FAMILIA_JUNTOS_RARAMENTE_APGAR_FAMILIAR_1': 0,
	
	# --- GDS-15 ---
	'ESTA_SATISFEIRO_COM_SUA_VIDA_NAO_GDS_15_1': 0,  
    'ESTA_SATISFEIRO_COM_SUA_VIDA_SIM_GDS_15_1': 1,
	'INTERROMPEU_MUITA_ATIVIDADES_NAO_GDS_15_1': 0,  
	'INTERROMPEU_MUITA_ATIVIDADES_SIM_GDS_15_1': 1,
	'ACHA_SUA_VIDA_VAZIA_NAO_GDS_15_1': 0,  
	'ACHA_SUA_VIDA_VAZIA_SIM_GDS_15_1': 1,
	'ABORRECE_SE_COM_FREQUENCIA_NAO_GDS_15_1': 0,  
	'ABORRECE_SE_COM_FREQUENCIA_SIM_GDS_15_1': 1,
	'SENTE_SE_BEM_COM_VIDA_MAIOR_PARTE_TEMPO_NAO_GDS_15_1': 0,  
	'SENTE_SE_BEM_COM_VIDA_MAIOR_PARTE_TEMPO_SIM_GDS_15_1': 1,
	'TEME_QUE_ALGO_RUIM_LHE_ACONTECA_NAO_GDS_15_1': 0,  
	'TEME_QUE_ALGO_RUIM_LHE_ACONTECA_SIM_GDS_15_1': 1,
	'SENTE_SE_ALEGRE_MAIOR_PARTE_TEMPO_NAO_GDS_15_1': 0,  
	'SENTE_SE_ALEGRE_MAIOR_PARTE_TEMPO_SIM_GDS_15_1': 1,
	'SENTE_SE_DESAMPARADO_COM_FREQUENCIA_NAO_GDS_15_1': 0,  
	'SENTE_SE_DESAMPARADO_COM_FREQUENCIA_SIM_GDS_15_1': 1,
	'PREFERE_FICAR_CASA_SAIR_FAZER_COISAS_NOVAS_NAO_GDS_15_1': 0,  
	'PREFERE_FICAR_CASA_SAIR_FAZER_COISAS_NOVAS_SIM_GDS_15_1': 1,
	'ACHA_MAIS_PROBLEMAS_MEMORIA_OUTRAS_PESSOAS_NAO_GDS_15_1': 0,  
	'ACHA_MAIS_PROBLEMAS_MEMORIA_OUTRAS_PESSOAS_SIM_GDS_15_1': 1,
	'ACHA_MARAVILHOSO_VIVO_NAO_GDS_15_1': 0,  
	'ACHA_MARAVILHOSO_VIVO_SIM_GDS_15_1': 1,
	'SENTE_SE_INUTIL_NAO_GDS_15_1': 0,  
	'SENTE_SE_INUTIL_SIM_GDS_15_1': 1,
	'SENTE_SE_CHEIO_DE_ENERGIA_NAO_GDS_15_1': 0,  
	'SENTE_SE_CHEIO_DE_ENERGIA_SIM_GDS_15_1': 1,
	'SENTE_SE_SEM_ESPERANCA_NAO_GDS_15_1': 0,  
	'SENTE_SE_SEM_ESPERANCA_SIM_GDS_15_1': 1,
	'ACHA_OUTROS_MAIS_SORTE_VOCE_NAO_GDS_15_1': 0,  
	'ACHA_OUTROS_MAIS_SORTE_VOCE_SIM_GDS_15_1': 1,
	

    # --- KCCQ-12 ---
    'TOMAR_BANHO_LIMITOU_MUITISSIMO_1': 1,
    'TOMAR_BANHO_LIMITOU_BASTANTE_1': 2,
    'TOMAR_BANHO_LIMITOU_UM_POUCO_1': 3,
    'TOMAR_BANHO_LIMITOU_MUITO_POUCO_1': 4,
    #'TOMAR_BANHO_NAO_LIMITOU_1': 5,
    'TOMAR_BANHO_NAO_LIMITOU_2': 5,
    'TOMAR_BANHO_FUI_LIMITADO_OUTRAS_RAZOES_1': 0, 
	'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_FUI_LIMITADO_OUTRAS_RAZOES_NAO_FIZ_TAL_ATIVIDADE_1': 0,  
	'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_BASTANTE_1': 2,
 	'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_MUITISSIMO_1': 1,  
	'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_MUITO_POUCO_1': 4,
 	'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_LIMITOU_UM_POUCO_1': 3,  
	'CAMINHAR_QUARTEIRAO_TERRENO_PLANO_NAO_LIMITOU_1': 5,
	'CORRER_ANDAR_APRESSADAMENTE_FUI_LIMITADO_POR_OUTRAS_RAZOES_1': 0,  
	'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_BASTANTE_1': 2,
 	'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_MUITISSIMO_1': 1,  
	'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_MUITO_POUCO_1': 4,
 	'CORRER_ANDAR_APRESSADAMENTE_LIMITOU_UM_POUCO_1': 3,  
	'CORRER_ANDAR_APRESSADAMENTE_NAO_LIMITOU_1': 5,
	'1_2_VEZES_POR_SEMANA_QUESTAO_2_1': 3, 
	'3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODOS_OS_DIAS_QUESTAO_2_1': 2, 
	'MENOS_DE_1_VEZ_POR_SEMANA_QUASTAO_2_1': 4,
    'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_2_1': 5, 
	'TODAS_AS_MANHAS_QUESTAO_2_1': 1,
	'1_2_VEZES_POR_SEMANA_QUESTAO_3_1': 5, 
	'3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODOS_OS_DIAS_QUESTAO_3_1': 4, 
	'MENOS_DE_1_POR_SEMANA_QUESTAO_3_1': 6,
    'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_3_1': 7, 
	'O_TEMPO_TODO_QUESTAO_3_1': 1, 
	'PELO_MENOS_1_VEZES_POR_DIA_QUESTAO_3_1': 3, 
	'VARIA_VEZES_POR_DIA_QUESTAO_3_1': 2,
	'1_2_VEZES_POR_SEMANA_QUESTAO_4_1': 5, 
	'3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODOS_OS_DIAS_QUESTAO_4_1': 4, 
	'MENOS_DE_1_POR_SEMANA_QUESTAO_4_1': 6, 
    'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_4_1': 7, 
	'O_TEMPO_TODO_QUESTAO_4_1': 1, 
	'PELO_MENOS_1_VEZES_POR_DIA_QUESTAO_4_1': 3, 
	'VARIA_VEZES_POR_DIA_QUESTAO_4_1': 2,
	'1_2_VEZES_POR_SEMANA_QUESTAO_5_1': 3,  
	'3_OU_MAIS_VEZES_POR_SEMANA_POREM_NAO_TODAS_AS_NOITES_QUESTAO_5_1': 2, 
	'MENOS_DE_1_POR_SEMANA_QUESTAO_5_1': 4,
    'NENHUMA_VEZ_NAS_2_ULTIMAS_SEMANAS_QUESTAO_5_1': 5, 
	'O_TEMPO_TODOS_QUESTAO_5_1': 1,
	'IMPEDIU_BASTANTE_QUESTAO_6_1': 2,  
	'IMPEDIU_MUITISSIMO_QUESTAO_6_1': 1,  
	'IMPEDIU_MUITO_POUCO_QUESTAO_6_1': 4,  
	'IMPEDIU_UM_POUCO_QUESTAO_6_1': 3, 
    'NAO_IMPEDIU_EM_NADA_QUESTAO_6_1': 5,
	'BASTANTE_INSATISFEITO_QUESTAO_7_1': 2,  
	'BASTANTE_SATISFEITO_QUESTAO_7_1': 4, 
	'TOTALMENTE_INSATISFEITO_QUESTAO_7_1': 1,  
	'TOTALMENTE_SATISFEITO_QUESTAO_7_1': 5,
    'UM_POUCO_SATISFEITO_QUESTAO_7_1': 3,
	'PASSATEMPOS_ATIVIDADES_RECREATIVAS_FUI_LIMITADO_POR_OUTROS_RAZOES_1': 0,  
	'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_BASTANTE_1': 2,
    'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_MUITISSIMO_1': 1,  
	'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_MUITO_POUCO_1': 4,
    'PASSATEMPOS_ATIVIDADES_RECREATIVAS_LIMITOU_UM_POUCO_1': 3,  
	'PASSATEMPOS_ATIVIDADES_RECREATIVAS_NAO_LIMITOU_1': 5,
	'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_FUI_LIMITADO_POR_OUTRAS_RAZOES_1': 0,  
	'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_BASTANTE_1': 2,
    'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_MUITISSIMO_1': 1,  
	'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_MUITO_POUCO_1': 4,
    'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_LIMITOU_UM_POUCO_1': 3,  
	'TRABALHAR_FAZER_TAREFAS_DOMESTICAS_NAO_LIMITOU_1': 5,
	'VISITAR_PARENTES_OU_AMIGOS_FUI_LIMITADO_POR_OUTRAS_RAZOES_1': 0,  
	'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_BASTANTE_1': 2,
    'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_MUITISSIMO_1': 1,  
	'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_MUITO_POUCO_1': 4,  
	'VISITAR_PARENTES_OU_AMIGOS_LIMITOU_UM_POUCO_1': 3,
    'VISITAR_PARENTES_OU_AMIGOS_NAO_LIMITOU_1': 5,
	

	# --- EQ5D ---
	'NAO_TENHO_PROBLEMAS_ANDAR_MOBILIDADE_EQ_5D_1': 1, 
	'TENHO_ALGUNS_PROBLEMAS_ANDAR_MOBILIDADE_EQ_5D_1': 2,
	'TENHO_DE_ESTAR_NA_CAMA_MOBILIDADE_EQ_5D_1': 3,
	'NAO_TENHO_PROBLEMAS_EM_CUIDAR_DE_MIM_CUIDADOS_PESSOAIS_EQ_5D_1': 1, 
	'TENHO_ALGUNS_PROBLEMAS_A_LAVAR_ME_OU_VESTIR_ME_CUIDADOS_PESSOAIS_EQ_5D_1': 2,
	'SOU_INCAPAZ_DE_ME_LAVAR_OU_VESTIR_SOZINHO_CUIDADOS_PESSOAIS_EQ_5D_1': 3,
	'NAO_TENHO_PROBLEMAS_EM_DESEMPENHAR_AS_MINHAS_ATIVIDADES_HABITUAIS_ATIVIDADES_HABITUAIS_5Q_5D_1': 1, 
	'TENHO_ALGUNS_PROBLEMAS_EM_DESEMPENHAR_AS_MINHAS_ATIVIDADES_HABITUAIS_ATIVIDADES_HABITUAIS_5Q_5D_1': 2,
	'SOU_INCAPAZ_DE_DESEMPENHAR_AS_MINHAS_ATIVIDADES_HABITUAIS_ATIVIDADES_HABITUAIS_5Q_5D_1': 3,
	'NAO_TENHO_DORES_OU_MAL_ESTAR_DOR_MAL_ESTAR_EQ_5D_1': 1, 
	'TENHO_DORES_OU_MAL_ESTAR_EXTREMOS_DOR_MAL_ESTAR__1': 3,
	'TENHO_DORES_OU_MAL_ESTAR_MODERADOS_DOR_MAL_ESTAR_5Q_5D_1': 2,
	'NAO_ESTOU_ANSIOSO_OU_DEPRIMIDO_ANSIEDADE_DEPRESSAO_EQ_5D_1': 1, 
	'ESTOU_EXTREMAMENTE_ANSIOSO_OU_DEPRIMIDO_ANSIEDADE_DEPRESSAO_EQ_5D_1': 3,
	'ESTOU_MODERAMENTE_ANSIOSO_OU_DEPRIMIDO_ANSIEDADE_DEPRESSAO_EQ_5D_1': 2,
	'MELHOR_COMPARACAO_COM_O_ULTIMO_ANO_1': 1, 
	'O_MESMO_COMPARACAO_COM_O_ULTIMO_ANO_1': 2, 
	'PIOR_COMPARACAO_COM_O_ULTIMO_ANO_1': 3,


}


In [229]:
dGer.columns

Index(['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID',
       'CD_DOCUMENTO', 'DS_DOCUMENTO', 'DS_LEITO', 'DS_IDENTIFICADOR',
       'DS_CAMPO', 'DS_RESPOSTA', 'CD_PRESTADOR', 'NM_PRESTADOR',
       'nome_paciente', 'tp_situacao', 'genero', 'etnia', 'data_nascimento',
       'nr_ddd_fone', 'nr_fone_1', 'nr_ddd_celular', 'nr_celular',
       'nr_ddd_fone_comercial', 'nr_fone_comercial', 'NM_SETOR'],
      dtype='object')

In [230]:
len(geronto_ls) #66

66

In [231]:
len(geronto_rn) #204

204

In [232]:
# 1. Lista de identificadores já mapeados
ident_tratados = [item for sublist in geronto_ls.values() for item in sublist]

# 2. Cria colunas para os identificadores mapeados
for key in tqdm(geronto_ls.keys(), position=0, leave=False):
    dGer[key] = None
    for item in geronto_ls[key]:
        with ipython_io.capture_output() as captured:
        	dGer[key][dGer['DS_IDENTIFICADOR'] == item] = item

# 3. Filtra os identificadores não tratados
nao_tratados = dGer[~dGer['DS_IDENTIFICADOR'].isin(ident_tratados)]

nao_tratados['DS_RESPOSTA'] = nao_tratados['DS_RESPOSTA'].apply(
    lambda x: 'SIM' if isinstance(x, str) and x.strip().lower() == 'true'
    else 'NAO' if isinstance(x, str) and x.strip().lower() == 'false'
    else x
)

# 4. Pivotando os identificadores não tratados
pivot_extra = nao_tratados.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    columns='DS_IDENTIFICADOR',
    values='DS_RESPOSTA',
    aggfunc='first'  # ou 'last', dependendo da lógica de negócio
).reset_index()

# 5. Junta com os dados principais (já com colunas criadas manualmente)
dGer1 = pd.merge(
    dGer.drop(columns=['DS_IDENTIFICADOR', 'DS_RESPOSTA', 'DS_CAMPO']),
    pivot_extra,
    on=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    how='left'
)

# 6. Substituir os valores conforme geronto_rn (string → string)
for column in dGer1.columns:
    if column not in ['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID',
       'CD_DOCUMENTO', 'DS_DOCUMENTO', 'DS_LEITO', 'nome_paciente', 'tp_situacao', 'genero', 'etnia', 'data_nascimento',
       'nr_ddd_fone', 'nr_fone_1', 'nr_ddd_celular', 'nr_celular', 'nr_ddd_fone_comercial', 'nr_fone_comercial']:
        dGer1[column].replace(geronto_rn, inplace=True)


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\881821607.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nao_tratados['DS_RESPOSTA'] = nao_tratados['DS_RESPOSTA'].apply(
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\881821607.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace

In [233]:
len(dGer1.columns.tolist()) #144

144

In [234]:
dGer2 = dGer1.rename(
    columns={
    'CD_PACIENTE': 'record_id', 

	# --- LF ---
	'CRITERIOS_FRAGILIDADE_LINDA_FRIED_1': 'lf_criterios_presentes',

	# --- SPPB ---
	'EQUILIBRIO_SPPB_1': 'sppb_equilibrio',
	'MARCHA_TEMPO_SPPB_1': 'sppb_marcha_tempo',
	'MARCHA_SPPB_1': 'sppb_marcha_pontos',
	'LEVANTAR_SENTAR_CADEIRA_TEMPO_SPPB_1': 'sppb_cadeira_tempo',
	'LEVANTAR_SENTAR_CADEIRA_SPPB_1': 'sppb_cadeira_pontos',
	'SPPB_SHORT_PHYSICAL_PERFORMANCE_BATTERY_1': 'sppb_escore_total',

	# --- BARTHEL ---
	'ATIVIDADE_BASICAS_VIDA_DIARIA_AVD_BARTHEL_1': 'barthel_score_total',
	
	# --- LAWTON ---
	'ATIVIDADES_INSTRUMENTAIS_VIDA_DIARIA_AIVD_LAWTON_1': 'lawton_score_total',

	# --- SARCF ---
	'SARC_F_RISCO__1': 'sarcf_score_total',

	# --- APGAR FAMILIAR ---
	'APGAR_FAMILIAR_1': 'apgar_score_total',

	# --- GDS-15 ---
	'ESCALA_DEPRESSAO_GERIATRICA_GDS_15_1': 'gds15_score_total',

	# --- KCCQ-12 ---
	'RESULTADO_KCCQ_12_1': 'kccq_score_resumo_geral',
    
	# --- EQ5D ---
	'DE_0_ATE_100_QUANTO_VOCE_AVALIA_SAUDE_HOJE_EQ5D_1': 'eq5d_escala_estado_saude',
    
	# --- MEEM ---
	'MINI_EXAME_ESTADO_MENTAL_MEEM_1': 'meem_score_total',
	'QUAL_DIA_SEMANA_ORIENTACAO_ESPACIAL_MEEM_1': 'meem_weekday',
	'QUAL_MES_ORIENTACAO_ESPACIAL_MEEM_1': 'meem_month',
	'QUAL_DIA_MES_ORIENTACAO_ESPACIAL_MEEM_1': 'meem_monthday',
	'QUAL_ANO_ORIENTACAO_ESPACIAL_MEEM_1': 'meem_year',
	'QUAL_HORA_APROXIMADA_ORIENTACAO_ESPACIAL_MEEM_1': 'meem_hour',
	'ONDE_ESTAMOS_LOCAL_ORIENTACAO_TEMPORAL_ESPACIAL_MEEM_1': 'meem_local',
	'ONDE_ESTAMOS_INSTITUICAO_ORIENTACAO_TEMPORAL_ESPACIAL_MEEM_1': 'meem_instituicao',
	'ONDE_ESTAMOS_BAIRRO_ORIENTACAO_TEMPORAL_ESPACIAL_MEEM_1': 'meem_bairro',
	'ONDE_ESTAMOS_CIDADE_ORIENTACAO_TEMPORAL_ESPACIAL_MEEM_1': 'meem_cidade',
	'ONDE_ESTAMOS_ESTADO_ORIENTACAO_TEMPORAL_ESPACIAL_MEEM_1': 'meem_estado',
	'MENCIONE_3_PALAVRAS_MEEM_1': 'meem_reg_word_1',
	'MENCIONE_3_PALAVRAS_MEEM_2': 'meem_reg_word_2',
	'MENCIONE_3_PALAVRAS_MEEM_3': 'meem_reg_word_3',
	'SETE_SERIADO_ATENCAO_CALCULO_MEEM_1': 'meem_calculo',
	'PERGUNTE_NOME_3_PALAVRAS_APRENDIDOS_MEEM_1': 'meem_evoc_word_1',
	'PERGUNTE_NOME_3_PALAVRAS_APRENDIDOS_MEEM_2': 'meem_evoc_word_2',
	'PERGUNTE_NOME_3_PALAVRAS_APRENDIDOS_MEEM_3': 'meem_evoc_word_3',
	'APONTE_LAPIS_RELOGIO_FACA_PACIENTE_DIZER_NOME_OBJETOS_CONFORME_APONTA_LIGUAGEM_MEEM_1': 'meem_objeto',
	'FACA_PACIENTE_REPETIR_LIGUAGEM_MEEM_1': 'meem_repeat_frase',
	'FACA_PACIENTE_SEGUIR_COMANDO_LINGUAGEM_MEEM_1': 'meem_comando_verbal',
	'FACA_PACIENTE_LER_OBEDECER_LINGUAGEM_MEEM_1': 'meem_ler_obedecer',
	'FACA_PACIENTE_ESCREVER_FRASE_PROPRIA_AUTORIA_LINGUAGEM_MEEM_1': 'meem_escrever_frase',
	'RESULTADO_COPIE_DESENHO_ABAIXO_MEEM_1': 'meem_copiar_desenho',
    
	# --- OUTRAS COLUNAS ---
	'ALTURA_MAPAR_1': 'altura',
    'BRANQUIAL_CM_1': 'circ_braquial_cm',
    'CINTURA_CM_1': 'circ_abdomen_cm',
    'PANTURRILHA_CM_1': 'circ_panturilha_cm',
    'PESO_MAPA_1': 'peso',
    'TC6M_TESTE_CAMINHADA_SEIS_MUNITOS_1': 'caminhada_metro',
    'METROS_POR_SEGUNDOS_1': 'caminhada_6min',
    'OBSERVACAO_AVALIACAO_GERONTOLOGICA_1':'final_observacao'

	}
)



In [ ]:
#dGer2 = dGer2[dGer2['record_id'].astype('Int64').isin(geronto_a_processar)]

In [235]:
# Organizando o df em linhas unicas por pacientes
# Definir a ordem das colunas desejadas
cols_df = [
        'CD_ATENDIMENTO', 'record_id', 'DH_DOCUMENTO', 
        'nome_paciente', 'tp_situacao', 'genero', 'etnia', 'data_nascimento', 'nr_ddd_fone', 'nr_fone_1', 'nr_ddd_celular', 'nr_celular', 'nr_ddd_fone_comercial', 'nr_fone_comercial',
        'DATAID', 'CD_DOCUMENTO', 'DS_DOCUMENTO', 'DS_LEITO', 'CD_PRESTADOR', 'NM_PRESTADOR', 'NM_SETOR',
        'lf_perda_peso', 'lf_fadiga', 'lf_dim_forca', 'lf_lentidao_marcha', 'lf_baixa_ativ_fisica', 'barthel_alimentacao', 'barthel_banho', 'barthel_hig_pessoal',
        'barthel_vestir', 'barthel_intestinos', 'barthel_bexiga', 'barthel_uso_vaso', 'barthel_transfer', 'barthel_mobilidade', 'barthel_escadas', 'lawton_telefone',
        'lawton_compras', 'lawton_refeicoes', 'lawton_casa', 'lawton_lavar_roupa', 'lawton_transporte', 'lawton_medicamentos', 'lawton_financas', 'sarcf_forca', 'sarcf_caminho',
        'sarcf_levanta_cadeira', 'sarcf_escadas', 'sarcf_queda', 'apgar_ajuda', 'apgar_discute_problemas', 'apgar_apoia_desejos', 'apgar_responde_emocoes', 'apgar_tempo_juntos',
        'gds15_1_satisfeito_vida', 'gds15_2_abandono_atividade', 'gds15_3_vida_vazia', 'gds15_4_aborrecido_freq', 'gds15_5_animado_humor', 'gds15_6_medo_ruim', 'gds15_7_feliz_todo',
        'gds15_8_desamparado_freq', 'gds15_9_prefere_casa', 'gds15_10_problema_memoria', 'gds15_11_maravilha_vivo', 'gds15_12_inutil_semvalor', 'gds15_13_energizado',
        'gds15_14_sem_esperanca', 'gds15_15_pessoas_melhores', 'kccq_1a_banho', 'kccq_1b_caminho', 'kcc1_1c_correr', 'tb_kccq_2', 'tb_kccq_3', 'tb_kccq_4', 'tb_kccq_5', 'tb_kccq_6',
        'tb_kccq_7', 'kccq_8a_passatempos', 'kccq_8b_tarefas', 'kccq_8c_visitas', 'eq5d_mobilidade', 'eq5d_cuidados_pessoais', 'eq5d_atividades_habituais', 'eq5d_dor', 'eq5d_ansiedade',
        'eq5d_comparacao_12meses', 'altura', 'apgar_score_total', 'meem_objeto', 'lawton_score_total', 'barthel_score_total', 'circ_braquial_cm', 'circ_abdomen_cm', 'lf_criterios_presentes',
        'eq5d_escala_estado_saude', 'sppb_equilibrio', 'gds15_score_total', 'meem_escrever_frase', 'meem_ler_obedecer', 'meem_repeat_frase', 'meem_comando_verbal', 'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_1',
        'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_2', 'IMC_AVALIACAO_GERONTOLOGICA_1', 'INTERPRETACAO_AVD_BARTHEL_1', 'sppb_cadeira_pontos', 'sppb_cadeira_tempo', 'sppb_marcha_pontos', 'sppb_marcha_tempo',
        'meem_reg_word_1', 'meem_reg_word_2', 'meem_reg_word_3', 'caminhada_6min', 'meem_score_total', 'final_observacao', 'meem_bairro', 'meem_cidade', 'meem_estado', 'meem_instituicao',
        'meem_local', 'circ_panturilha_cm', 'meem_evoc_word_1', 'meem_evoc_word_2', 'meem_evoc_word_3', 'peso', 'QUADRILATERO_1', 'meem_year', 'meem_monthday', 'meem_weekday', 'meem_hour',
        'meem_month', 'meem_copiar_desenho', 'kccq_score_resumo_geral', 'sarcf_score_total', 'meem_calculo', 'sppb_escore_total', 'caminhada_metro', 'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_1',
        'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_2'
    ]

# Aplicar a função ao DataFrame Comorb
dGer3 = organizar_dados_paciente(
    dGer2,
    chaves_agrupamento=['CD_ATENDIMENTO', 'record_id', 'DH_DOCUMENTO'],
    colunas_desejadas=cols_df[3:]
)

C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\3918263019.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_organizado = df.groupby(todas_chaves, dropna=False, as_index=False).first()
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\3918263019.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_organizado = df.groupby(todas_chaves, dropna=False, as_index=False).first()
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\3918263019.py:39: PerformanceWarning: DataFrame is highly f

In [236]:
# --- DEFINIÇÃO DE COLUNAS ---

# 1. Colunas numéricas/mistas que precisam trocar vírgula por ponto
# Incluímos 'final_observacao' aqui, o que é correto para padronizar
cols_para_ajustar = [
    'peso', 'altura', 'circ_braquial_cm', 'circ_abdomen_cm', 
    'meem_repeat_frase', 'sppb_cadeira_tempo', 'sppb_marcha_tempo',  
    'caminhada_6min',  'circ_panturilha_cm', 'meem_copiar_desenho', 
    'caminhada_metro', 'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_1', 
    'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_2', 'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_1',
    'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_2', 'final_observacao'
]

# 2. Colunas que compõem o texto final (Obs)
# AJUSTE: Adicionei 'final_observacao' aqui para ela passar pela limpeza
cols_obs = [
    'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_1', 
    'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_2',
    'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_1',
    'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_2',
    'final_observacao' 
]

# --- PASSO 1: Tratamento Inicial (Vírgula para Ponto) ---
# O .astype(str) converte nulos para "nan". Isso evita erro de tipo, mas suja o texto.
dGer3[cols_para_ajustar] = dGer3[cols_para_ajustar].astype(str).apply(lambda x: x.str.replace(',', '.'))


# --- PASSO 2: Cálculos Numéricos (Força) ---
cols_forca = [
    'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_1', 
    'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_2'
]
# Calculamos o máximo convertendo para numérico temporariamente (errors='coerce' ignora textos)
dGer3['preensao_palmar'] = dGer3[cols_forca].apply(pd.to_numeric, errors='coerce').max(axis=1)


# --- PASSO 3: Limpeza de Texto (CRÍTICO) ---
# Agora limpamos as strings "nan", "0", "0.0" geradas pelo passo 1 ou importação
for col in cols_obs:
    if col in dGer3.columns:
        # 1. Substituir valores indesejados por string vazia
        # Removemos 'nan' (string), 'None', '0' e '0.0' para o texto ficar limpo
        dGer3[col] = dGer3[col].replace(['nan', 'None', '0', '0.0', 'nan'], '')
        
        # 2. Remover ".0" final (ex: "Texto da obs.0" -> "Texto da obs")
        dGer3[col] = dGer3[col].str.replace(r'\.0$', '', regex=True)
        
        # 3. Trim (remove espaços extras nas pontas)
        dGer3[col] = dGer3[col].str.strip()
    else:
        dGer3[col] = ''

# --- PASSO 4: Concatenação Segura ---
# Agora garantimos que tudo é string limpa, sem "nan" e sem "0"
dGer3['lf_observacoes'] = (
    "Força: " + dGer3['FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_1'] + "/" + dGer3['FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_2'] + 
    "\n Velocidade: " + dGer3['VELOCIDADE_MARCHA_LENTA_OBSERVACAO_1'] + "/" + dGer3['VELOCIDADE_MARCHA_LENTA_OBSERVACAO_2'] +
    "\n Observações: " + dGer3['final_observacao']
)

C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2902923072.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dGer3['preensao_palmar'] = dGer3[cols_forca].apply(pd.to_numeric, errors='coerce').max(axis=1)
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2902923072.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dGer3['lf_observacoes'] = (


In [237]:
dGer4 = vincular_instancias_temporal(
    df_principal=dGer3,
    df_referencia=regvalv_cirurgia[['record_id', 'redcap_repeat_instance', 'data_cirurgia']],
    col_data_principal='DH_DOCUMENTO',    # Data do documento
    col_data_referencia='data_cirurgia',  # Data da cirurgia
    tolerancia_dias=180
)

print(dGer4['record_id'].nunique())

155


In [ ]:
# 155 pacientes

In [238]:
# 3. Executando a classificação
dGer4 = classificar_eventos_redcap(
    df_entrada=dGer4,
    evento_antes='fase1preoperatorio_arm_1',
    evento_depois='fase6segalta30dias_arm_1'
)

dGer4['redcap_repeat_instrument'] = 'qualidade_vida'

In [239]:
# Se o tempo não for vazio, marca 1, caso contrário 0
dGer4['teste_de_marcha_4m_realizado'] = np.where(dGer4['sppb_marcha_tempo'] != "", 1, 0)

dGer4['teste_da_cadeira_realizado'] = np.where(dGer4['sppb_cadeira_tempo'] != "", 1, 0)

In [240]:
# Remove diretamente no dataframe original (sem precisar atribuir a 'df = ...')
dGer4.drop(columns=['CD_ATENDIMENTO', 'DH_DOCUMENTO', 'DATAID',
 'CD_DOCUMENTO',
 'DS_DOCUMENTO',
 'DS_LEITO',
 'CD_PRESTADOR',
 'NM_PRESTADOR',
 'NM_SETOR', 'apgar_score_total', 'lawton_score_total',
 'barthel_score_total', 'lf_criterios_presentes',
 'gds15_score_total', 'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_1',
 'FORCA_MUSCULAR_REDUZIDA_OBSERVACAO_2',
 'IMC_AVALIACAO_GERONTOLOGICA_1',
 'INTERPRETACAO_AVD_BARTHEL_1',
 'sppb_cadeira_pontos', 'sppb_marcha_pontos', 'meem_score_total', 'QUADRILATERO_1', 'kccq_score_resumo_geral',
 'sarcf_score_total','sppb_escore_total', 'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_1',
 'VELOCIDADE_MARCHA_LENTA_OBSERVACAO_2', 'data_cirurgia', 'final_observacao'] , inplace=True)

In [241]:
# =====================================================
# --- 1. PREPARAÇÃO DOS DADOS DO REDCAP (EXISTENTES) ---

# Garantir que record_id seja string em ambos os lados para evitar erros de comparação ( "100" != 100 )
regvalv_geronto['record_id'] = regvalv_geronto['record_id'].astype(str)

# Criamos um "Set" de tuplas com (ID, EVENTO). 
# Isso funciona como uma lista de presença: "O paciente X já tem ficha no evento Y"
geronto_existentes = set(zip(
    regvalv_geronto['record_id'].astype(str), 
    regvalv_geronto['redcap_event_name'].astype(str),
    regvalv_geronto['redcap_repeat_instance'].astype(str)
    ))

print(f"Total de registros já existentes no Redcap: {len(geronto_existentes)}")


# --- 2. FILTRANDO OS DADOS LOCAIS ---

# Filtra cada fase
dGer4 = filtrar_novos_registros(dGer4, geronto_existentes)


# Exibe o resultado
print('Qtde de pctes únicos: ', dGer4['record_id'].nunique())
print('Qtde de colunas: ', len(dGer4.columns))
print('Colunas dGer4: ', dGer4.columns.tolist())
print('Registros: ', dGer4['record_id'].unique())


Total de registros já existentes no Redcap: 745
📊 Filtro aplicado:
   - Registros originais no DataFrame: 157
   - Registros ignorados (já existem no REDCap): 156
   - Novos registros para importar: 1
Qtde de pctes únicos:  1
Qtde de colunas:  121
Colunas dGer4:  ['record_id', 'altura', 'apgar_ajuda', 'apgar_apoia_desejos', 'apgar_discute_problemas', 'apgar_responde_emocoes', 'apgar_tempo_juntos', 'barthel_alimentacao', 'barthel_banho', 'barthel_bexiga', 'barthel_escadas', 'barthel_hig_pessoal', 'barthel_intestinos', 'barthel_mobilidade', 'barthel_transfer', 'barthel_uso_vaso', 'barthel_vestir', 'caminhada_6min', 'caminhada_metro', 'circ_abdomen_cm', 'circ_braquial_cm', 'circ_panturilha_cm', 'data_nascimento', 'eq5d_ansiedade', 'eq5d_atividades_habituais', 'eq5d_comparacao_12meses', 'eq5d_cuidados_pessoais', 'eq5d_dor', 'eq5d_escala_estado_saude', 'eq5d_mobilidade', 'etnia', 'gds15_10_problema_memoria', 'gds15_11_maravilha_vivo', 'gds15_12_inutil_semvalor', 'gds15_13_energizado', 'gds1

In [242]:
dGer5 = dGer4[['record_id',
 'lf_perda_peso',
 'lf_fadiga',
 'lf_dim_forca',
 'lf_lentidao_marcha',
 'lf_baixa_ativ_fisica',
 'barthel_alimentacao',
 'barthel_banho',
 'barthel_hig_pessoal',
 'barthel_vestir',
 'barthel_intestinos',
 'barthel_bexiga',
 'barthel_uso_vaso',
 'barthel_transfer',
 'barthel_mobilidade',
 'barthel_escadas',
 'lawton_telefone',
 'lawton_compras',
 'lawton_refeicoes',
 'lawton_casa',
 'lawton_lavar_roupa',
 'lawton_transporte',
 'lawton_medicamentos',
 'lawton_financas',
 'sarcf_forca',
 'sarcf_caminho',
 'sarcf_levanta_cadeira',
 'sarcf_escadas',
 'sarcf_queda',
 'apgar_ajuda',
 'apgar_discute_problemas',
 'apgar_apoia_desejos',
 'apgar_responde_emocoes',
 'apgar_tempo_juntos',
 'gds15_1_satisfeito_vida',
 'gds15_2_abandono_atividade',
 'gds15_3_vida_vazia',
 'gds15_4_aborrecido_freq',
 'gds15_5_animado_humor',
 'gds15_6_medo_ruim',
 'gds15_7_feliz_todo',
 'gds15_8_desamparado_freq',
 'gds15_9_prefere_casa',
 'gds15_10_problema_memoria',
 'gds15_11_maravilha_vivo',
 'gds15_12_inutil_semvalor',
 'gds15_13_energizado',
 'gds15_14_sem_esperanca',
 'gds15_15_pessoas_melhores',
 'kccq_1a_banho',
 'kccq_1b_caminho',
 'kcc1_1c_correr',
 'tb_kccq_2',
 'tb_kccq_3',
 'tb_kccq_4',
 'tb_kccq_5',
 'tb_kccq_6',
 'tb_kccq_7',
 'kccq_8a_passatempos',
 'kccq_8b_tarefas',
 'kccq_8c_visitas',
 'eq5d_mobilidade',
 'eq5d_cuidados_pessoais',
 'eq5d_atividades_habituais',
 'eq5d_dor',
 'eq5d_ansiedade',
 'eq5d_comparacao_12meses',
 'meem_objeto',
 'circ_braquial_cm',
 'circ_abdomen_cm',
 'eq5d_escala_estado_saude',
 'meem_escrever_frase',
 'meem_ler_obedecer',
 'meem_repeat_frase',
 'meem_comando_verbal',
 'sppb_cadeira_tempo',
 'sppb_marcha_tempo',
 'sppb_equilibrio',
 'meem_reg_word_1',
 'meem_reg_word_2',
 'meem_reg_word_3',
 'caminhada_6min',
 'meem_bairro',
 'meem_cidade',
 'meem_estado',
 'meem_instituicao',
 'meem_local',
 'circ_panturilha_cm',
 'meem_evoc_word_1',
 'meem_evoc_word_2',
 'meem_evoc_word_3',
 'meem_year',
 'meem_monthday',
 'meem_weekday',
 'meem_hour',
 'meem_month',
 'meem_copiar_desenho',
 'meem_calculo',
 'caminhada_metro',
 'preensao_palmar',
 'lf_observacoes',
 'teste_de_marcha_4m_realizado',
 'teste_da_cadeira_realizado',
 'redcap_repeat_instance',
 'redcap_event_name',
 'redcap_repeat_instrument']]

In [243]:
dPcte = dGer4[['record_id',
 'nome_paciente',
 'genero',
 'etnia',
 'data_nascimento',
 'nr_ddd_fone',
 'nr_fone_1',
 'nr_ddd_celular',
 'nr_celular',
 'nr_ddd_fone_comercial',
 'nr_fone_comercial',
 ]]

In [244]:
dHist = dGer4[['record_id', 'redcap_repeat_instance', 
            'peso', 'altura']]

In [245]:
dHist['redcap_event_name'] = 'fase1preoperatorio_arm_1'
#dHist['redcap_repeat_instance'] = 1
dHist['redcap_repeat_instrument'] = 'fatores_risco'

C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\969659657.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dHist['redcap_event_name'] = 'fase1preoperatorio_arm_1'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\969659657.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dHist['redcap_repeat_instrument'] = 'fatores_risco'


In [246]:
# 1. Definição do Evento
dPcte['redcap_event_name'] = 'fase1preoperatorio_arm_1'

# 2. Mapeamento de Gênero e Etnia
mapa = {'MASCULINO': 1, 'FEMININO': 2, 'AMARELA': 4, 'BRANCA': 1, 'NEGRA': 2, 'PARDA': 3}

# Dica: Verifique se 'genero' e 'etnia' não estão nulos antes do map, 
# senão fillna(3) vai preencher tudo que não casou com 3.
dPcte['genero'] = dPcte['genero'].map(mapa).fillna('').astype(int)
dPcte['etnia'] = dPcte['etnia'].map(mapa).fillna('').astype(int)

# 3. Conversão Segura dos Telefones
cols_telefones = [
    'nr_ddd_fone', 
    'nr_fone_1', 
    'nr_ddd_celular', 
    'nr_celular', 
    'nr_ddd_fone_comercial', 
    'nr_fone_comercial'
]

for col in cols_telefones:
    if col in dPcte.columns:
        # A. (Opcional) Remove caracteres comuns de telefone se existirem, como (-) ou ( )
        # Se os dados já vierem limpos (apenas dígitos), pode remover essa linha 'A'
        dPcte[col] = dPcte[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        
        # B. Converte para Numérico
        # 'coerce' transforma vazios, erros e textos em NaN
        dPcte[col] = pd.to_numeric(dPcte[col], errors='coerce')
        
        # C. Arredonda e Converte para Int64
        # .round() garante que 9999.0 vire 9999 antes de virar inteiro
        # 'Int64' (com maiúscula) aceita NaN (o 'int' normal quebraria com NaN)
        dPcte[col] = dPcte[col].round().astype('Int64')

# Cria a data e já formata como texto (dia/mes/ano)
dPcte['data_inclusao'] = pd.to_datetime('today').strftime('%d/%m/%Y')


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\4110686166.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dPcte['redcap_event_name'] = 'fase1preoperatorio_arm_1'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\4110686166.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dPcte['genero'] = dPcte['genero'].map(mapa).fillna('').astype(int)
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\4110686166.py:10: SettingWithCopyWarning: 
A value is trying to be set on 

In [247]:
# Estas colunas NÃO serão arredondadas nem transformadas em Inteiros
colunas_medidas_decimais = [
    'altura', 
    'circ_braquial_cm', 
    'circ_abdomen_cm', 
    'meem_escrever_frase', 
    'meem_repeat_frase', 
    'sppb_cadeira_tempo', 
    'sppb_marcha_tempo', 
    'caminhada_6min', 
    'circ_panturilha_cm', 
    'peso', 
    'meem_copiar_desenho', 
    'caminhada_metro', 
    'preensao_palmar'
]

# Certifique-se de que 'colunas_medidas_decimais' está definida no seu código
dGer5 = limpar_dados_para_importacao(
	dGer5, 
	cols_decimais=colunas_medidas_decimais
)

dGer5['qualidade_vida_complete'] = 2


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\3260891506.py:67: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2023128435.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dGer5['qualidade_vida_complete'] = 2


In [248]:
dPcte.columns.to_list()

['record_id',
 'nome_paciente',
 'genero',
 'etnia',
 'data_nascimento',
 'nr_ddd_fone',
 'nr_fone_1',
 'nr_ddd_celular',
 'nr_celular',
 'nr_ddd_fone_comercial',
 'nr_fone_comercial',
 'redcap_event_name',
 'data_inclusao']

In [ ]:
#dHist.to_excel(f"C://Users/priscilla.sequetin/Downloads/dHist.xlsx", index=False)

In [249]:
# Verifica se sobrou alguém para importar (Checagem Inicial)
if dGer5 is not None and not dGer5.empty:
    print(f"--- INICIANDO ANÁLISE DE {len(dGer5)} REGISTROS ---")
    print(f"Novos registros reais para importar: {len(dGer5)}")

    # Executa a importação USANDO O DATAFRAME FILTRADO
    try:
        # Atribuímos o retorno à variável results_Geronto (seu batch_error)
        results_Geronto = import_redcap(
            df_sorted=dGer5,     
            api_url=api_url,
            api_key=api_key,
            batch_size=10,
            overwrite=False
        )
        
        # 1. Tratamento de erros de registros bloqueados (Locked)
        if results_Geronto:
            ids_bloqueados = extract_locked_ids_only(results_Geronto)
            if ids_bloqueados:
                print(f"⚠️ Atenção: {len(ids_bloqueados)} registros estão bloqueados e não foram importados.")
                print(f"IDs Bloqueados: {ids_bloqueados}")
            else:
                print("✅ Nenhum erro de bloqueio retornado pela API.")
        
        # 2. Confirmação dos IDs que tentamos enviar
        ids_enviados = dGer5['record_id'].unique().tolist()
        print(f"🚀 Processamento finalizado para os IDs: {ids_enviados}")
            
    except Exception as e:
        print(f"❌ Erro crítico durante a importação: {e}")

else:
    # Este else agora cobre tanto o DataFrame vazio quanto o filtro de duplicados
    print("DataFrame vazio ou todos os registros já existem no REDCap. Nada a importar.")
    

--- INICIANDO ANÁLISE DE 1 REGISTROS ---
Novos registros reais para importar: 1
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
🚀 Processamento finalizado para os IDs: ['1156847']


In [ ]:
# dGer5.loc[dGer5['record_id'] == 1142415, 'eq5d_dor'] = '1'

# Visualizar o resultado
# print(dGer5.loc[dGer5['record_id'] == 1142415, ['record_id', 'eq5d_dor']])

#dGer5.to_excel('C://Users/priscilla.sequetin/Downloads/dGer5_erro.xlsx', index=False)

In [250]:
dGer5

,record_id,lf_perda_peso,lf_fadiga,lf_dim_forca,lf_lentidao_marcha,lf_baixa_ativ_fisica,barthel_alimentacao,barthel_banho,barthel_hig_pessoal,barthel_vestir,barthel_intestinos,barthel_bexiga,barthel_uso_vaso,barthel_transfer,barthel_mobilidade,barthel_escadas,lawton_telefone,lawton_compras,lawton_refeicoes,lawton_casa,lawton_lavar_roupa,lawton_transporte,lawton_medicamentos,lawton_financas,sarcf_forca,sarcf_caminho,sarcf_levanta_cadeira,sarcf_escadas,sarcf_queda,apgar_ajuda,apgar_discute_problemas,apgar_apoia_desejos,apgar_responde_emocoes,apgar_tempo_juntos,gds15_1_satisfeito_vida,gds15_2_abandono_atividade,gds15_3_vida_vazia,gds15_4_aborrecido_freq,gds15_5_animado_humor,gds15_6_medo_ruim,gds15_7_feliz_todo,gds15_8_desamparado_freq,gds15_9_prefere_casa,gds15_10_problema_memoria,gds15_11_maravilha_vivo,gds15_12_inutil_semvalor,gds15_13_energizado,gds15_14_sem_esperanca,gds15_15_pessoas_melhores,kccq_1a_banho,kccq_1b_caminho,kcc1_1c_correr,tb_kccq_2,tb_kccq_3,tb_kccq_4,tb_kccq_5,tb_kccq_6,tb_kccq_7,kccq_8a_passatempos,kccq_8b_tarefas,kccq_8c_visitas,eq5d_mobilidade,eq5d_cuidados_pessoais,eq5d_atividades_habituais,eq5d_dor,eq5d_ansiedade,eq5d_comparacao_12meses,meem_objeto,circ_braquial_cm,circ_abdomen_cm,eq5d_escala_estado_saude,meem_escrever_frase,meem_ler_obedecer,meem_repeat_frase,meem_comando_verbal,sppb_cadeira_tempo,sppb_marcha_tempo,sppb_equilibrio,meem_reg_word_1,meem_reg_word_2,meem_reg_word_3,caminhada_6min,meem_bairro,meem_cidade,meem_estado,meem_instituicao,meem_local,circ_panturilha_cm,meem_evoc_word_1,meem_evoc_word_2,meem_evoc_word_3,meem_year,meem_monthday,meem_weekday,meem_hour,meem_month,meem_copiar_desenho,meem_calculo,caminhada_metro,preensao_palmar,lf_observacoes,teste_de_marcha_4m_realizado,teste_da_cadeira_realizado,redcap_repeat_instance,redcap_event_name,redcap_repeat_instrument,qualidade_vida_complete
87,1156847,0,1,0,0,1,10,5,5,10,10,5,10,15,15,5,1,0,1,1,1,1,1,1,1,0,0,1,2,2,2,2,2,2,1,1,1,0,1,1,1,0,1,1,1,0,1,0,0,5,3,2,1,4,7,5,3,1,3,3,5,1,1,2,2,3,1,2,25,113,70,1,1,1,2,11.72,4.39,2,1,1,1,0.76,0,1,1,1,1,38,1,1,1,1,1,1,1,1,1,2,275,19.8,Força: 19.8/15.4\n Velocidade: 4.70/4.39\n Obs...,1,1,1,fase1preoperatorio_arm_1,qualidade_vida,2


In [251]:
print(f"--- PREPARANDO IMPORTAÇÃO DE DADOS DEMOGRÁFICOS ---")

# --- PASSO 1: FILTRO DE FASE (Documento 1172) ---
# Extraímos os IDs que tiveram documentos novos
ids_com_novos_docs = dGer5[dGer5['redcap_event_name'] == 'fase1preoperatorio_arm_1'].copy()
ids_com_novos_docs = ids_com_novos_docs['record_id'].unique().tolist()
aPct1_ids = aPct1['record_id'].unique().tolist()

# --- PASSO 2: FILTRO DE EXCLUSÃO (Check contra aPct1) ---
# Só importamos de Geronto se o ID NÃO estiver na lista aPct1 (já importados)
ids_para_importar = [id for id in ids_com_novos_docs if id not in aPct1_ids]

print(f"IDs encontrados na Geronto Fase 1: {ids_com_novos_docs}")
print(f"IDs que serão ignorados (já existem em aPct1): {[id for id in ids_com_novos_docs if id in aPct1_ids]}")
print(f"IDs prontos para importar: {ids_para_importar}")

# --- PASSO 3: EXECUÇÃO DA IMPORTAÇÃO ---
# Criamos o DataFrame final filtrado (dPcte2)
dPcte2 = dPcte[dPcte['record_id'].isin(ids_para_importar)].copy()

if not dPcte2.empty:
    print(f"🚀 Iniciando importação de {len(dPcte2)} novos registros demográficos...")
    
    try:
        # Executa e captura possíveis bloqueios (batch_error)
        batch_error_pcte = import_redcap(
            df_sorted=dPcte2,
            api_url=api_url,
            api_key=api_key,
            batch_size=10,
            overwrite=False
        )
        
        # Ajuste do batch_error: Extração de IDs bloqueados no REDCap
        if batch_error_pcte:
            ids_bloqueados = extract_locked_ids_only(batch_error_pcte)
            if ids_bloqueados:
                print(f"⚠️ Registros bloqueados (LOCKED) no REDCap: {ids_bloqueados}")
            else:
                print("✅ Importação realizada com sucesso (sem bloqueios).")
        else:
            print("✅ Importação concluída com sucesso!")

    except Exception as e:
        print(f"❌ Erro crítico na comunicação com a API: {e}")

else:
    print("ℹ️ Nada a importar. Todos os pacientes de Geronto já foram processados via Pré-Op.")
    

--- PREPARANDO IMPORTAÇÃO DE DADOS DEMOGRÁFICOS ---
IDs encontrados na Geronto Fase 1: ['1156847']
IDs que serão ignorados (já existem em aPct1): []
IDs prontos para importar: ['1156847']
🚀 Iniciando importação de 1 novos registros demográficos...
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Importação concluída com sucesso!


In [252]:
print(f"--- PREPARANDO IMPORTAÇÃO DE FATORES DE RISCO (Peso, Altura) ---")

# --- PASSO 1: FILTRO DE FASE (Documento 1172) ---
# --- PASSO 2: FILTRO DE EXCLUSÃO  ---

#print(f"IDs com documentos novos importados: {ids_para_importar}")

# --- PASSO 3: EXECUÇÃO DA IMPORTAÇÃO ---
# Criamos o DataFrame final filtrado (dHist1)

dHist1 = dHist.copy()

if not dHist1.empty and 'altura' in dHist1.columns and 'peso' in dHist1.columns:
    # 1. Ajuste da Altura para 3 dígitos (cm)
    dHist1['altura'] = dHist1['altura'].astype(str).str.replace('.', '', regex=False).str.replace('nan', '')
    # Garante 3 dígitos (ex: '17' -> '170', '1.75' -> '175')
    dHist1['altura'] = dHist1['altura'].apply(lambda x: x.ljust(3, '0')[:3] if x not in ['', 'None'] else '')

    # 2. Preparação de valores numéricos para o cálculo
    # Convertemos para float: peso em kg e altura de cm para metros
    peso_val = pd.to_numeric(dHist1['peso'], errors='coerce')
    altura_m = pd.to_numeric(dHist1['altura'], errors='coerce') / 100

    # 3. Cálculo do IMC
    # O uso do np.where evita erros de divisão por zero caso a altura seja nula
    dHist1['imc'] = np.where(altura_m > 0, peso_val / (altura_m ** 2), np.nan)

    # 4. Função de Categorização
    def categorizar_imc(imc):
        if pd.isna(imc) or np.isinf(imc) or imc <= 0:
            return ""
        if imc < 18.5: return "Magreza grau I"
        if imc < 25.0: return "Eutrofia"
        if imc < 30.0: return "Pré-obesidade"
        if imc < 35.0: return "Obesidade moderada (grau I)"
        if imc < 40.0: return "Obesidade severa (grau II)"
        return "Obesidade muito severa (grau III)"

    # 5. Aplicação da Classificação
    dHist1['imc_classificacao'] = dHist1['imc'].apply(categorizar_imc)

    # 6. Limpeza
    # Se você quiser manter apenas a classificação e remover o valor numérico do IMC:
    dHist1 = dHist1.drop(columns='imc', errors='ignore')

# 2. Verifica se sobrou alguém para importar
#if not dHist1.empty:
    print(f"🚀 Iniciando importação de {len(dHist1)} novos registros fatores de risco ---")
    
    try:
        # Executa e captura possíveis bloqueios (batch_error)
        batch_error_ftrisk = import_redcap(
            df_sorted=dHist1,
            api_url=api_url,
            api_key=api_key,
            batch_size=10,
            overwrite=False
        )
        
        # Ajuste do batch_error: Extração de IDs bloqueados no REDCap
        if batch_error_ftrisk:
            ids_bloqueados = extract_locked_ids_only(batch_error_ftrisk)
            if ids_bloqueados:
                print(f"⚠️ Registros bloqueados (LOCKED) no REDCap: {ids_bloqueados}")
            else:
                print("✅ Importação realizada com sucesso (sem bloqueios).")
        else:
            print("✅ Importação concluída com sucesso!")

    except Exception as e:
        print(f"❌ Erro crítico na comunicação com a API: {e}")

else:
    print("ℹ️ Nada a importar.")
    

--- PREPARANDO IMPORTAÇÃO DE FATORES DE RISCO (Peso, Altura) ---
🚀 Iniciando importação de 1 novos registros fatores de risco ---
Batch 1/1 imported successfully. Records in this batch: 1. Total records imported: 1
✅ Importação concluída com sucesso!


In [ ]:
dHist1

In [ ]:
#dGer4.to_excel(f"C://Users/priscilla.sequetin/Downloads/geronto_datadante_7.xlsx", index=False)
#dPcte.to_csv(f"C://Users/priscilla.sequetin/Downloads/pcte_2.csv", sep="|", index=False)
#dHist.to_csv(f"C://Users/priscilla.sequetin/Downloads/dHist.csv", sep="|", index=False)

# CIRURGIAS
## descrição cirurgica
## valvulas utilizadas
## perfusao, cec e anoxia

In [297]:
# Buscando cirurgias na ft_aviso_cirurgico

query_CIRG =f"""
SET DATEFORMAT ymd;

WITH dic AS (
    SELECT NK_CD_CIRURGIA, DS_CIRURGIA, DS_PROCEDIMENTO_PSIH
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY NK_CD_CIRURGIA ORDER BY DT_ATUAL DESC) as rn FROM dim_cirurgia) t 
    WHERE rn = 1
),
-- Agregando a Equipe Médica com o Anestesista
equipe_consolidada AS (
    SELECT 
        CD_AVISO_CIRURGIA,
        -- Cirurgião Principal
        MAX(CASE WHEN DS_ATI_MED = 'CIRURGIAO' THEN NM_PRESTADOR END) AS CIRURGIAO_PRINCIPAL,
        -- Anestesista (Ajuste o termo 'ANESTESISTA' se o seu banco usar outra nomenclatura)
        MAX(CASE WHEN DS_ATI_MED LIKE '%ANESTESISTA%' THEN NM_PRESTADOR END) AS ANESTESISTA,
        -- Auxiliares (Apenas primeiro nome)
        STRING_AGG(CASE WHEN DS_ATI_MED LIKE '%AUXILIAR%' THEN LEFT(NM_PRESTADOR, CHARINDEX(' ', NM_PRESTADOR + ' ') - 1) END, ', ') 
            WITHIN GROUP (ORDER BY NM_PRESTADOR) AS AUXILIARES
    FROM ft_aviso_cirurgico_equipe
    GROUP BY CD_AVISO_CIRURGIA
)

SELECT 
    fac.CD_PACIENTE,
    fac.CD_ATENDIMENTO,
    fac.CD_AVISO_CIRURGIA,
    dip.NM_PACIENTE,
    dic.DS_PROCEDIMENTO_PSIH,
    CAST(COALESCE(fac.DT_REALIZACAO, fac.DT_INICIO_CIRURGIA) AS DATE) AS DATA_CIRURGIA,
    fdc.DS_CIRURGIA AS DESCRICAO_CIRURGIA,
    fac.DT_INICIO_CIRURGIA,
    fac.DT_FIM_CIRURGIA,
    fac.DT_SAIDA_SAL_CIR,
    DATEDIFF(MINUTE, fac.DT_INICIO_CIRURGIA, fac.DT_FIM_CIRURGIA) AS DURACAO_CIRURGIA_MINUTOS,
    ec.CIRURGIAO_PRINCIPAL,
    ISNULL(ec.ANESTESISTA, '') AS ANESTESISTA,
    ISNULL(ec.AUXILIARES, '') AS AUXILIARES,
    dsc.DS_CEN_CIR
FROM 
    ft_aviso_cirurgico fac
    INNER JOIN dim_paciente dip ON dip.NK_CD_PACIENTE = fac.CD_PACIENTE
    LEFT JOIN dim_cirurgia dic ON dic.NK_CD_CIRURGIA = fac.CD_CIRURGIA
    LEFT JOIN ft_descricao_cirurgica fdc ON fdc.NK_CD_DESCRICAO_AVISO_CIRURGIA = fac.CD_DESCRICAO_AVISO_CIRURGIA
    LEFT JOIN dim_sala_cirurgica dsc ON dsc.NK_CD_SAL_CIR = fac.CD_SAL_CIR
    LEFT JOIN equipe_consolidada ec ON ec.CD_AVISO_CIRURGIA = fac.CD_AVISO_CIRURGIA
WHERE 1=1
    --and fac.CD_PACIENTE = 1128345 
    AND ISNULL(dsc.DS_CEN_CIR, '') != 'HEMODINAMICA'
	AND YEAR(fac.DT_AVISO_CIRURGIA) >= 2026
ORDER BY 
    fac.DT_INICIO_CIRURGIA DESC;
""" 

# Executa a função para buscar os dados
df_CIRG = query_sql_to_dataframe(conn_string, query_CIRG)

Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [298]:
# Copiando a df
aCir = df_CIRG.copy()

# Trabalhando com o ano corrente
# aCir = aCir[aCir['DT_REALIZACAO'].dt.year >= 2025]

# Removendo pacientes de teste
aCir = remove_test_patients(aCir, 'CD_PACIENTE')

# --- Passo de Limpeza solicitado ---
# Remove linhas onde DESCRICAO_CIRURGIA é nula (NaN) ou apenas espaços vazios
aCir = aCir[aCir['DESCRICAO_CIRURGIA'].notna() & (aCir['DESCRICAO_CIRURGIA'].astype(str).str.strip() != "")]



In [299]:
# --- PREPARAÇÃO DOS FILTROS ---

# 1. Recupera os IDs novos do passo anterior (Geronto)
lista_novos_geronto = []
if 'dGer5' in locals() and not dGer5.empty:
    filtro_fase1 = dGer5['redcap_event_name'] == 'fase1preoperatorio_arm_1'
    lista_novos_geronto = dGer5.loc[filtro_fase1, 'record_id'].astype('Int64').unique().tolist()

# 2. Extrai IDs da Ficha Pré-Operatória (dVal5)
lista_preop = []
if 'dVal5' in locals() and not dVal5.empty:
    # Extraímos os IDs únicos de record_id
    lista_preop = dVal5['record_id'].astype('Int64').unique().tolist()

# 3. Filtra registros onde o status é "1" (Unverified) ou Vazio (Nunca salvo) na API
filtro_pendentes = (regvalv_cirurgia['procedimento_complete'] == "1") | (regvalv_cirurgia['procedimento_complete'] == "")

lista_api_pendentes = []
if not regvalv_cirurgia[filtro_pendentes].empty:
    lista_api_pendentes = regvalv_cirurgia.loc[filtro_pendentes, 'record_id'].astype('Int64').tolist()

# --- 4. Consolidação (União) ---
# Usamos set.union para aceitar tanto listas quanto sets sem erro de concatenação
lista_final_set = set().union(
    lista_a_processar, 
    lista_api_pendentes, 
    lista_novos_geronto, 
    lista_preop,
    obito_a_verificar # Aqui ele aceita sendo set ou lista
)

# CORREÇÃO AQUI: 
# Converte todos os itens para número inteiro (ignorando eventuais valores nulos) antes de ordenar
lista_limpa = [int(x) for x in lista_final_set if pd.notna(x)]
lista_para_processar = sorted(lista_limpa)

# --- RESUMO DOS RESULTADOS ---
print(f"\n--- RESUMO FINAL ---")
print(f"Novos s/ Válvula: {len(lista_a_processar)} , {lista_a_processar}")
print(f"Pré-Op (dVal5): {len(lista_preop)}, {lista_preop}")
print(f"Novos Geronto (Apenas Fase 1): {len(lista_novos_geronto)}, {lista_novos_geronto}")
print(f"Pendentes/Vazios API: {len(lista_api_pendentes)}, {lista_api_pendentes}")

# CORREÇÃO DO PRINT AQUI:
print(f"Lista de Obitos: {obito_a_verificar}") 

print("-" * 30)
print(f"TOTAL ÚNICO para processar: {len(lista_para_processar)}")
print(f"Lista Final: {lista_para_processar}")


--- RESUMO FINAL ---
Novos s/ Válvula: 87 , ['887664', '1161235', '1144992', '571572', '1152195', '1152246', '1145924', '335458', '556037', '1111616', '550953', '1161988', '1129915', '793042', '883945', '1155826', '1023415', '1121141', '1114593', '1106694', '1158377', '741733', '1089242', '677960', '1140434', '1110802', '950526', '1104809', '1140620', '1151700', '1149962', '985224', '1146111', '1129508', '1112267', '1043905', '1020833', '440359', '1156393', '1145734', '1143719', '1159923', '678566', '336893', '712755', '949704', '618437', '446949', '1073969', '1163216', '939793', '587546', '834779', '1005872', '341220', '846884', '904606', '1160564', '1085090', '1156346', '989233', '1154874', '988625', '391604', '910528', '401562', '1103315', '1156847', '1146922', '472525', '1075661', '1150098', '349574', '1131889', '1134420', '1050096', '1056201', '600762', '1157043', '593227', '1148693', '612689', '822752', '584721', '1133920', '903485', '999140']
Pré-Op (dVal5): 3, [1128722, 113023

In [300]:

# Iniciando a busca dos procedimentos cirurgicos
aCir1 = aCir[aCir['CD_PACIENTE'].astype('Int64').isin(lista_para_processar)].copy()

In [301]:
aCir1.columns

Index(['CD_PACIENTE', 'CD_ATENDIMENTO', 'CD_AVISO_CIRURGIA', 'NM_PACIENTE',
       'DS_PROCEDIMENTO_PSIH', 'DATA_CIRURGIA', 'DESCRICAO_CIRURGIA',
       'DT_INICIO_CIRURGIA', 'DT_FIM_CIRURGIA', 'DT_SAIDA_SAL_CIR',
       'DURACAO_CIRURGIA_MINUTOS', 'CIRURGIAO_PRINCIPAL', 'ANESTESISTA',
       'AUXILIARES', 'DS_CEN_CIR'],
      dtype='object')

In [302]:
colunas_data = ['DATA_CIRURGIA', 'DT_INICIO_CIRURGIA', 'DT_FIM_CIRURGIA', 'DT_SAIDA_SAL_CIR']
for col in colunas_data:
    aCir1[col] = pd.to_datetime(aCir1[col], errors='raise')

In [ ]:
# # 1. Filtre o DataFrame para conter APENAS os auxiliares
# #    O .str.contains() é robusto e pegará '1 AUXILIAR', '2 AUXILIAR', etc.
# df_auxiliares = aCir1[aCir1['DS_ATI_MED'].str.contains('AUXILIAR', na=False)].copy()

# # 2. Na cópia filtrada, crie uma coluna com apenas o PRIMEIRO nome
# #    .str.split(' ') cria uma lista (ex: ['NATANAEL', 'PONTE', ...])
# #    .str[0] pega o primeiro item dessa lista
# df_auxiliares['PRIMEIRO_NOME'] = df_auxiliares['NM_PRESTADOR'].str.split(' ').str[0]

# # 3. Agrupe por 'ID_CIRURGIA' e junte os primeiros nomes
# #    Isso cria um "resumo" (uma pd.Series) onde cada ID_CIRURGIA
# #    tem uma string única com os nomes dos seus auxiliares.
# resumo_auxiliares = df_auxiliares.groupby('CD_CIRURGIA_AVISO')['PRIMEIRO_NOME'].apply(', '.join)

# # 4. Mapeie esse resumo de volta para o DataFrame original
# #    Isso cria a coluna 'AUXILIARES' no aCir1 original,
# #    alinhada pelo 'ID_CIRURGIA'
# aCir1['AUXILIARES'] = aCir1['CD_CIRURGIA_AVISO'].map(resumo_auxiliares)

# # O .fillna('') é opcional, caso queira preencher cirurgias
# # sem auxiliares com uma string vazia em vez de NaN.
# aCir1['AUXILIARES'] = aCir1['AUXILIARES'].fillna('')

In [303]:
aCir1.columns

Index(['CD_PACIENTE', 'CD_ATENDIMENTO', 'CD_AVISO_CIRURGIA', 'NM_PACIENTE',
       'DS_PROCEDIMENTO_PSIH', 'DATA_CIRURGIA', 'DESCRICAO_CIRURGIA',
       'DT_INICIO_CIRURGIA', 'DT_FIM_CIRURGIA', 'DT_SAIDA_SAL_CIR',
       'DURACAO_CIRURGIA_MINUTOS', 'CIRURGIAO_PRINCIPAL', 'ANESTESISTA',
       'AUXILIARES', 'DS_CEN_CIR'],
      dtype='object')

In [304]:
aCir2 = aCir1[['CD_PACIENTE', 'CD_ATENDIMENTO', 'CD_AVISO_CIRURGIA', 'DS_PROCEDIMENTO_PSIH',
        'DATA_CIRURGIA', 'DESCRICAO_CIRURGIA', 'DT_INICIO_CIRURGIA', 'DT_FIM_CIRURGIA', 'DT_SAIDA_SAL_CIR', 
        'DURACAO_CIRURGIA_MINUTOS', 'CIRURGIAO_PRINCIPAL', 'ANESTESISTA',
       	'AUXILIARES', 'DS_CEN_CIR']]

In [305]:
def identificar_abordagem(texto):
    if pd.isna(texto):
        return ""  # Caso esteja vazio, define como "Outra" ou deixe vazio se preferir
    
    texto = str(texto).upper()
    
    # 5. Robótica (Prioridade alta pois pode conter outros termos)
    if any(x in texto for x in ['ROBOTICA', 'ROBOTICA', 'ROBÔ']):
        return "5"
    
    # 4. Minimamente invasiva
    if any(x in texto for x in ['MINIMAMENTE', 'VIDEO', 'VÍDEO', 'MINI-INVASIVA', 'MINI-TORACOTOMIA']):
        return "4"
    
    # 2. Toracotomia
    if 'TORACOTOMIA' in texto:
        return "2"
    
    # 1. Esternotomia (Abordagem padrão em cirurgias abertas)
    if any(x in texto for x in ['ESTERNOTOMIA', 'ESTERNO', 'MEDIANA']):
        return "1"
    
    # 9. Outra
    return ""

# Aplicando a função no seu DataFrame acir
aCir2['abordagem_cirurgica'] = aCir2['DESCRICAO_CIRURGIA'].apply(identificar_abordagem)

In [306]:
aCir2.columns = aCir2.columns.str.lower()

In [307]:
print(aCir2['ds_procedimento_psih'].unique())

['CORREÇÃO DE ANEURISMA / DISSECÇÃO DA AORTA TORACO-ABDOMINAL'
 'IMPLANTE DE PRÓTESE VALVAR'
 'IMPLANTE DE MARCAPASSO DE CÂMARA DUPLA TRANSVENOSO']


In [308]:
aCir3 = aCir2[(aCir2['ds_procedimento_psih']!='TORACOTOMIA EXPLORADORA') & (aCir2['ds_procedimento_psih']!='RESSUTURA DE PAREDE ABDOMINAL (POR DEISCENCIA TOTAL / EVISCERAÇÃO)') 
            & (aCir2['ds_procedimento_psih']!='TRATAMENTO CIRÚRGICO DE VARIZES (UNILATERAL)') & (aCir2['ds_procedimento_psih']!='DRENAGEM C/ BIOPSIA DE PERICÁRDIO')
            & (aCir2['ds_procedimento_psih']!='EXCISAO DE LESAO E/OU SUTURA DE FERIMENTO DA PELE ANEXOS E MUCOSA') & (aCir2['ds_procedimento_psih']!='IMPLANTE DE MARCAPASSO DE CÂMARA DUPLA TRANSVENOSO')
            & (aCir2['ds_procedimento_psih']!='LAPAROTOMIA EXPLORADORA') & (aCir2['ds_procedimento_psih']!='IMPLANTE DE MARCAPASSO CARDÍACO MULTI-SITIO EPIMIOCÁRDICO POR TORACOTOMIA P/IMPLANTE DE ELETRODO')
            & (aCir2['ds_procedimento_psih']!='DESCORTICACAO PULMONAR') & (aCir2['ds_procedimento_psih']!='DESCORTICAÇÃO PULMONAR') & (aCir2['ds_procedimento_psih']!='CATETERISMO CARDIACO')
            & (aCir2['ds_procedimento_psih']!='FIBRINÓLISE PARA EMBOLIA PULMONAR MACICA INTRAVASCULAR POR CATETER (INCLUI FIBRINOLÍTICO)')
            & (aCir2['ds_procedimento_psih']!='IMPLANTE DE MARCAPASSO DE CÂMARA ÚNICA TRANSVENOSO') & (aCir2['ds_procedimento_psih']!='BIOPSIAS MULTIPLAS INTRA-ABDOMINAIS EM ONCOLOGIA')
            & (aCir2['ds_procedimento_psih']!='IMPLANTE DE CARDIOVERSOR DESFIBRILADOR DE CÃMARA DUPLA TRANSVENOSO') & (aCir2['ds_procedimento_psih']!='LAPAROTOMIA EXPLORADORA COM RESSECÇÃO COMPLETA OU INCOMPLETA DO TUMOR EM ONCOLOGIA')
            & (aCir2['ds_procedimento_psih']!='BIÓPSIAS MÚLTIPLAS INTRA-ABDOMINAIS EM ONCOLOGIA') & (aCir2['ds_procedimento_psih']!='RESSUTURA DE PAREDE ABDOMINAL (POR DEISCENCIA TOTAL / EVISCERACAO)')
            ]

In [309]:
print(aCir3['ds_procedimento_psih'].unique())

['CORREÇÃO DE ANEURISMA / DISSECÇÃO DA AORTA TORACO-ABDOMINAL'
 'IMPLANTE DE PRÓTESE VALVAR']


In [310]:
aCir3['redcap_repeat_instance'] = aCir3.groupby('cd_paciente')['data_cirurgia'].rank(method='dense', ascending=True).astype('Int64')


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\1816148216.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aCir3['redcap_repeat_instance'] = aCir3.groupby('cd_paciente')['data_cirurgia'].rank(method='dense', ascending=True).astype('Int64')


In [270]:
aCir3.columns

Index(['cd_paciente', 'cd_atendimento', 'cd_aviso_cirurgia',
       'ds_procedimento_psih', 'data_cirurgia', 'descricao_cirurgia',
       'dt_inicio_cirurgia', 'dt_fim_cirurgia', 'dt_saida_sal_cir',
       'duracao_cirurgia_minutos', 'cirurgiao_principal', 'anestesista',
       'auxiliares', 'ds_cen_cir', 'abordagem_cirurgica',
       'redcap_repeat_instance'],
      dtype='object')

In [311]:
# --- 1. Renomear a coluna de data em aCir4 para bater com aVal ---
# Isso é essencial para o merge funcionar
aCir4 = aCir3.rename(columns={'cd_paciente': 'record_id', 'anestesista': 'anestesista_principal', 'duracao_cirurgia_minutos': 'duracao_cirurgia'})

# --- 2. Garantir que os tipos de dados são iguais ---

# Ex: Se cd_paciente for número em um e texto em outro, o merge falha
# Vamos forçar os dois para string (ou int, o que for melhor)
# (Usar string é mais seguro se houver zeros à esquerda)
#aVal1['record_id'] = aVal1['record_id'].astype(str)
aCir4['record_id'] = aCir4['record_id'].astype(str)

# Garantir que as datas são datas (sem hora!)
# Isso é CRUCIAL, pois '2025-10-10 08:00' é diferente de '2025-10-10'
#aVal1['data_cirurgia'] = pd.to_datetime(aVal1['data_cirurgia']).dt.date
aCir4['data_cirurgia'] = pd.to_datetime(aCir4['data_cirurgia']).dt.date

print("Preparação concluída. Colunas de chave renomeadas e tipos ajustados.")


Preparação concluída. Colunas de chave renomeadas e tipos ajustados.


In [312]:
aCir4 = aCir4.drop_duplicates(subset=['record_id', 'cd_atendimento', 'cd_aviso_cirurgia',])

In [313]:
aCir4['duracao_cirurgia'] = aCir4['duracao_cirurgia'].round(0).astype('Int64')

# Converte para float (para lidar com o '.0') e depois para Int64 (que aceita nulos)
aCir4['record_id'] = aCir4['record_id'].astype(float).astype('Int64')
aCir4['cd_atendimento'] = aCir4['cd_atendimento'].astype(float).astype('Int64')

In [314]:
aCir4['redcap_event_name'] = 'fase3procedcirurgi_arm_1'
aCir4['redcap_repeat_instrument'] = 'procedimento'
aCir4['procedimento_complete'] = '0'

In [315]:
# Remove diretamente no dataframe original (sem precisar atribuir a 'df = ...')
aCir4.drop(columns=['ds_procedimento_psih', 'dt_inicio_cirurgia', 'dt_fim_cirurgia', 'dt_saida_sal_cir', 'ds_cen_cir'], inplace=True)


In [316]:
# O argumento regex=False evita avisos futuros e garante substituição literal
aCir4['descricao_cirurgia'] = aCir4['descricao_cirurgia'].str.replace(';', '-', regex=False)
aCir4['descricao_cirurgia'] = aCir4['descricao_cirurgia'].str.replace(',', '-', regex=False)


# VALVULAS 
### tipo cirurgia, modelo, tamanho, protese

In [278]:
# Buscando valvulas tipo, modelo, tamanho

query_VALV =f"""
SET DATEFORMAT ymd;
WITH 
	dp AS (
		SELECT 
			NK_CD_PRODUTO,
			DS_CLASSE,
			DS_SUB_CLA
		FROM dim_produto dp1
		WHERE DT_ATUAL = (
			SELECT MAX(DT_ATUAL)
			FROM dim_produto dp2
			WHERE dp2.NK_CD_PRODUTO = dp1.NK_CD_PRODUTO
		)
	)
	SELECT
		
		fa.CD_PACIENTE as record_id
        ,fc.[CD_ATENDIMENTO] as cd_atendimento
		,fc.[CD_AVISO_CIRURGIA]
        ,CAST(COALESCE(fac.DT_REALIZACAO, fac.DT_INICIO_CIRURGIA) AS DATE) AS data_cirurgia
		,[DS_PRODUTO]
		,[DS_ESPECIE]
		,[NM_SETOR]
		,[DT_MVTO_ESTOQUE]
		,DS_CLASSE
		,DS_SUB_CLA
	from ft_consumo_centro_custo fc
    left join ft_aviso_cirurgico fac ON fac.CD_AVISO_CIRURGIA = fc.CD_AVISO_CIRURGIA
	left join dp ON dp.NK_CD_PRODUTO = fc.CD_PRODUTO
	join ft_atendimento fa ON fa.NK_CD_ATENDIMENTO = fc.CD_ATENDIMENTO
	WHERE 1=1
	and DS_PRODUTO LIKE '%VALV%' 
	and DS_CLASSE in ('ORTESE', 'PROTESE', 'HEMODINAMICA')
	AND DS_SUB_CLA NOT IN ('TORNEIRA', 'CATETER', 'ACESSORIO')
	and DS_PRODUTO != 'VALVULA DE ALIVIO PARA VACUO'
	and NM_SETOR != 'HEMODINAMICA'
    and YEAR(DT_MVTO_ESTOQUE) >= 2026
    
""" 

# Executa a função para buscar os dados
df_VALV = query_sql_to_dataframe(conn_string, query_VALV)

Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [279]:
aVal = df_VALV[df_VALV['record_id'].astype('Int64').isin(aCir4['record_id'].astype('Int64'))].copy()

In [280]:
# Garantir tipos consistentes antes do processamento
aVal['record_id'] = aVal['record_id'].astype('Int64')
aVal['cd_atendimento'] = aVal['cd_atendimento'].astype('Int64')
aCir4['record_id'] = aCir4['record_id'].astype('Int64')
aCir4['cd_atendimento'] = aCir4['cd_atendimento'].astype('Int64')

In [281]:
aVal.columns

Index(['record_id', 'cd_atendimento', 'CD_AVISO_CIRURGIA', 'data_cirurgia',
       'DS_PRODUTO', 'DS_ESPECIE', 'NM_SETOR', 'DT_MVTO_ESTOQUE', 'DS_CLASSE',
       'DS_SUB_CLA'],
      dtype='object')

In [282]:
def processar_modelo_valvula(df_sql):
    # Criamos uma cópia para evitar SettingWithCopyWarning
    df_sql = df_sql.copy()
    
    # Tratamento inicial de string
    df_sql['DS_PRODUTO'] = df_sql['DS_PRODUTO'].astype(str).str.upper().str.strip()
    
    mapa_check = {
        'tipo_cirurgia___1': ['AORT', 'AOR '], # Note o espaço em 'AO ' para evitar pegar 'AORTA' duas vezes ou outras palavras
        'tipo_cirurgia___2': ['MIT', 'MITRAL'], 
        'tipo_cirurgia___3': ['TRICUSP', 'TRI '], 
        'tipo_cirurgia___4': ['PULM'],
        'tipo_cirurgia___5': ['PLASTIA', 'ANULOPLASTIA', 'REPARO'], # Exemplo de múltiplos termos
        'tipo_cirurgia___9': ['TRANSC', 'TAVI']
    }

    def extrair_individuais(texto):
        # Definição do Tipo
        # Prioridade para Biológica (2), depois Mecânica (1), senão Outros (3)
        tipo = '2' if any(x in texto for x in ['BIOL', 'BIO', 'EPIC', 'MAGNA', 'MITRIS', 'INSPIRIS', 'RESILIA']) \
               else ('1' if any(x in texto for x in ['MECAN', 'ABBOTT', 'ONX', 'ST JUDE']) else '3')
        
        # Extração do Tamanho (procura 2 dígitos, opcionalmente com decimal)
        match = re.search(r'(\d{2}(?:[.,]\d)?)', texto)
        tamanho = match.group(1).replace(',', '.') if match else None
        
		# Segurança extra: se extraiu "00" ou "0", força None
        if tamanho and float(tamanho) == 0:
            tamanho = None
            
        # Checks Ativos (Procedimentos)
        checks_ativos = [
            col for col, lista_termos in mapa_check.items() 
            if any(termo in texto for termo in lista_termos)
        ]
        
        return [tipo, tamanho, checks_ativos]

    # --- CORREÇÃO DO ERRO ---
    # 1. Removemos a linha confusa 'df_res = ... result_type'
    # 2. Aplicamos a função e convertemos para lista de listas (muito mais rápido que pd.Series linha a linha)
    dados_extraidos = df_sql['DS_PRODUTO'].apply(extrair_individuais).tolist()
    
    # 3. Criamos um DataFrame temporário com as colunas corretas e o mesmo índice
    df_temp = pd.DataFrame(dados_extraidos, columns=['tipo_temp', 'tam_temp', 'proceds_list'], index=df_sql.index)
    
    # 4. Concatenamos ao dataframe original
    df_sql = pd.concat([df_sql, df_temp], axis=1)

    # --- AGRUPAMENTO ---
    # Agrupamento por Evento Único
    cols_group = ['record_id', 'cd_atendimento', 'CD_AVISO_CIRURGIA', 'data_cirurgia']
    
    # Garantir que não existam NaNs nas colunas de agrupamento (opcional, mas recomendado)
    # df_sql = df_sql.dropna(subset=cols_group) 

    df_consolidado = df_sql.groupby(cols_group).agg({
        'DS_PRODUTO': lambda x: ' / '.join(x.unique()),
        # Pega o primeiro valor não nulo encontrado para tamanho e tipo
        # Lógica ajustada:
        # 1. Filtra valores que não são None E não são '0'
        # 2. Se a lista estiver vazia (não achou nada), retorna None (em vez de "" ou "0")
        'tam_temp': lambda x: next((v for v in x if v is not None and v != '0' and v != ''), None), 
        'tipo_temp': lambda x: next((v for v in x if v is not None), "3"),
        # Achata a lista de listas de procedimentos
        'proceds_list': lambda x: [item for sublist in x for item in sublist]
    }).reset_index()

    # Criar colunas de checkbox (Flag 0 ou 1)
    for col in mapa_check.keys():
        df_consolidado[col] = df_consolidado['proceds_list'].apply(lambda x: 1 if col in x else 0)
        
    # Regra para múltiplas válvulas (Flag tipo_cirurgia___8)
    cols_troca = ['tipo_cirurgia___1', 'tipo_cirurgia___2', 'tipo_cirurgia___3', 'tipo_cirurgia___4']
    
    # Se a soma das flags de troca for maior que 1, marca 'Outros/Múltiplos' como True
    df_consolidado['tipo_cirurgia___8'] = (df_consolidado[cols_troca].sum(axis=1) > 1).astype(int)
	
    # Renomear e limpar
    return df_consolidado.rename(columns={
        'DS_PRODUTO': 'modelo_protese',
        'tam_temp': 'tamanho_protese',
        'tipo_temp': 'tipo_valvula_prostetica'
    }).drop(columns=['proceds_list'])



In [283]:
# 1. Gerar os dados das válvulas

aVal1 = processar_modelo_valvula(aVal)

In [284]:
# 2. Realizar o Merge Final
# Como aVal1 tem record_id e cd_atendimento, o merge 'on' funcionará
aCir5 = pd.merge(
    aCir4,
    aVal1.drop(columns=['CD_AVISO_CIRURGIA']),
    on=['record_id', 'cd_atendimento', 'data_cirurgia'], 
    how='left'
)

In [ ]:
#aVal1.to_excel(f"C://Users/priscilla.sequetin/Downloads/aval1.xlsx", index=False)

In [ ]:
aCir5

# PERFUSAO 
## TA (anoxia, pinçamento) E TP (cec, perfusão)

In [285]:
#1175

# Buscando docText.PERFUSAO

sql_PERFUS = """
SET DATEFORMAT ymd;
SELECT 
    fde.CD_ATENDIMENTO as cd_atendimento, 
    fde.CD_PACIENTE as record_id, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS data_cirurgia, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA
    
FROM ft_doc_eletronico_texto fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 

-- Filtros
WHERE fde.CD_DOCUMENTO IN (1175)  
  	AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
	
ORDER BY fde.DH_DOCUMENTO DESC;
""" 

# Executa a função para buscar os dados
txPERFUS = query_sql_to_dataframe(conn_string, sql_PERFUS)

Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [286]:
dPer = txPERFUS.copy()

In [287]:
dPer1 = dPer[dPer['record_id'].astype('Int64').isin(aCir5['record_id'].astype('Int64'))]

In [288]:
dPer1["DS_IDENTIFICADOR"] = dPer1["DS_IDENTIFICADOR"].str[3:-2]

C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2798843893.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dPer1["DS_IDENTIFICADOR"] = dPer1["DS_IDENTIFICADOR"].str[3:-2]


In [289]:
# 4. Pivotando os identificadores não tratados
dPer2 = dPer1.pivot_table(
    index=['cd_atendimento', 'record_id', 'DH_DOCUMENTO', 'data_cirurgia'],
    columns='DS_IDENTIFICADOR',
    values='DS_RESPOSTA',
    aggfunc='first'  # ou 'last', dependendo da lógica de negócio
).reset_index()

In [290]:
# --- 1. SEU PASSO DE LIMPEZA (Mantido) ---
def limpar_apenas_numeros(valor):
    if pd.isna(valor):
        return None
    # Converte para string e busca a primeira sequência de dígitos
    match = re.search(r'(\d+)', str(valor))
    if match:
        return int(match.group(1))
    return None

colunas_para_limpar = ['TEMPO_ANOXIA', 'TEMPO_PERFUSAO']

# --- CONDICIONAL DE SEGURANÇA ---
# Se NÃO estiver vazio, faz a limpeza pesada (Regex).
# Se estiver vazio, apenas converte o tipo para evitar erro de schema depois.
if not dPer2.empty:
    for col in colunas_para_limpar:
        # Verifica se a coluna existe para evitar KeyError
        if col in dPer2.columns:
            dPer2[col] = dPer2[col].apply(limpar_apenas_numeros).astype('Int64')
else:
    # Mesmo vazio, forçamos o tipo Int64 para garantir consistência nas colunas
    for col in colunas_para_limpar:
        if col in dPer2.columns:
            dPer2[col] = dPer2[col].astype('Int64')

# --- 2. RENOMEAR E CRIAR FLAG (Executa sempre, vazio ou cheio) ---

# Renomeando (Preparando para o dPer3)
dPer3 = dPer2.rename(columns={
    'TEMPO_ANOXIA': 'tempo_pinc_ao', 
    'TEMPO_PERFUSAO': 'tempo_cec'
})

# Criação da flag
# Se o DF for vazio, isso vai gerar uma coluna vazia do tipo Int64, sem erros.
cols_alvo = ['tempo_pinc_ao', 'tempo_cec']

# Verificamos se as colunas renomeadas existem antes de criar a flag (segurança extra)
if set(cols_alvo).issubset(dPer3.columns):
    dPer3['circulacao_extracorporea'] = dPer3[cols_alvo].notna().any(axis=1).astype('Int64')
else:
    # Fallback caso as colunas não existam (ex: dPer2 veio sem elas)
    dPer3['circulacao_extracorporea'] = pd.Series(dtype='Int64')

# Opcional: print para validar
print(f"Status dPer3: {dPer3.shape[0]} linhas processadas.")


# Exemplo de resultado:
# '150 min' -> 150
# "123'"    -> 123
# '77 MIN'  -> 77

Status dPer3: 2 linhas processadas.


In [291]:
dPer3.columns

Index(['cd_atendimento', 'record_id', 'DH_DOCUMENTO', 'data_cirurgia',
       'HEPARINA', 'PERFUSATO', 'PROTAMINA', 'TCA_FINAL', 'TCA_INICIAL',
       'tempo_pinc_ao', 'tempo_cec', 'TIPO_CARDIOPLEGIA', 'adado_P_208306',
       'circulacao_extracorporea'],
      dtype='object', name='DS_IDENTIFICADOR')

In [292]:
# 2. Realizar o Merge Final
# Como aVal1 tem record_id e cd_atendimento, o merge 'on' funcionará
# Lista de colunas que você deseja remover (se existirem)
cols_perfusao = [
    'DH_DOCUMENTO', 'TXT_TIPO_CARDIOPLAGIA', 'CTX_OBSERVACAO', 
    'HEPARINA', 'PERFUSATO', 'PROTAMINA', 
    'TCA_FINAL', 'TCA_INICIAL', 'TIPO_CARDIOPLEGIA'
]

# 2. Realizar o Merge Final
aCir6 = pd.merge(
    aCir5,
    # O errors='ignore' garante que o código não quebre se uma coluna faltar
    dPer3.drop(columns=cols_perfusao, errors='ignore'), 
    on=['record_id', 'cd_atendimento', 'data_cirurgia'], 
    how='left'
)

In [293]:
cols = ['tipo_cirurgia___1', 'tipo_cirurgia___2',
       'tipo_cirurgia___3', 'tipo_cirurgia___4', 'tipo_cirurgia___5',
       'tipo_cirurgia___9', 'tipo_cirurgia___8', 'tamanho_protese']

for col in cols:
    # 1. Garante que o pandas entenda como número (converte string '1.0' para float 1.0)
    aCir6[col] = pd.to_numeric(aCir6[col], errors='coerce')
    
    # 2. Preenche vazios (NaN) com 0 e converte para Inteiro
    # Isso elimina o ".0" automaticamente (ex: 1.0 vira 1)
    aCir6[col] = aCir6[col].fillna(0).astype('Int64')

In [294]:
aCir6.columns

Index(['record_id', 'cd_atendimento', 'cd_aviso_cirurgia', 'data_cirurgia',
       'descricao_cirurgia', 'duracao_cirurgia', 'cirurgiao_principal',
       'anestesista_principal', 'auxiliares', 'abordagem_cirurgica',
       'redcap_repeat_instance', 'redcap_event_name',
       'redcap_repeat_instrument', 'procedimento_complete', 'modelo_protese',
       'tamanho_protese', 'tipo_valvula_prostetica', 'tipo_cirurgia___1',
       'tipo_cirurgia___2', 'tipo_cirurgia___3', 'tipo_cirurgia___4',
       'tipo_cirurgia___5', 'tipo_cirurgia___9', 'tipo_cirurgia___8',
       'tempo_pinc_ao', 'tempo_cec', 'adado_P_208306',
       'circulacao_extracorporea'],
      dtype='object')

In [295]:
# 1. Criar o Dicionário Mestre (Nome -> [Volume, Experiencia])
# Note que juntei os nomes que estavam quebrados no seu texto original
referencia_cirurgioes = {
    'ALMIRO CARLOS FERRO JUNIOR': [200, 3],
    'ALOYSIO ABDO SILVA CAMPOS': [150, 2],
    'ANDRE LUIZ MENDES MARTINS': [200, 2],
    'ANDREA CRISTINA OLIVEIRA FREITAS': [200, 3],
    'ANTONIO FLAVIO SANCHEZ DE ALMEIDA': [200, 3],
    'DANIEL CHAGAS DANTAS': [200, 3],
    'DIEGO GAMARRA MOREIRA': [50, 2],
    'GIOVANNA PAULA MACEDO DE LACERDA GUEDES': [100, 1],
    'JANAYNA THAINA RABELATO': [200, 3],
    'MARCELA DALLA BERNARDINA SENA': [30, 1],
    'MARIO ISSA': [200, 3],
    'OMAR ALONZO POZO IBANEZ': [200, 3],
    'PAULO HENRIQUE D PAULISTA': [200, 3],
    'RENATO TAMBELLINI ARNONI': [200, 3],
    'VITOR LUCENA CARNEIRO': [150, 2],
    'LEANDRO PICHETH ELOY PEREIRA': [30, 1]
}

# 2. Padronização (Crucial para garantir o "match")
# Removemos espaços extras nas pontas e convertemos para maiúsculo
# (Isso evita que "Mario Issa " não bata com "MARIO ISSA")
def extrair_primeiro_cirurgiao(texto):
    # 1. Se for nulo ou vazio, retorna None (ignora)
    if pd.isna(texto) or str(texto).strip() == '':
        return np.nan
    
    # 2. Converte para string e maiúsculo
    texto_tratado = str(texto).upper()
    
    # 3. Remove quebras de linha (\n ou \r) substituindo por espaço
    # Isso corrige o caso: "ANTONIO \n FLAVIO" -> "ANTONIO FLAVIO"
    texto_tratado = texto_tratado.replace('\n', ' ').replace('\r', ' ')
    
    # 4. Pega apenas a primeira parte antes da vírgula
    primeiro_nome = texto_tratado.split(',')[0]
    
    # 5. Remove espaços extras nas pontas e espaços duplos no meio
    return " ".join(primeiro_nome.split())

# Aplicando a função
aCir6['cirurgiao_principal_clean'] = aCir6['cirurgiao_principal'].apply(extrair_primeiro_cirurgiao)

# 3. Aplicação do Mapeamento
# Criamos as colunas mapeando o dicionário. 
# O [0] pega o volume e o [1] pega a experiência.

aCir6['volume_casos_anual'] = aCir6['cirurgiao_principal_clean'].map(
    lambda x: referencia_cirurgioes.get(x, [None, None])[0]
).astype('Int64') # Int64 aceita NaN (caso não encontre o nome)

aCir6['experiencia_cirurgiao'] = aCir6['cirurgiao_principal_clean'].map(
    lambda x: referencia_cirurgioes.get(x, [None, None])[1]
).astype('Int64')

# 4. Limpeza (Opcional)
# Removemos a coluna auxiliar de limpeza, se desejar
aCir6.drop(columns=['cirurgiao_principal_clean', 'adado_P_208306', 'cd_aviso_cirurgia'], inplace=True)

# --- VERIFICAÇÃO DE ERROS ---
# É importante ver se sobrou alguém sem preencher (nomes escritos errados no banco)
sem_match = aCir6[aCir6['volume_casos_anual'].isna()]['cirurgiao_principal'].unique()

if len(sem_match) > 0:
    print("ATENÇÃO: Os seguintes cirurgiões não foram encontrados no dicionário:")
    print(sem_match)
else:
    print("Sucesso! Todos os cirurgiões foram mapeados.")

Sucesso! Todos os cirurgiões foram mapeados.


In [ ]:
#aCir6.to_excel(f"C://Users/priscilla.sequetin/Downloads/aCir6.xlsx", index=False)

In [ ]:
aCir6

In [296]:
aCir6.to_excel(f"C://Users/priscilla.sequetin/Downloads/aCir6_1.xlsx", index=False)

In [326]:
tsts3 = regvalv_cirurgia[regvalv_cirurgia['procedimento_complete']=='1']

In [331]:
tsts3 = tsts3.drop(columns='idade_calc', errors='ignore')

In [328]:
tsts3.columns.to_list()

['record_id',
 'redcap_event_name',
 'redcap_repeat_instrument',
 'redcap_repeat_instance',
 'idade_calc',
 'cd_atendimento',
 'data_cirurgia',
 'descricao_cirurgia',
 'tipo_cirurgia___1',
 'tipo_cirurgia___2',
 'tipo_cirurgia___3',
 'tipo_cirurgia___4',
 'tipo_cirurgia___10',
 'tipo_cirurgia___5',
 'tipo_cirurgia___6',
 'tipo_cirurgia___7',
 'tipo_cirurgia___8',
 'tipo_cirurgia___9',
 'tipo_cirurgia_outros',
 'cirurgica_associada_encam___1',
 'cirurgica_associada_encam___2',
 'cirurgica_associada_encam___3',
 'cirurgica_associada_encam___4',
 'cirurgica_associada_encam___5',
 'cirurgica_associada_encam___6',
 'cirurgica_associada_encam___7',
 'abordagem_cirurgica',
 'abordagem_cirurgica_outros',
 'duracao_cirurgia',
 'tipo_valvula_prostetica',
 'modelo_protese',
 'tamanho_protese',
 'circulacao_extracorporea',
 'tempo_cec',
 'tempo_pinc_ao',
 'complicacoes_intraop',
 'tipo_complicacao_intraop___1',
 'tipo_complicacao_intraop___2',
 'tipo_complicacao_intraop___3',
 'tipo_complicacao_in

In [330]:
# 1. Lista exata das colunas que receberam dados indevidos
colunas_para_limpar = [
    'cd_atendimento', 'data_cirurgia', 'descricao_cirurgia', 
    'tipo_cirurgia___1', 'tipo_cirurgia___2', 'tipo_cirurgia___3', 'tipo_cirurgia___4', 'tipo_cirurgia___10', 
    'tipo_cirurgia___5', 'tipo_cirurgia___6', 'tipo_cirurgia___7', 'tipo_cirurgia___8', 'tipo_cirurgia___9', 'tipo_cirurgia_outros', 
    'cirurgica_associada_encam___1', 'cirurgica_associada_encam___2', 'cirurgica_associada_encam___3', 'cirurgica_associada_encam___4', 
    'cirurgica_associada_encam___5', 'cirurgica_associada_encam___6', 'cirurgica_associada_encam___7', 
    'abordagem_cirurgica', 'abordagem_cirurgica_outros', 'duracao_cirurgia', 'tipo_valvula_prostetica', 'modelo_protese', 
    'tamanho_protese', 'circulacao_extracorporea', 'tempo_cec', 'tempo_pinc_ao', 'complicacoes_intraop', 
    'tipo_complicacao_intraop___1', 'tipo_complicacao_intraop___2', 'tipo_complicacao_intraop___3', 'tipo_complicacao_intraop___4', 
    'tipo_complicacao_intraop___5', 'tipo_complicacao_intraop___6', 'complicacoes_intraop_outros', 
    'cirurgiao_principal', 'anestesista_principal', 'auxiliares', 'experiencia_cirurgiao', 'volume_casos_anual'
]

# 2. Criar a máscara (filtro) para isolar apenas os registros que precisam de limpeza
# Garante que seja lido como string para evitar falhas de tipagem ('1' vs 1)
mascara_limpeza = tsts3['procedimento_complete'].astype(str) == '1'

# 3. Aplicar a limpeza respeitando as regras do REDCap
for col in colunas_para_limpar:
    if col in tsts3.columns:
        if '___' in col:
            # É checkbox: força o zero para desmarcar
            tsts3.loc[mascara_limpeza, col] = 0
        else:
            # É texto/data/radio: envia string vazia
            tsts3.loc[mascara_limpeza, col] = ""

In [332]:
batch_error_cirurgia = import_redcap(
            df_sorted=tsts3,
            api_url=api_url,
            api_key=api_key,
            batch_size=20, 
            overwrite=True
        )

Batch 1/5 imported successfully. Records in this batch: 20. Total records imported: 20
Batch 2/5 imported successfully. Records in this batch: 20. Total records imported: 40
Batch 3/5 imported successfully. Records in this batch: 20. Total records imported: 60
Batch 4/5 imported successfully. Records in this batch: 20. Total records imported: 80
Batch 5/5 imported successfully. Records in this batch: 1. Total records imported: 81


In [ ]:
from IPython.display import display, HTML

# 1. Pega a lista atualizada do DataFrame
lista_ids = aCir6['record_id'].to_list()

# 2. Monta o HTML usando f-string (f"""...""") para injetar a variável {lista_ids}
html_lembrete = f"""
<div style="background-color: #fff3cd; border: 1px solid #ffeeba; color: #856404; padding: 15px; border-radius: 5px; font-family: sans-serif;">
  <h3 style="margin-top: 0; display: flex; align-items: center;">
    📌 Lembrete Dinâmico
  </h3>
  <ul style="margin-bottom: 0;">
    <li>Verificar o envio das informações de cirurgia para o Redcap, ver valores faltantes</li>
    <li>Implementar mapeamento de <code>tipo_complicacao_intraop</code> e <code>complicacoes_intraop_outros</code></li>
    
    <li style="margin-top: 10px; padding-top: 10px; border-top: 1px dashed #d3bd82;">
      <strong>Verifique RECORD_ID no REDCAP:</strong><br>
      <code style="background-color: #856404; padding: 2px 5px; border-radius: 3px; word-wrap: break-word;">
        {lista_ids}
      </code>
    </li>
  </ul>
</div>
"""

# 3. Renderiza o HTML na saída da célula
display(HTML(html_lembrete))

In [ ]:
#dPer4.to_excel(f"C://Users/priscilla.sequetin/Downloads/dPer4.xlsx", index=False)

# FISIOTERAPIA

In [ ]:
# Buscando docText.FISIO

sql_FISIO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    fde.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM ft_doc_eletronico_texto fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros
WHERE fde.CD_DOCUMENTO IN (62, 240, 764, 895, 898, 899, 900, 901, 902, 903, 904, 905, 906, 908, 1041, 1136, 1137, 1138, 1140, 1141, 1142, 1145)  
  	AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
	AND DS_RESPOSTA NOT IN ('false')
  	--AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND (YEAR(fde.DH_DOCUMENTO) >= 2025 AND MONTH(fde.DH_DOCUMENTO) >= 6)

ORDER BY fde.DH_DOCUMENTO DESC;

""" 

# Executa a função para buscar os dados
txFISIO = query_sql_to_dataframe(conn_string, sql_FISIO)

txFISIO['DS_LEITO'] = txFISIO['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)


In [ ]:
# Buscando docRadio.Fisio

sql_FISIO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    fde.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.resposta as DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM FT_DOC_ELETRONICO fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    --AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros
WHERE fde.CD_DOCUMENTO IN (62, 240, 764, 895, 898, 899, 900, 901, 902, 903, 904, 905, 906, 908, 1041, 1136, 1137, 1138, 1140, 1141, 1142, 1145)  
  	AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
    AND resposta = 'SIM'
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND (YEAR(fde.DH_DOCUMENTO) >= 2025 AND MONTH(fde.DH_DOCUMENTO) >= 6)

ORDER BY fde.DH_DOCUMENTO DESC;

""" 

# Executa a função para buscar os dados
rbFISIO= query_sql_to_dataframe(conn_string, sql_FISIO)

rbFISIO['DS_LEITO'] = txFISIO['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
# Buscando presc.Fisio

sql_FISIO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT 
    [NK_CD_ORI_ATE],
    [DS_ORI_ATE],
    [DT_ATUAL]
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fpe.CD_ATENDIMENTO,
    ate.CD_PACIENTE,
    fpe.HR_PRE_MED AS DH_DOCUMENTO,
    CAST(fpe.HR_PRE_MED AS DATE) AS DATAID,
    fpe.CD_PRE_MED AS CD_DOCUMENTO,
    fpe.TP_PRE_MED AS DS_DOCUMENTO,

    -- Fallback: se LEITO atual for vazio ou nulo, busca último anterior válido
    COALESCE(NULLIF(fiu.LEITO, ''), prev_leito.LEITO) AS DS_LEITO,

    'TX_FISIOTERAPIA_EVOLUCAO_1' AS DS_IDENTIFICADOR,
    'FISIOTERAPIA_EVOLUCAO' AS DS_CAMPO,
    fpe.DS_EVOLUCAO AS DS_RESPOSTA,
    fpe.CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Fallback: se UNIDADE_PASSAGEM estiver vazia, busca última anterior; senão origem ambulatorial
    COALESCE(NULLIF(fiu.UNIDADE_PASSAGEM, ''), prev_unid.UNIDADE_PASSAGEM, doa.DS_ORI_ATE COLLATE Latin1_General_CI_AS) AS NM_SETOR

FROM [DATADANTE].[dbo].[ft_prescricao_evolucao] fpe with(nolock)
INNER JOIN dim_prestador dpt 
    ON dpt.NK_CD_PRESTADOR = fpe.CD_PRESTADOR

-- Internação principal (Mantido para pegar leito/setor de internados)
LEFT JOIN ft_internacao_unidade fiu
    ON fpe.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   AND fpe.HR_PRE_MED >= fiu.HR_MOV_INT
   AND (fpe.HR_PRE_MED <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Fallback: leito anterior válido (Mantido)
OUTER APPLY (
    SELECT TOP 1 LEITO
    FROM ft_internacao_unidade f2
    WHERE f2.CD_ATENDIMENTO = fpe.CD_ATENDIMENTO
      AND f2.HR_MOV_INT < fpe.HR_PRE_MED
      AND f2.LEITO IS NOT NULL AND f2.LEITO <> ''
    ORDER BY f2.HR_MOV_INT DESC
) prev_leito

-- Fallback: unidade anterior válida (Mantido)
OUTER APPLY (
    SELECT TOP 1 UNIDADE_PASSAGEM
    FROM ft_internacao_unidade f3
    WHERE f3.CD_ATENDIMENTO = fpe.CD_ATENDIMENTO
      AND f3.HR_MOV_INT < fpe.HR_PRE_MED
      AND f3.UNIDADE_PASSAGEM IS NOT NULL AND f3.UNIDADE_PASSAGEM <> ''
    ORDER BY f3.HR_MOV_INT DESC
) prev_unid

-- CORREÇÃO AQUI: Trazemos o atendimento SEM restrição de tipo para garantir o CD_PACIENTE
LEFT JOIN ft_atendimento ate 
    ON fpe.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO

-- Join com origem continua, mas agora ele pode vir null ou preenchido dependendo do atendimento
LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

WHERE dpt.CONSELHO = 'CREFITO'
--AND YEAR(fpe.HR_PRE_MED) in ({ano})
AND (YEAR(fpe.HR_PRE_MED) >= 2025 AND MONTH(fpe.HR_PRE_MED) >= 6)


ORDER BY fpe.HR_PRE_MED DESC;

""" 

# Executa a função para buscar os dados
evFISIO = query_sql_to_dataframe(conn_string, sql_FISIO)

evFISIO['DS_DOCUMENTO'] = evFISIO['DS_DOCUMENTO'].apply(lambda x: 'FISIOTERAPIA')

In [ ]:
#Copiando os dfs para processamento

tDocF = txFISIO.copy()

rDocF = rbFISIO.copy()

ePrsF = evFISIO.copy()

In [ ]:
dFis = pd.concat([df for df in [tDocF, rDocF, ePrsF] if not df.empty], ignore_index=True) if any(not df.empty for df in [tDocF, rDocF, ePrsF]) else pd.DataFrame()

In [ ]:
# 🔧 PADRONIZAÇÃO DE TIPOS (ANTES do filtro)
print("🔧 Padronizando cd_atendimento/CD_ATENDIMENTO...")

# Padroniza TODOS os DataFrames para Int64 nullable
aCir6['cd_atendimento'] = pd.to_numeric(aCir6['cd_atendimento'], errors='raise').astype('Int64')
regvalv_cirurgia['cd_atendimento'] = pd.to_numeric(regvalv_cirurgia['cd_atendimento'], errors='raise').astype('Int64')
dFis['CD_ATENDIMENTO'] = pd.to_numeric(dFis['CD_ATENDIMENTO'], errors='raise').astype('Int64')

print("✅ Todos os DFs padronizados para Int64")
print(f"dFis CD_ATENDIMENTO: {dFis['CD_ATENDIMENTO'].dtype}")
print(f"aCir6 cd_atendimento: {aCir6['cd_atendimento'].dtype}")
print(f"regvalv_cirurgia cd_atendimento: {regvalv_cirurgia['cd_atendimento'].dtype}")

# 🎯 AGORA o filtro funciona perfeitamente
mask_atend = (
    regvalv_cirurgia['cd_atendimento'].notna() & 
    (regvalv_cirurgia['procedimento_complete'] != 1)
)

filtro_df = regvalv_cirurgia.loc[mask_atend, ['record_id', 'cd_atendimento', 'data_cirurgia', 'redcap_repeat_instance']].copy()
filtro_df['record_id'] = pd.to_numeric(filtro_df['record_id'], errors='raise').astype('Int64')

print(f"📊 Registros filtrados: {len(filtro_df)}")

# Concat com Int64 consistente
ids_brutos = pd.concat([
    filtro_df['cd_atendimento'].astype('Int64'), 
    aCir6['cd_atendimento'].astype('Int64')
])

set_atendimentos = set(ids_brutos.dropna().astype('Int64').unique())
print(f"🎯 Total de atendimentos únicos: {len(set_atendimentos)}")

# ✅ Filtro final NO dFis (agora compatível)
dFis1 = dFis[dFis['CD_ATENDIMENTO'].astype('Int64').isin(set_atendimentos)].copy()
print(f"✅ dFis1 filtrado: {len(dFis1)} registros")


In [ ]:
len(set_atendimentos)

In [ ]:
dFis1["DS_IDENTIFICADOR"] = dFis1["DS_IDENTIFICADOR"].str[3:-2]

In [ ]:
fisio_ls = {
 
# --- LF ---
'tosse_evol_fisio':['TOSSE_PRESENTE', 'TOSSE_AUSENTE',],
'tosse_evol_fisio_2':[ 'TOSSE_PRODUTIVA', 'TOSSE_SECA',],
'tosse_evol_fisio_3':['TOSSE_EFICAZ', 'TOSSE_INEFICAZ', 'TOSSE_POUCO_EFICAZ'],

'BIA_SN':                       		['BIA_NAO', 'BIA_SIM'],
'CUFF_RL_SN':                   		['CUFF_RL_NAO', 'CUFF_RL_SIM'],
'CUFF_TQT_SN':                  		['CUFF_TQT_NAO', 'CUFF_TQT_SIM'],
'DROGAS_VASOATIVAS_SN':      			['DROGAS_VASOATIVAS_NAO', 'DROGAS_VASOATIVAS_SIM'],
'ECMO_SN':                      		['ECMO_NAO', 'ECMO_SIM'],
'EOT_CENTRO_CIRURGICO_SN':      		['EOT_CENTRO_CIRURGICO_NAO', 'EOT_CENTRO_CIRURGICO_SIM'],
'EXTUBACAO_SN':                 		['EXTUBACAO_NAO', 'EXTUBACAO_SIM'],
'NO_SN':                        		['NO_NAO', 'NO_SIM'],
'OXIGENOTERAPIA_SN':            		['OXIGENOTERAPIA_NAO', 'OXIGENOTERAPIA_SIM'],
'POS_OPERATORIO_SN':            		['POS_OPERATORIO_NAO', 'POS_OPERATORIO_SIM'],
'VENTILACAO_MECANICA_SN':       		['VENTILACAO_MECANICA_INVASIVA_NAO', 'VENTILACAO_MECANICA_INVASIVA_SIM'],
'VENTILACAO_NAO_INVASIVA_SN':   		['VENTILACAO_NAO_INVASIVA_NAO', 'VENTILACAO_NAO_INVASIVA_SIM'],
'VIA_AEREA_ARTIFICIAL_SN':      		['VIA_AEREA_ARTIFICIAL_NAO', 'VIA_AEREA_ARTIFICIAL_SIM'],
'REINTUBACAO_SN':               		['REINTUBACAO_NAO', 'REINTUBACAO_SIM'],
'RESPIRACAO_ESPONTANEA_SN':     		['RESPIRACAO_ESPONTANEA_NAO', 'RESPIRACAO_ESPONTANEA_SIM'],
'SINAIS_DESCONFORTO_SN':        		['SINAIS_DESCONFORTO_RESPIRATORIO_NAO', 'SINAIS_DESCONFORTO_RESPIRATORIO_SIM'],
'SEDACAO_SN':							['SEDACAO_NAO_NEUROLOGICO', 'SEDACAO_SIM_NEUROLOGICO'],
'TREINAMENTO_MUSCULAR_RESPORATORIO_SN':	['TREINAMENTO_MUSCULAR_RESPORATORIO_NAO', 'TREINAMENTO_MUSCULAR_RESPORATORIO_SIM',],
'PREENSAO_PALMAR':						['NAO_TESTADO_PREENSAO_PALMAR', 'NAO_TESTADO_PRESSAO_PALMAR', 'PRESSAO_PALMAR_MEMBRO_DOMINANTE_DIREITO', 'PRESSAO_PALMAR_MEMBRO_DOMINANTE_ESQUERDO']

}

In [ ]:
fisio_rn = {
    # --- ---
    'TOSSE_PRESENTE': 1,
    'TOSSE_AUSENTE': 2,
    'TOSSE_PRODUTIVA': 1,
    'TOSSE_SECA': 2,
    'TOSSE_EFICAZ': 1,
    'TOSSE_INEFICAZ': 2,
    'TOSSE_POUCO_EFICAZ': 3,
    
	'BIA_NAO': 0, 'BIA_SIM': 1,
    'CUFF_RL_NAO': 0, 'CUFF_RL_SIM': 1,
    'CUFF_TQT_NAO': 0, 'CUFF_TQT_SIM': 1,
    'DROGAS_VASOATIVAS_NAO': 0, 'DROGAS_VASOATIVAS_SIM': 1,
    'ECMO_NAO': 0, 'ECMO_SIM': 1,
    'EOT_CENTRO_CIRURGICO_NAO': 0, 'EOT_CENTRO_CIRURGICO_SIM': 1,
    'EXTUBACAO_NAO': 0, 'EXTUBACAO_SIM': 1,
    'NO_NAO': 0, 'NO_SIM': 1,
    'OXIGENOTERAPIA_NAO': 0, 'OXIGENOTERAPIA_SIM': 1,
    'POS_OPERATORIO_NAO': 0, 'POS_OPERATORIO_SIM': 1,
    'VENTILACAO_MECANICA_INVASIVA_NAO': 0, 'VENTILACAO_MECANICA_INVASIVA_SIM': 1,
    'VENTILACAO_NAO_INVASIVA_NAO': 0, 'VENTILACAO_NAO_INVASIVA_SIM': 1,
    'VIA_AEREA_ARTIFICIAL_NAO': 0, 'VIA_AEREA_ARTIFICIAL_SIM': 1,
    'REINTUBACAO_NAO': 0, 'REINTUBACAO_SIM': 1,
    'RESPIRACAO_ESPONTANEA_NAO': 0, 'RESPIRACAO_ESPONTANEA_SIM': 1,
    'SINAIS_DESCONFORTO_RESPIRATORIO_NAO': 0, 'SINAIS_DESCONFORTO_RESPIRATORIO_SIM': 1,
    'SEDACAO_NAO_NEUROLOGICO': 0, 'SEDACAO_SIM_NEUROLOGICO': 1,
    'TREINAMENTO_MUSCULAR_RESPORATORIO_NAO': 0, 'TREINAMENTO_MUSCULAR_RESPORATORIO_SIM': 1,
    
	'NAO_TESTADO_PREENSAO_PALMAR': 0, 'NAO_TESTADO_PRESSAO_PALMAR': 0, 
    'PRESSAO_PALMAR_MEMBRO_DOMINANTE_DIREITO': 1, 'PRESSAO_PALMAR_MEMBRO_DOMINANTE_ESQUERDO': 2,
    
}

In [ ]:
# 1. Lista de identificadores já mapeados
ident_tratados = [item for sublist in fisio_ls.values() for item in sublist]

# 2. Cria colunas para os identificadores mapeados
for key in tqdm(fisio_ls.keys(), position=0, leave=False):
    dFis1[key] = None
    for item in fisio_ls[key]:
        with ipython_io.capture_output() as captured:
        	dFis1[key][dFis1['DS_IDENTIFICADOR'] == item] = item

# 3. Filtra os identificadores não tratados
nao_tratados = dFis1[~dFis1['DS_IDENTIFICADOR'].isin(ident_tratados)]

nao_tratados['DS_RESPOSTA'] = nao_tratados['DS_RESPOSTA'].apply(
    lambda x: 'SIM' if isinstance(x, str) and x.strip().lower() == 'true'
    else 'NAO' if isinstance(x, str) and x.strip().lower() == 'false'
    else x
)

# 4. Pivotando os identificadores não tratados
pivot_extra = nao_tratados.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    columns='DS_IDENTIFICADOR',
    values='DS_RESPOSTA',
    aggfunc='first'  # ou 'last', dependendo da lógica de negócio
).reset_index()

# 5. Junta com os dados principais (já com colunas criadas manualmente)
dFis2 = pd.merge(
    dFis1.drop(columns=['DS_IDENTIFICADOR', 'DS_RESPOSTA', 'DS_CAMPO']),
    pivot_extra,
    on=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    how='left'
)

# 6. Substituir os valores conforme fisio_rn (string → string)
for column in dFis2.columns:
    if column not in ['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID',
       ]:
        dFis2[column].replace(fisio_rn, inplace=True)


In [ ]:

# Aplicação
dFis3 = organizar_dados_paciente(
    dFis2,
    chaves_agrupamento=['CD_DOCUMENTO', 'CD_PACIENTE', 'CD_ATENDIMENTO', 'DH_DOCUMENTO'],
    colunas_fixas=['NM_PRESTADOR']
)

In [ ]:
# Listas de colunas de evento e hora
cols_status = [f'evento_{i}_status' for i in range(1, 15)]
cols_hora = [f'hora_{i}' for i in range(1, 15)]

# Identifica se "Intubação orotraqueal" aparece em qualquer coluna de status
# Usamos .stack() para busca eficiente em múltiplas colunas
mask_iot = regvalv_anest[cols_status].apply(lambda x: x.str.contains('Intubação orotraqueal', na=False, case=False))

# Cria flag de intubação (1 se aparecer em qualquer lugar)
regvalv_anest['iot_anest_flag'] = mask_iot.any(axis=1).astype('Int64')

# Captura a hora correspondente ao evento de IOT
def extrair_hora_iot(row):
    # Encontra o índice da primeira coluna de status que contém IOT
    indices = np.where(row[cols_status].str.contains('Intubação orotraqueal', na=False, case=False))[0]
    if len(indices) > 0:
        # Retorna a hora na mesma posição (ex: se status_3 é IOT, pega hora_3)
        return row[cols_hora[indices[0]]]
    return np.nan

regvalv_anest['hora_iot_anest'] = regvalv_anest.apply(extrair_hora_iot, axis=1)

# Reduzimos o DataFrame para o essencial antes do merge
df_anest_resumo = regvalv_anest[['record_id', 'iot_anest_flag', 'hora_iot_anest']].copy()
df_anest_resumo['record_id'] = df_anest_resumo['record_id'].astype(str)

In [ ]:
# 1. Renomear CD_PACIENTE para record_id de forma correta
if 'CD_PACIENTE' in dFis3.columns:
    dFis3 = dFis3.rename(columns={'CD_PACIENTE': 'record_id'})

# 2. Garantir que record_id seja string em ambos para o merge não falhar
dFis3['record_id'] = dFis3['record_id'].astype(str)
df_anest_resumo['record_id'] = df_anest_resumo['record_id'].astype(str)

# 3. Traz a informação da anestesia para a dFis3
# Usamos left merge para manter todos os registros da dFis3
dFis3 = dFis3.merge(df_anest_resumo, on='record_id', how='left')

# --- Ajuste da Lógica 1: IOT (Híbrida) ---
col_iot = "DATA_IOT"
if col_iot in dFis3.columns:
    # Intubado se: Tem DATA_IOT na dFis3 OU flag de IOT na anestesia é 1
    dFis3["paciente_intubado"] = (
        (dFis3[col_iot].notna() & (dFis3[col_iot].astype(str).str.strip() != "")) | 
        (dFis3["iot_anest_flag"] == 1)
    ).astype('Int64')

# --- Ajuste da Lógica 3: HORA (Híbrida) ---
if 'HORA_IOT' in dFis3.columns and 'MINUTOS_IOT' in dFis3.columns:
    # Calcula a hora vinda da dFis3
    hora_fisio = (
        dFis3['HORA_IOT'].fillna(0).astype(str).str.split('.').str[0].str.zfill(2) + ":" +
        dFis3['MINUTOS_IOT'].fillna(0).astype(str).str.split('.').str[0].str.zfill(2)
    )
    
    # Se a hora da fisio for "00:00", tentamos usar a hora capturada da anestesia
    dFis3['hora_iot'] = np.where(
        (hora_fisio == "00:00") & (dFis3['hora_iot_anest'].notna() & (dFis3['hora_iot_anest'] != "")),
        dFis3['hora_iot_anest'],
        hora_fisio
    )
    
    # Removendo colunas temporárias para manter o DF limpo
    cols_para_remover = ['iot_anest_flag', 'hora_iot_anest']
    dFis3.drop(columns=[c for c in cols_para_remover if c in dFis3.columns], inplace=True)


In [ ]:
def unificar_avaliacao(row):
    # 1. Defina as colunas que deseja mesclar
    colunas = ['AVALIACAO_GERAL', 'AVALIACAO_GERAL_EVOLUCAO_FISIOTERAPIA', 'ANDENDO_FISIOTERAPIA', 
            'CONDUTA_RESPIRATORIA', 'DESCRICAO_CONDUTA_MOTORA_CAIXA_TEXTO', 'META_TERAPEUTICA', 
            'VISITA_MULTIPROFISSIONAL_EVOLUCAO',]
    
    valores_tratados = []
    
    for col in colunas:
        val = row.get(col) # .get evita erro se a coluna não existir
        
        # 2. Verifica se é válido (não nulo e não vazio)
        if pd.notna(val) and str(val).strip() != "":
            # 3. Normalização: Tudo para maiúsculo e sem espaços nas pontas
            texto_limpo = str(val).upper().strip()
            valores_tratados.append(texto_limpo)
    
    # 4. Set para remover duplicatas exatas e conversão de volta para lista
    # Ex: ["BOM", "BOM"] vira ["BOM"]
    # Ex: ["REGULAR", "BOM"] vira ["REGULAR", "BOM"]
    valores_unicos = sorted(list(set(valores_tratados)))
    
    # 5. Concatena os valores únicos com uma barra ou espaço
    if not valores_unicos:
        return np.nan # Ou "" se preferir string vazia
        
    return " / ".join(valores_unicos)

# --- APLICAÇÃO NO DATAFRAME ---

# Supondo que seu df se chame 'dFis3' ou 'dFis5'
# Certifique-se que as colunas existem antes de aplicar
cols_origem = ['AVALIACAO_GERAL', 'AVALIACAO_GERAL_EVOLUCAO_FISIOTERAPIA', 'ANDENDO_FISIOTERAPIA', 
        'CONDUTA_RESPIRATORIA', 'DESCRICAO_CONDUTA_MOTORA_CAIXA_TEXTO', 'META_TERAPEUTICA',
        'VISITA_MULTIPROFISSIONAL_EVOLUCAO',]

# Verifica se as colunas existem para evitar KeyError
cols_existentes = [c for c in cols_origem if c in dFis3.columns]

if cols_existentes:
    dFis3['avaliacao_geral'] = dFis3.apply(unificar_avaliacao, axis=1)
    
    # Opcional: Se quiser remover as colunas originais antigas para limpar
    dFis3.drop(columns=cols_origem, inplace=True, errors='ignore')
    
    print("Colunas unificadas com sucesso!")
    #print(dFis3[['record_id', 'avaliacao_geral']].head())
else:
    print("Colunas de avaliação não encontradas no DataFrame.")

In [ ]:
map_cols = {
    "SEDACAO_UTILIZADA":                     "sedacao_utilizada_evol_fisio",
    "DROGAS_VASOATIVAS":                     "quais_drogas_vasoativas_evol_fisio",
    "SINAL":                                 "sinal_evol_fisio",
    "TIPO_OXIGENOTERAPIA":                   "tipo_oxigenoterapia_evol_fisio",
    "OFERTA_TIPO":                           "oferta_oxigenoterapia_evol_fisio",
    "MODO_VENTILACAO_NAO_INVASIVA":          "modo_ventilacao_n_invasiva_evol_fisio",
    "MODO_VENTILACAO_MECANICA_INVASIVA":     "modo_vent_inv_evol_fisio",
    "DATA_EXTUBACAO":                        "fisio_dt_eot",
    "SUPORTE_APOS_EOT":                      "fisio_suporte_eot",
    'SEDACAO_SN':							 'sedacao_evol_fisio',
    'DROGAS_VASOATIVAS_SN':					 'drogas_vasoativas_evol_fisio',
    'RESPIRACAO_ESPONTANEA_SN':				 'respiracao_espontanea_evol_fisio',
 	'SINAIS_DESCONFORTO_SN':				 'sinais_desconforto_resp_evol_fisio',
    'OXIGENOTERAPIA_SN':					 'oxigenoterapia_evol_fisio',
    'VENTILACAO_NAO_INVASIVA_SN':			 'ventilacao_nao_invasiva_evol_fisio',
    'VIA_AEREA_ARTIFICIAL_SN':				 'via_aerea_artificial_evol_fisio',
 	'VENTILACAO_MECANICA_SN':				 'vent_mec_inv_evol_fisio',
    'EXTUBACAO_SN':							 'fisio_eot',
    'BIA_SN':								 'fisio_bia',
 	'NO_SN':								 'fisio_no',
 	'ECMO_SN':								 'fisio_ecmo',
    'TA_EVOLUCAO': 							 'tempo_pinc_ao',
    'TP_EVOLUCAO': 							 'tempo_cec',
    'CD_ATENDIMENTO': 						 'cd_atendimento',
    'INTERFACE':							 'interface_evol_fisio',
    'TREINAMENTO_MUSCULAR_RESPORATORIO_SN':	 'treino_musc_respiratorio',
    'REINTUBACAO_SN':						 'fisio_reiot',
    'PREENSAO_PALMAR':						 'preensao_palmar_fisio',
    
}

In [ ]:
dFis3 = dFis3.rename(columns=map_cols).copy()

In [ ]:
# Verifica se a coluna existe antes de tentar processar
if 'MOTIVO_EXTUBACAO' in dFis3.columns:
    
    # 1. Converte para string e quebra onde houver '||', pegando a primeira parte
    dFis3['MOTIVO_EXTUBACAO'] = dFis3['MOTIVO_EXTUBACAO'].astype(str).str.split('||').str[0]

    # 2. Converte para número (o que for 'nan' ou erro vira NaN real do Pandas)
    dFis3['MOTIVO_EXTUBACAO'] = pd.to_numeric(dFis3['MOTIVO_EXTUBACAO'], errors='coerce')

    # 3. Converte para inteiro que aceita nulos (remove o .0 visual)
    dFis3['MOTIVO_EXTUBACAO'] = dFis3['MOTIVO_EXTUBACAO'].astype('Int64')
    
else:
    # Opcional: Aviso caso a coluna não exista
    print("A coluna 'MOTIVO_EXTUBACAO' não foi encontrada no DataFrame.")
    


In [ ]:
dFis3['redcap_repeat_instrument'] = 'fisioterapia'

In [ ]:
COLUNAS_PARA_REMOVER = [
    'NM_PRESTADOR', 'DS_LEITO', 'CD_PRESTADOR', 'CUFF_RL_SN', 'ANO',
    'CUFF_TQT_SN', 'EOT_CENTRO_CIRURGICO_SN', 'POS_OPERATORIO_SN', 'ADEQUADO_DESENVOLVIMENTO', 'ADMISSAO',
    'ALIMENTACAO_FSS', 'ALTURA_TESTE_CAMINHADA', 'AMPLITUDE_MOVIMENTO', 'ANTECEDENTES_PESSOAIS_EVOLUCAO',
    'ANTECEDENTES_PESSOAIS_SETOR_ORIGEM', 'ASSISTENCIA', 'ATENDIMENTO', 'AUSCULTA_PULMONAR_EVOLUCAO', 
    'BORG_DISPNEIA_DURANTE', 'BORG_DISPNEIA_DURANTE_CONDUTA_MOTORA', 'BORG_DISPNEIA_FINAL', 'BORG_DISPNEIA_FINAL_CONDUTA_MOTORA',
    'BORG_DISPNEIA_INICIO', 'BORG_DISPNEIA_INICIO_CONDUTA_MOTORA', 'BORG_FADIGA_DURANTE_CONDUTA_MOTORA', 
    'BORG_FADIGA_FINAL_CONDUTA_MOTORA', 'BORG_FADIGA_INICIO_CONDUTA_MOTORA', 'CARGA_TREINAMENTO_MUSCULAR', 
    'CIRURGIA_EVOLUCAO', 'COAGULOGRAMA_EXAMES_LABORATORIAIS', 'COMUNICACAO_FSS', 'DEFICIT_MOTOR', 'DEFICIT_MOTOR_AUSENTE',
    'DEFICIT_MOTOR_NAO_AVALIADO', 'DEFICIT_MOTOR_PRESENTE', 'DIAGNOSTICO_BASE', 'ESTADO_MENTAL_FSS', 'FADIGA_DURANTE',
    'FADIGA_FINAL', 'FADIGA_INICIO', 'FC_EVOLUCAO', 'FIO2_SWEEP', 'FIO2_VENTILACAO_NAO_INVASIVA', 'FLUXO_VPM',
    'FORCA_MUSCALAR_MIE', 'FORCA_MUSCULAR_MID', 'FORCA_MUSCULAR_MSD', 'FORCA_MUSCULAR_MSE', 'FR_AUSCULTA_PULMONAR', 'FR_VENTILACAO_NAO_INVASIVA',
    'FUNCAO_MOTORA_FSS', 'GASONOMETRIA_ARTERIAL', 'GLASGOW_EVOLUCAO', 'HEMOGRAMA_EXAMES_LABORITARIAIS',
    'HORARIO_HORA', 'HORARIO_MINUTO', 'NAO_AVALIADO_DESENVOLVIMENTO', 'NAO_TESTADO_PREENSAO_PALMAR', 'NAO_TESTADO_PRESSAO_PALMAR',
    'P0_1', 'PAM_EVOLUCAO', 'PAO2_FIO2_VPM', 'PA_EVOLUCAO', 'PEEP_MODO_PARAMETROS', 'PEEP_VENTILACAO_NAO_INVASIVA', 
    'PERME', 'PESO_TESTE_CAMINHA', 'PINSP_VENTILACAO_NAO_INVASIVA', 'PPICO_VPM',
    'PRESSAO_CUFF', 'PRESSAO_PALMAR_MEMBRO_DOMINANTE_DIREITO', 'RAIO_X_EXAMES_IMAGEM', 'RASS_EVOLUCAO', 'REPETICAO_TREINAMENTO_MUSCULAR',
    'RESPIRATORIO_FSS', 'RL', 'SENSORIAL_FSS', 'SERIE_TREINAMENTO_MUSCULAR', 'SETOR_DE_ORIGEM', 'SITUACAO_ATUAL',
    'SPO2_ALVO', 'SPO2_EVOLUCAO', 'SWEEP', 'TINSP_VENTILACAO_NAO_INVASIVA', 'TONUS_MUSCULAR', 'TOTAL_FSS',
    'VALOR_PREDITO_TESTE_CAMINHADA', 'VO2', 'VOLUME_CORRENTE_ALVO_VPM', 
    'DS_DOCUMENTO', 'NM_SETOR', 'SENSIBILIDADE_VPM', 'BIA', 'NO', 'DATA_INICIO_VPM', 'DATA_INTERRUPCAO_VPM', 'DESMAME_VENTILATORIO',
    'EOT_PO', 'GRAU_DEPENDENCIA', 'INDICE_TOBIM', 'JOHN_HOPKINS', 'MUSCULAR_OUTROS', 'OUTROS_EXAMES_LABORITORIAIS',
    'POS_OPERATORIO_DATA', 'TIPO_VENO_ARTERIAL', 'NUMERO_COT', 'NUMERO_TQT', 'TUBO_T', 'MINUTOS_IOT'


]

In [ ]:
# 1. Vincular (Traz data_cirurgia e redcap_repeat_instance para dentro do dFis)
dFis4 = vincular_cirurgia_por_atendimento(
    df_itens=dFis3,
    df_cirurgia=regvalv_cirurgia,
    col_atendimento_item='cd_atendimento',
    col_atendimento_cirurgia='cd_atendimento', # Verifique se no seu CSV está minúsculo ou maiúsculo
    col_data_item='DH_DOCUMENTO',
    col_data_cirurgia='data_cirurgia',
    col_instancia_cirurgia='redcap_repeat_instance'
)

# 2. Classificar (Define o redcap_event_name baseado no tempo e local)
dFis5 = classificar_eventos_por_atendimento(
    df_entrada=dFis4,
    evento_pre='fase2preprocedimen_arm_1',
    evento_uti='fase4posprocedimed_arm_1',
    evento_enfermaria='fase5recuperacaoin_arm_1',
    evento_amb='fase6segalta30dias_arm_1',
    col_criterio_local='CD_DOCUMENTO',     # Pode ser 'CD_DOCUMENTO' ou 'NM_SETOR'
    val_uti=(1140, 1145, 1137),            # Pode ser tupla de IDs ou string 'UTI'
    val_enfermaria=(1136, 1141)      	   # Ajuste seus códigos aqui
)

In [ ]:
# Sintaxe: df.loc[FILTRO, 'COLUNA'] = NOVO_VALOR
# PAliativo, fisioterapia nao vai atender fase pre

dFis5.loc[dFis5["redcap_event_name"] == "fase2preprocedimen_arm_1", "redcap_event_name"] = "fase4posprocedimed_arm_1"

In [ ]:
# 1. Garantir a ordenação sequencial
# Mesmo não filtrando por data, ordenamos para que o python saiba qual é a "última" linha
# Se 'dataid' for sequencial, usamos ele. Caso contrário, pode-se usar o índice original.
dFis5 = dFis5.sort_values(['record_id', 'redcap_event_name', 'DATAID'], ascending=True)

# 2. Agrupamento e Consolidação
# Agrupamos por Paciente e Fase
# O método .last() pega o último valor não-nulo encontrado na coluna para aquele grupo.
# Isso resolve o problema de ter dados espalhados em linhas diferentes (como na sua imagem).
dFis5 = dFis5.groupby(['record_id', 'redcap_event_name'], as_index=False).last()

In [ ]:
# --- AQUI É O MELHOR LUGAR PARA LIMPAR ---
print(">>> 3.1 Limpando colunas desnecessárias...")
# Nota: Removi o inplace=True e usei atribuição direta, é mais seguro em pipelines
dFis5 = dFis5.drop(columns=COLUNAS_PARA_REMOVER, errors='ignore')

for i in COLUNAS_PARA_REMOVER:
	print(f'{i},')


In [ ]:
dFis5.columns = dFis5.columns.str.lower()

In [ ]:
# --- 1. PREPARAÇÃO DOS DADOS DO REDCAP (EXISTENTES) ---

# Garantir que record_id seja string em ambos os lados para evitar erros de comparação ( "100" != 100 )
regvalv_fisio['record_id'] = regvalv_fisio['record_id'].astype(str)

# Criamos um "Set" de tuplas com (ID, EVENTO). 
# Isso funciona como uma lista de presença: "O paciente X já tem ficha no evento Y"
registros_existentes = set(zip(
    regvalv_fisio['record_id'].astype(str), 
    regvalv_fisio['redcap_event_name'].astype(str),
    regvalv_fisio['redcap_repeat_instance'].astype(str)
    ))

print(f"Total de registros já existentes no Redcap: {len(registros_existentes)}")

In [ ]:
# --- 3. FILTRANDO OS DADOS LOCAIS ---

# Filtra cada fase
dFis5 = filtrar_novos_registros(dFis5, registros_existentes)

# Exibe o resultado
print(f"Novos registros: {len(dFis5)}")


In [ ]:
posop_uti = dFis5.copy()

cols_uti = [
    'record_id',
    'redcap_repeat_instrument',
    'redcap_event_name',
    'redcap_repeat_instance',
    'dataid',
    'paciente_intubado',
    'sedacao_evol_fisio',
    'sedacao_utilizada_evol_fisio',
    'drogas_vasoativas_evol_fisio',
    'quais_drogas_vasoativas_evol_fisio',
    'fisio_bia',
    'fisio_no',
    'fisio_ecmo',
]

cols_uti = [c for c in cols_uti if c in posop_uti.columns]
posop_uti = posop_uti[cols_uti]

# 1. Substituir 'fisioterapia' por 'pos_operatorio' na coluna redcap_repeat_instrument
posop_uti.loc[posop_uti['redcap_repeat_instrument'] == 'fisioterapia', 'redcap_repeat_instrument'] = 'pos_operatorio'

# 2. Substituir o event_name específico
posop_uti.loc[posop_uti['redcap_event_name'] == 'fase5recuperacaoin_arm_1', 'redcap_event_name'] = 'fase4posprocedimed_arm_1'


posop_uti = posop_uti.sort_values(['record_id', 'dataid'], ascending=True)

posop_uti = posop_uti.groupby(['record_id'], as_index=False).last()

posop_uti = posop_uti.drop(columns='dataid', errors='ignore')


In [ ]:
primeiras_fases = dFis5.copy()

cols = [
    'record_id',
    'redcap_repeat_instrument',
    'redcap_event_name',
    'redcap_repeat_instance',
    'avaliacao_geral',
    'respiracao_espontanea_evol_fisio',
    'interface_evol_fisio',
    'sinais_desconforto_resp_evol_fisio',
    'sinal_evol_fisio',
    'tosse_evol_fisio',
 	'tosse_evol_fisio_2',
 	'tosse_evol_fisio_3',
    'oxigenoterapia_evol_fisio',
	'tipo_oxigenoterapia_evol_fisio',
    'oferta_oxigenoterapia_evol_fisio',
    'ventilacao_nao_invasiva_evol_fisio',
    'modo_ventilacao_n_invasiva_evol_fisio',
    'via_aerea_artificial_evol_fisio',
    'vent_mec_inv_evol_fisio',
    'modo_vent_inv_evol_fisio',
    'fisio_eot',
    'fisio_dt_eot',
    'fisio_suporte_eot',
    'peep_vpm',
 	'pemax',
 	'pimax',
 	'pinsp_vpm',
    'fr_vpm',
    'feeling_scale',
 	'fio2_vpm',
 	'hora_iot',
 	'tinsp_vpm',
 	'volume_corrente_vpm',
 	'escala_ims',
    'preensao_palmar_fisio',
]

cols = [c for c in cols if c in primeiras_fases.columns]
primeiras_fases = primeiras_fases[cols]

In [ ]:
# Lista das colunas que deram erro no log
colunas_categoricas = [
    'sedacao_evol_fisio',
    'drogas_vasoativas_evol_fisio',
    'respiracao_espontanea_evol_fisio',
    'sinais_desconforto_resp_evol_fisio',
    'tosse_evol_fisio',
    'tosse_evol_fisio_2',
    'tosse_evol_fisio_3',
    'oxigenoterapia_evol_fisio',
    'ventilacao_nao_invasiva_evol_fisio',
    'via_aerea_artificial_evol_fisio',
    'fisio_bia',
    'fisio_no',
    'fisio_ecmo',
    'fisio_eot',
    'vent_mec_inv_evol_fisio',
    'paciente_intubado',
    'preensao_palmar_fisio'
]

# Lista específica para o replace de 0 por vazio
colunas_tosse = ['tosse_evol_fisio', 'tosse_evol_fisio_2', 'tosse_evol_fisio_3']

# Aplicar a correção
for col in colunas_categoricas:
    if col in primeiras_fases.columns:
        # 1. Aplica a limpeza padrão (remove .0, etc)
        primeiras_fases[col] = primeiras_fases[col].apply(limpar_formato_redcap)
        
        # 2. Se for uma coluna de tosse, remove os zeros
        if col in colunas_tosse:
            # Substitui tanto o número 0 quanto a string '0' por vazio
            primeiras_fases[col] = primeiras_fases[col].replace(['0', 0, '0.0'], '')

print("✅ Correção aplicada e valores '0' removidos das colunas de tosse.")

In [ ]:
for col in colunas_categoricas:
    if col in posop_uti.columns:
        posop_uti[col] = posop_uti[col].apply(limpar_formato_redcap)

In [ ]:
#primeiras_fases.to_excel(f"C://Users/priscilla.sequetin/Downloads/fisio_fases_9.xlsx", index=False)

In [ ]:
# # Lista expandida com campos dropdown fisio
# CAMPOS_SEM_ZERO = {
    
#     # Campos fisio (NOVOS)
#     'tosse_evol_fisio', 'tosse_evol_fisio_2', 'tosse_evol_fisio_3',
#     # Adicione outros conforme necessário: 'dor_evol_fisio', 'mobilidade_evol_fisio', etc.
# }

# # A lógica de limpeza permanece a MESMA (já funciona!)
# payload = primeiras_fases.to_dict(orient='records')  # seu DataFrame de fisio
# payload_limpo = []

# for registro in payload:
#     clean_reg = {}
#     for k, v in registro.items():
#         # Campos de controle SEMPRE incluídos
#         if k in ['record_id', 'redcap_repeat_instance', 'redcap_event_name', 
#                 'redcap_repeat_instrument', 'procedimento_complete']:
#             clean_reg[k] = str(v)
        
#         # Checkbox (___): mantém "0" ou "1"
#         elif "___" in k:
#             clean_reg[k] = str(v) if pd.notna(v) else "0"
        
#         # CAMPOS_SEM_ZERO: NUNCA envia "0" - só valor válido ou OMITE
#         elif k in CAMPOS_SEM_ZERO:
#             if pd.notna(v) and v is not None and str(v).strip() != "" and str(v).strip() != "0":
#                 clean_reg[k] = str(v)
#             # Senão, OMITE completamente (não inclui a chave)
        
#         # Outros campos: só inclui se válido
#         else:
#             if pd.notna(v) and v is not None and str(v).strip() != "" and str(v).strip() != "0":
#                 clean_reg[k] = str(v)
    
#     payload_limpo.append(clean_reg)

# print(f"📋 Exemplo payload (1º registro): {payload_limpo[0]}")
# print("🚀 Enviando para REDCap...")


In [ ]:
# --- COMO USAR NO SEU FLUXO ---

# Agora você importa 
import_redcap(
    df_sorted=primeiras_fases,
    api_url=api_url,
    api_key=api_key,
    batch_size=10,
    overwrite=False
)

#primeiras_fases['record_id'].unique().tolist()

In [ ]:

batch_error = import_redcap(primeiras_fases, api_url, api_key)
ids = extract_locked_ids_only(batch_error)
primeiras_fases_final = primeiras_fases[~primeiras_fases['record_id'].isin(ids)]
import_redcap(primeiras_fases_final, api_url, api_key)

print(ids)

In [ ]:
# --- COMO USAR NO SEU FLUXO ---

# Agora você importa 
import_redcap(
    df_sorted=posop_uti,
    api_url=api_url,
    api_key=api_key,
    batch_size=10,
    overwrite=False
)

#primeiras_fases['record_id'].unique().tolist()

## 'TA_EVOLUCAO', 'TP_EVOLUCAO' Tempo de CEC e Anoxia da ficha de fisioterapia

In [ ]:
# Lista de colunas desejadas
cols_tcec = ['record_id', 'redcap_repeat_instance', 'cd_atendimento', 
             'dh_documento', 'tempo_pinc_ao', 'tempo_cec']

# Verifica se TODAS as colunas da lista existem em dFis5
if set(cols_tcec).issubset(dFis5.columns):
    tCec = dFis5[cols_tcec].copy()
else:
    print("Aviso: Colunas para tCec não encontradas. Criando DataFrame vazio para evitar erros.")
    # Cria vazio apenas com os cabeçalhos para não quebrar códigos futuros
    tCec = pd.DataFrame(columns=cols_tcec)
    

In [ ]:
# 1. Renomear colunas (Mantido)


# 2. Função de limpeza (Mantido)
def limpar_valor_robusto(valor):
    s_val = str(valor).lower().strip()
    # Se for lixo ou vazio, retorna NaN
    if s_val in ['', 'nan', 'na', 'none', 'nat', 'ni', 'None']:
        return np.nan

    # Busca padrão numérico
    match = re.search(r'(\d+([.,]\d+)?)', s_val)
    
    if match:
        numero_limpo = match.group(0).replace(',', '.') 
        try:
            return int(float(numero_limpo))
        except ValueError:
            return np.nan     
    return np.nan

# 3. Aplicar a limpeza
cols_metricas = ['tempo_pinc_ao', 'tempo_cec']
for col in cols_metricas:
    tCec[col] = tCec[col].apply(limpar_valor_robusto)

# 4. Ordenar e Agrupar
tCec.sort_values(by=['record_id', 'cd_atendimento', 'dh_documento'], inplace=True)

tCec1 = tCec.groupby(
    ['record_id', 'redcap_repeat_instance', 'cd_atendimento'], 
    as_index=False
)[cols_metricas].first()

# --- CORREÇÃO PRINCIPAL AQUI ---

# 5. REMOVER LINHAS VAZIAS (O Pulo do Gato)
# Removemos a linha se AMBAS as colunas de tempo forem NaN (how='all')
tCec1.dropna(subset=cols_metricas, how='all', inplace=True)

# 6. Calcular a flag de Circulação Extracorpórea
# Como já removemos quem não tem nada, verificamos a regra de negócio:
# Se tem tempo de CEC ou Pinçamento, consideramos flag 1.
# (Aqui usamos max(..., 0) para garantir que NaN vire 0 na conta da flag)
tCec1['circulacao_extracorporea'] = tCec1[cols_metricas].notna().any(axis=1).astype(int)

# 7. Formatação final para String (Apenas para o que sobrou)
# Convertemos para Int64 primeiro (para tirar o .0) e depois para string
for col in cols_metricas:
    tCec1[col] = tCec1[col].astype('Int64').astype(str).replace('<NA>', '')

# Visualização para conferência
print(f"Total de registros limpos: {len(tCec1)}")

In [ ]:
tCec1['redcap_event_name'] = 'fase3procedcirurgi_arm_1'
tCec1['redcap_repeat_instrument'] = 'procedimento'



In [ ]:
# --- 1. PREPARAR O ALVO (REDCap) ---
# Filtramos no DF do Redcap apenas as linhas onde 'tempo_cec' está vazio
# Se quiser considerar também o status, adicione: | (regvalv_cirurgia['procedimento_complete'] != 2)
mask_vazios = (regvalv_cirurgia['tempo_cec'] == "") | (regvalv_cirurgia['tempo_cec'].isna())
df_alvos_vazios = regvalv_cirurgia.loc[mask_vazios, ['record_id', 'redcap_repeat_instance']].copy()

# --- 2. PADRONIZAR TIPAGEM (CRUCIAL) ---
# Para o cruzamento funcionar, ID deve ser string e Instância deve ser Inteiro em AMBOS
def padronizar_chaves(df):
    df['record_id'] = df['record_id'].astype(str)
    # Converte para numérico primeiro para garantir, depois para int
    df['redcap_repeat_instance'] = pd.to_numeric(df['redcap_repeat_instance'], errors='coerce').fillna(1).astype('Int64')
    return df

df_alvos_vazios = padronizar_chaves(df_alvos_vazios)
tCec1 = padronizar_chaves(tCec1)

# --- 3. FILTRAGEM VIA MERGE ---
# O 'inner' join funciona como um filtro: mantém apenas as linhas de tCec1
# que possuem correspondência exata de (id + instancia) na tabela df_alvos_vazios
tCec2 = tCec1.merge(
    df_alvos_vazios, 
    on=['record_id', 'redcap_repeat_instance'], 
    how='inner'
)

# --- VALIDAÇÃO ---
print(f"Total original tCec1: {len(tCec1)}")
print(f"Instâncias vazias no Redcap: {len(df_alvos_vazios)}")
print(f"Total filtrado para envio (tCec2): {len(tCec2)}")

In [ ]:
# --- COMO USAR NO SEU FLUXO ---


# Agora você importa 
import_redcap(
    df_sorted=tCec2,
    api_url=api_url,
    api_key=api_key,
    batch_size=10,
    overwrite=False
)

print(tCec2['record_id'].tolist())

In [ ]:
#tCec1.to_excel(f"C://Users/priscilla.sequetin/Downloads/fisio_tCec1.xlsx", index=False)

## Fisio Manejo Respiratório e Motor

In [ ]:
dFis5.columns.to_list()

In [ ]:
# Reorganizamos para separar Mobilidade Real de Exercícios
# Nota: Baixei 'Exercício Resistido' para nível 1 pois pode ser feito no leito.
# O ideal é que exercício resistido não defina o marco motor, mas sim a postura.
# 1. Configuração dos Pesos e Descrições (Hierarquia Fina)
# Usamos decimais para desempatar prioridades dentro do mesmo nível
config_marcos = {
    # Nível 5 - Deambulação
    "deambulacao":          {'peso': 5.1, 'desc': "Deambulação"},
    "deambulacao_evolucao": {'peso': 5.0, 'desc': "Deambulação"},
    
    # Nível 4 - Ortostatismo
    "ortostatismo":         {'peso': 4.1, 'desc': "Ortostatismo"},
    "ortostase":            {'peso': 4.0, 'desc': "Ortostatismo"},
    
    # Nível 3 - Sedestação Fora (Ciclo ganha de Poltrona simples)
    "cicloergometro":       {'peso': 3.5, 'desc': "Cicloergômetro"},
    "sedestacao_poltrona":  {'peso': 3.1, 'desc': "Sedestação em Poltrona"},
    "poltrona":             {'peso': 3.0, 'desc': "Sedestação em Poltrona"},
    
    # Nível 2 - Sedestação Beira
    "sedestacao_beira_leito": {'peso': 2.1, 'desc': "Sedestação Beira Leito"},
    "sedestacao_beira":       {'peso': 2.0, 'desc': "Sedestação Beira Leito"},
    
    # Nível 1 - Leito (Exercício ganha de passivo)
    #"EXERCICIO_ATIVO_LIVRE": {'peso': 1.5, 'desc': "Restrito ao Leito (Com Exercícios)"},
    #"EXERCICIO_RESISTIDO":   {'peso': 1.5, 'desc': "Restrito ao Leito (Com Exercícios)"},
    "mudanca_decubito":      {'peso': 1.0, 'desc': "Mudança de Decúbito"},
    "dependente_leito":      {'peso': 0.5, 'desc': "Dependente ao Leito"}
}

# Lista apenas das colunas que existem no seu DataFrame atual
cols_interesse = [c for c in config_marcos.keys() if c in dFis5.columns]

# --- PASSO 1: UNPIVOT (MELT) ---
# Criamos um DF "Long" apenas com ID e as colunas de motor
# Assumindo que 'record_id' e 'redcap_repeat_instance' (ou dh_documento) são seus identificadores únicos
cols_id = ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance'] # Adicione 'dh_documento' se ele for chave única

# --- PASSO A: UNPIVOT (MELT) ---
df_melt = dFis5[cols_id + cols_interesse].melt(
    id_vars=cols_id, 
    value_vars=cols_interesse, 
    var_name='variavel_motor', 
    value_name='valor'
)

# --- PASSO B: CALCULAR O "VENCEDOR" ---
# Filtra apenas SIM
df_melt = df_melt[df_melt['valor'].astype(str).str.upper() == 'SIM'].copy()

# Mapeia pesos e descrições
df_melt['peso'] = df_melt['variavel_motor'].map(lambda x: config_marcos.get(x, {}).get('peso', 0))
df_melt['descricao_marco'] = df_melt['variavel_motor'].map(lambda x: config_marcos.get(x, {}).get('desc', ''))

# Ordena: ID, Evento, Instância... e PESO Decrescente
df_melt.sort_values(by=cols_id + ['peso'], ascending=(True, True, True, True, False), inplace=True)

# Pega o campeão de cada linha (remove duplicatas mantendo o primeiro/maior peso)
df_campeoes = df_melt.drop_duplicates(subset=cols_id, keep='first')

# --- PASSO C: MERGE E SELEÇÃO FINAL ---
# Faz o merge de volta com o original
dFis6 = dFis5.merge(
    df_campeoes[cols_id + ['descricao_marco']], 
    on=cols_id, 
    how='left'
)

# Preenche vazios
dFis6['descricao_marco'] = dFis6['descricao_marco'].fillna("Não Avaliado")

# --- PASSO D: FILTRAR COLUNAS FINAIS ---
colunas_saida = [
    'record_id',
    'redcap_repeat_instrument', 
    'redcap_event_name', 
    'redcap_repeat_instance',
    'descricao_marco'
]

# Cria o DF final apenas com o que você pediu
dFis6 = dFis6[colunas_saida]

print("Classificação concluída!")

In [ ]:
lista_fisio = primeiras_fases['record_id'].unique().tolist()

dFis7 = dFis6[dFis6['record_id'].isin(lista_fisio)]

In [ ]:
# --- COMO USAR NO SEU FLUXO ---


# Agora você importa 
import_redcap(
    df_sorted=dFis7,
    api_url=api_url,
    api_key=api_key,
    batch_size=10,
    overwrite=False
)

## INTERCORRENCIAS FISIO

In [ ]:
posop_uti.columns.tolist()

In [ ]:
primeiras_fases.columns.tolist()

In [ ]:
tCec2.columns.tolist()

In [ ]:
dFis7.columns.tolist()

In [ ]:
# 2. Definição das Chaves (O que DEVE ficar)
cols_chaves = ['record_id', 'redcap_repeat_instrument', 'redcap_event_name', 'redcap_repeat_instance']

# 3. Construção da Lista de Exclusão (O que DEVE sair)
# Começa com as colunas dos DataFrames já processados
lista_exclusao = (
    primeiras_fases.columns.tolist() + 
    tCec2.columns.tolist() + 
    dFis7.columns.tolist() +
    posop_uti.columns.tolist()
)

# Transforma em SET para otimizar e remover duplicatas
cols_para_excluir = set(lista_exclusao)

# Adiciona as colunas MOTORAS (Garante Maiúsculas e Minúsculas para segurança)
chaves_motor = list(config_marcos.keys())
cols_para_excluir.update(chaves_motor)                 # Adiciona ex: "DEAMBULACAO"
cols_para_excluir.update([k.lower() for k in chaves_motor]) # Adiciona ex: "deambulacao"

# 4. Proteção das Chaves
# Remove as colunas chave da lista de exclusão, caso elas tenham entrado lá acidentalmente
for chave in cols_chaves:
    if chave in cols_para_excluir:
        cols_para_excluir.remove(chave)

# 5. Filtragem Final no dFis5
# Mantém a coluna SE for chave OU SE NÃO estiver na lista de exclusão
cols_finais = [col for col in dFis5.columns if col in cols_chaves or col not in cols_para_excluir]

dFis8 = dFis5[cols_finais].copy()

# --- Verificação ---
print(f"Total de colunas originais (dFis5): {dFis5.shape[1]}")
print(f"Total de colunas excluidas (lista_exclusao): {len(cols_para_excluir)}")
print(f"Total de colunas restantes (dFis8): {dFis8.shape[1]}")
print("\nColunas disponíveis para a análise final:")
print(dFis8.columns.tolist())

In [ ]:
# --- ETAPA 1: Mapeamento Completo (Ativando todos os campos) ---

# Note que descomentamos tudo. Se a coluna 'estridor' não existir no dFis8,
# o código vai criar a coluna de destino preenchida com 0.

map_reintubacao = {
    'motivo_desconforto_ventilatorio': 'motivo_reintubacao___1',
    'motivo_estridor': 'motivo_reintubacao___2',                      # Ativado
    'motivo_eletivo': 'motivo_reintubacao___3',
    'motivo_intervencao_cirurgica': 'motivo_reintubacao___4',
    'motivo_atelectasia': 'motivo_reintubacao___5',                   # Ativado
    'motivo_pcr': 'motivo_reintubacao___6',                           # Ativado PCR reintubação
    'motivo_rebaixamento_nivel_consciencia': 'motivo_reintubacao___7',
    'motivo_causa_cardiogenica': 'motivo_reintubacao___8',
    'motivo_outros': 'motivo_reintubacao___9'
}

map_intercorrencias = {
    'eot_acidental': 'tipo_intercorrencias___1',
    'pcr': 'tipo_intercorrencias___2',                         # PCR intercorrencia
    'queda_spo2': 'tipo_intercorrencias___3',
    'hipertensao_evolucao': 'tipo_intercorrencias___4',
    'hipotensao': 'tipo_intercorrencias___5',
    'hipotensao_postural': 'tipo_intercorrencias___6',
    'taquicardia': 'tipo_intercorrencias___7',
    'bradicardia': 'tipo_intercorrencias___8',
    'arritmia_evolucao': 'tipo_intercorrencias___9',
    'precordialgia': 'tipo_intercorrencias___10',              # Ativado
    'sincope': 'tipo_intercorrencias___11',                    # Ativado
    'cianose': 'tipo_intercorrencias___12'
}

# --- ETAPA 2: Criação e Preenchimento (CORREÇÃO AQUI) ---

def tratar_binario(df, col_origem):
    """
    Converte colunas que podem conter 'SIM', 'Sim', 'sim', '1' ou 1 para o inteiro 1.
    Qualquer outra coisa vira 0.
    """
    if col_origem in df.columns:
        # Converte para string, remove espaços e coloca em minúsculo
        col_str = df[col_origem].astype(str).str.strip().str.lower()
        
        # Mapeia: se for 'sim', 's', '1' ou 'true' vira 1, senão 0
        return col_str.apply(lambda x: 1 if x in ['sim', 's', '1', 'true'] else 0)
    else:
        return 0

# Aplica o Mapeamento de Reintubação
for origem, destino in map_reintubacao.items():
    dFis8[destino] = tratar_binario(dFis8, origem)

# Aplica o Mapeamento de Intercorrências
for origem, destino in map_intercorrencias.items():
    dFis8[destino] = tratar_binario(dFis8, origem)

# Garante outras colunas essenciais
dFis8['fisio_reiot'] = tratar_binario(dFis8, 'fisio_reiot')
dFis8['paciente_traqueo'] = tratar_binario(dFis8, 'paciente_traqueo')


# --- ETAPA 3: Agrupamento por Fase ---

cols_agrupamento = ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance']
cols_agrupamento_validas = [c for c in cols_agrupamento if c in dFis8.columns]

agg_dict = {
    'fisio_reiot': 'max',
    'paciente_traqueo': 'max'
}

cols_redcap_checkboxes = list(map_reintubacao.values()) + list(map_intercorrencias.values())
for col in cols_redcap_checkboxes:
    agg_dict[col] = 'max'

if 'motivo_data' in dFis8.columns:
    agg_dict['motivo_data'] = 'first'
if 'data_tqt_evolucao' in dFis8.columns:
    agg_dict['data_tqt_evolucao'] = 'first'

# Lista das colunas que queremos verificar e unir
cols_to_merge = ['outros_intercorrencias', 'outros_intercorrencias1', 'outras_intercorrencias_durante_atendimento']

# Verifica quais dessas colunas realmente existem no DataFrame
existing_cols = [c for c in cols_to_merge if c in dFis8.columns]

if existing_cols:
    # 1. PRÉ-PROCESSAMENTO: Unifica o texto das colunas existentes em uma só ('outros_intercorrencias')
    # Usamos apply(axis=1) para juntar o texto linha a linha
    dFis8['outros_intercorrencias'] = dFis8[existing_cols].apply(
        lambda row: " | ".join([str(val) for val in row if pd.notna(val) and str(val).strip() != ""]),
        axis=1
    )

    # 2. AGREGAÇÃO: Define a regra apenas para a coluna unificada
    # O set() aqui garante que não haja repetição de frases iguais ao agrupar as linhas
    agg_dict['outros_intercorrencias'] = lambda x: " | ".join(sorted(set(x.dropna().astype(str)[x != ""])))

dFis9 = dFis8.groupby(cols_agrupamento_validas).agg(agg_dict).reset_index()


# --- ETAPA 4: Cálculos Finais e Seleção (CORREÇÃO AQUI) ---

# Calcula "Nenhuma Intercorrência"
cols_intercorrencias_destino = list(map_intercorrencias.values())

# SOMA: Verifica quantas intercorrências aconteceram na linha
soma_intercorrencias = dFis9[cols_intercorrencias_destino].sum(axis=1)

# LÓGICA INVERTIDA PEDIDA:
# Se a soma for 0 (Nenhuma intercorrência aconteceu - "SIM"), o valor final deve ser 0.
# Se a soma for > 0 (Alguma intercorrência aconteceu), o valor final deve ser 1.
dFis9['nenhuma_intercorrencias'] = soma_intercorrencias.apply(lambda x: 0 if x == 0 else 1)


# Ordenação Final
colunas_finais_ordenadas = cols_agrupamento_validas + [
    'fisio_reiot',
    'motivo_data',
    'motivo_reintubacao___1', 'motivo_reintubacao___2', 'motivo_reintubacao___3',
    'motivo_reintubacao___4', 'motivo_reintubacao___5', 'motivo_reintubacao___6',
    'motivo_reintubacao___7', 'motivo_reintubacao___8', 'motivo_reintubacao___9',
    'outros_motivo_reintubacao',
    'paciente_traqueo',
    'data_tqt_evolucao',
    'nenhuma_intercorrencias',
    'tipo_intercorrencias___1', 'tipo_intercorrencias___2', 'tipo_intercorrencias___3',
    'tipo_intercorrencias___4', 'tipo_intercorrencias___5', 'tipo_intercorrencias___6',
    'tipo_intercorrencias___7', 'tipo_intercorrencias___8', 'tipo_intercorrencias___9',
    'tipo_intercorrencias___10', 'tipo_intercorrencias___11', 'tipo_intercorrencias___12',
    'outros_intercorrencias',
    'treino_musc_respiratorio',
    'distancia_percorrida_teste_caminhada',
    'valor_pressao_palmar',
]

cols_finais_existentes = [c for c in colunas_finais_ordenadas if c in dFis9.columns]
dFis9 = dFis9[cols_finais_existentes].copy()

print(f"Colunas finais geradas: {dFis9.shape[1]}")
print(dFis9.columns.tolist())

In [ ]:
dFis9['fisioterapia_complete'] = 2

In [ ]:
dFis10 = dFis9[dFis9['record_id'].isin(lista_fisio)]

In [ ]:
cols = ['data_tqt_evolucao', 'motivo_data']

# 2. Se for uma coluna de tosse, remove os zeros
for col in cols:
	if col in cols:
		# Substitui tanto o número 0 quanto a string '0' por vazio
		dFis10[col] = dFis10[col].replace(['0', 0, '0.0'], '')

In [ ]:
# --- COMO USAR NO SEU FLUXO ---


# Agora você importa 
import_redcap(
    df_sorted=dFis10,
    api_url=api_url,
    api_key=api_key,
    batch_size=10,
    overwrite=False
)

In [ ]:
batch_error = import_redcap(dFis10, api_url, api_key)
ids = extract_locked_ids_only(batch_error)
dFis10_final = dFis10[~dFis10['record_id'].isin(ids)]
import_redcap(dFis10_final, api_url, api_key)

print(ids)

# PSICOLOGIA

In [ ]:
# Buscando docText Psico

sql_PSICO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM ft_doc_eletronico_texto fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros
WHERE dcd.CD_DOCUMENTO IN (64, 919, 931, 932, 939, 1148, 1165)
	AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
	AND DS_RESPOSTA NOT IN ('false')
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND (YEAR(fde.DH_DOCUMENTO) >= 2025 AND MONTH(fde.DH_DOCUMENTO) >=6)
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
txPSICO = query_sql_to_dataframe(conn_string, sql_PSICO)

In [ ]:
txPSICO['DS_LEITO'] = txPSICO['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
# Buscando fichas fono

sql_PSICO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.resposta as DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM FT_DOC_ELETRONICO fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros 
WHERE dcd.CD_DOCUMENTO IN (64, 919, 931, 932, 939, 1148, 1165)
	AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
    AND resposta = 'SIM'
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND (YEAR(fde.DH_DOCUMENTO) >= 2025 AND MONTH(fde.DH_DOCUMENTO) >=6)
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
rbPSICO = query_sql_to_dataframe(conn_string, sql_PSICO)

In [ ]:
rbPSICO['DS_LEITO'] = rbPSICO['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
# Buscando cirurgias na ft_prescricao_evolucao

query_PSICO =f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT 
    [NK_CD_ORI_ATE],
    [DS_ORI_ATE],
    [DT_ATUAL]
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fpe.CD_ATENDIMENTO,
    ate.CD_PACIENTE,
    fpe.HR_PRE_MED AS DH_DOCUMENTO,
    CAST(fpe.HR_PRE_MED AS DATE) AS DATAID,
    fpe.CD_PRE_MED AS CD_DOCUMENTO,
    fpe.TP_PRE_MED AS DS_DOCUMENTO,

    -- Fallback: se LEITO atual for vazio ou nulo, busca último anterior válido
    COALESCE(NULLIF(fiu.LEITO, ''), prev_leito.LEITO) AS DS_LEITO,

    'TX_PSICOLOGIA_EVOLUCAO_1' AS DS_IDENTIFICADOR,
    'PSICOLOGIA_EVOLUCAO' AS DS_CAMPO,
    fpe.DS_EVOLUCAO AS DS_RESPOSTA,
    fpe.CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Fallback: se UNIDADE_PASSAGEM estiver vazia, busca última anterior; senão origem ambulatorial
    COALESCE(NULLIF(fiu.UNIDADE_PASSAGEM, ''), prev_unid.UNIDADE_PASSAGEM, doa.DS_ORI_ATE COLLATE Latin1_General_CI_AS) AS NM_SETOR

FROM [DATADANTE].[dbo].[ft_prescricao_evolucao] fpe with(nolock)
INNER JOIN dim_prestador dpt 
    ON dpt.NK_CD_PRESTADOR = fpe.CD_PRESTADOR

-- Internação principal (Mantido para pegar leito/setor de internados)
LEFT JOIN ft_internacao_unidade fiu
    ON fpe.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   AND fpe.HR_PRE_MED >= fiu.HR_MOV_INT
   AND (fpe.HR_PRE_MED <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Fallback: leito anterior válido (Mantido)
OUTER APPLY (
    SELECT TOP 1 LEITO
    FROM ft_internacao_unidade f2
    WHERE f2.CD_ATENDIMENTO = fpe.CD_ATENDIMENTO
      AND f2.HR_MOV_INT < fpe.HR_PRE_MED
      AND f2.LEITO IS NOT NULL AND f2.LEITO <> ''
    ORDER BY f2.HR_MOV_INT DESC
) prev_leito

-- Fallback: unidade anterior válida (Mantido)
OUTER APPLY (
    SELECT TOP 1 UNIDADE_PASSAGEM
    FROM ft_internacao_unidade f3
    WHERE f3.CD_ATENDIMENTO = fpe.CD_ATENDIMENTO
      AND f3.HR_MOV_INT < fpe.HR_PRE_MED
      AND f3.UNIDADE_PASSAGEM IS NOT NULL AND f3.UNIDADE_PASSAGEM <> ''
    ORDER BY f3.HR_MOV_INT DESC
) prev_unid

-- CORREÇÃO AQUI: Trazemos o atendimento SEM restrição de tipo para garantir o CD_PACIENTE
LEFT JOIN ft_atendimento ate 
    ON fpe.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO

-- Join com origem continua, mas agora ele pode vir null ou preenchido dependendo do atendimento
LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

WHERE dpt.CONSELHO = 'CRP'
--AND YEAR(fpe.HR_PRE_MED) in ({ano})
AND (YEAR(fpe.HR_PRE_MED) >= 2025 AND YEAR(fpe.HR_PRE_MED) >= 6)

ORDER BY HR_PRE_MED DESC;
""" 

# Executa a função para buscar os dados
evPSICO = query_sql_to_dataframe(conn_string, query_PSICO)




In [ ]:
evPSICO['DS_DOCUMENTO'] = evPSICO['DS_DOCUMENTO'].apply(lambda x: 'PSICOLOGIA')

In [ ]:
#Copiando os dfs para processamento

tDocPs = txPSICO.copy()

rDocPs = rbPSICO.copy()

eEvlPs = evPSICO.copy()

In [ ]:
tDocPs.info()

In [ ]:
dPsi = pd.concat([tDocPs, rDocPs, eEvlPs], ignore_index=True)

In [ ]:
# --- 1. PREPARAÇÃO DOS DADOS ---

# Filtro inicial: Só quero psico de pacientes que existem no meu estudo
# Isso já elimina ruído de pacientes de outras especialidades

dPsi1 = dPsi[dPsi['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))].copy()


In [ ]:
psico_ls = {
 
# --- LF ---
'em_tratamento':['DS_NAO_EM_TRATAMENTO_SAUDE_MENTAL_1', 'DS_SIM_EM_TRATAMENTO_SAUDE_MENTAL_1'],
'frequencia_pensamento_doenca':['RB_QUASE_NUNCA_FREQUENCIA_PENSAMENTOS_SOBRE_DOENCA_1',
                                'RB_AS_VEZES_FREQUENCIA_PENSAMENTOS_SOBRE_DOENCA_1',
                                'RB_FREQUENTEMENTE_PENSAMENTOS_SOBRE_DOENCA_1'],
'compreensao_saude':['RB_AUSENCIA_COMPREENSAO_REACOES_EMOCIONAIS_FRENTE_DIAGNOSTICO_CIRURGICA_CATEGORIAS_1',
                     'RB_BOA_COMPREENSAO_REACOES_EMOCIONAIS_FRENTE_DIAGNOSTICO_CIRURGICA_CATEGORIAS_1',
                     'RB_COMPREENSAO_PARCIAL_REACOES_EMOCIONAIS_FRENTE_DIAGNOSTICO_CIRURGICA_CATEGORIAS_1'],
'tipo_protese':['RB_SIM_PACIENTE_ALGUMA_INFORMACAO_NOCAO_HA_TIPOS_PROTESE_1',
                'RB_NAO_PACIENTE_ALGUMA_INFORMACAO_NOCAO_HA_TIPOS_PROTESE_1'],




}

In [ ]:
psico_rn = {
    # --- ---
    'DS_NAO_EM_TRATAMENTO_SAUDE_MENTAL_1': 0, 'DS_SIM_EM_TRATAMENTO_SAUDE_MENTAL_1': 1,
    'RB_QUASE_NUNCA_FREQUENCIA_PENSAMENTOS_SOBRE_DOENCA_1': 0, 
    'RB_AS_VEZES_FREQUENCIA_PENSAMENTOS_SOBRE_DOENCA_1': 1,
    'RB_FREQUENTEMENTE_PENSAMENTOS_SOBRE_DOENCA_1': 2,
    'RB_AUSENCIA_COMPREENSAO_REACOES_EMOCIONAIS_FRENTE_DIAGNOSTICO_CIRURGICA_CATEGORIAS_1': 0,
    'RB_BOA_COMPREENSAO_REACOES_EMOCIONAIS_FRENTE_DIAGNOSTICO_CIRURGICA_CATEGORIAS_1': 1,
    'RB_COMPREENSAO_PARCIAL_REACOES_EMOCIONAIS_FRENTE_DIAGNOSTICO_CIRURGICA_CATEGORIAS_1': 2,
    'RB_NAO_PACIENTE_ALGUMA_INFORMACAO_NOCAO_HA_TIPOS_PROTESE_1': 0,
    'RB_SIM_PACIENTE_ALGUMA_INFORMACAO_NOCAO_HA_TIPOS_PROTESE_1': 1,
    

}

In [ ]:
dPsi1.columns

In [ ]:
# 1. Lista de identificadores já mapeados
ident_tratados = [item for sublist in psico_ls.values() for item in sublist]

# 2. Cria colunas para os identificadores mapeados
for key in tqdm(psico_ls.keys(), position=0, leave=False):
    dPsi1[key] = None
    for item in psico_ls[key]:
        with ipython_io.capture_output() as captured:
        	dPsi1[key][dPsi1['DS_IDENTIFICADOR'] == item] = item

# 3. Filtra os identificadores não tratados
nao_tratados = dPsi1[~dPsi1['DS_IDENTIFICADOR'].isin(ident_tratados)]

nao_tratados['DS_RESPOSTA'] = nao_tratados['DS_RESPOSTA'].apply(
    lambda x: 'SIM' if isinstance(x, str) and x.strip().lower() == 'true'
    else 'NAO' if isinstance(x, str) and x.strip().lower() == 'false'
    else x
)

# 4. Pivotando os identificadores não tratados
pivot_extra = nao_tratados.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    columns='DS_IDENTIFICADOR',
    values='DS_RESPOSTA',
    aggfunc='first'  # ou 'last', dependendo da lógica de negócio
).reset_index()

# 5. Junta com os dados principais (já com colunas criadas manualmente)
dPsi2 = pd.merge(
    dPsi1.drop(columns=['DS_IDENTIFICADOR', 'DS_RESPOSTA', 'DS_CAMPO']),
    pivot_extra,
    on=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    how='left'
)

# 6. Substituir os valores conforme psico_rn (string → string)
for column in dPsi2.columns:
    if column not in ['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID',
       ]:
        dPsi2[column].replace(psico_rn, inplace=True)

In [ ]:

# reutilizando função para reorganizar df

dPsi3 = organizar_df_otimizado(dPsi2)

In [ ]:
map_cols_psy = {
'META_TERAPEUTICA_PSICOLOGIA_1':						'terapeutica_psicologia',
'DS_COMO_DESCOBRIU_PROBLEMA_CARDIOLOGICO_1':			'problema_cardiologico',
'DS_COMO_REAGIU_SABER_DIAGNOSTICO_1':					'saber_diagnostico',
'DS_COMO_SENTIU_QUANDO_MEDICO_EXPLICOU_CIRURGIA_SERIA_NECESSARIA_CUIDAR_SAUDE_1':			'cuidar_saude',
'DS_HA_ALGO_MAIS_GOSTARIA_COMPARTILHAR_SOBRE_DOENCA_SENTIMENTOS_ENVOLVIDOS_1':				'sentimentos_envolvidos',
'DS_APOIO_AFETIVO_PONTUACAO_ESCALA_APOIO_SOCIAL_1':				'apoio_efetivo',
'DS_QUEM_SERA_REDE_APOIO_APOS_CIRURGIA_ESCALA_APOIO_SOCIAL_1':	'rede_apoio',
'DS_CONSEGUE_TOMAR_MEDICACOES_CONFORME_ORIENTADO_1':			'medicacao',
'DS_QUE_OU_QUEM_CONSIDERA_APOIADO_NESTE_MOMENTO_ENFRENTAMENTO_IMPACTO_PESSOAL_1':			'apoio_momento',
'DS_QUE_MUDOU_VIDA_DEPOIS_PROBLEMA_CARDIOLOGICO_ENFRENTAMENTO_IMPACTO_PESSOAL_1':			'problema_cardiologico_2',
'DS_DEIXOU_REALIZAR_ALGUMA_ATIVIDADE_ENFRENTAMENTO_IMPACTO_PESSOAL_1':						'realizar_atividade',
'DS_QUAIS_TEM_SIDO_PENSAMENTOS_ATITUDES_TEM_AJUDADO_CONVIVER_PROBLEMA_ENFRENTAMENTO_IMPACTO_PESSOAL_1':	'pensamento_atitudes',
'DS_QUE_CONSIDERA_SIDO_MAIS_DIFICIL_NESTA_CONVIVENCIA_COM_DOENCA_ENFRENTAMENTO_IMPACTO_PESSOAL_1':		'convivencia_doenca',
'DS_COSTUMA_PENSAR_SUA_DOENCA_COMO_FICA_FAZ_QUANDO_PENSAMENTOS_SURGEM_ENFRENTAMENTO_IMPACTO_PESSOAL_1':	'pensar_doenca',
'DS_CONDUTA_ENCAMINHAMENTO_ESCALA_HADS_1':						'conduta_encaminhamento',
'DS_TEM_FEITO_ALGO_PARA_SOLUCIONALAS_1':						'algo_solucao',
'DS_SUAS_EXPECTATIVAS_DIANTE_PROCEDIMENTO_CIRURGICO_1':			'expectativas_procedimento',
'DS_PREOCUPACOES_FRENTE_CIRURGIA_1':							'preocupacao_cirurgia',
'DS_QUADROS_PSICOPATOLOGICOS_PREVIOS_ATUAIS_1':					'psicopatologicos',
'DS_MEDICACOES_EM_USO_SAUDE_MENTAL_1':							'medicacao_uso',
'DS_ESCORE_SINTOMAS_ANSIEDADE_PONTO_CORTE_8_1':					'hads_ansiedade',
'DS_ESCORE_SINTOMAS_ANSIEDADE_PONTO_CORTE_9_1':					'hads_depressao',
'DS_ME_ENFRENTAMENTO_FOCALIZADO_PROBLEMA_1':					'enf_problema',
'DS_ME_ENFRENTAMENTO_FOCALIZADO_EMOCAO_1':						'enf_emocao',
'RB_ME_BUSCA_PRATICAS_RELIGIOSAS_PENSAMENTO_FANTASIOSO_1':		'busca_religiosa',
'RB_ME_BUSCA_SUPORTE_SOCIAL_1':									'busca_social',
'DS_RECEBEU_APOIO_FAMILIAR_APOS_ADOECIMENTO_DE_QUEM_1':			'apoio_familiar',
'DS_SUA_RELACOES_SOCIAIS_MUDARAM_APOS_ADOECIMENTO_1':			'relacao_social',
'DS_APOIO_MATERIAL_PONTUACAO_ESCALA_APOIO_SOCIAL_1':			'apoio_material',
'DS_APOIO_EMOCIONAL_INFORMACIONAL_PONTUACAO_ESCALA_APOIO_SOCIAL_1':							'apoio_emocional',
'DS_INTERACAO_SOCIAL_AFETIVA_PONTUACAO_ESCALA_APOIO_SOCIAL_1':	'interacao_social',
'DS_EXERCE_ATIVIDADE_REMUNERADA_ATUALMENTE_1':					'atividade_remunerada',
'DS_SE_PACIENTE_CONTINUA_TRABALHANDO_ALGO_MUDOU_TRABALHO_1':	'alteracao_trabalho',
'DS_SE_PAROU_TRABALHAR_COMO_FOI_PARA_VOCE_PARAR_TABALHAR_1':	'parou_trabalhar',
'DS_SUAS_PREOCUPACOES_MOMENTO_1':								'preocupacao_momento',
'CK_NEGACAO_PSC_1':												'reacoes___0', 
'CK_MEDO_PSC_1':												'reacoes___1', 
'CK_ANSIEDADE_PSC_1':											'reacoes___2', 
'CK_TRISTEZA_PSC_1':											'reacoes___3',
'CK_ALIVIO_PSC_1':												'reacoes___4', 
'CK_ACEITACAO_PSC_1':											'reacoes___5', 
'CK_ESPERANCA_PSC_1':											'reacoes___6', 
'CK_INDIFERENCA_PSC_1':											'reacoes___7',
'CK_OUTRAS_REACOES_EMOCIONAIS_1':								'reacoes___9',
'TX_PSICOLOGIA_EVOLUCAO_1':										'evolucao_psicologia',
'DS_SE_SIM_ESPECIFIQUE_SEU_CONHECIMENTO_PACIENTE_ALGUMA_INFORMACAO_NOCAO_HA_TIPOS_PROTESE_1':	'conhece_protese',
}

In [ ]:
dPsi3 = dPsi3.rename(columns=map_cols_psy).copy()

In [ ]:
dPsi3.columns

In [ ]:
# 1. Ordenar (Mantido para garantir ordem cronológica)
dPsi4 = dPsi3.sort_values(by=['CD_PACIENTE', 'DATAID'], ascending=True)
cols_concat = ['terapeutica_psicologia', 'evolucao_psicologia']

# 2. Função personalizada (Mantida)
def join_unique_strings(series):
    valid_strings = series.dropna().astype(str).unique()
    if len(valid_strings) > 0:
        return ' / '.join(valid_strings)
    return np.nan

# 3. Construção do dicionário de agregação
agg_dict = {}

# As chaves do agrupamento serão Paciente e Data, então não precisamos incluí-las no dicionário
chaves_agrupamento = ['CD_PACIENTE', 'DATAID']

for col in dPsi4.columns:
    if col in chaves_agrupamento:
        continue
    
    # Se for a coluna de prescrição, CONCATENA
    if col == cols_concat:
        agg_dict[col] = join_unique_strings
    
    # Para as demais colunas (ex: nome do prestador do dia), pega o primeiro
    else:
        agg_dict[col] = 'first'

# 4. O GRANDE TRUQUE: Agrupar por UMA LISTA de colunas
# Ao agrupar por [CD_PACIENTE, DATAID], você garante que as datas não se misturam
dPsi5 = dPsi4.groupby(chaves_agrupamento).agg(agg_dict).reset_index()

In [ ]:
# Buscando prescrições.PSICO

sql_PSICO = f"""
SET DATEFORMAT ymd;

WITH 
dpt AS (
    SELECT *
    FROM dim_prestador dpt1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_prestador dpt2
        WHERE dpt2.NK_CD_PRESTADOR = dpt1.NK_CD_PRESTADOR
    )
)

SELECT 
	fpe.NK_CD_ITPRED_MED,
	fpe.CD_PRE_MED,
	fpe.CD_ATENDIMENTO,
	fat.CD_PACIENTE,
	fpe.HR_PRE_MED,
	fpe.TP_PRE_MED,
    fpe.CD_TIP_PRESC,
	DS_TIP_PRESC,
    fpe.QT_ITPRE_MED,
    fpe.CD_TIP_ESQ,
    DS_TIP_ESQ,
	fat.TP_ATENDIMENTO,
    fpe.TP_SITUACAO,
    fpe.CD_PRESTADOR,
	dpt.NM_PRESTADOR,
	dpt.TIPO_PRESTADOR,
    dpt.CONSELHO,
    dpt.TP_SITUACAO AS SITUACAO_PRESTADOR,
	CAST(HR_PRE_MED  AS DATE) AS DATAID,
	DH_CRIACAO,
	DH_CANCELADO
FROM [DATADANTE].[dbo].[ft_prescricao] fpe
INNER JOIN ft_atendimento fat ON fat.NK_CD_ATENDIMENTO = fpe.CD_ATENDIMENTO
INNER JOIN dpt ON dpt.NK_CD_PRESTADOR = fpe.CD_PRESTADOR
INNER JOIN dim_item_prescricao DP ON fpe.CD_TIP_PRESC = DP.NK_CD_TIP_PRESC
WHERE 
	dpt.CONSELHO = 'CRP'
    --AND YEAR(fpe.HR_PRE_MED) in ({ano})
	AND (YEAR(fpe.HR_PRE_MED) >= 2025 AND YEAR(fpe.HR_PRE_MED) >= 6)
    ORDER BY HR_PRE_MED DESC
""" 

# Executa a função para buscar os dados
pcPSICO = query_sql_to_dataframe(conn_string, sql_PSICO)


In [ ]:
pPsc = pcPSICO.copy()

In [ ]:
pPsc1 = pPsc[pPsc['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))]


In [ ]:
pPsc1.columns

In [ ]:
# 1. Ordenar (Mantido para garantir ordem cronológica)
pPsc1 = pPsc1.sort_values(by=['CD_PACIENTE', 'DATAID'], ascending=True)

# 2. Função personalizada (Mantida)
def join_unique_strings(series):
    valid_strings = series.dropna().astype(str).unique()
    if len(valid_strings) > 0:
        return ' / '.join(valid_strings)
    return np.nan

# 3. Construção do dicionário de agregação
agg_dict = {}

# As chaves do agrupamento serão Paciente e Data, então não precisamos incluí-las no dicionário
chaves_agrupamento = ['CD_PACIENTE', 'DATAID']

for col in pPsc1.columns:
    if col in chaves_agrupamento:
        continue
    
    # Se for a coluna de prescrição, CONCATENA
    if col == 'DS_TIP_PRESC':
        agg_dict[col] = join_unique_strings
    
    # Para as demais colunas (ex: nome do prestador do dia), pega o primeiro
    else:
        agg_dict[col] = 'first'

# 4. O GRANDE TRUQUE: Agrupar por UMA LISTA de colunas
# Ao agrupar por [CD_PACIENTE, DATAID], você garante que as datas não se misturam
pPsc2 = pPsc1.groupby(chaves_agrupamento).agg(agg_dict).reset_index()


In [ ]:
dPsi6 = pd.merge(
    dPsi5,
    pPsc2[['CD_ATENDIMENTO', 'CD_PACIENTE', 'DATAID', 'DS_TIP_PRESC']],
    on=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DATAID'],
    how='left'
)

In [ ]:
dPsi6.columns

In [ ]:
dPsi6.columns = dPsi6.columns.str.lower()

In [ ]:
dPsi6 = dPsi6.rename(columns={'cd_paciente': 'record_id', 'ds_tip_presc': 'tipo_atend_psico'})


In [ ]:
dPsi6.columns

In [ ]:
dPsi6['record_id'] = dPsi6['record_id'].astype('Int64')

In [ ]:
# 1. Vincular (Traz data_cirurgia e redcap_repeat_instance para dentro do dFis)
dPsi7 = vincular_instancias_temporal(
    df_principal=dPsi6,
    df_referencia=regvalv_cirurgia[['record_id', 'cd_atendimento', 'redcap_repeat_instance', 'data_cirurgia']],
    col_data_principal='dh_documento',    # Data do documento
    col_data_referencia='data_cirurgia',  # Data da cirurgia
    tolerancia_dias=180,
    # --- NOVOS ARGUMENTOS OPCIONAIS ---
    col_atend_principal='cd_atendimento', 
    col_atend_referencia='cd_atendimento'
)

# 2. Classificar (Define o redcap_event_name baseado no tempo e local)
dPsi8 = classificar_eventos_por_atendimento(
    df_entrada=dPsi7,
    evento_pre='fase1preoperatorio_arm_1',
    evento_uti='fase4posprocedimed_arm_1',
    evento_enfermaria='fase5recuperacaoin_arm_1',
    evento_amb='fase6segalta30dias_arm_1',
    col_data_documento='dh_documento',
    col_criterio_local='nm_setor',     # Pode ser 'CD_DOCUMENTO' ou 'NM_SETOR'
    val_uti='UTI',            # Pode ser tupla de IDs ou string 'UTI'
    val_enfermaria='ENF',      	   # Ajuste seus códigos aqui
	val_amb='AMBULATORIO'
)

In [ ]:
dPsi8['redcap_event_name'].value_counts()

In [ ]:
# Sintaxe: df.loc[FILTRO, 'COLUNA'] = NOVO_VALOR
# PAliativo, fisioterapia nao vai atender fase pre

dPsi8.loc[dPsi8["redcap_event_name"] == "fase4posprocedimed_arm_1", "redcap_event_name"] = "fase1preoperatorio_arm_1"
dPsi8.loc[dPsi8["redcap_event_name"] == "fase5recuperacaoin_arm_1", "redcap_event_name"] = "fase6segalta30dias_arm_1"

In [ ]:
# 1. Garantir a ordenação sequencial
# Mesmo não filtrando por data, ordenamos para que o python saiba qual é a "última" linha
# Se 'dataid' for sequencial, usamos ele. Caso contrário, pode-se usar o índice original.
dPsi8 = dPsi8.sort_values(['record_id', 'redcap_event_name', 'dataid'], ascending=True)

# 2. Agrupamento e Consolidação
# Agrupamos por Paciente e Fase
# O método .last() pega o último valor não-nulo encontrado na coluna para aquele grupo.
# Isso resolve o problema de ter dados espalhados em linhas diferentes (como na sua imagem).
dPsi8 = dPsi8.groupby(['record_id', 'redcap_event_name'], as_index=False).last()

In [ ]:
cols_to_drop = ['cd_documento', 'dh_documento', 'nm_prestador', 'ds_documento',
                'ds_leito', 'cd_prestador', 'nm_setor', 'data_cirurgia']

cols_to_replace = [ 'reacoes___0', 'reacoes___1', 'reacoes___2', 'reacoes___3', 'reacoes___4', 'reacoes___5', 
                  'reacoes___6', 'reacoes___7', 'reacoes___9',] # SIM = 1, NAO = 0

cols_int = [ 'record_id', 'em_tratamento', 'frequencia_pensamento_doenca', 'compreensao_saude', 'reacoes___0', 
            'reacoes___1', 'reacoes___2', 'reacoes___3', 'reacoes___4', 'reacoes___5', 'reacoes___6', 
            'reacoes___7', 'reacoes___9', 'tipo_protese']

cols_nan = [ 'compreensao_saude', 'tipo_protese', 'frequencia_pensamento_doenca', 'enf_problema', 
            'enf_emocao', 'busca_religiosa', 'busca_social', 'em_tratamento']



In [ ]:
# --- APLICAÇÃO DAS TRANSFORMAÇÕES ---

# 1. Remover colunas desnecessárias
# errors='ignore' evita erro caso a coluna já tenha sido removida antes
dPsi8 = dPsi8.drop(columns=cols_to_drop, errors='ignore')

# 2. Substituição de Valores (SIM/NAO -> 1/0)
# Criamos um dicionário para garantir que pegue variações
mapa_valores = {
    'SIM': 1, 'Sim': 1, 'sim': 1,
    'NAO': 0, 'NÃO': 0, 'Não': 0, 'nao': 0
}

# Verificamos quais colunas da lista realmente existem no df para evitar erro
cols_rep_existentes = [c for c in cols_to_replace if c in dPsi8.columns]
dPsi8[cols_rep_existentes] = dPsi8[cols_rep_existentes].replace(mapa_valores)

# Verificamos quais colunas dessa lista existem no DataFrame
cols_target_existentes = [c for c in cols_nan if c in dPsi8.columns]

for col in cols_target_existentes:
    # 1. pd.to_numeric: Transforma texto em número e erros em NaN
    # 2. .astype('Int64'): Transforma em inteiro, mas mantendo o NaN (não preenche com 0)
    dPsi8[col] = pd.to_numeric(dPsi8[col], errors='coerce').astype('Int64')

# --- Para as DEMAIS colunas que você AINDA quer preencher com 0 ---
# (Caso existam outras colunas fora dessa lista que precisem da lógica antiga)
# IMPORTANTE: Para converter para 'int', não pode haver NaN (vazios).
# Abaixo preenchemos vazios com 0. Se 0 não for o ideal, use .fillna(-1) ou outra lógica.
cols_restantes = [c for c in cols_int if c in dPsi8.columns and c not in cols_nan]

for col in cols_restantes:
    dPsi8[col] = pd.to_numeric(dPsi8[col], errors='coerce').fillna(0).astype(int)

In [ ]:
dPsi8['redcap_repeat_instrument'] = 'psicologia'

In [ ]:
# --- 1. PREPARAÇÃO DOS DADOS DO REDCAP (EXISTENTES) ---

# Garantir que record_id seja string em ambos os lados para evitar erros de comparação ( "100" != 100 )
regvalv_psico['record_id'] = regvalv_psico['record_id'].astype(str)

# Criamos um "Set" de tuplas com (ID, EVENTO). 
# Isso funciona como uma lista de presença: "O paciente X já tem ficha no evento Y"
psico_existentes = set(zip(
    regvalv_psico['record_id'].astype(str), 
    regvalv_psico['redcap_event_name'].astype(str),
    regvalv_psico['redcap_repeat_instance'].astype(str)
))

print(f"Total de registros já existentes no Redcap: {len(psico_existentes)}")

In [ ]:
cols_ajuste_pontuacao = ['enf_emocao', 'enf_problema', 'busca_religiosa', 'busca_social']

for col in cols_ajuste_pontuacao:
    if col in dPsi8.columns:
        # Converte para string (garantia) e substitui vírgula por ponto
        # O regex=False é para garantir que o ponto seja lido como caractere literal e não comando regex
        dPsi8[col] = dPsi8[col].astype(str).str.replace(',', '.', regex=False)

In [ ]:
cols_drop = ['dataid', 'cd_atendimento']
psy_fases = dPsi8.drop(columns=cols_drop, errors='ignore')

In [ ]:
psy_fases['psicologia_complete'] = 2

In [ ]:
#psy_fases = psy_fases[(psy_fases['redcap_event_name'] == 'fase1preoperatorio_arm_1') | (psy_fases['redcap_event_name'] == 'fase6segalta30dias_arm_1')]

In [ ]:
# --- 3. FILTRANDO OS DADOS LOCAIS ---

# Filtra cada fase
psy_fases = filtrar_novos_registros(psy_fases, psico_existentes)

# Exibe o resultado
print(f"Novos registros Fase 2: {len(psy_fases)}")




In [ ]:
#dPsi8.to_excel(f"C://Users/priscilla.sequetin/Downloads/proced_psico_6.xlsx", index=False)
#psy_fases.to_excel(f"C://Users/priscilla.sequetin/Downloads/proced_psico_8.xlsx", index=False)

In [ ]:
# 3. Executa a importação USANDO O DATAFRAME FILTRADO dPsi8
results_Psico = import_redcap(
        df_sorted=psy_fases,     # <--- MUDEI AQUI: De dPcte para dPsi8
        api_url=api_url,
        api_key=api_key,
        batch_size=10,
        overwrite=False        
    )

In [ ]:
#dPsi8.to_excel(f"C://Users/priscilla.sequetin/Downloads/proced_psico_3.xlsx", index=False)

# ODONTOLOGIA

In [ ]:
# Buscando docText.ODONTO

sql_ODONTO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM ft_doc_eletronico_texto fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros
WHERE dcd.CD_DOCUMENTO IN (241, 710, 891, 893, 894)
	AND fde.DS_CAMPO not in ( 'Chave do Documento', 'Código da Empresa', 'Código do Atendimento', 'Código do Documento', 'Data de Registro', 
    'Registro de Documento', 'Registro do editor', 'Identificador', 'Código do Item na Prescrição', 'Código do Paciente', 'Código do Usuário', 
    'Sistema responsável', 'Último Resultado') 
    AND DS_RESPOSTA NOT IN ('false')
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND YEAR(fde.DH_DOCUMENTO) >= 2025
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
txODONTO = query_sql_to_dataframe(conn_string, sql_ODONTO)

In [ ]:
txODONTO['DS_LEITO'] = txODONTO['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
sql_ODONTO = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


FROM FT_DOC_ELETRONICO fde 
LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação (aplica-se apenas se DS_LEITO estiver presente)
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
   	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
   	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
   	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
   	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- Filtros 
WHERE dcd.CD_DOCUMENTO IN (241, 710, 891, 893, 894)
	AND fde.DS_CAMPO not in ( 'Chave do Documento', 'Código da Empresa', 'Código do Atendimento', 'Código do Documento', 'Data de Registro', 
    'Registro de Documento', 'Registro do editor', 'Identificador', 'Código do Item na Prescrição', 'Código do Paciente', 'Código do Usuário', 
    'Sistema responsável', 'Último Resultado') 
    AND resposta = 'SIM'
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
    AND YEAR(fde.DH_DOCUMENTO) >= 2025
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
rbODONTO = query_sql_to_dataframe(conn_string, sql_ODONTO)


In [ ]:
rbODONTO['DS_LEITO'] = rbODONTO['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
sql_ODONTO = f"""
SET DATEFORMAT ymd;

WITH 
dpt AS (
    SELECT *
    FROM dim_prestador dpt1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_prestador dpt2
        WHERE dpt2.NK_CD_PRESTADOR = dpt1.NK_CD_PRESTADOR
    )
)

SELECT 
	fpe.NK_CD_ITPRED_MED,
	fpe.CD_PRE_MED,
	fpe.CD_ATENDIMENTO,
	fat.CD_PACIENTE,
	fpe.HR_PRE_MED,
	fpe.TP_PRE_MED,
    fpe.CD_TIP_PRESC,
	UPPER(DS_TIP_PRESC) AS PRESCRICOES,
    fpe.QT_ITPRE_MED,
    fpe.CD_TIP_ESQ,
    DS_TIP_ESQ,
	fat.TP_ATENDIMENTO,
    fpe.TP_SITUACAO,
    fpe.CD_PRESTADOR,
	dpt.NM_PRESTADOR,
	dpt.TIPO_PRESTADOR,
    dpt.CONSELHO,
    dpt.TP_SITUACAO AS SITUACAO_PRESTADOR,
	CAST(HR_PRE_MED  AS DATE) AS DATAID,
	DH_CRIACAO,
	DH_CANCELADO
FROM [DATADANTE].[dbo].[ft_prescricao] fpe
INNER JOIN ft_atendimento fat ON fat.NK_CD_ATENDIMENTO = fpe.CD_ATENDIMENTO
INNER JOIN dpt ON dpt.NK_CD_PRESTADOR = fpe.CD_PRESTADOR
INNER JOIN dim_item_prescricao DP ON fpe.CD_TIP_PRESC = DP.NK_CD_TIP_PRESC
WHERE 
	(
		dpt.CONSELHO = 'CRO'
	OR
		(fpe.CD_TIP_ESQ IN ('POD', 'MOD', 'PRO') AND (dpt.CONSELHO = 'CRM'))
	)
    --AND YEAR(fpe.HR_PRE_MED) in ({ano})
    AND YEAR(fpe.HR_PRE_MED) >= 2025
ORDER BY HR_PRE_MED DESC
""" 

# Executa a função para buscar os dados
pcODONTO = query_sql_to_dataframe(conn_string, sql_ODONTO)


In [ ]:
#Copiando os dfs para processamento

tDocOd = txODONTO.copy()

rDocOd = rbODONTO.copy()

pPrcOd = pcODONTO.copy()


In [ ]:
dOdt = pd.concat([tDocOd, rDocOd], ignore_index=True)

In [ ]:
dOdt["DS_IDENTIFICADOR"] = dOdt["DS_IDENTIFICADOR"].str[3:-2]

In [ ]:
#tst = dOdt['DS_IDENTIFICADOR'].value_counts().sort_index()

In [ ]:
# --- 1. PREPARAÇÃO DOS DADOS ---

# Filtro inicial: Só quero psico de pacientes que existem no meu estudo
# Isso já elimina ruído de pacientes de outras especialidades
dOdt1 = dOdt[dOdt['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))].copy()

pOdt = pPrcOd[pPrcOd['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))].copy()


In [ ]:
# Dicionário das fichas odonto
odonto_ls = {
    'EFEO_CONTACT': ['CONTACTUANTE', 'CONTACTUANTE_NAO_VERBAL', 'CONTACTUANTE_VERBAL'],
    'EFEO_DEAMB': ['DEAMBULANTE', 'DEAMBULANTE_COM_AUXILIO', 'CADEIRANTE', 'ACAMADO'],
    'EFEO_DIETA': ['DIETA_ORAL', 'SNE', 'GTT', 'JEJUM'],
    'EFEO_RESPIR': ['SUPORTE_O2', 'ENTUBADO', 'TQT'],
    'EFIO_HIG_DONE':  ['HIGIENE_ORAL_SATISFATORIA', 'HIGIENE_ORAL_REGULAR', 'HIGIENE_ORAL_INSATISFATORIA'],
	'EFIO_HIG_WHO':  ['HIEGIENE_ORAL_PELO_PACIENTE', 'HIGIENE_ORAL_PELO_ACOMPANHANTE_CUIDADOR', 'HIGIENE_ORAL_PELA_ENFERMAGEM'],
    'EFIO_MUCOSA':  ['MUCOSA_INTEGRA', 'MUCOSA_RESSECADA', 'MUCOSA_CROSTA'],
    'EFIO_LABIO':  ['LABIO_INTEGRO', 'LABIO_RESSECADO', 'LABIO_CROSTA'],
    'EFIO_LINGUA':  ['LINGUA_INTEGRA', 'LINGUA_RESSECADA', 'LINGUA_COM_CROSTA', 'LINGUA_SABURRA', 'LINGUA_DESPAPILADA'],
    'EFIO_LESAO_TM':  ['TECIDOS_MOLES_COM_MUCOSA', 'TECIDOS_MOLES_COM_PALATO', 'TECIDO_MOLES_LINGUA', 'LABIO'],
                      
    'PCTE_URGENTE':  ['NAO_URG_EMERGENCIA', 'SIM_URG_EMERGENCIA', 'URGENCIA_E_EMERGENCIA_NAO', 'URGENCIA_E_EMERGENCIA_SIM'],
	'LIB_CIRURGIA':  ['LIBERACAO_CIRURGICA_NAO', 'LIBERACAO_CIRURGICA_SIM', 'NAO_LIBERACAO_CIRURGIA', 'SIM_LIBERACAO_CIRURGIA'],
    'PROTESE_DENTARIA': ['PROTESE_DENTARIA_NAO', 'PROTESE_DENTARIA_SIM'],
    'USO_ANTICOAG': ['FAZ_USO_DE_ANTICOAGULANTE', 'NAO_FAZ_USO_DE_ANTICOAGULANTE'],
    'USO_ANTIBIOT': ['PROFILAXIA_ANTIBIOTICA_NAO', 'PROFILAXIA_ANTIBIOTICA_SIM'],
    
    
}


# Dicionário de comorbidades fichas serviço social
odonto_rn = {
    
    'CONTACTUANTE':'Não Contactuante', 'CONTACTUANTE_NAO_VERBAL':'Contactuante Não Verbal', 'CONTACTUANTE_VERBAL':'Contactuante Verbal', 'DEAMBULANTE':'Deambulante', 
    'DEAMBULANTE_COM_AUXILIO':'Deambulante com Auxílio', 'CADEIRANTE':'Cadeirante', 'ACAMADO':'Acamado', 'DIETA_ORAL':'Dieta Oral', 'SNE':'SNE', 'SUPORTE_O2':'Suporte O2', 'GTT':'GGT', 
    'JEJUM':'Jejum', 'ENTUBADO':'Entubado', 'TQT':'TQT',
    'HIGIENE_ORAL_SATISFATORIA':'Satisfatória', 'HIGIENE_ORAL_REGULAR':'Regular', 'HIGIENE_ORAL_INSATISFATORIA':'Insatisfatória', 'HIEGIENE_ORAL_PELO_PACIENTE':'Paciente', 
    'HIGIENE_ORAL_PELO_ACOMPANHANTE_CUIDADOR':'Acompanhante/Cuidador', 'HIGIENE_ORAL_PELA_ENFERMAGEM':'Enfermagem',         
    'MUCOSA_INTEGRA':'Mucosa Integra', 'MUCOSA_RESSECADA':'Mucosa Ressecada', 'MUCOSA_CROSTA':'Mucosa com Crosta',
    'LABIO_INTEGRO':'Lábio Integro', 'LABIO_RESSECADO':'Lábio Ressecado', 'LABIO_CROSTA':'Lábio com Crosta',
    'LINGUA_INTEGRA':'Língua Integra', 'LINGUA_RESSECADA':'Língua Ressecada', 'LINGUA_COM_CROSTA':'Língua com Crosta', 'LINGUA_SABURRA':'Língua Saburra', 'LINGUA_DESPAPILADA':'Lingua Despapilada',
    'TECIDOS_MOLES_COM_MUCOSA':'Lesão Mucosa', 'TECIDOS_MOLES_COM_PALATO':'Lesão Palato', 'TECIDO_MOLES_LINGUA':'Lesão Língua', 'LABIO':'Lesão Lábio',
     
    'LIBERACAO_CIRURGICA_NAO': 0, 'LIBERACAO_CIRURGICA_SIM': 1, 'NAO_LIBERACAO_CIRURGIA': 0, 'SIM_LIBERACAO_CIRURGIA': 1,
    'NAO_URG_EMERGENCIA': 0, 'SIM_URG_EMERGENCIA': 1, 'URGENCIA_E_EMERGENCIA_NAO': 0, 'URGENCIA_E_EMERGENCIA_SIM': 1,
    'PROTESE_DENTARIA_NAO': 0, 'PROTESE_DENTARIA_SIM': 1,
    'FAZ_USO_DE_ANTICOAGULANTE': 1, 'NAO_FAZ_USO_DE_ANTICOAGULANTE': 0, 
    'PROFILAXIA_ANTIBIOTICA_SIM': 1, 'PROFILAXIA_ANTIBIOTICA_NAO': 0,

}

In [ ]:
# 1. Lista de identificadores já mapeados
ident_tratados = [item for sublist in odonto_ls.values() for item in sublist]

# 2. Cria colunas para os identificadores mapeados
for key in tqdm(odonto_ls.keys(), position=0, leave=False):
    dOdt1[key] = None
    for item in odonto_ls[key]:
        with ipython_io.capture_output() as captured:
        	dOdt1[key][dOdt1['DS_IDENTIFICADOR'] == item] = item

# 3. Filtra os identificadores não tratados
nao_tratados = dOdt1[~dOdt1['DS_IDENTIFICADOR'].isin(ident_tratados)]

nao_tratados['DS_RESPOSTA'] = nao_tratados['DS_RESPOSTA'].apply(
    lambda x: 'SIM' if isinstance(x, str) and x.strip().lower() == 'true'
    else 'NAO' if isinstance(x, str) and x.strip().lower() == 'false'
    else x
)

# 4. Pivotando os identificadores não tratados
pivot_extra = nao_tratados.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    columns='DS_IDENTIFICADOR',
    values='DS_RESPOSTA',
    aggfunc='first'  # ou 'last', dependendo da lógica de negócio
).reset_index()

# 5. Junta com os dados principais (já com colunas criadas manualmente)
dOdt2 = pd.merge(
    dOdt1.drop(columns=['DS_IDENTIFICADOR', 'DS_RESPOSTA', 'DS_CAMPO']),
    pivot_extra,
    on=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'],
    how='left'
)

# 6. Substituir os valores conforme odonto_rn (string → string)
for column in dOdt2.columns:
    if column not in ['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID',
       'CD_DOCUMENTO', 'DS_DOCUMENTO', 'DS_LEITO']:
        dOdt2[column].replace(odonto_rn, inplace=True)


In [ ]:
# reutilizando função para reorganizar df
dOdt3 = organizar_dados_paciente(
    dOdt2,
    chaves_agrupamento=['CD_DOCUMENTO', 'CD_PACIENTE', 'CD_ATENDIMENTO', 'DH_DOCUMENTO'],
    colunas_fixas=['NM_PRESTADOR']
)

In [ ]:
map_cols_odonto = {
'CARIE': 						'clinica_odonto___1',
'RAIZ_RESIDUAL': 				'clinica_odonto___2',
'CALCULO_DENTARIO': 			'clinica_odonto___3',
'GENGIVITE': 					'clinica_odonto___4',
'MOBILIDADE_DENTARIA': 			'clinica_odonto___5',
'PERIODONTITE': 				'clinica_odonto___6',
'LESAO_PERIAPICAL': 			'clinica_odonto___7',
'NECROSE_PULPAR': 				'clinica_odonto___8',
'PULPITE': 						'clinica_odonto___9',
'ABCESSO_DENTARIO': 			'clinica_odonto___10',
'ALVEOLITE': 					'clinica_odonto___11',
'PERICORONARITE': 				'clinica_odonto___12',
'RESTAURACAO_INSATISFATORIA': 	'clinica_odonto___13',
'PROTESE_FIXA': 				'clinica_odonto___14',
'IMPLANTE': 					'clinica_odonto___15',
'EDENTADO_TOTAL': 				'clinica_odonto___16',
'DENTADO_TOTAL': 				'clinica_odonto___17',
'DENTADO_PARCIAL': 				'clinica_odonto___18',
'FRATURA_DENTARIA': 			'clinica_odonto___19',

'PT_SUPERIOR': 					'tipo_protese_dentaria___1',
'PT_INFERIOR': 					'tipo_protese_dentaria___2',
'PPR_SUPERIOR': 				'tipo_protese_dentaria___3',
'PPR_INFERIOR': 				'tipo_protese_dentaria___4',

'IHO_S': 						'indice_ihos',
'SSIFICACAO': 					'classificacao_ihos',
'PREVALENCIA':					'prevalencia_cpod',
'NIVEL_PREVALENCIA':			'nivel_prevalencia_cpod',
'CARIADO':						'cariado_cpod',
'PERDIDO':						'perdido_cpod',
'OBTURADO':						'obturado_cpod',
'DENTE':						'dentes_cpod',

'SANGRAMENTO_SIM':				'odonto_outras_avaliacoes___1',
'FLUXO_SALIVAR_NORMAL':			'odonto_outras_avaliacoes___2',
'FLUXO_SALIVAR_SIALORREIA':		'odonto_outras_avaliacoes___3',
'FLUXO_SALIVAR_BABACAO':		'odonto_outras_avaliacoes___4',
'FLUXO_SALIVAR_XEROSTOMIA':		'odonto_outras_avaliacoes___5',
'FLUXO_SALIVAR_HIPOSSALIVACAO':	'odonto_outras_avaliacoes___6',
'CANDIASE_ORAL_SIM':			'odonto_outras_avaliacoes___7',
'HERPES_ORAL_SIM':				'odonto_outras_avaliacoes___8',
'DISFAGIA':						'odonto_outras_avaliacoes___9',
'DISTURBIO_PALADAR':			'odonto_outras_avaliacoes___10',
'TRISMO':						'odonto_outras_avaliacoes___11',
'DISTURBIOS_NEUROLOGICOS':		'odonto_outras_avaliacoes___12',
'PACIENTE_ALERGICO_SIM':		'odonto_outras_avaliacoes___13',
'PALIATIVO_SIM':				'odonto_outras_avaliacoes___14',
'DROGAS_ILICITAS':				'odonto_outras_avaliacoes___15',
'HIV':							'odonto_outras_avaliacoes___16',
'ALTERACOES_HEPATICAS':			'odonto_outras_avaliacoes___17',
'ALTERACOES_PULMONARES':		'odonto_outras_avaliacoes___18',
'ALTERACOES_RENAIS':			'odonto_outras_avaliacoes___19',

'GLICEMIA':						'exames_realizados_odonto___1', 
'TP':							'exames_realizados_odonto___2',
'RX_PANORAMICO':				'exames_realizados_odonto___3', 
'RX_PERIAPICAL':				'exames_realizados_odonto___4',

}

In [ ]:
dOdt3 = dOdt3.rename(columns=map_cols_odonto).copy()

In [ ]:
cols_drop_odonto_inicial = ['CARDIOGERIATRIA', 'CONGENITO', 'CORONARIA', 'ELETROFISIOLOGIA', 'PEDIATRIA', 'SETOR_HIPERTENSAO', 'MARCA_PASSO', 
                'MIOCARDIO', 'TX_CARDIACO', 'TX_RENAL', 'VALVULA', 'VASCULAR', 'ANTICOAGULACAO', 'AMBULATORIO', 'INTERNADO', 'CENTRO_CIRURGICO', 
                'INTERNADO_UTI', 'INTERNADO_NA_UCO', 'INTERNADO_NA_ENFERMARIA', 'INTERNADO_NO_PRONTO_SOCORRO', 'PACIENTE_PRE_CIRURGICO_NAO', 
                'PACIENTE_PRE_CIRURGICO_SIM', 'PRE_CIRURGICO_NAO', 'PRE_CIRURGICO_SIM', 'PROBLEMA_COM_TRATAMENTO_ODONTOLOGICO_SIM',
                'PALIATIVO_NAO', 'PACIENTE_ALERGICO_NAO', 'SANGRAMENTO_NAO', 'CANDIASE_ORAL_NAO', 'HERPES_ORAL_NAO', 'VESTIBULAR_11_0',
                'VESTIBULAR_11_1', 'VESTIBULAR_11_2', 'VESTIBULAR_16_0', 'VESTIBULAR_16_1', 'VESTIBULAR_26_0', 'VESTIBULAR_26_1',
       			'VESTIBULAR_31_0', 'VESTIBULAR_31_1', 'VESTIBULAR_31_2', 'VESTIBULAR_31_3', 'LINGUAL_36_0',  'LINGUAL_36_1', 'LINGUAL_36_2',
                'LINGUAL_46_0', 'LINGUAL_46_1', 'LINGUAL_46_2', 'LISTA_EXODONTIA', 'LISTA_MEDICAMENTOS', 'LISTA_QTD_GENGIVECTOMIA', 'LISTA_QTD_RESTAURACAO',
                'LISTA_QTD_RX_PERIAPICAL', 'LISTA_QTD_SELAMENTO_PROVISORIO', 'LISTA_QTD_TRATAMENTO_ENDODONTICO', 'LISTA_QTD_TRATAMENTO_PERIODONTAL',
                'MEDICACAO', 'AVALIACAO_ODONTOLOGICA', 'ORIENTACOES_DE_HIGIENE_ORAL', 'REMOCAO_DE_SUTURA', 'BIOPSIA', 'LASERTERAPIA',
                'SELANTE_FLUOR', 'SELAMENTO_PROVISORIO', 'CONDICIONAMENTO', 'EXAMES_COMPLEMENTARES', 'EXODONTIA', 'EX_ETILISTA', 'EX_TABAGISTA',
                'REGISTRO_ODONTOLOGICO', 'REGULARIZACAO_OSSEA', 'REGULARIZAZAO_OSSEA', 'DLP', 'DM', 'DPOC', 'ETILISMO', 'GENGIVECTOMIA',
                'FREQUENCIA_HIGIENE_ORAL', 'OUTRAS_DOENCAS', 'OUTROS_PROCEDIMENTOS', 'PESO', 'PESO', 'PROBLEMA_ASSOCIADO_AO_TRATAMENTO_ODONTOLOGICO',
                'PT', 'QUEIXA_PRINCIPAL', 'RESTAURACAO', 'SEDENTARISMO', 'TABAGISMO_1', 'TELEFONE', 'TIPO_DE_CIRURGIA',
                'TRATAMENTO_ENDODONTICO', 'TRATAMENTO_ODONTOLOGICO', 'TRATAMENTO_ODONTOLOGICO_NAO', 'TRATAMENTO_PERIODONTAL', 
                'ATIVIDADE_FISICA', 'AVC_CEBEBRO_VASCULAR', 'A_TERAPEUTICA_ODONTO', 'ANTECEDENTES_PESSOAIS', 'HAS', 'PPR', 
                'VESTIBULAR_11_0', 'VESTIBULAR_11_1', 'VESTIBULAR_11_2', 'VESTIBULAR_11_3', 'VESTIBULAR_16_0', 'VESTIBULAR_16_1',
                'VESTIBULAR_16_2', 'VESTIBULAR_16_3', 'VESTIBULAR_26_0', 'VESTIBULAR_26_1', 'VESTIBULAR_26_2', 'VESTIBULAR_26_3',
                'VESTIBULAR_31_0', 'VESTIBULAR_31_1', 'VESTIBULAR_31_2', 'VESTIBULAR_31_3', 'LINGUAL_36_0', 'LINGUAL_36_1', 'LINGUAL_36_2',
                'LINGUAL_36_3', 'LINGUAL_46_0', 'LINGUAL_46_1', 'LINGUAL_46_2', 'LINGUAL_46_3', 'LISTA_EXODONTIA', 'LISTA_MEDICAMENTOS', 'LISTA_QTD_GENGIVECTOMIA',
                'LISTA_QTD_RESTAURACAO', 'LISTA_QTD_RX_PERIAPICAL', 'LISTA_QTD_SELAMENTO_PROVISORIO', 'LISTA_QTD_TRATAMENTO_ENDODONTICO', 'LISTA_QTD_TRATAMENTO_PERIODONTAL',
                'MEDICACAO', 'NAO_SE_APLICA_IHOS',

                ]

dOdt3 = dOdt3.drop(columns=cols_drop_odonto_inicial, errors='ignore')


In [ ]:
dOdt3 = dOdt3.rename(columns={'CD_PACIENTE': 'record_id'})

In [ ]:
dOdt3.columns = dOdt3.columns.str.lower()

In [ ]:
# 1. Defina a lista completa de colunas que você deseja
colunas_alvo = ['cd_documento', 'record_id', 'cd_atendimento', 'dh_documento', 'dataid', 'nm_setor', 'efeo_contact', 'efeo_deamb', 'efeo_dieta', 'efeo_respir',
       'efio_hig_done', 'efio_hig_who', 'efio_mucosa', 'efio_labio', 'efio_lingua', 'efio_lesao_tm', 'pcte_urgente', 'lib_cirurgia', 'protese_dentaria', 
       'uso_anticoag', 'uso_antibiot', 'clinica_odonto___10', 'odonto_outras_avaliacoes___17', 'odonto_outras_avaliacoes___18', 'odonto_outras_avaliacoes___19',
       'anticoagulante', 'clinica_odonto___3', 'candiase_oral', 'odonto_outras_avaliacoes___7', 'cariado_cpod', 'clinica_odonto___1', 'conduta_odonto', 
       'clinica_odonto___18', 'clinica_odonto___17', 'dentes_cpod', 'diagnostico_odonto', 'odonto_outras_avaliacoes___12', 'odonto_outras_avaliacoes___10', 
       'odonto_outras_avaliacoes___15', 'clinica_odonto___16', 'odonto_outras_avaliacoes___4', 'odonto_outras_avaliacoes___6', 'odonto_outras_avaliacoes___2',
       'odonto_outras_avaliacoes___3', 'odonto_outras_avaliacoes___5', 'clinica_odonto___19', 'clinica_odonto___4', 'exames_realizados_odonto___1', 
       'odonto_outras_avaliacoes___16', 'indice_ihos', 'clinica_odonto___15', 'clinica_odonto___7', 'lesao_tecidos_moles', 'clinica_odonto___5', 'clinica_odonto___8',
       'nivel_prevalencia_cpod', 'obturado_cpod', 'paciente_alergico', 'odonto_outras_avaliacoes___13', 'odonto_outras_avaliacoes___14', 'perdido_cpod', 
       'clinica_odonto___6', 'planejamento_do_tratamento_odontologico', 'plano_de_alta', 'tipo_protese_dentaria___4', 'tipo_protese_dentaria___3', 'prevalencia_cpod', 
       'profilaxia_antibiotica', 'clinica_odonto___14', 'tipo_protese_dentaria___2', 'tipo_protese_dentaria___1', 'clinica_odonto___9', 'clinica_odonto___2', 
       'clinica_odonto___13', 'exames_realizados_odonto___3', 'exames_realizados_odonto___4', 'sangramento', 'odonto_outras_avaliacoes___1', 'classificacao_ihos',
       'exames_realizados_odonto___2', 'urgencia_e_emergencia']

# 2. O "IF": Cria uma lista apenas com as colunas que REALMENTE existem no dOdt3
# Isso evita o KeyError
colunas_validas = [col for col in colunas_alvo if col in dOdt3.columns]

# 3. Cria o dOdt4 usando apenas a lista validada
dOdt4 = dOdt3[colunas_validas].copy()

# (Opcional) Printar para você saber o que ficou de fora
colunas_ignoradas = set(colunas_alvo) - set(colunas_validas)
if colunas_ignoradas:
    print(f"Atenção: {len(colunas_ignoradas)} colunas não foram encontradas no dOdt3 e foram ignoradas.")
    print(f"Colunas ignoradas: {list(colunas_ignoradas)}")
else:
    print("Todas as colunas foram encontradas!")


In [ ]:
dOdt4 = dOdt4.drop_duplicates()

In [ ]:
#pOdt = pPrcOd[pPrcOd['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))].copy()

In [ ]:
# Dicionário de Para -> De

mapa_procedimentos = {
    # --- diagnostico_triagem ---
    'PRIMEIRA CONSULTA ODONTOLOGICA PROGRAMATICA': 'diagnostico_triagem',
    'CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA ATENÇÃO ESPEC': 'diagnostico_triagem',
    'TRATAMENTO ODONT. PARA PACIENTES COM NECESSIDADES ESPECIAIS': 'diagnostico_triagem',
    'CONSULTA OFTALMOLOGIA': 'diagnostico_triagem', # (Outlier, mas é consulta)
    'RX PANORAMICA DE MANDIBULA (ORTOPANTOMOGRAFIA)': 'diagnostico_triagem',
    'RADIOGRAFIA PERIAPICAL - INTERPROXIMAL': 'diagnostico_triagem',
    'TC DE SEIOS DA FACE': 'diagnostico_triagem',
    'TP TEMPO DE PROTROMBINA': 'diagnostico_triagem',
    'COLETA DE RASPADO EM LESOES OU SITIOS ESPE DA REGI': 'diagnostico_triagem',

    # --- prevencao_higiene ---
    'ORIENTAÇÃO DE HIGIENE BUCAL': 'prevencao_higiene',
    'REALIZAR HIGIENE ORAL - ESCOVAÇÃO COM CLOREXIDINA 0,12%': 'prevencao_higiene',
    'PROFILAXIA REMOÇÃO DA PLACA BACTERIANA': 'prevencao_higiene',
    'ORIENTAÇÃO DE HIGIENIZAÇÃO DAS PROTESES DENTARIAS': 'prevencao_higiene',
    'REALIZAR AUXILIAR HIGIENE ORAL DO PACIENTE PARA CONFORTO': 'prevencao_higiene',
    'SALIVA ARTIFICAL (GEL UMECTANTE)': 'prevencao_higiene',

    # --- periodontia (GENGIVA) ---
    'RASPAGEM CORONO-RADICULAR (POR SEXTANTE)': 'periodontia',
    'RASPAGEM ALISAMENTO E POLIMENTO SUPRAGENGIVAL (POR SEXTANTE)': 'periodontia',
    'RASPAGEM ALISAMENETO SUBGENGIVAL (POR SEXTANTE)': 'periodontia',

    # --- dentistica (RESTAURAÇÃO) ---
    'RESTAURACAO DE DENTE PERMANENTE ANTERIOR': 'dentistica',
    'RESTAURAÇÃO DENTE PERMANENTE POSTERIOR': 'dentistica',
    'RESTAURAÇÃO DENTE PERMANENTE ANTERIOR': 'dentistica', # Repetido na lista original
    'RESTAURACAO EM RESINA FOTOPOLIMERIZAVEL 1 FACE': 'dentistica',
    'RESTAURACAO EM RESINA FOTOPOLIMERIZAVEL 2 FACES': 'dentistica',
    'SELAMENTO PROVISÓRIO CAVIDADE': 'dentistica',

    # --- endodontia (CANAL) ---
    'TRATAMENTO ENDODONTICO BIRRADICULAR': 'endodontia',
    'TRATAMENTO ENDODONTICO UNIRRADICULAR': 'endodontia',
    'TRATAMENTO ENDODONTICO MULTIRRADICULAR': 'endodontia',
    'RETRATAMENTO ENDODONTICO EM DENTE PERMANENTE UNI-RADICULAR': 'endodontia',
    'RETRATAMENTO ENDODONTICO EM DENTE PERMANENTE BI-RADICULAR': 'endodontia',
    'ACESSO A POLPA DENTÁRIA E MEDICAÇÃO (POR DENTE)': 'endodontia',
    'CAPEAMENTO PULPAR': 'endodontia',
    'CURATIVO DEMORA COM OU SEM PREPARO BIOMECÂNICO': 'endodontia',
    'SELAMENTO DE PERFURAÇÃO RADICULAR': 'endodontia',

    # --- cirurgia_oral ---
    'EXODONTIA DENTES PERMANENTES': 'cirurgia_oral',
    'EXODONTIA DE RAIZ RESIDUAL': 'cirurgia_oral',
    'EXODONTIA A RETALHO': 'cirurgia_oral',
    'CURETAGEM PERIAPICAL': 'cirurgia_oral',
    'CURETAGEM APICAL': 'cirurgia_oral',
    'ODONTOSECÇÃO / RADILECTOMIA / TUNELIZAÇÃO': 'cirurgia_oral',
    'ODONTO-SECCAO': 'cirurgia_oral',
    'CORREÇÃO DE IRREGULARIDADES DE REBORDO ALVEOLAR': 'cirurgia_oral',
    'BIOPSIA DOS TECIDOS MOLES DA BOCA': 'cirurgia_oral',
    'TRATAMENTO CIRURGICO DE HEMORRAGIA BUCAL DENTAL': 'cirurgia_oral',
    'CONTR DE HEMORRAGIA C/APLIC DE AGENTE HEMOST EM REG BUCO-M': 'cirurgia_oral',
    'RETIRADA DE PONTOS DE CIRURGIA BASICA (POR PACIENTE)': 'cirurgia_oral',

    # --- farmacoterapia E ANESTESIA ---
    'AMOXICILINA 500 MG CP': 'farmacoterapia',
    'CLINDAMICINA 300 MG CP': 'farmacoterapia',
    'DIPIRONA 500 MG CP': 'farmacoterapia',
    'DEXAMETASONA 4 MG/ML AMP 2,5 ML': 'farmacoterapia',
    'CAPTOPRIL 25 MG CP': 'farmacoterapia',
    'ACIDO EPSILON  AMINOCAPROICO 50 MG/ML (1G) FAMP 20 ML': 'farmacoterapia',
    'ACIDO EPSILON AMINOCAPROICO 50 MG/ML (1G) FAMP 20 ML': 'farmacoterapia',
    'ACIDOS GRAXOS ESSENCIAIS OLEO CICATRIZANTE FR 200 ML': 'farmacoterapia',
    'LIDOCAINA+EPINEFRINA(2% + 1:100000) TUBETE 1,8 ML ODONTO': 'farmacoterapia',
    'LIDOCAINA + EPINEFRINA (2%+1:100000) TUBETE 1,8 ML ODONTO': 'farmacoterapia',

    # --- TERAPIAS ADJUVANTES / SUPORTE ---
    'APLICAÇÃO DE LASER TERAPEUTICO': 'laserterapia',
    'APLICACAO DE LASER TERAPEUTICO': 'laserterapia',
    'DIETA PASTOSA': 'suporte_hospitalar',
    'DIETA PASTOSA DIABETICO 40G': 'suporte_hospitalar',
    
	# --- LIBERAÇÃO CIRURGICA ---
    'LIBERAÇÃO ODONTOLOGICA PARA CIRURGIA CARDIACA': 'lib_cirurgia'
}


In [ ]:
pOdt = pOdt.rename(columns={'CD_PACIENTE': 'record_id'})

In [ ]:
pOdt.columns

In [ ]:
#tst = pOdt[pOdt['DH_CANCELADO'] == ""]

In [ ]:
df = pOdt.copy()

# 1. Preparação de Tipos
df['DATAID'] = pd.to_datetime(df['DATAID'], errors='coerce')
df['PRESCRICOES'] = df['PRESCRICOES'].astype(str).str.strip()

# --- CRIAÇÃO PRÉVIA DA COLUNA (Necessário para o index do Pivot) ---
# Identifica linhas de liberação
condicao_liberacao = (df['CD_TIP_PRESC'] == 45759)

# # Cria data temporária
# df['dt_libera_temp'] = np.where(condicao_liberacao, df['DATAID'], pd.NaT)

# # Propaga para todas as linhas do mesmo horário (para agrupar corretamente)
# df['dt_libera_odonto'] = df.groupby(['record_id', 'DATAID'])['dt_libera_temp'].transform('max')

# -------------------------------------------------------------------

# 2. Aplicar o Mapeamento
df['categoria_procedimento'] = df['PRESCRICOES'].map(mapa_procedimentos)

# 3. Filtrar apenas o que foi mapeado
df_mapeado = df.dropna(subset=['categoria_procedimento']).copy()

# 4. Criar a Pivot Table
df_organizado = df_mapeado.pivot_table(
    index=['record_id', 'CD_ATENDIMENTO', 'DATAID'], 
    columns='categoria_procedimento',
    values='PRESCRICOES',
    aggfunc=lambda x: 1, 
    fill_value=0 
).reset_index()

# 5. Garantir colunas zeradas
categorias_possiveis = set(mapa_procedimentos.values())
for cat in categorias_possiveis:
    if cat not in df_organizado.columns:
        df_organizado[cat] = 0

# 6. Reordenar e Selecionar Colunas
cols_fixas = ['record_id', 'CD_ATENDIMENTO', 'DATAID']
cols_dinamicas = sorted(list(categorias_possiveis))

# Filtro de segurança
cols_finais = [c for c in cols_fixas + cols_dinamicas if c in df_organizado.columns]
pOdt1 = df_organizado[cols_finais].copy()

# # --- 7. AJUSTE FINAL (Seu pedido específico) ---
# # Aqui garantimos que a coluna pegue apenas a DATA (dia/mês/ano) do DATAID
# # A coluna 'Liberado para cirurgia' vem do mapeamento. Verificamos se ela existe primeiro.

# col_liberacao = 'Liberado para cirurgia'

# if col_liberacao in df_final.columns:
#     # Garante que é datetime
#     df_final['DATAID'] = pd.to_datetime(df_final['DATAID'])
    
#     # Lógica: Onde 'Liberado para cirurgia' == 1, pega a DATA de DATAID
#     df_final['dt_libera_odonto'] = np.where(
#         df_final[col_liberacao] == 1,
#         df_final['DATAID'].dt.date,  # <--- AQUI PEGA APENAS A DATA
#         pd.NaT
#     )

# Visualização do resultado
#print(df_final.head())

In [ ]:
pOdt1.columns = pOdt1.columns.str.lower()

In [ ]:
dOdt4.info()

In [ ]:
pOdt1.columns

In [ ]:
map_cols_podonto = {
	'cirurgia_oral': 			'atividades_odonto___1',
	'dentistica': 				'atividades_odonto___2',
	'diagnostico_triagem': 		'atividades_odonto___3',
	'endodontia': 				'atividades_odonto___4',
	'farmacoterapia': 			'atividades_odonto___5',
	'laserterapia': 			'atividades_odonto___6',
	'periodontia': 				'atividades_odonto___7',
	'prevencao_higiene': 		'atividades_odonto___8',
	'suporte_hospitalar': 		'atividades_odonto___9',

}

In [ ]:
pOdt1 = pOdt1.rename(columns=map_cols_podonto).copy()

In [ ]:
dOdt4['dataid'] = pd.to_datetime(dOdt4['dataid'])


dOdt5 = pd.merge(
    dOdt4,
    pOdt1,
    on=['record_id', 'cd_atendimento', 'dataid','lib_cirurgia'],
	how='left'
)


In [ ]:
dOdt5.columns

In [ ]:
# 1. Garantir a ordenação sequencial
# Mesmo não filtrando por data, ordenamos para que o python saiba qual é a "última" linha
# Se 'dataid' for sequencial, usamos ele. Caso contrário, pode-se usar o índice original.
dOdt5 = dOdt5.sort_values(['record_id', 'dataid'], ascending=True)

# 2. Agrupamento e Consolidação
# Agrupamos por Paciente e Fase
# O método .last() pega o último valor não-nulo encontrado na coluna para aquele grupo.
# Isso resolve o problema de ter dados espalhados em linhas diferentes (como na sua imagem).
dOdt5 = dOdt5.groupby(['record_id'], as_index=False).last()

In [ ]:
dOdt5.columns

In [ ]:

dOdt5['cd_atendimento'] = dOdt5['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
regvalv_cirurgia['cd_atendimento'] = regvalv_cirurgia['cd_atendimento'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# 1. Vincular (Traz data_cirurgia e redcap_repeat_instance para dentro do dFis)
dOdt6 = vincular_instancias_temporal(
    df_principal=dOdt5,
    df_referencia=regvalv_cirurgia[['record_id', 'cd_atendimento', 'redcap_repeat_instance', 'data_cirurgia']],
    col_data_principal='dh_documento',    # Data do documento
    col_data_referencia='data_cirurgia',  # Data da cirurgia
    tolerancia_dias=180,
    # --- NOVOS ARGUMENTOS OPCIONAIS ---
    col_atend_principal='cd_atendimento', 
    col_atend_referencia='cd_atendimento'
)

# # 2. Classificar (Define o redcap_event_name baseado no tempo e local)
# dOdt7 = classificar_eventos_por_atendimento(
#     df_entrada=dOdt6,
#     evento_pre='fase1preoperatorio_arm_1',
#     evento_uti='fase4posprocedimed_arm_1',
#     evento_enfermaria='fase5recuperacaoin_arm_1',
#     evento_amb='fase6segalta30dias_arm_1',
#     col_data_documento='dh_documento',
#     col_criterio_local='nm_setor',     # Pode ser 'CD_DOCUMENTO' ou 'NM_SETOR'
#     val_uti='UTI',            # Pode ser tupla de IDs ou string 'UTI'
#     val_enfermaria='ENF',      	   # Ajuste seus códigos aqui
# 	val_amb='AMBULATORIO'
# )

In [ ]:
dOdt6['redcap_repeat_instrument'] = 'odontologia'

In [ ]:
dOdt6.columns

In [ ]:
dOdt6['redcap_event_name'] = 'fase1preoperatorio_arm_1'

In [ ]:
# --- 1. PREPARAÇÃO DOS DADOS DO REDCAP (EXISTENTES) ---

# Garantir que record_id seja string em ambos os lados para evitar erros de comparação ( "100" != 100 )
regvalv_odonto['record_id'] = regvalv_odonto['record_id'].astype(str)

# Criamos um "Set" de tuplas com (ID, EVENTO). 
# Isso funciona como uma lista de presença: "O paciente X já tem ficha no evento Y"
odonto_existentes = set(zip(
    regvalv_odonto['record_id'].astype(str), 
    regvalv_odonto['redcap_event_name'].astype(str),
    regvalv_odonto['redcap_repeat_instance'].astype(str)
    ))

print(f"Total de registros já existentes no Redcap: {len(odonto_existentes)}")

In [ ]:

# --- 2. FILTRANDO OS DADOS LOCAIS ---

# Filtra cada fase
dOdt7 = filtrar_novos_registros(dOdt6, odonto_existentes)


In [ ]:
odonto_drop = ['cd_documento', 'cd_atendimento', 'dh_documento', 'dataid', 'nm_setor', 'efeo_contact', 'efeo_deamb', 'efeo_dieta', 'efeo_respir',
       'efio_hig_done', 'efio_hig_who', 'efio_mucosa', 'efio_labio', 'efio_lingua', 'efio_lesao_tm', 'pcte_urgente', 'data_cirurgia',
       'uso_anticoag', 'uso_antibiot', 'anticoagulante', 'candiase_oral', 'conduta_odonto', 'diagnostico_odonto', 'lesao_tecidos_moles', 'paciente_alergico', 
       'planejamento_do_tratamento_odontologico', 'plano_de_alta', 'profilaxia_antibiotica', 'sangramento', 'urgencia_e_emergencia', 'cirurgia_oral']

odonto_replace = [ 	'clinica_odonto___1', 'clinica_odonto___2', 'clinica_odonto___3', 'clinica_odonto___4', 'clinica_odonto___5', 'clinica_odonto___6',
        'clinica_odonto___7', 'clinica_odonto___8', 'clinica_odonto___9', 'clinica_odonto___10', 'clinica_odonto___11', 'clinica_odonto___12', 'clinica_odonto___13',
        'clinica_odonto___14', 'clinica_odonto___15', 'clinica_odonto___16', 'clinica_odonto___17', 'clinica_odonto___18', 'clinica_odonto___19', 
        'odonto_outras_avaliacoes___1', 'odonto_outras_avaliacoes___2', 'odonto_outras_avaliacoes___3', 'odonto_outras_avaliacoes___4', 'odonto_outras_avaliacoes___5',
        'odonto_outras_avaliacoes___6', 'odonto_outras_avaliacoes___7', 'odonto_outras_avaliacoes___8', 'odonto_outras_avaliacoes___9', 'odonto_outras_avaliacoes___10',
        'odonto_outras_avaliacoes___11', 'odonto_outras_avaliacoes___12', 'odonto_outras_avaliacoes___13', 'odonto_outras_avaliacoes___14', 'odonto_outras_avaliacoes___15',
        'odonto_outras_avaliacoes___16', 'odonto_outras_avaliacoes___17', 'odonto_outras_avaliacoes___18', 'odonto_outras_avaliacoes___19', 'tipo_protese_dentaria___1',
        'tipo_protese_dentaria___2', 'tipo_protese_dentaria___3', 'tipo_protese_dentaria___4', 'exames_realizados_odonto___1', 'exames_realizados_odonto___2',
        'exames_realizados_odonto___3', 'exames_realizados_odonto___4',] # SIM = 1, NAO = 0

odonto_int = [ 'pcte_urgente', 'lib_cirurgia', 'protese_dentaria', 'uso_anticoag', 'uso_antibiot', 'atividades_odonto___1', 'atividades_odonto___2', 'atividades_odonto___3', 'atividades_odonto___4', 'atividades_odonto___5', 'atividades_odonto___6',
       	'atividades_odonto___7', 'atividades_odonto___8', 'atividades_odonto___9', 'clinica_odonto___1', 'clinica_odonto___2', 'clinica_odonto___3', 'clinica_odonto___4', 'clinica_odonto___5', 'clinica_odonto___6',
        'clinica_odonto___7', 'clinica_odonto___8', 'clinica_odonto___9', 'clinica_odonto___10', 'clinica_odonto___11', 'clinica_odonto___12', 'clinica_odonto___13',
        'clinica_odonto___14', 'clinica_odonto___15', 'clinica_odonto___16', 'clinica_odonto___17', 'clinica_odonto___18', 'clinica_odonto___19', 
        'odonto_outras_avaliacoes___1', 'odonto_outras_avaliacoes___2', 'odonto_outras_avaliacoes___3', 'odonto_outras_avaliacoes___4', 'odonto_outras_avaliacoes___5',
        'odonto_outras_avaliacoes___6', 'odonto_outras_avaliacoes___7', 'odonto_outras_avaliacoes___8', 'odonto_outras_avaliacoes___9', 'odonto_outras_avaliacoes___10',
        'odonto_outras_avaliacoes___11', 'odonto_outras_avaliacoes___12', 'odonto_outras_avaliacoes___13', 'odonto_outras_avaliacoes___14', 'odonto_outras_avaliacoes___15',
        'odonto_outras_avaliacoes___16', 'odonto_outras_avaliacoes___17', 'odonto_outras_avaliacoes___18', 'odonto_outras_avaliacoes___19', 'tipo_protese_dentaria___1',
        'tipo_protese_dentaria___2', 'tipo_protese_dentaria___3', 'tipo_protese_dentaria___4', 'exames_realizados_odonto___1', 'exames_realizados_odonto___2',
        'exames_realizados_odonto___3', 'exames_realizados_odonto___4'
        ]


dOdt8 = dOdt7.drop(columns=odonto_drop, errors='ignore')

mapa_valores = {
    'SIM': 1, 'Sim': 1, 'sim': 1,
    'NAO': 0, 'NÃO': 0, 'Não': 0, 'nao': 0
}

# Verificamos quais colunas da lista realmente existem no df para evitar erro
cols_rep_existentes = [c for c in odonto_replace if c in dOdt8.columns]
dOdt8[cols_rep_existentes] = dOdt8[cols_rep_existentes].replace(mapa_valores)

cols_int_existentes = [c for c in odonto_int if c in dOdt8.columns]
for col in cols_int_existentes:
    dOdt8[col] = pd.to_numeric(dOdt8[col], errors='coerce').fillna(0).astype('Int64')


In [ ]:
dOdt8['odontologia_complete'] = 2

In [ ]:
dOdt8['redcap_event_name'].value_counts()

In [ ]:
# 3. Executa a importação USANDO O DATAFRAME FILTRADO dPsi8
results_Odonto = import_redcap(
        df_sorted=dOdt8,     # <--- MUDEI AQUI: De dPcte para dPsi8
        api_url=api_url,
        api_key=api_key,
        batch_size=10,
        overwrite=False        
    )

# Auditoria da tabela pre-op

In [ ]:
# # 1. Limpeza Prévia (Essencial para o Merge não falhar por tipos diferentes)
# for df in [regvalv_cirurgia, aPrc]:
#     if 'record_id' in df.columns:
#         df['record_id'] = df['record_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# # 2. Alinhamento por record_id (Inner Merge)
# # Usamos inner para ver apenas quem existe em ambos. 
# # Se quiser ver quem foi deletado ou inserido, usaria how='outer'
# df_comparacao = pd.merge(
#     regvalv_cirurgia, 
#     aPrc, 
#     on='record_id', 
#     how='inner', 
#     suffixes=('_antigo', '_novo')
# )

# log_mudancas = []

# # 3. Lista de colunas para comparar
# # Filtramos apenas colunas que possuem a versão _antigo e _novo no merge
# cols_para_comparar = [c.replace('_antigo', '') for c in df_comparacao.columns if c.endswith('_antigo')]

# for col in cols_para_comparar:
#     col_a = f"{col}_antigo"
#     col_n = f"{col}_novo"
    
#     # Se a coluna 'novo' não existir por algum motivo no merge, pula
#     if col_n not in df_comparacao.columns:
#         continue

#     # Normalização dos valores para a comparação
#     v_antigo = df_comparacao[col_a].astype(str).str.strip().replace(['nan', 'None', 'NaN', 'nat', '<NA>'], '')
#     v_novo = df_comparacao[col_n].astype(str).str.strip().replace(['nan', 'None', 'NaN', 'nat', '<NA>'], '')
    
#     # Identifica onde houve mudança real
#     mudou = (v_antigo != v_novo)
    
#     if mudou.any():
#         detalhe = df_comparacao[mudou][['record_id', col_a, col_n]].copy()
#         detalhe.columns = ['record_id', 'valor_anterior', 'valor_atualizado']
#         detalhe['campo'] = col
#         log_mudancas.append(detalhe)

# # 4. Gerar o Relatório Final
# if log_mudancas:
#     df_audit = pd.concat(log_mudancas).reset_index(drop=True)
#     df_audit = df_audit[['record_id', 'campo', 'valor_anterior', 'valor_atualizado']]
    
#     # OPCIONAL: Remover mudanças onde apenas a precisão decimal mudou (ex: 1 vs 1.0)
#     # Isso é um refino extra caso o astype(str) ainda pegue diferenças sutis
#     print(f"✅ Relatório gerado: {len(df_audit)} divergências encontradas.")
# else:
#     print("✨ Nenhuma divergência encontrada entre as bases.")
    

# # 1. Criar uma cópia para não perder a base original caso algo dê errado
# df_atualizado = regvalv_cirurgia.copy()


# # 2. Definir o record_id como índice (garantindo que seja string para bater com o audit)
# df_atualizado['record_id'] = df_atualizado['record_id'].astype(str).str.replace(r'\.0$', '', regex=True)
# df_atualizado.set_index('record_id', inplace=True)    
	

In [ ]:
# # Iterar sobre o log de auditoria que você já filtrou/ajustou
# for _, linha in df_audit.iterrows():
#     rid = str(linha['record_id'])
#     campo = linha['campo']
#     novo_valor = linha['valor_atualizado']
    
#     # Verifica se o paciente existe na base destino
#     if rid in df_atualizado.index:
#         # Verifica se a coluna existe na base destino
#         if campo in df_atualizado.columns:
#             df_atualizado.at[rid, campo] = novo_valor
#         else:
#             # Se for uma coluna nova (ex: tipo_cirurgia___9), o pandas a cria
#             df_atualizado.loc[rid, campo] = novo_valor

# # 3. Resetar o índice para voltar ao formato original do REDCap
# df_atualizado.reset_index(inplace=True)


# # Tenta converter colunas numéricas automaticamente
# df_atualizado = df_atualizado.infer_objects()

# # Se você sabe que certas colunas DEVEM ser inteiras (ex: as de checkbox)
# cols_int = [c for c in df_atualizado.columns if '___' in c or 'tipo' in c]
# for c in cols_int:
#     if c in df_atualizado.columns:
#         df_atualizado[c] = pd.to_numeric(df_atualizado[c], errors='coerce').astype('Int64')



# # 3. Reimportação de demograficos
# import_redcap(
#     df_atualizado,
#     api_url=api_url,
#     api_key=api_key,
#     batch_size=10,
#     overwrite=True
# )


In [ ]:
# # # Lista de record_ids que você quer ignorar
# excluir_ids = ['1161700', '1160432', '1122999', '1110344', '1112534', '1142415', '904606', '1159923', '310758', '1106191',
#             '1141935', '1134480', '1156346', '1133920', '1005872', '1013190', '1015033', '1022380'] 

# # # O símbolo ~ inverte a lógica (lê-se: "onde o record_id NÃO está na lista")
# df_audit = df_audit[~df_audit['record_id'].isin(excluir_ids)].copy()

In [ ]:
#df_import_completo.to_excel(f"C://Users/priscilla.sequetin/Downloads/df_import_completo.xlsx", index=False)

In [ ]:
# def processar_indicadores_movimentacao(dados_entrada):
#     """
#     Processa DataFrames de movimentação de leitos para gerar indicadores
#     de tempo de permanência, readmissão em UTI e desfechos (Alta/Óbito).

#     Args:
#         dados_entrada: Pode ser um único pd.DataFrame ou uma lista de DataFrames.

#     Returns:
#         pd.DataFrame: DataFrame consolidado por atendimento com os indicadores calculados.
#     """
    
#     # 1. CONCATENAÇÃO E CÓPIA DE SEGURANÇA
#     if isinstance(dados_entrada, list):
#         # Se for uma lista de DFs, junta todos
#         df = pd.concat(dados_entrada, ignore_index=True).copy()
#     else:
#         # Se for um único DF, apenas cria cópia
#         df = dados_entrada.copy()

#     # ==============================================================================
#     # 2. PRÉ-PROCESSAMENTO
#     # ==============================================================================
    
#     # Converter colunas de data
#     date_cols = ['HR_ATENDIMENTO', 'HR_MOV_INT', 'HR_SAIDA_MOV_INT', 'HR_ALTA', 'HR_ALTA_MEDICA']
#     for col in date_cols:
#         if col in df.columns:
#             df[col] = pd.to_datetime(df[col], errors='raise')

#     # Ordenação (Essencial para o shift funcionar)
#     df = df.sort_values(['CD_ATENDIMENTO', 'HR_MOV_INT'])

#     # Remover duplicatas de CD_MOV_INT
#     if 'CD_MOV_INT' in df.columns:
#         df = df.drop_duplicates(subset=['CD_MOV_INT'])

#     # Filtro de Unidades de Teste
#     df = df[
#         (~df['UNIDADE_PASSAGEM'].isin(['UNIDADE TESTE'])) & 
#         (~df['UNIDADE_SAIDA'].isin(['UNIDADE TESTE'])) & 
#         (df['LEITO'] != 'TESTE01')
#     ]

#     # --- FUNÇÃO INTERNA: LIMPEZA DE TEXTO ---
#     def limpar_unidade(texto):
#         if pd.isna(texto): return texto
#         texto = str(texto).upper()
#         texto = re.sub(r'HOSP\d+\s*-\s*', '', texto)
#         texto = texto.replace('ENFERMARIA', 'ENF.')
#         return texto

#     df['UNIDADE_PASSAGEM'] = df['UNIDADE_PASSAGEM'].apply(limpar_unidade)
#     df['UNIDADE_SAIDA'] = df['UNIDADE_SAIDA'].apply(limpar_unidade)

#     # ==============================================================================
#     # 3. CÁLCULOS ROW LEVEL (Linha a Linha)
#     # ==============================================================================

#     # Tempo de permanência na movimentação específica
#     df['TPO_PERM_LEITO'] = (df['HR_SAIDA_MOV_INT'] - df['HR_MOV_INT']).dt.total_seconds() / 86400       # Dias
#     df['TPO_PERM_LEITO_HORAS'] = (df['HR_SAIDA_MOV_INT'] - df['HR_MOV_INT']).dt.total_seconds() / 3600 # Horas

#     # Flags de Identificação
#     df['EH_UTI'] = df['UNIDADE_PASSAGEM'].str.contains('UTI|UCPS|UCO', case=False, na=False)
#     df['EH_CC'] = df['UNIDADE_PASSAGEM'].str.contains('CIRURG|CC|CENTRO', case=False, na=False)
#     df['EH_ENF'] = df['UNIDADE_PASSAGEM'].str.contains('ENF|PRONTO', case=False, na=False)

#     # ==============================================================================
#     # 4. LÓGICA DE READMISSÃO
#     # ==============================================================================

#     # Verifica se o status de UTI mudou em relação à linha anterior do MESMO atendimento
#     df['mudou_status_uti'] = df['EH_UTI'] != df.groupby('CD_ATENDIMENTO')['EH_UTI'].shift(1)

#     # Um novo episódio começa se: É UTI agora E (o anterior não era OU é a primeira linha)
#     df['inicio_episodio_uti'] = df['EH_UTI'] & df['mudou_status_uti']

#     # Agregamos para contar quantos episódios existiram
#     df_readmissao = df.groupby('CD_ATENDIMENTO')['inicio_episodio_uti'].sum().reset_index()
#     df_readmissao.rename(columns={'inicio_episodio_uti': 'QTD_EPISODIOS_UTI'}, inplace=True)

#     # Define Flag de Readmissão (se teve > 1 episódio)
#     df_readmissao['readmissao_em_uti'] = df_readmissao['QTD_EPISODIOS_UTI'].apply(lambda x: '1' if x > 1 else '0')

#     # ==============================================================================
#     # 5. CÁLCULO DE DATAS E TEMPOS (Consolidado)
#     # ==============================================================================

#     # --- UTI ---
#     df_apenas_uti = df[df['EH_UTI']].copy()
#     df_datas_uti = df_apenas_uti.groupby('CD_ATENDIMENTO').agg({
#         'HR_MOV_INT': 'min',           
#         'HR_SAIDA_MOV_INT': 'max',     
#         'TPO_PERM_LEITO': 'sum',       
#         'TPO_PERM_LEITO_HORAS': 'sum'  
#     }).reset_index().rename(columns={
#         'HR_MOV_INT': 'admissao_uti',
#         'HR_SAIDA_MOV_INT': 'saida_uti',
#         'TPO_PERM_LEITO': 'TEMPO_TOTAL_UTI_DIAS',
#         'TPO_PERM_LEITO_HORAS': 'tempo_uti'
#     })

#     # --- ENFERMARIA ---
#     df_apenas_enf = df[df['EH_ENF']].copy()
#     df_datas_enf = df_apenas_enf.groupby('CD_ATENDIMENTO').agg({
#         'HR_MOV_INT': 'min',           
#         'HR_SAIDA_MOV_INT': 'max',     
#         'TPO_PERM_LEITO': 'sum',       
#         'TPO_PERM_LEITO_HORAS': 'sum'  
#     }).reset_index().rename(columns={
#         'HR_MOV_INT': 'admissao_enf',
#         'HR_SAIDA_MOV_INT': 'saida_enf',
#         'TPO_PERM_LEITO': 'tempo_permanencia_alta',
#         'TPO_PERM_LEITO_HORAS': 'TEMPO_TOTAL_ENF_HORAS'
#     })

#     # ==============================================================================
#     # 6. CONSOLIDAÇÃO FINAL
#     # ==============================================================================

#     # Dados gerais do atendimento
#     df1 = df.sort_values('HR_MOV_INT').groupby('CD_ATENDIMENTO').agg({
#         'CD_PACIENTE': 'first',
#         'HR_ATENDIMENTO': 'first',
#         'HR_ALTA_MEDICA': 'last',
#         'DS_MOT_ALT': 'last',
#         'UNIDADE_SAIDA': 'last'
#     }).reset_index()

#     # Merges
#     df1 = pd.merge(df1, df_readmissao, on='CD_ATENDIMENTO', how='left')
#     df1 = pd.merge(df1, df_datas_uti, on='CD_ATENDIMENTO', how='left')
#     df1 = pd.merge(df1, df_datas_enf, on='CD_ATENDIMENTO', how='left')

#     # Tratamento de Nulos
#     df1['TEMPO_TOTAL_UTI_DIAS'] = df1['TEMPO_TOTAL_UTI_DIAS'].fillna(0)
#     df1['tempo_uti'] = df1['tempo_uti'].fillna(0)
#     df1['QTD_EPISODIOS_UTI'] = df1['QTD_EPISODIOS_UTI'].fillna(0).astype(int)
#     df1['readmissao_em_uti'] = df1['readmissao_em_uti'].fillna('0')
#     df1['tempo_permanencia_alta'] = df1['tempo_permanencia_alta'].fillna(0)
#     df1['TEMPO_TOTAL_ENF_HORAS'] = df1['TEMPO_TOTAL_ENF_HORAS'].fillna(0)

#     # LOS Hospitalar Total
#     df1['tempo_permanencia_hosp_calc'] = (df1['HR_ALTA_MEDICA'] - df1['HR_ATENDIMENTO']).dt.total_seconds() / 86400

#     # ==============================================================================
#     # 7. NOVAS VARIÁVEIS DE DESFECHO (REDCap)
#     # ==============================================================================

#     def definir_desfechos_redcap(row):
#         motivo = str(row['DS_MOT_ALT']).upper()
#         unidade = str(row['UNIDADE_SAIDA']).upper()
        
#         mortalidade = 0
#         local_obito = ''
#         data_obito = ''
        
#         if 'OBITO' in motivo or 'ÓBITO' in motivo:
#             mortalidade = '1'
#             if pd.notnull(row['HR_ALTA_MEDICA']):
#                 data_obito = pd.to_datetime(row['HR_ALTA_MEDICA']).strftime('%d/%m/%Y')
            
#             if 'CIRURG' in unidade or 'CC' in unidade or 'CENTRO' in unidade:
#                 local_obito = '1'
#             elif 'UTI' in unidade or 'UCO' in unidade:
#                 local_obito = '2'
#             else:
#                 local_obito = '3'
                
#         return pd.Series([mortalidade, local_obito, data_obito])

#     colunas_redcap = ['mortalidade_hospitalar', 'local_obito', 'data_obito']
#     df1[colunas_redcap] = df1.apply(definir_desfechos_redcap, axis=1)

#     # ==============================================================================
#     # 8. FORMATAÇÃO FINAL
#     # ==============================================================================

#     cols_datas_final = ['admissao_uti', 'saida_uti', 'admissao_enf', 'saida_enf']
#     cols_numericas_final = ['TEMPO_TOTAL_UTI_DIAS', 'tempo_uti', 'tempo_permanencia_alta', 
#                             'TEMPO_TOTAL_ENF_HORAS', 'tempo_permanencia_hosp_calc']

#     for col in cols_datas_final:
#         if col in df1.columns:
#             df1[col] = pd.to_datetime(df1[col]).dt.date

#     for col in cols_numericas_final:
#         if col in df1.columns:
#             df1[col] = df1[col].astype(float).round(2)
#             df1[col] = df1[col].astype(str).str.replace(',', '.')

#     # Seleção de colunas conforme solicitado
#     colunas_view = [
#         'CD_ATENDIMENTO', 'CD_PACIENTE', 'admissao_uti', 'saida_uti', 'tempo_uti', 
#         'tempo_permanencia_alta', 'TEMPO_TOTAL_ENF_HORAS', 'readmissao_em_uti',
#         'tempo_permanencia_hosp_calc', 'mortalidade_hospitalar', 'local_obito', 'data_obito'
#     ]
    
#     # Retorna apenas as colunas que existem no df final (segurança)
#     cols_existentes = [c for c in colunas_view if c in df1.columns]
    
#     return df1[cols_existentes]

# # --- COMO USAR ---

# aEnd1 = processar_indicadores_movimentacao(aEnd)


In [ ]:
# def processar_indicadores_movimentacao(dados_entrada):
#     """
#     Calcula tempo de permanência POR UNIDADE_PASSAGEM usando:
#     HR_ATENDIMENTO (início internação) → HR_ALTA (saída hospitalar)
#     Agrupado por CD_PACIENTE + CD_ATENDIMENTO + UNIDADE_PASSAGEM
#     """
    
#     # 1. CONCATENAÇÃO SEGURA
#     if isinstance(dados_entrada, list):
#         dfs_nao_vazios = [df for df in dados_entrada if not df.empty]
#         if not dfs_nao_vazios:
#             return pd.DataFrame()
#         df = pd.concat(dfs_nao_vazios, ignore_index=True).copy()
#     else:
#         df = dados_entrada.copy()
    
#     if df.empty:
#         return df

#     # 2. PRÉ-PROCESSAMENTO - Colunas do seu SELECT
#     date_cols = ['HR_ATENDIMENTO', 'HR_ALTA', 'HR_MOV_INT', 'HR_SAIDA_MOV_INT', 'HR_ALTA_MEDICA']
#     for col in date_cols:
#         if col in df.columns:
#             df[col] = pd.to_datetime(df[col], errors='coerce')

#     # Padronização de códigos
#     df['CD_ATENDIMENTO'] = pd.to_numeric(df['CD_ATENDIMENTO'], errors='coerce').astype('Int64')
#     df['CD_PACIENTE'] = pd.to_numeric(df['CD_PACIENTE'], errors='coerce').astype('Int64')

#     # 3. ✅ CÁLCULO TEMPO NA UNIDADE_PASSAGEM (HR_ATENDIMENTO → HR_ALTA)
#     df['TEMPO_UNIDADE_HORAS'] = (
#         (df['HR_ALTA'] - df['HR_ATENDIMENTO']).dt.total_seconds() / 3600
#     ).round(2)
    
#     df['TEMPO_UNIDADE_DIAS'] = (
#         df['TEMPO_UNIDADE_HORAS'] / 24
#     ).round(2)

#     # Flags de unidade (para análise complementar)
#     df['EH_UTI'] = df['UNIDADE_PASSAGEM'].str.contains('UTI|UCPS|UCO', case=False, na=False)
#     df['EH_CC'] = df['UNIDADE_PASSAGEM'].str.contains('CIRURG|CC|CENTRO|HEMODINAMICA', case=False, na=False)
#     df['EH_ENF'] = df['UNIDADE_PASSAGEM'].str.contains('ENF|PRONTO', case=False, na=False)

#     # 4. ✅ AGRUPAMENTO FINAL: CD_PACIENTE + CD_ATENDIMENTO + UNIDADE_PASSAGEM
#     df_final = df.groupby(['CD_PACIENTE', 'CD_ATENDIMENTO', 'UNIDADE_PASSAGEM']).agg({
#         'HR_ATENDIMENTO': 'min',           # Início internação hospitalar
#         'HR_ALTA': 'max',                  # Saída hospitalar (alta/óbito)
#         'TEMPO_UNIDADE_HORAS': 'first',    # Tempo total na unidade
#         'TEMPO_UNIDADE_DIAS': 'first',     # Tempo total na unidade (dias)
#         'LEITO': 'first',
#         'HR_MOV_INT': 'min',               # Entrada no leito (complementar)
#         'HR_SAIDA_MOV_INT': 'max',         # Saída do leito (complementar)
#         'EH_UTI': 'first',
#         'EH_CC': 'first',
#         'EH_ENF': 'first'
#     }).round(2).reset_index()

#     # LOS Hospitalar Total (se disponível HR_ALTA_MEDICA)
#     if 'HR_ALTA_MEDICA' in df.columns:
#         df_los = df.groupby(['CD_PACIENTE', 'CD_ATENDIMENTO']).agg({
#             'HR_ATENDIMENTO': 'min',
#             'HR_ALTA_MEDICA': 'max'
#         }).reset_index()
#         df_los['LOS_HOSPITALAR_DIAS'] = (
#             (df_los['HR_ALTA_MEDICA'] - df_los['HR_ATENDIMENTO']).dt.total_seconds() / 86400
#         ).round(2)
#         df_final = df_final.merge(
#             df_los[['CD_PACIENTE', 'CD_ATENDIMENTO', 'LOS_HOSPITALAR_DIAS']], 
#             on=['CD_PACIENTE', 'CD_ATENDIMENTO'], 
#             how='left'
#         )

#     # 5. FORMATAÇÃO REDCap
#     # Datas
#     for col in ['HR_ATENDIMENTO', 'HR_ALTA', 'HR_MOV_INT', 'HR_SAIDA_MOV_INT']:
#         if col in df_final.columns:
#             df_final[col] = pd.to_datetime(df_final[col], errors='coerce').dt.strftime('%d/%m/%Y %H:%M')

#     # Números
#     num_cols = ['TEMPO_UNIDADE_HORAS', 'TEMPO_UNIDADE_DIAS', 'LOS_HOSPITALAR_DIAS']
#     for col in num_cols:
#         if col in df_final.columns:
#             df_final[col] = df_final[col].fillna(0).astype(str).str.replace(',', '.')

#     return df_final

# # === USO ===
# aEnd2 = processar_indicadores_movimentacao(aEnd)
# #print(f"✅ {len(aEnd1)} registros de tempo por unidade")
# #print(aEnd1[aEnd1['CD_PACIENTE'] == 1124671][['CD_PACIENTE', 'CD_ATENDIMENTO', 'UNIDADE_PASSAGEM', 'TEMPO_UNIDADE_HORAS']])


In [ ]:
# def processar_indicadores_movimentacao_v2(dados_entrada):
#     """✅ Tempo real por unidade + múltiplos atendimentos"""
    
#     # 1. CONCAT SEGURO
#     if isinstance(dados_entrada, list):
#         dfs = [df for df in dados_entrada if not df.empty]
#         if not dfs: return pd.DataFrame()
#         df = pd.concat(dfs, ignore_index=True).copy()
#     else:
#         df = dados_entrada.copy()
    
#     # 2. DATAS + CÓDIGOS
#     date_cols = ['HR_ATENDIMENTO', 'HR_MOV_INT', 'HR_SAIDA_MOV_INT', 'HR_ALTA', 'HR_ALTA_MEDICA']
#     for col in date_cols:
#         if col in df.columns:
#             df[col] = pd.to_datetime(df[col], errors='coerce')
    
#     df['CD_ATENDIMENTO'] = pd.to_numeric(df['CD_ATENDIMENTO'], errors='coerce').astype('Int64')
#     df['CD_PACIENTE'] = pd.to_numeric(df['CD_PACIENTE'], errors='coerce').astype('Int64')
    
#     # 3. TEMPO REAL POR PASSAGEM (CORRETO!)
#     df['TPO_PERM_LEITO_HORAS'] = (df['HR_SAIDA_MOV_INT'] - df['HR_MOV_INT']).dt.total_seconds() / 3600
    
#     # 4. FLAGS + FILTRO TESTE
#     df['EH_UTI'] = df['UNIDADE_PASSAGEM'].str.contains('UTI|UCPS|UCO', case=False, na=False)
#     df = df[~df['UNIDADE_PASSAGEM'].isin(['UNIDADE TESTE'])]
    
#     # 5. CONSOLIDAÇÃO POR ATENDIMENTO + UNIDADE
#     df_uti = df[df['EH_UTI']].groupby(['CD_PACIENTE', 'CD_ATENDIMENTO']).agg({
#         'TPO_PERM_LEITO_HORAS': 'sum'
#     }).round(2).reset_index().rename(columns={'TPO_PERM_LEITO_HORAS': 'TEMPO_TOTAL_UTI_HORAS'})
    
#     df_enf = df[~df['EH_UTI']].groupby(['CD_PACIENTE', 'CD_ATENDIMENTO']).agg({
#         'TPO_PERM_LEITO_HORAS': 'sum'
#     }).round(2).reset_index().rename(columns={'TPO_PERM_LEITO_HORAS': 'TEMPO_TOTAL_ENF_HORAS'})
    
#     # 6. MERGE FINAL
#     df_final = df.groupby(['CD_PACIENTE', 'CD_ATENDIMENTO']).agg({
#         'HR_ATENDIMENTO': 'min',
#         'HR_ALTA': 'max'
#     }).reset_index()
    
#     df_final = df_final.merge(df_uti, on=['CD_PACIENTE', 'CD_ATENDIMENTO'], how='left')
#     df_final = df_final.merge(df_enf, on=['CD_PACIENTE', 'CD_ATENDIMENTO'], how='left')
    
#     # Formatação
#     df_final[['TEMPO_TOTAL_UTI_HORAS', 'TEMPO_TOTAL_ENF_HORAS']] = df_final[['TEMPO_TOTAL_UTI_HORAS', 'TEMPO_TOTAL_ENF_HORAS']].fillna(0).astype(str).str.replace(',', '.')
    
#     return df_final

# # === USO ===
# aEnd3 = processar_indicadores_movimentacao(aEnd)

In [ ]:
# # Buscando docText.DESFECHOS

# sql_DESFECHO = f"""
# SET DATEFORMAT ymd;

# WITH 
# doa AS (
#     SELECT *
#     FROM dim_origem_atendimento doa1
#     WHERE DT_ATUAL = (
#         SELECT MAX(DT_ATUAL)
#         FROM dim_origem_atendimento doa2
#         WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
#     )
# )

# SELECT 
#     fde.CD_ATENDIMENTO, 
#     fde.CD_PACIENTE, 
#     fde.DH_DOCUMENTO,
#     CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
#     fde.CD_DOCUMENTO, 
#     dcd.DS_DOCUMENTO, 
#     fde.DS_LEITO,
#     fde.DS_IDENTIFICADOR, 
#     fde.DS_CAMPO, 
#     fde.DS_RESPOSTA, 
#     fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
#     dpt.NM_PRESTADOR,

#     -- Lógica de fallback para ambulatorial
#     COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR


# FROM ft_doc_eletronico_texto fde 
# LEFT JOIN dim_campo_documento dcd 	ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
# LEFT JOIN dim_prestador dpt 		ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

# -- Internação (aplica-se apenas se DS_LEITO estiver presente)
# LEFT JOIN ft_internacao_unidade fiu 
#     ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
#    	AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
#    	AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
#    	AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
#    	AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

# -- Ambulatorial
# LEFT JOIN ft_atendimento ate 
#     ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
#     AND ate.TP_ATENDIMENTO <> 'I'

# LEFT JOIN doa 
#     ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

# -- Filtros
# WHERE fde.CD_DOCUMENTO IN (1043, 1037)  
#   	AND fde.DS_CAMPO NOT IN (
#         'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
#         'Código do Documento', 'Data de Registro', 'Registro de Documento',
#         'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
#         'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
#         'Último Resultado'
#     )
# 	AND DS_RESPOSTA NOT IN ('false')
#     AND YEAR(fde.DH_DOCUMENTO) in ({ano})

# ORDER BY fde.DH_DOCUMENTO DESC;
# """ 

# # Executa a função para buscar os dados
# txDESFECHO = query_sql_to_dataframe(conn_string, sql_DESFECHO)

In [ ]:
# # Copiando a df
# tEnd = txDESFECHO.copy()

# # Removendo pacientes de teste
# tEnd = remove_test_patients(tEnd, 'CD_PACIENTE')

In [ ]:
# tEnd1 = tEnd.pivot_table(
#     index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID', 'CD_DOCUMENTO', 'DS_DOCUMENTO'],
#     columns='DS_IDENTIFICADOR',
#     values='DS_RESPOSTA',
#     aggfunc='last'  # ou 'last', dependendo da lógica de negócio
# ).reset_index()

In [ ]:
# tEnd1 = tEnd1.sort_values(['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'], ascending=True)
# tEnd1 = tEnd1.groupby(['CD_ATENDIMENTO', 'CD_PACIENTE'], as_index=False).last()

In [ ]:
# tEnd1.columns.tolist()

In [ ]:
# aObt = pd.merge(
#     tEnd1[['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID', 'CD_DOCUMENTO', 'DS_DOCUMENTO',
#            'DS_CAUSA_DA_MORTE_1', 'DT_ADMISSAO_1', 'DT_ALTA_1', 'DS_TEMPO_PERMANENCIA_1']],
#     aEnd1,
#     on=['CD_ATENDIMENTO', 'CD_PACIENTE'],
#     how='left'
# )


In [ ]:
# aObt1 = aObt.rename(columns={'CD_PACIENTE': 'record_id', 'DS_CAUSA_DA_MORTE_1': 'causa_obito', 
#                              'DT_ADMISSAO_1': 'admissao_enf',  'DT_ALTA_1': 'data_alta', 'DS_TEMPO_PERMANENCIA_1': 'tempo_permanencia_hosp'})

In [ ]:
# aObt1.columns = aObt1.columns.str.lower()

In [ ]:
# aObt1.columns.tolist()

In [ ]:
# # ==============================================================================
# # 1. PADRONIZAÇÃO DE TIPOS E CRIAÇÃO DA LISTA DE BLOQUEIO
# # ==============================================================================

# # --- A. Tratamento da base de Óbitos (regvalv_obito) ---
# # Converte record_id para Int64
# regvalv_obito['record_id'] = pd.to_numeric(regvalv_obito['record_id'], errors='raise').astype('Int64')

# # Converte mortalidade para numérico para garantir que 1 funcione, seja string '1' ou int 1
# regvalv_obito['mortalidade_hospitalar'] = pd.to_numeric(regvalv_obito['mortalidade_hospitalar'], errors='raise')

# # --- B. Identificar quem JÁ MORREU (Lista de Bloqueio) ---
# # Pacientes com mortalidade = 1 são "intocáveis"
# mask_obito_confirmado = (regvalv_obito['mortalidade_hospitalar'] == 1)
# ids_com_obito = regvalv_obito.loc[mask_obito_confirmado, 'record_id'].unique()

# print(f"🔒 Pacientes com óbito consolidado (serão ignorados): {len(ids_com_obito)}")

# # ==============================================================================
# # 2. PREPARAÇÃO DA ENTRADA (aObt1) COM FILTRO DE PROTEÇÃO
# # ==============================================================================

# # Padroniza tipos da entrada
# aObt1['record_id'] = pd.to_numeric(aObt1['record_id'], errors='coerce').astype('Int64')
# aObt1['cd_atendimento'] = pd.to_numeric(aObt1['cd_atendimento'], errors='coerce').astype('Int64')

# # --- C. O FILTRO DE SEGURANÇA ---
# # Remove do dataframe de entrada qualquer paciente que já esteja na lista de óbitos
# # Se o ID está em ids_com_obito, ele é removido de aObt1
# aObt1_filtrado = aObt1[~aObt1['record_id'].isin(ids_com_obito)].copy()

# print(f"Entrada original: {len(aObt1)} -> Após remover óbitos prévios: {len(aObt1_filtrado)}")

# # ==============================================================================
# # 3. PADRONIZAÇÃO DO HISTÓRICO (filtro_df)
# # ==============================================================================
# # filtro_df: Histórico de instâncias já existentes no REDCap para os vivos

# filtro_df['record_id'] = pd.to_numeric(filtro_df['record_id'], errors='coerce').astype('Int64')
# filtro_df['cd_atendimento'] = pd.to_numeric(filtro_df['cd_atendimento'], errors='coerce').astype('Int64')
# filtro_df['redcap_repeat_instance'] = pd.to_numeric(filtro_df['redcap_repeat_instance'], errors='coerce').fillna(0).astype(int)

# # ==============================================================================
# # 4. O MERGE ESTRATÉGICO (Usando a base filtrada aObt1_filtrado)
# # ==============================================================================

# df_processamento = pd.merge(
#     aObt1_filtrado, 
#     filtro_df,
#     on=['record_id', 'cd_atendimento'], 
#     how='left', 
#     indicator=True
# )

# # ==============================================================================
# # 5. TRATAMENTO: ATUALIZAÇÕES (CASO 'both')
# # ==============================================================================
# atualizacoes = df_processamento[df_processamento['_merge'] == 'both'].copy()
# atualizacoes = atualizacoes.drop(columns=['_merge'])

# print(f"🔄 Registros para ATUALIZAR (mesmo atendimento em pacientes vivos): {len(atualizacoes)}")

# # ==============================================================================
# # 6. TRATAMENTO: NOVOS EVENTOS (CASO 'left_only')
# # ==============================================================================
# novos_eventos_bruto = df_processamento[df_processamento['_merge'] == 'left_only'].copy()
# novos_eventos_bruto = novos_eventos_bruto.drop(columns=['_merge', 'redcap_repeat_instance'])

# if not novos_eventos_bruto.empty:
    
#     # A. Descobrir o último atendimento registrado no REDCap para cada paciente
#     df_ref_max = filtro_df.groupby('record_id')['cd_atendimento'].max().reset_index()
#     df_ref_max.rename(columns={'cd_atendimento': 'max_atend_redcap'}, inplace=True)
    
#     # B. Merge para comparação cronológica
#     novos_eventos_validados = pd.merge(
#         novos_eventos_bruto,
#         df_ref_max,
#         on='record_id',
#         how='inner' 
#     )
    
#     # C. Filtro: Só aceita se Atendimento Novo > Último Atendimento no REDCap
#     mask_futuro = novos_eventos_validados['cd_atendimento'] > novos_eventos_validados['max_atend_redcap']
#     novos_eventos = novos_eventos_validados[mask_futuro].copy()
    
#     print(f"📉 Filtro Cronológico: Restaram {len(novos_eventos)} novos eventos válidos.")

#     # --- CÁLCULO DA NOVA INSTÂNCIA ---
#     if not novos_eventos.empty:
#         df_max_instances = filtro_df.groupby('record_id')['redcap_repeat_instance'].max().reset_index()
#         df_max_instances.rename(columns={'redcap_repeat_instance': 'max_inst_atual'}, inplace=True)
        
#         novos_eventos = pd.merge(novos_eventos, df_max_instances, on='record_id', how='left')
        
#         # Define instância = Max + 1
#         novos_eventos['max_inst_atual'] = novos_eventos['max_inst_atual'].fillna(0).astype(int)
#         novos_eventos['redcap_repeat_instance'] = novos_eventos['max_inst_atual'] + 1
        
#         # Limpeza
#         novos_eventos = novos_eventos.drop(columns=['max_atend_redcap', 'max_inst_atual'])
#         print(f"🆕 Novos eventos prontos: {len(novos_eventos)}")
#     else:
#         print("ℹ️ Todos os candidatos eram antigos. Nada a inserir.")

# else:
#     print("🆕 Nenhum candidato a novo evento encontrado.")
#     novos_eventos = pd.DataFrame()

# # ==============================================================================
# # 7. CONSOLIDAÇÃO FINAL (SEPARADA POR SEGURANÇA)
# # ==============================================================================

# # A. PREPARA APENAS AS ATUALIZAÇÕES PARA IMPORTAÇÃO
# # O aObt2 agora contém EXCLUSIVAMENTE registros que já existem (update)
# aObt2 = atualizacoes.copy()

# print(f"🚀 Total de registros prontos para UPDATE no REDCap (aObt2): {len(aObt2)}")

# # B. SINALIZAÇÃO DE NOVOS EVENTOS (APENAS ALERTA)
# # Se houver novos eventos, mostramos na tela mas NÃO colocamos no aObt2
# if not novos_eventos.empty:
#     print("\n" + "!"*80)
#     print(f"⚠️  ATENÇÃO: Foram identificados {len(novos_eventos)} NOVOS EVENTOS/INSTÂNCIAS.")
#     print(f"    Eles NÃO foram adicionados ao 'aObt2' e não serão importados agora.")
#     print("!"*80)
    
#     # Seleciona colunas principais para facilitar sua leitura no print
#     cols_visualizacao = [c for c in ['record_id', 'cd_atendimento', 'redcap_repeat_instance', 'data_obito'] if c in novos_eventos.columns]
    
#     print("\n--- Amostra dos Novos Eventos para Avaliação ---")
#     print(novos_eventos[cols_visualizacao].head(15).to_string(index=False))
#     print("------------------------------------------------\n")
    
#     # Dica: Você pode inspecionar o df 'novos_eventos' manualmente agora
# else:
#     print("\nℹ️  Nenhum novo evento pendente. Apenas atualizações foram processadas.")

In [ ]:
#aObt4.to_excel(f"C://Users/priscilla.sequetin/Downloads/obito5.xlsx", index=False)

In [ ]:
#dVal5_test.to_excel(f"C://Users/priscilla.sequetin/Downloads/aval2.xlsx", index=False)

In [ ]:
# # Cria um novo DataFrame onde a segunda coluna contém a lista de textos
# docs_valvs = dVal.groupby('CD_DOCUMENTO')['DS_IDENTIFICADOR'].apply(lambda x: sorted(x.dropna().unique().tolist())).reset_index()

# # Renomeia para ficar claro
# docs_valvs.columns = ['CD_DOCUMENTO', 'lista_textos']

# print(docs_valvs.head())

In [ ]:
# print(len(dVal['DS_IDENTIFICADOR'].unique()))

# PESO CB CP NUTRICAO

In [ ]:
# Buscando docText NUTRI

sql_NUTRI = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR

FROM ft_doc_eletronico_texto fde 
LEFT JOIN dim_campo_documento dcd   ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt         ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
    AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
    AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
    AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
    AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- FILTROS ATUALIZADOS
WHERE (
        LOWER(dcd.DS_CAMPO) LIKE '%altura%' 
        OR LOWER(dcd.DS_CAMPO) LIKE '%peso%'
        OR dcd.DS_IDENTIFICADOR LIKE '%DS_CB_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CB_D_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CB_E_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%DS_CMB_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%DS_CP_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CP_D_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CP_E_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_CB_CIRCUNFERENCIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CMB_CIRCUNFERENCIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_CP_CIRCUNFERENCIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CAB_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CB_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CMB_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CP_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_DCT_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_CB_CLASSIFICACAO_DINAMOMETRIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CLASSIFICACAO_MME_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_DAC_FAMILIAR_NAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_DAC_FAMILIAR_SIM_1%'
    )
    AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
    AND DS_RESPOSTA NOT IN ('false')
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
txNUTRI = query_sql_to_dataframe(conn_string, sql_NUTRI)

txNUTRI['DS_LEITO'] = txNUTRI['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
# Buscando rbText NUTRI

sql_NUTRI = f"""
SET DATEFORMAT ymd;

WITH 
doa AS (
    SELECT *
    FROM dim_origem_atendimento doa1
    WHERE DT_ATUAL = (
        SELECT MAX(DT_ATUAL)
        FROM dim_origem_atendimento doa2
        WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
    )
)

SELECT 
    fde.CD_ATENDIMENTO, 
    fde.CD_PACIENTE, 
    fde.DH_DOCUMENTO,
    CAST(fde.DH_DOCUMENTO AS DATE) AS DATAID, 
    fde.CD_DOCUMENTO, 
    dcd.DS_DOCUMENTO, 
    fde.DS_LEITO,
    dcd.DS_IDENTIFICADOR, 
    fde.DS_CAMPO, 
    fde.resposta AS DS_RESPOSTA, 
    fde.CD_PRESTADOR_CRIOU AS CD_PRESTADOR,
    dpt.NM_PRESTADOR,

    -- Lógica de fallback para ambulatorial
    COALESCE(fiu.UNIDADE_PASSAGEM COLLATE Latin1_General_BIN, doa.DS_ORI_ATE) AS NM_SETOR

FROM FT_DOC_ELETRONICO fde 
LEFT JOIN dim_campo_documento dcd   ON fde.NK_CD_CAMPO = dcd.NK_CD_CAMPO 
LEFT JOIN dim_prestador dpt         ON fde.CD_PRESTADOR_CRIOU = dpt.NK_CD_PRESTADOR

-- Internação
LEFT JOIN ft_internacao_unidade fiu 
    ON fde.CD_ATENDIMENTO = fiu.CD_ATENDIMENTO
    AND fde.DS_LEITO IS NOT NULL AND fde.DS_LEITO <> ''
    AND fde.DS_LEITO COLLATE Latin1_General_CI_AS = fiu.LEITO COLLATE Latin1_General_CI_AS
    AND fde.DH_DOCUMENTO >= fiu.HR_MOV_INT
    AND (fde.DH_DOCUMENTO <= fiu.HR_SAIDA_MOV_INT OR fiu.HR_SAIDA_MOV_INT IS NULL)

-- Ambulatorial
LEFT JOIN ft_atendimento ate 
    ON fde.CD_ATENDIMENTO = ate.NK_CD_ATENDIMENTO
    AND ate.TP_ATENDIMENTO <> 'I'

LEFT JOIN doa 
    ON ate.CD_ORI_ATE = doa.NK_CD_ORI_ATE

-- FILTROS ATUALIZADOS
WHERE (
        LOWER(dcd.DS_CAMPO) LIKE '%altura%' 
        OR LOWER(dcd.DS_CAMPO) LIKE '%peso%'
        OR dcd.DS_IDENTIFICADOR LIKE '%DS_CB_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CB_D_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CB_E_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%DS_CMB_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%DS_CP_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CP_D_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_CP_E_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_CB_CIRCUNFERENCIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CMB_CIRCUNFERENCIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_CP_CIRCUNFERENCIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CAB_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CB_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CMB_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CP_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_DCT_CLASSIFICACAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_CB_CLASSIFICACAO_DINAMOMETRIA_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%CLA_TXT_CLASSIFICACAO_MME_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_DAC_FAMILIAR_NAO_1%'
        OR dcd.DS_IDENTIFICADOR LIKE '%RB_DAC_FAMILIAR_SIM_1%'
    )
    AND fde.DS_CAMPO NOT IN (
        'Chave do Documento', 'Código da Empresa', 'Código do Atendimento',
        'Código do Documento', 'Data de Registro', 'Registro de Documento',
        'Registro do editor', 'Identificador', 'Código do Item na Prescrição',
        'Código do Paciente', 'Código do Usuário', 'Sistema responsável',
        'Último Resultado'
    )
	AND DS_RESPOSTA NOT IN ('false')
    --AND YEAR(fde.DH_DOCUMENTO) in ({ano})
ORDER BY DH_DOCUMENTO DESC
""" 

# Executa a função para buscar os dados
rbNUTRI = query_sql_to_dataframe(conn_string, sql_NUTRI)

rbNUTRI['DS_LEITO'] = rbNUTRI['DS_LEITO'].apply(
    lambda x: 'AMBULATORIO' if pd.isna(x) or str(x).strip() == '' else x
)

In [ ]:
#Copiando os dfs para processamento

tDocNt = txNUTRI.copy()

rDocNt = rbNUTRI.copy()

In [ ]:
dNtr = pd.concat([tDocNt, rDocNt], ignore_index=True)

In [ ]:
# --- 1. IDENTIFICAR RECORD_IDS COM CAMPOS VAZIOS ---

# No instrumento FT Risk (Peso e Altura)
ids_vazios_ftrisk = regvalv_ftrisk.loc[
    (regvalv_ftrisk['peso'].isin(["", None]) | regvalv_ftrisk['peso'].isna()) |
    (regvalv_ftrisk['altura'].isin(["", None]) | regvalv_ftrisk['altura'].isna()),
    'record_id'
].unique()

# No instrumento Geronto (Circunferências)
ids_vazios_geronto = regvalv_geronto.loc[
    (regvalv_geronto['circ_abdomen_cm'].isin(["", None]) | regvalv_geronto['circ_abdomen_cm'].isna()) |
    (regvalv_geronto['circ_braquial_cm'].isin(["", None]) | regvalv_geronto['circ_braquial_cm'].isna()) |
    (regvalv_geronto['circ_panturilha_cm'].isin(["", None]) | regvalv_geronto['circ_panturilha_cm'].isna()),
    'record_id'
].unique()

# --- 2. UNIFICAR OS IDs (UNIÃO DOS CONJUNTOS) ---
# Usamos set().union para pegar todos os IDs sem duplicatas
todos_ids_vazios = set(ids_vazios_ftrisk).union(set(ids_vazios_geronto))

# Converter para lista de inteiros para garantir compatibilidade com o SQL
lista_final_ids = [int(i) for i in todos_ids_vazios if i is not None]

# --- 3. FILTRAR O DATAFRAME DE ORIGEM (SQL) ---
dNtr1 = dNtr[dNtr['CD_PACIENTE'].astype('Int64').isin(lista_final_ids)]

In [ ]:
dNtr1.columns

In [ ]:
dNtr2 = dNtr1.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'DATAID'],
    columns='DS_IDENTIFICADOR',
    values='DS_RESPOSTA',
    aggfunc='last'  # ou 'last', dependendo da lógica de negócio
).reset_index()

In [ ]:
dNtr2['DH_DOCUMENTO'] = pd.to_datetime(dNtr2['DH_DOCUMENTO'], errors='raise')
dNtr2['ANO'] = dNtr2['DH_DOCUMENTO'].dt.year


# 1. Garantir a ordenação sequencial
# Mesmo não filtrando por data, ordenamos para que o python saiba qual é a "última" linha
# Se 'dataid' for sequencial, usamos ele. Caso contrário, pode-se usar o índice original.
dNtr2 = dNtr2.sort_values(['CD_PACIENTE', 'DH_DOCUMENTO'], ascending=True)

# 2. Agrupamento e Consolidação
# Agrupamos por Paciente e Fase
# O método .last() pega o último valor não-nulo encontrado na coluna para aquele grupo.
# Isso resolve o problema de ter dados espalhados em linhas diferentes (como na sua imagem).
dNtr3 = dNtr2.groupby(['CD_PACIENTE', 'ANO'], as_index=False).last()

In [ ]:
dNtr3.columns

In [ ]:
# 1. Limpeza inicial: Converter para string, trocar vírgula por ponto e remover espaços
dNtr3['DS_PESO_2'] = dNtr3['DS_PESO_2'].astype(str).str.replace(',', '.', regex=False).str.strip()
dNtr3['DS_PESO_MAPA_1'] = dNtr3['DS_PESO_MAPA_1'].astype(str).str.replace(',', '.', regex=False).str.strip()

# Converter para numérico (NaN para o que for 'nan' ou vazio)
peso2 = pd.to_numeric(dNtr3['DS_PESO_2'], errors='coerce')
peso_mapa = pd.to_numeric(dNtr3['DS_PESO_MAPA_1'], errors='coerce')

# 2. Executar o Coalesce (Prioriza o peso_mapa, se nulo busca no peso2)
dNtr3['peso'] = peso_mapa.combine_first(peso2)

# 3. Aplicar o mesmo para Altura (conforme suas imagens mostram variações como '1,52' e '170')
dNtr3['altura'] = dNtr3['DS_ALTURA_MAPAR_1'].astype(str).str.replace(',', '.', regex=False).str.strip()



In [ ]:
# --- CRIAR COLUNA HISTORICO_FAMILIAR_DCV ---
# Verificamos se as colunas existem para evitar erros de KeyError
cols_presentes = dNtr3.columns

condicoes = []
valores = []

# Se o campo SIM estiver preenchido (não for nulo e for diferente de vazio)
if 'RB_DAC_FAMILIAR_SIM_1' in cols_presentes:
    # Ajuste o filtro dependendo se o SQL retorna '1', 'true' ou o nome do campo
    condicoes.append(
        (dNtr3['RB_DAC_FAMILIAR_SIM_1'].notna()) & 
        (dNtr3['RB_DAC_FAMILIAR_SIM_1'] != "") & 
        (dNtr3['RB_DAC_FAMILIAR_SIM_1'] != "false")
    )
    valores.append(1)

# Se o campo NÃO estiver preenchido
if 'RB_DAC_FAMILIAR_NAO_1' in cols_presentes:
    condicoes.append(
        (dNtr3['RB_DAC_FAMILIAR_NAO_1'].notna()) & 
        (dNtr3['RB_DAC_FAMILIAR_NAO_1'] != "") & 
        (dNtr3['RB_DAC_FAMILIAR_NAO_1'] != "false")
    )
    valores.append(0)

# Aplica a lógica: se houver condições válidas, cria a coluna
if condicoes:
    dNtr3['historico_familiar_dcv'] = np.select(condicoes, valores, default=np.nan)
    

In [ ]:
# Lista de colunas para remover (incluindo as de sistema e as raw de peso/altura)
cols_para_remover = [
    'CD_ATENDIMENTO', 'DATAID',
    'DS_ALTURA_MAPAR_1', 'DS_PESO_2', 'DS_PESO_MAPA_1',
    'RB_PERDA_PESO_NAO_INTENCIONAL_NAO_1', 'RB_DAC_FAMILIAR_NAO_1', 'RB_DAC_FAMILIAR_SIM_1',
    'CB_PESO_1', 'CB_PESO_2', 'CB_PESO_3',
    'DS_ALTURA_1', 'DS_ALTURA_AFERIDA_1', 'DS_ALTURA_AVALIACAO_1',
    'DS_ALTURA_DO_JOELHO_1', 'DS_ALTURA_ESTIMADA_1',
    'DS_ALTURA_EXAMES_FISICOS_1', 'DS_ALTURA_TESTE_CAMINHADA_1',
    'DS_NECESSIDADES_PESO_SAUDAVEL_1', 'DS_PERC_PERDA_PESO_HABITUAL_1',
    'DS_PERDA_DE_PESO_1', 'DS_PESO_1', 'DS_PESO_AFERIDO_1',
    'DS_PESO_ANTERIOR_1', 'DS_PESO_ASCITE_1', 'DS_PESO_ATUAL_1',
    'DS_PESO_ATUAL_AVAL_1', 'DS_PESO_CORRIGIDO_1', 'DS_PESO_EDEMA_1',
    'DS_PESO_EXAMES_FISICOS_1', 'DS_PESO_HABITUAL_1',
    'DS_PESO_HABITUAL_AVALAIACAO_1', 'DS_PESO_IDEAL_1',
    'DS_PESO_SAUDAVEL_1', 'DS_PESO_TESTE_CAMINHA_1', 'NR_PESO_PACIENTE_1',
    'PERDA_PESO_AMPUTACAO_KILO_1', 'PERDA_PESO_AMPUTACAO_PERCENTUAL_1',
    'PESO_DRENADO_1', 'PESO_SINAIS_VITAIS_1', 'PESO_SNV_1', 'PESO_VISITA_1',
    'RB_CB_D_1', 'RB_CB_E_1', 'RB_CP_D_1', 'RB_CP_E_1',
    'RB_HOUVE_PERDA_DE_PESO_NOS_ULTIMOS_3_MESES_SIM_1',
    'RB_HOUVE_PERDA_DE_PESO_NOS_ULTIMOS_TRES_MESES_NAO_1',
    'RB_HOUVE_PERDA_DE_PESO_NOS_ULTIMOS_TRES_MESES_SIM_1',
    'RB_PERDA_DE_PESO_MODERADO_1', 'RB_PERDA_PESO_NAO_INTENCIONAL_SIM_1',
    'RB_PERDA_PESO_OU_GANHO_INSUFICIENTE_ULTIMAS_SEMANAS_MESES_NAO_1',
    'SN_BAIXO_PESO_1', 'SN_QUEDA_DA_PROPRIA_ALTURA_1', 'SN_SOBREPESO_1',
]

# Executa o drop com segurança
dNtr4 = dNtr3.drop(columns=[c for c in cols_para_remover if c in dNtr3.columns])

In [ ]:
dNtr4 = dNtr4.rename(columns={'CD_PACIENTE': 'record_id', 'DS_CB_1': 'circ_braquial_cm',
       'DS_CP_1': 'circ_panturilha_cm',})



In [ ]:
dNtr4['peso'] = dNtr4['peso'].astype(str).str.replace(',', '.', regex=False)
dNtr4['altura'] = dNtr4['altura'].astype(str).str.replace(',', '.', regex=False)
dNtr4['circ_braquial_cm'] = dNtr4['circ_braquial_cm'].astype(str).str.replace(',', '.', regex=False)
dNtr4['circ_panturilha_cm'] = dNtr4['circ_panturilha_cm'].astype(str).str.replace(',', '.', regex=False)


In [ ]:
# 2. Lógica para converter Altura em 3 dígitos (ex: 1.6 -> 160)
def formatar_altura_redcap(val):
    try:
        if val == 'nan' or not val:
            return ""
        
        # Converte para float para normalizar (ex: "1.6" ou "1.60" ou "160")
        num = float(val)
        
        # Se o número for menor que 3 (ex: 1.6 ou 1.75), multiplica por 100
        if num < 3:
            num = num * 100
            
        # Converte para inteiro e remove qualquer decimal restante
        return str(int(num))
    except:
        return ""

dNtr4['altura'] = dNtr4['altura'].apply(formatar_altura_redcap)

In [ ]:
dNtr4.columns

In [ ]:
# 1. Preparação das colunas
dNtr4['record_id'] = dNtr4['record_id'].astype(str)
regvalv_cirurgia['record_id'] = regvalv_cirurgia['record_id'].astype(str)

# Criar coluna de ano para o casamento (join)
regvalv_cirurgia['data_cirurgia'] = pd.to_datetime(regvalv_cirurgia['data_cirurgia'], errors='raise')
regvalv_cirurgia['ANO'] = regvalv_cirurgia['data_cirurgia'].dt.year

# 2. Merge por record_id E ano_cirurgia
# Selecionamos as colunas necessárias da base de referência
cols_ref = ['record_id', 'ANO', 'redcap_event_name', 'redcap_repeat_instance', 'data_cirurgia']

dNtr5 = regvalv_cirurgia[cols_ref].merge(
    dNtr4,
    on=['record_id', 'ANO'],
    how='left'
)

# 3. Tratamento da Instância e Cirurgia Anterior
# Preenche padrão 1 para quem não teve match
dNtr5['redcap_repeat_instance'] = dNtr5['redcap_repeat_instance'].fillna(1).astype('Int64')

# Verificação de Cirurgia Anterior (Lista de pacientes específicos)
#pct_cir_anterior = [str(x) for x in [587546, 458634, 1056201, 1089242, 1082666]]
#dNtr5.loc[dNtr5['record_id'].isin(pct_cir_anterior), 'redcap_repeat_instance'] += 1

In [ ]:
# Remove a linha se o record_id ou o DS_CAMPO forem nulos
dNtr6 = dNtr5.dropna(subset=['circ_braquial_cm', 'circ_panturilha_cm', 'peso', 'altura'])

In [ ]:
dNtr6.info()

In [ ]:
cols_ger = ['circ_braquial_cm', 'circ_panturilha_cm']

cols_ftr = ['peso', 'altura']

# =====================================================
# 1. PREPARAÇÃO E TRATAMENTO DE DATAS
# =====================================================
dNtr6['DH_DOCUMENTO'] = pd.to_datetime(dNtr6['DH_DOCUMENTO'], errors='coerce') # Use 'coerce' para evitar erros se houver data inválida
dNtr6['data_cirurgia'] = pd.to_datetime(dNtr6['data_cirurgia'], errors='coerce')

# CORREÇÃO DO ERRO: Garantir que as colunas técnicas existam antes da extração
if 'redcap_repeat_instrument' not in dNtr6.columns:
    dNtr6['redcap_repeat_instrument'] = ""

if 'redcap_event_name' not in dNtr6.columns:
    dNtr6['redcap_event_name'] = ""

# Colunas base
cols_base = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'redcap_repeat_instrument']

# =====================================================
# 2. EXTRAÇÃO DE FATORES DE RISCO (cFtr)
# =====================================================
cFtr = extrair_instrumento(dNtr6, cols_ftr, 'fatores_risco')
if not cFtr.empty:
    cFtr['redcap_event_name'] = 'fase1preoperatorio_arm_1'
    cFtr['redcap_repeat_instrument'] = 'fatores_risco'

# =====================================================
# 3. EXTRAÇÃO DE GERONTO (cGer) com Lógica de Data
# =====================================================
cGer = extrair_instrumento(dNtr6, cols_ger, 'qualidade_vida')

if not cGer.empty:
    # Trazemos as datas para o cGer para comparação
    # Importante: incluímos 'redcap_repeat_instance' no merge se necessário
    cGer = cGer.merge(
        dNtr6[['record_id', 'DH_DOCUMENTO', 'data_cirurgia']].drop_duplicates(), 
        on='record_id', 
        how='left'
    )

    # Lógica de Evento
    condicoes_ger = [
        (cGer['DH_DOCUMENTO'] <= cGer['data_cirurgia']),
        (cGer['DH_DOCUMENTO'] > cGer['data_cirurgia'])
    ]
    escolhas_eventos = [
        'fase1preoperatorio_arm_1', 
        'fase6segalta30dias_arm_1'
    ]

    cGer['redcap_event_name'] = np.select(condicoes_ger, escolhas_eventos, default='fase1preoperatorio_arm_1')
    cGer['redcap_repeat_instrument'] = 'qualidade_vida'
    
    # Limpeza de colunas auxiliares
    cGer = cGer.drop(columns=['DH_DOCUMENTO', 'data_cirurgia'], errors='ignore')

# =====================================================
# 4. LIMPEZA DA dNtr6 E CÁLCULOS DE IMC (Seu código original)
# =====================================================

# Lista de colunas para remover da dNtr6 (Residual)
todas_cols_mapeadas = cols_ftr + cols_ger
cols_remover = list(set(todas_cols_mapeadas)) + ['cd_atendimento', 'DH_DOCUMENTO', 'CD_DOCUMENTO', 'data_cirurgia']

dNtr6_limpa = dNtr6.drop(columns=[c for c in cols_remover if c in dNtr6.columns], errors='ignore')

# --- Lógica de IMC para cFtr ---
if not cFtr.empty and 'altura' in cFtr.columns and 'peso' in cFtr.columns:
    # (Mantenha seu bloco de cálculo de IMC aqui conforme postado)
    cFtr['altura'] = cFtr['altura'].astype(str).str.replace('.', '', regex=False).str.replace('nan', '')
    cFtr['altura'] = cFtr['altura'].apply(lambda x: x.ljust(3, '0')[:3] if x not in ['', 'None'] else '')
    cFtr['peso'] = cFtr['peso'].astype(str).str.replace('nan', '')
    peso_val = pd.to_numeric(cFtr['peso'], errors='coerce')
    altura_m = pd.to_numeric(cFtr['altura'], errors='coerce') / 100
    cFtr['imc'] = np.where(altura_m > 0, peso_val / (altura_m ** 2), np.nan)
    
    def categorizar_imc(imc):
        if pd.isna(imc) or np.isinf(imc) or imc <= 0: return ""
        if imc < 18.5: return "Magreza grau I"
        if imc < 25.0: return "Eutrofia"
        if imc < 30.0: return "Pré-obesidade"
        if imc < 35.0: return "Obesidade moderada (grau I)"
        if imc < 40.0: return "Obesidade severa (grau II)"
        return "Obesidade muito severa (grau III)"

    cFtr['imc_classificacao'] = cFtr['imc'].apply(categorizar_imc)
    cFtr = cFtr.drop(columns='imc', errors='ignore')



print("✅ Processamento concluído.")

print(f"Colunas restantes na dNtr6: {len(dNtr6.columns)}")

print("✅ Instrumentos REDCap prontos:")
print(f"Nutri: {len(dNtr6)} registros")
print(f"Fatores Risco: {len(cFtr)} registros")
print(f"✅ Geronto repartido: {len(cGer)} registros processados com lógica de data.")





In [ ]:
import_redcap(
            cFtr,  # <-- Importante usar o DF LIMPO aqui
            api_url=api_url,
            api_key=api_key,
            batch_size=10,
            overwrite=False
        )

In [ ]:
# Supondo que você já baixou os dados atuais do REDCap em 'regvalv_geronto'
chaves_ger = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'qualidade_vida_complete']

# Se o instrumento for repetível, adicione 'redcap_repeat_instrument'
cGer_consolidado = mesclar_preservando_redcap(regvalv_geronto, cGer, chaves_ger)

cGer_consolidado['circ_braquial_cm'] = cGer_consolidado['circ_braquial_cm'].astype(str).str.replace(['None','nan'], '')
cGer_consolidado['circ_panturilha_cm'] = cGer_consolidado['circ_panturilha_cm'].astype(str).str.replace(['None','nan'], '')

# Agora você pode aplicar filtros de status ou cálculos extras
#cGer_consolidado['qualidade_vida_complete'] = 2

In [ ]:
import_redcap(
            cGer_consolidado,  # <-- Importante usar o DF LIMPO aqui
            api_url=api_url,
            api_key=api_key,
            batch_size=10,
            overwrite=False
        )

# PESO ANESTESIA

In [ ]:
# 1. CARREGAR O NOVO ARQUIVO (Ajuste o caminho se necessário)
# Se estiver no Jupyter/Local, mantenha o caminho. Se for no Sandbox, use apenas o nome do arquivo.
caminho_excel = r"C:\Users\priscilla.sequetin\Downloads\dados_paciente_anest.xlsx"
df_anest = pd.read_excel(caminho_excel)

# Limpeza inicial de nomes de colunas (remover espaços e garantir minúsculas se preferir)
df_anest.columns = df_anest.columns.str.strip()

# 2. PADRONIZAÇÃO DE TIPOS E DATAS
df_anest['record_id'] = df_anest['record_id'].astype(str)
df_anest['cd_atendimento'] = df_anest['cd_atendimento'].astype(str)
df_anest['data_cirurgia'] = pd.to_datetime(df_anest['data_cirurgia'], errors='coerce').dt.date

# Fazemos o mesmo para as bases do REDCap para garantir o match
regvalv_cirurgia['record_id'] = regvalv_cirurgia['record_id'].astype(str)
regvalv_cirurgia['cd_atendimento'] = regvalv_cirurgia['cd_atendimento'].astype(str)
regvalv_cirurgia['data_cirurgia'] = pd.to_datetime(regvalv_cirurgia['data_cirurgia'], errors='coerce').dt.date

regvalv_ftrisk['record_id'] = regvalv_ftrisk['record_id'].astype(str)

# =================================================================
# --- 1. IDENTIFICAR RECORD_IDS COM CAMPOS VAZIOS ---

# No instrumento FT Risk (Peso e Altura)
ids_vazios_ftrisk = regvalv_ftrisk.loc[
    (regvalv_ftrisk['peso'].isin(["", None]) | regvalv_ftrisk['peso'].isna()) |
    (regvalv_ftrisk['altura'].isin(["", None]) | regvalv_ftrisk['altura'].isna()),
    'record_id'
].unique()

# --- 2. UNIFICAR OS IDs (UNIÃO DOS CONJUNTOS) ---
# Usamos set().union para pegar todos os IDs sem duplicatas
todos_ids_vazios = set(ids_vazios_ftrisk)

# Converter para lista de inteiros para garantir compatibilidade com o SQL
lista_final_ids = [int(i) for i in todos_ids_vazios if i is not None]

# --- 3. FILTRAR O DATAFRAME DE ORIGEM (SQL) ---
df_anest_1 = df_anest[df_anest['record_id'].astype('Int64').isin(lista_final_ids)]



In [ ]:
# 1. Preparação das colunas
df_anest_1['record_id'] = df_anest_1['record_id'].astype(str)
regvalv_cirurgia['record_id'] = regvalv_cirurgia['record_id'].astype(str)

# Criar coluna de ano para o casamento (join)
regvalv_cirurgia['data_cirurgia'] = pd.to_datetime(regvalv_cirurgia['data_cirurgia'], errors='raise')
regvalv_cirurgia['ANO'] = regvalv_cirurgia['data_cirurgia'].dt.year
df_anest_1['data_cirurgia'] = pd.to_datetime(df_anest_1['data_cirurgia'], errors='raise')
df_anest_1['ANO'] = df_anest_1['data_cirurgia'].dt.year

# 2. Merge por record_id E ano_cirurgia
# Selecionamos as colunas necessárias da base de referência
cols_ref = ['record_id', 'ANO', 'redcap_event_name', 'redcap_repeat_instance', 'data_cirurgia']

df_anest_1 = regvalv_cirurgia[cols_ref].merge(
    df_anest_1,
    on=['record_id', 'ANO'],
    how='left'
)

# 3. Tratamento da Instância e Cirurgia Anterior
# Preenche padrão 1 para quem não teve match
df_anest_1['redcap_repeat_instance'] = df_anest_1['redcap_repeat_instance'].fillna(1).astype('Int64')

In [ ]:
# =====================================================
# 1. PREPARAÇÃO E TRATAMENTO DE DATAS
# =====================================================

# CORREÇÃO DO ERRO: Garantir que as colunas técnicas existam antes da extração
if 'redcap_repeat_instrument' not in df_anest_1.columns:
    df_anest_1['redcap_repeat_instrument'] = ""

if 'redcap_event_name' not in df_anest_1.columns:
    df_anest_1['redcap_event_name'] = ""

# Colunas base
cols_base = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'redcap_repeat_instrument']

# =====================================================
# 2. EXTRAÇÃO DE FATORES DE RISCO (df_anest_1)
# =====================================================
df_anest_1 = extrair_instrumento(df_anest_1, cols_ftr, 'fatores_risco')
if not df_anest_1.empty:
    df_anest_1['redcap_event_name'] = 'fase1preoperatorio_arm_1'
    df_anest_1['redcap_repeat_instrument'] = 'fatores_risco'


# =====================================================
# 4. LIMPEZA DA df_anest_1 E CÁLCULOS DE IMC (Seu código original)
# =====================================================

# Lista de colunas para remover da df_anest_1 (Residual)
todas_cols_mapeadas = cols_ftr + cols_ger
cols_remover = list(set(todas_cols_mapeadas)) + ['cd_atendimento', 'data_cirurgia']

df_anest_1_limpa = df_anest_1.drop(columns=[c for c in cols_remover if c in df_anest_1.columns], errors='ignore')

# --- Lógica de IMC para df_anest_1 ---
if not df_anest_1.empty and 'altura' in df_anest_1.columns and 'peso' in df_anest_1.columns:
    # (Mantenha seu bloco de cálculo de IMC aqui conforme postado)
    df_anest_1['altura'] = df_anest_1['altura'].astype(str).str.replace('.', '', regex=False).str.replace('nan', '')
    df_anest_1['altura'] = df_anest_1['altura'].apply(lambda x: x.ljust(3, '0')[:3] if x not in ['', 'None'] else '')
    df_anest_1['peso'] = df_anest_1['peso'].astype(str).str.replace('nan', '')
    peso_val = pd.to_numeric(df_anest_1['peso'], errors='coerce')
    altura_m = pd.to_numeric(df_anest_1['altura'], errors='coerce') / 100
    df_anest_1['imc'] = np.where(altura_m > 0, peso_val / (altura_m ** 2), np.nan)
    
    def categorizar_imc(imc):
        if pd.isna(imc) or np.isinf(imc) or imc <= 0: return ""
        if imc < 18.5: return "Magreza grau I"
        if imc < 25.0: return "Eutrofia"
        if imc < 30.0: return "Pré-obesidade"
        if imc < 35.0: return "Obesidade moderada (grau I)"
        if imc < 40.0: return "Obesidade severa (grau II)"
        return "Obesidade muito severa (grau III)"

    df_anest_1['imc_classificacao'] = df_anest_1['imc'].apply(categorizar_imc)
    df_anest_1 = df_anest_1.drop(columns='imc', errors='ignore')

# =====================================================
# 5. CONSOLIDAÇÃO E AJUSTES FINAIS
# =====================================================

# 1. Definir as chaves corretas. 
# Se o formulário fatores_risco NÃO for repetitivo no REDCap, 
# remova 'redcap_repeat_instance' da lista abaixo.
chaves_ftr = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'fatores_risco_complete'] 

# Se houver colunas que você SABE que não quer que o combine_first tente misturar, remova-as aqui
# Exemplo: evitar que a coluna de status 'complete' antiga do SQL sobrescreva a do REDCap
df_anest_1 = df_anest_1.drop(columns=['qualidade_vida_complete'], errors='ignore')

# Executa a mesclagem (Preserva o que já está no REDCap, preenche o que está vazio com o Excel)
df_anest_1_consolidado = mesclar_preservando_redcap(regvalv_ftrisk, df_anest_1, chaves_ftr)

# 2. Limpeza de formatação para o REDCap (Remover .0 de números transformados em string)
if 'altura' in df_anest_1_consolidado.columns:
    df_anest_1_consolidado['altura'] = (
        df_anest_1_consolidado['altura']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .replace(['nan', 'None', 'NaN'], '')
    )

if 'peso' in df_anest_1_consolidado.columns:
    df_anest_1_consolidado['peso'] = (
        df_anest_1_consolidado['peso']
        .astype(str)
        .replace(['nan', 'None', 'NaN'], '')
    )

# 3. Garantir que o IMC Classificação também foi consolidado
# (Opcional: você pode rodar a função de categorização novamente aqui se preferir)

print("✅ Consolidação concluída com sucesso.")
print(f"Total de registros para upload: {len(df_anest_1_consolidado)}")



In [ ]:
import_redcap(
            df_anest_1_consolidado,  # <-- Importante usar o DF LIMPO aqui
            api_url=api_url,
            api_key=api_key,
            batch_size=20,
            overwrite=False
        )


# PESO SINAIS VITAIS

In [ ]:
# Buscando docText SVIT

sql_SVIT = f"""
SET DATEFORMAT ymd;
SELECT *
  FROM [DATADANTE].[dbo].ft_sinais_vitais
  where lower(DS_SINAL_VITAL) like '%altura%' or lower(DS_SINAL_VITAL) like '%peso%'
""" 

# Executa a função para buscar os dados
txSVIT = query_sql_to_dataframe(conn_string, sql_SVIT)



In [ ]:
# --- 3. FILTRAR O DATAFRAME DE ORIGEM (SQL) ---
dSvt = txSVIT[txSVIT['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))]


In [ ]:
dSvt.columns

In [ ]:
dSvt1 = dSvt.pivot_table(
    index=['CD_ATENDIMENTO', 'CD_PACIENTE', 'DATA_COLETA'],
    columns='DS_SINAL_VITAL',
    values='VALOR',
    aggfunc='last'  # ou 'last', dependendo da lógica de negócio
).reset_index()

In [ ]:
dSvt1['DATA_COLETA'] = pd.to_datetime(dSvt1['DATA_COLETA'], errors='raise')
dSvt1['ANO'] = dSvt1['DATA_COLETA'].dt.year


# 1. Garantir a ordenação sequencial
# Mesmo não filtrando por data, ordenamos para que o python saiba qual é a "última" linha
# Se 'dataid' for sequencial, usamos ele. Caso contrário, pode-se usar o índice original.
dSvt1 = dSvt1.sort_values(['CD_PACIENTE', 'DATA_COLETA'], ascending=True)

# 2. Agrupamento e Consolidação
# Agrupamos por Paciente e Fase
# O método .last() pega o último valor não-nulo encontrado na coluna para aquele grupo.
# Isso resolve o problema de ter dados espalhados em linhas diferentes (como na sua imagem).
dSvt2 = dSvt1.groupby(['CD_PACIENTE', 'ANO'], as_index=False).last()

In [ ]:
dSvt2.columns

In [ ]:
# 1. Limpeza inicial: Converter para string, trocar vírgula por ponto e remover espaços
dSvt2['PESO - ATUAL'] = dSvt2['PESO - ATUAL'].astype(str).str.replace(',', '.', regex=False).str.strip()
dSvt2['PESO HABITUAL'] = dSvt2['PESO HABITUAL'].astype(str).str.replace(',', '.', regex=False).str.strip()

# Converter para numérico (NaN para o que for 'nan' ou vazio)
peso3 = pd.to_numeric(dSvt2['PESO - ATUAL'], errors='coerce')
peso4 = pd.to_numeric(dSvt2['PESO HABITUAL'], errors='coerce')

# 2. Executar o Coalesce (Prioriza o peso4, se nulo busca no peso3)
dSvt2['peso'] = peso3.combine_first(peso4)

# 3. Aplicar o mesmo para Altura (conforme suas imagens mostram variações como '1,52' e '170')
dSvt2['altura'] = dSvt2['ALTURA'].astype(str).str.replace(',', '.', regex=False).str.strip()

sv_drop = [ 'ALTURA',
       'PESO - ATUAL', 'PESO DRENADO', 'PESO HABITUAL', 'PESO IDEAL',
       'PESO POS - HEMODIALISE', 'PESO PRE - HEMODIALISE',
       'PESO SECO - HEMODIALISE']

dSvt3 = dSvt2.drop(columns=[c for c in sv_drop if c in dSvt2.columns])

dSvt3 = dSvt3.rename(columns={'CD_PACIENTE': 'record_id',})

dSvt3['altura'] = dSvt3['altura'].apply(formatar_altura_redcap)

In [ ]:
# 1. Preparação das colunas
dSvt3['record_id'] = dSvt3['record_id'].astype(str)
regvalv_cirurgia['record_id'] = regvalv_cirurgia['record_id'].astype(str)

# Criar coluna de ano para o casamento (join)
regvalv_cirurgia['data_cirurgia'] = pd.to_datetime(regvalv_cirurgia['data_cirurgia'], errors='raise')
#regvalv_cirurgia['ANO'] = regvalv_cirurgia['data_cirurgia'].dt.year

# 2. Merge por record_id E ano_cirurgia
# Selecionamos as colunas necessárias da base de referência
cols_ref = ['record_id', 'ANO', 'redcap_event_name', 'redcap_repeat_instance', 'data_cirurgia']

dSvt4 = regvalv_cirurgia[cols_ref].merge(
    dSvt3,
    on=['record_id', 'ANO'],
    how='left'
)

# 3. Tratamento da Instância e Cirurgia Anterior
# Preenche padrão 1 para quem não teve match
dSvt4['redcap_repeat_instance'] = dSvt4['redcap_repeat_instance'].fillna(1).astype('Int64')

In [ ]:
# Remove a linha se o record_id ou o DS_CAMPO forem nulos
dSvt4 = dSvt4.dropna(subset=['peso', 'altura'])

In [ ]:
dSvt4.columns

In [ ]:
cols_svt = ['peso', 'altura']

# =====================================================
# 1. PREPARAÇÃO E TRATAMENTO DE DATAS
# =====================================================

# CORREÇÃO DO ERRO: Garantir que as colunas técnicas existam antes da extração
if 'redcap_repeat_instrument' not in dSvt4.columns:
    dSvt4['redcap_repeat_instrument'] = ""

if 'redcap_event_name' not in dSvt4.columns:
    dSvt4['redcap_event_name'] = ""

# Colunas base
cols_base = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'redcap_repeat_instrument']

# =====================================================
# 2. EXTRAÇÃO DE FATORES DE RISCO (dSvt4)
# =====================================================
dSvt4 = extrair_instrumento(dSvt4, cols_ftr, 'fatores_risco')
if not dSvt4.empty:
    dSvt4['redcap_event_name'] = 'fase1preoperatorio_arm_1'
    dSvt4['redcap_repeat_instrument'] = 'fatores_risco'


# =====================================================
# 4. LIMPEZA DA dSvt4 E CÁLCULOS DE IMC (Seu código original)
# =====================================================

# Lista de colunas para remover da dSvt4 (Residual)
todas_cols_mapeadas = cols_ftr + cols_ger
cols_remover = list(set(todas_cols_mapeadas)) + ['cd_atendimento', 'data_cirurgia']

dSvt4_limpa = dSvt4.drop(columns=[c for c in cols_remover if c in dSvt4.columns], errors='ignore')

# --- Lógica de IMC para dSvt4 ---
if not dSvt4.empty and 'altura' in dSvt4.columns and 'peso' in dSvt4.columns:
    # (Mantenha seu bloco de cálculo de IMC aqui conforme postado)
    dSvt4['altura'] = dSvt4['altura'].astype(str).str.replace('.', '', regex=False).str.replace('nan', '')
    dSvt4['altura'] = dSvt4['altura'].apply(lambda x: x.ljust(3, '0')[:3] if x not in ['', 'None'] else '')
    dSvt4['peso'] = dSvt4['peso'].astype(str).str.replace('nan', '')
    peso_val = pd.to_numeric(dSvt4['peso'], errors='coerce')
    altura_m = pd.to_numeric(dSvt4['altura'], errors='coerce') / 100
    dSvt4['imc'] = np.where(altura_m > 0, peso_val / (altura_m ** 2), np.nan)
    
    def categorizar_imc(imc):
        if pd.isna(imc) or np.isinf(imc) or imc <= 0: return ""
        if imc < 18.5: return "Magreza grau I"
        if imc < 25.0: return "Eutrofia"
        if imc < 30.0: return "Pré-obesidade"
        if imc < 35.0: return "Obesidade moderada (grau I)"
        if imc < 40.0: return "Obesidade severa (grau II)"
        return "Obesidade muito severa (grau III)"

    dSvt4['imc_classificacao'] = dSvt4['imc'].apply(categorizar_imc)
    dSvt4 = dSvt4.drop(columns='imc', errors='ignore')

# =====================================================
# 5. CONSOLIDAÇÃO E AJUSTES FINAIS
# =====================================================

# 1. Definir as chaves corretas. 
# Se o formulário fatores_risco NÃO for repetitivo no REDCap, 
# remova 'redcap_repeat_instance' da lista abaixo.
chaves_ftr = ['record_id', 'redcap_event_name', 'redcap_repeat_instance', 'fatores_risco_complete'] 

# Se houver colunas que você SABE que não quer que o combine_first tente misturar, remova-as aqui
# Exemplo: evitar que a coluna de status 'complete' antiga do SQL sobrescreva a do REDCap
dSvt4 = dSvt4.drop(columns=['qualidade_vida_complete'], errors='ignore')

# Executa a mesclagem (Preserva o que já está no REDCap, preenche o que está vazio com o Excel)
dSvt4_consolidado = mesclar_preservando_redcap(regvalv_ftrisk, dSvt4, chaves_ftr)

# 2. Limpeza de formatação para o REDCap (Remover .0 de números transformados em string)
if 'altura' in dSvt4_consolidado.columns:
    dSvt4_consolidado['altura'] = (
        dSvt4_consolidado['altura']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .replace(['nan', 'None', 'NaN'], '')
    )

if 'peso' in dSvt4_consolidado.columns:
    dSvt4_consolidado['peso'] = (
        dSvt4_consolidado['peso']
        .astype(str)
        .replace(['nan', 'None', 'NaN'], '')
    )

# 3. Garantir que o IMC Classificação também foi consolidado
# (Opcional: você pode rodar a função de categorização novamente aqui se preferir)

print("✅ Consolidação concluída com sucesso.")
print(f"Total de registros para upload: {len(dSvt4_consolidado)}")


In [ ]:
dSvt4_consolidado = dSvt4_consolidado.drop(columns=['imc'])

In [ ]:
import_redcap(
            dSvt4_consolidado,  # <-- Importante usar o DF LIMPO aqui
            api_url=api_url,
            api_key=api_key,
            batch_size=20,
            overwrite=False
        )


# ECOS

In [31]:
d2Eco1 = d2Eco.copy()

# Chamando a função para remover pacientes de teste
d2Eco1 = remove_test_patients(d2Eco1, 'CD_PACIENTE')

d2Eco1 = d2Eco1[d2Eco1['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))]

In [32]:
# 1. Converter para datetime (necessário para extrair o ano)
d2Eco1['DT_REVISADO'] = pd.to_datetime(d2Eco1['DT_REVISADO'], errors='raise')

# 2. Extrair o ANO
d2Eco1['ANO'] = d2Eco1['DT_REVISADO'].dt.year

# 3. Transformar em DATE apenas (remove H:M:S)
# Usamos o acessor .dt.date
d2Eco1['DT_REVISADO'] = d2Eco1['DT_REVISADO'].dt.date

# 4. Ordenação e Agrupamento
d2Eco1 = d2Eco1.sort_values(['CD_PACIENTE', 'DT_REVISADO'], ascending=True)

# 5. Consolidação
# Nota: .first() pegará a primeira data encontrada no grupo (a menor, devido ao sort acima)
d2Eco2 = d2Eco1.groupby(['CD_PACIENTE', 'DT_REVISADO'], as_index=False).last()

In [33]:
d2Eco2.columns.to_list()

['CD_PACIENTE',
 'DT_REVISADO',
 'ID',
 'CD_ITEM_PEDIDO',
 'CD_ATENDIMENTO',
 'DS_LAUDO_TXT',
 'DT_CADASTRO',
 'DT_ENTRADA_EXAME',
 'DT_PEDIDO',
 'DT_ALTERACAO',
 'DT_STUDY',
 'DT_IMPRESSO',
 'DT_ASSINADO',
 'DT_DIGITADO',
 'DT_DITADO',
 'DT_PREVISAO_ENTREGA',
 'ALTURA',
 'PESO',
 'ASC',
 'JANELA_LIMITADA',
 'RAIZ_DA_AORTA_SEIOS_DE_VALSALVA',
 'AORTA_ASCENDENTE_PROXIMAL',
 'DIAMETRO_ATRIO_ESQUERDO',
 'VOLUME_INDEXADO_ATRIO_ESQUERDO',
 'DIAMETRO_DIASTOLICO_FINAL_DO_VE',
 'DIAMETRO_SISTOLICO_FINAL_DO_VE',
 'ESPESSURA_DIASTOLICA_DO_SEPTO',
 'ESPESSURA_DIASTOLICA_DA_PAREDE_POSTERIOR',
 'FRACAO_DE_EJECAO_VE_SIMPSON',
 'FRACAO_DE_EJECAO_VE_3D',
 'FRACAO_DE_EJECAO_VE_TEICHHOLZ',
 'STRAIN_LONGITUDINAL_GLOBAL_VE',
 'MASSA_VENTRICULAR_ESQUERDA',
 'MASSA_DO_VE_INDEXADA',
 'ESPESSURA_RELATIVA_DA_PAREDE_POSTERIOR',
 'VOLUME_DIASTOLICO_FINAL_VE',
 'VOLUME_SISTOLICO_FINAL_VE',
 'STRAIN_LONGITUDINAL_PAREDE_LIVRE_VD',
 'FAC',
 'TAPSE',
 'TROMBO',
 'S',
 'FC',
 'PSAP',
 'AV_MITRAL',
 'AV_AORTICA',
 'GD_

In [34]:
# 1. Mapeamento Direto (Incluindo record_id)
mapping_eco = {
    'CD_PACIENTE': 'record_id',
    'CD_ATENDIMENTO': 'cd_atendimento',
    'DT_REVISADO': 'data_do_eco',
	'RAIZ_DA_AORTA_SEIOS_DE_VALSALVA': 'raiz_aorta',
	'AORTA_ASCENDENTE_PROXIMAL': 'aorta_ascendente',
	'DISFUNCAO_VENTRICULAR_SEVERA': 'disfuncao_ventricular_encam',
	'DISFUNCAO_DIASTOLICA': 'disfuncao_ventricular_encam_2',
	'FRACAO_DE_EJECAO_VE_SIMPSON': 'feve_encam',
	'FAC':'fac_encam',
	'TAPSE': 'tapse_encam',
	'PSAP': 'resultado_eco_psap',
	'DIAMETRO_DIASTOLICO_FINAL_DO_VE': 'resultado_eco_dimensao_ve',
	'DIAMETRO_ATRIO_ESQUERDO': 'resultado_eco_dimensao_ae',
    'GD_MITRAL_MAX': 'gd_mitral_max' ,
	'GD_MITRAL_MED': 'gd_mitral_med',
	'GS_MITRAL_MAX': 'gs_mitral_max',
	'GS_MITRAL_MED': 'gs_mitral_med',
	'GD_AORTICO_MAX': 'gd_aortico_max',
	'GD_AORTICO_MED': 'gd_aortico_med',
	'GS_AORTICO_MAX': 'gs_aortico_max',
	'GS_AORTICO_MED': 'gs_aortico_med',
	'GD_TRICUSPIDE_MAX': 'gd_tricuspide_max',
	'GD_TRICUSPIDE_MED': 'gd_tricuspide_med',
	'GS_TRICUSPIDE_MAX': 'gs_tricuspide_max',
	'GS_TRICUSPIDE_MED': 'gs_tricuspide_med',

}

# Renomeando as colunas (CD_PACIENTE vira record_id aqui)
d2Eco2 = d2Eco2.rename(columns=mapping_eco)

# 6. Seleção final das colunas (Ordenação RedCap style)
colunas_finais = [
    'record_id', 'cd_atendimento', 'ANO', 'data_do_eco', 
    'raiz_aorta', 'aorta_ascendente', 'disfuncao_ventricular_encam', 
    'disfuncao_ventricular_encam_2', 'feve_encam', 'fac_encam', 
    'tapse_encam', 'resultado_eco_psap', 'resultado_eco_dimensao_ve', 
    'resultado_eco_dimensao_ae', 'gd_mitral_max' , 'gd_mitral_med', 'gs_mitral_max',
    'gs_mitral_med', 'gd_aortico_max', 'gd_aortico_med', 'gs_aortico_max', 
    'gs_aortico_med', 'gd_tricuspide_max', 'gd_tricuspide_med', 'gs_tricuspide_max',
    'gs_tricuspide_med',
]

# Criar o DF final aLbs2
d2Eco3 = d2Eco2[[c for c in colunas_finais if c in d2Eco2.columns]].copy()

In [35]:
# ==============================================================================
# PASSO 7: Vincular Eco à Cirurgia e pegar o ÚLTIMO antes da cirurgia
# ==============================================================================

# 1. Garantir que as chaves sejam do mesmo tipo (string ajuda a evitar conflitos de Int/Float nulos)
d2Eco3['record_id'] = d2Eco3['record_id'].astype(str)
regvalv_cirurgia['record_id'] = regvalv_cirurgia['record_id'].astype(str)

# 2. Garantir que as datas sejam objetos do tipo 'date' para comparação correta
d2Eco3['data_do_eco'] = pd.to_datetime(d2Eco3['data_do_eco'], errors='raise').dt.date
if 'data_cirurgia' in regvalv_cirurgia.columns:
    regvalv_cirurgia['data_cirurgia'] = pd.to_datetime(regvalv_cirurgia['data_cirurgia'], errors='raise').dt.date
else:
    # Se a coluna não existir, cria uma vazia para não quebrar o código
    regvalv_cirurgia['data_cirurgia'] = pd.NaT

# 3. Fazer o cruzamento (Merge) partindo dos Ecos para as Cirurgias
# Usamos 'left' para garantir que não vamos perder pacientes que só têm Eco e não têm cirurgia
df_eco_cirurgia = d2Eco3.merge(
    regvalv_cirurgia[['record_id', 'redcap_repeat_instance', 'data_cirurgia']], 
    on='record_id', 
    how='left'
)

# 4. Tratar pacientes que não cruzaram com nenhuma cirurgia (instância vira 1)
df_eco_cirurgia['redcap_repeat_instance'] = df_eco_cirurgia['redcap_repeat_instance'].fillna(1).astype(int)

# 5. Filtrar apenas os exames feitos ANTES ou NO MESMO DIA da cirurgia.
# O '| df_eco_cirurgia['data_cirurgia'].isna()' garante que pacientes sem cirurgia mantenham seus exames.
condicao_data = (df_eco_cirurgia['data_do_eco'] <= df_eco_cirurgia['data_cirurgia']) | df_eco_cirurgia['data_cirurgia'].isna()
df_eco_valido = df_eco_cirurgia[condicao_data].copy()

# 6. Ordenar para pegar a ÚLTIMA ocorrência antes da cirurgia
# Ordenamos por Paciente Crescente, Instância Crescente e Data Decrescente (o primeiro será o mais novo)
df_eco_valido = df_eco_valido.sort_values(
    by=['record_id', 'redcap_repeat_instance', 'data_do_eco'], 
    ascending=[True, True, False]
)

# 7. Remover as duplicatas para ficar apenas com o mais recente de cada instância de cirurgia
d2Eco4 = df_eco_valido.drop_duplicates(
    subset=['record_id', 'redcap_repeat_instance'], 
    keep='first'
)

# (Opcional) Visualizar o resultado e colunas finais
#print(f"Total de Ecos pré-operatórios vinculados: {len(df_eco_final)}")

C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_28540\1698610732.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_eco_cirurgia['redcap_repeat_instance'] = df_eco_cirurgia['redcap_repeat_instance'].fillna(1).astype(int)


In [36]:
d2Eco4['redcap_repeat_instrument'] = 'ecocardiograma'
d2Eco4['redcap_event_name'] = 'fase1preoperatorio_arm_1'


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_28540\3495931153.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d2Eco4['redcap_repeat_instrument'] = 'ecocardiograma'
C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_28540\3495931153.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d2Eco4['redcap_event_name'] = 'fase1preoperatorio_arm_1'


In [37]:
d2Eco4 = d2Eco4.drop(columns=['ANO', 'data_cirurgia', 'cd_atendimento'], errors='ignore')

In [ ]:
d2Eco4

In [38]:

d2Eco4.to_excel('C://Users/priscilla.sequetin/Downloads/d2Eco4_2.xlsx', index=False)

In [ ]:
results_ecos = import_redcap(
            d2Eco4, 
            api_url=api_url,
            api_key=api_key,
            batch_size=30,
            overwrite=False
        )

# LABS

In [62]:
# Buscando docText LABS

sql_LABS = f"""
SELECT 
        CD_PACIENTE,
        CD_ATENDIMENTO,
        CD_EXAME,
        NM_CAMPO,
        NM_EXA_LAB,
        DS_RESULTADO,
        DS_UNIDADE,
        DT_REVISAO
    FROM
        ft_resultado_laboratorial
    WHERE 
		CD_EXAME IN (230, 778, 1012, 736, 1012, 938, 727, 379, 977, 356, 954, 886 )
""" 

# Executa a função para buscar os dados
bLABS = query_sql_to_dataframe(conn_string, sql_LABS)

Conexão estabelecida com sucesso!


C:\Users\priscilla.sequetin\AppData\Local\Temp\ipykernel_21860\2274903237.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexao)


In [85]:
aLbs = bLABS.copy()

# Chamando a função para remover pacientes de teste
aLbs = remove_test_patients(aLbs, 'CD_PACIENTE')

aLbs = aLbs[aLbs['CD_PACIENTE'].astype('Int64').isin(regvalv_pcte['record_id'].astype('Int64'))]

In [86]:
aLbs = aLbs.pivot_table(index=('CD_ATENDIMENTO', 'CD_PACIENTE', 'DT_REVISAO', 'DS_UNIDADE'), 
                            columns= 'NM_CAMPO',
                            values= 'DS_RESULTADO', 
                            aggfunc='first').reset_index()

In [75]:
# 1. Converter para datetime (necessário para extrair o ano)
aLbs['DT_REVISAO'] = pd.to_datetime(aLbs['DT_REVISAO'], errors='raise')

# 2. Extrair o ANO
aLbs['ANO'] = aLbs['DT_REVISAO'].dt.year

# 3. Transformar em DATE apenas (remove H:M:S)
# Usamos o acessor .dt.date
aLbs['DT_REVISAO'] = aLbs['DT_REVISAO'].dt.date

# 4. Ordenação e Agrupamento
aLbs = aLbs.sort_values(['CD_PACIENTE', 'DT_REVISAO'], ascending=True)

# 5. Consolidação
# Nota: .first() pegará a primeira data encontrada no grupo (a menor, devido ao sort acima)
aLbs1 = aLbs.groupby(['CD_PACIENTE', 'DT_REVISAO'], as_index=False).first()

In [76]:

# 1. Mapeamento Direto (Incluindo record_id)
mapping = {
    'CD_PACIENTE': 'record_id',
    'CD_ATENDIMENTO': 'cd_atendimento',
    'DT_REVISAO': 'dt_lab',
    'Hemoglobina - hemograma': 'hb_pre',
    'Leucocitos totais - HEMOGRAMA': 'leucocitos_pre',
    'Plaquetas': 'plaquetas_pre',
    'Creatinina serica': 'creatinina_pre',
    'RESULTADO': 'ureia_pre',
    'Pro Peptideo Natriuretico Tipo B': 'bnp_pre',
    'Hemoglobina Glicada (A1C)': 'hba1c_encam',
    'TSH - Hormonio Tireoestimulante': 'tsh_encam'
}

# 2. Lista de colunas que compõem o bloco de Sorologia
cols_sorologia_fontes = [
    ' ', 
    'VDRL', 
    'Anticorpos Anti HBc IgM', 
    'Antigeno Australia (HBsAG)', 
    'Resultado (S/CO) - HEPATITE C',
    'HIV Resultado Quantitativo (Etapa I'
]

# 3. Função de Categorização (1: Pendente, 2: Negativa, 3: Positiva)
def categorizar_sorologia(row):
    valores = [str(row[col]).upper() for col in cols_sorologia_fontes if col in row.index and pd.notnull(row[col])]
    
    if not valores or all(v.strip() == '' for v in valores):
        return 1
    
    positivos = ['POSI', 'REAGEN', 'DETECTA', 'PRESENTE', 'SIM', ' +']
    negativos = ['NEGA', 'NAO REAGEN', 'NÃO REAGEN', 'AUSENTE', 'INDETECTA', ' -']

    if any(any(p in v for p in positivos) for v in valores):
        return 3
    if any(any(n in v for n in negativos) for v in valores):
        return 2
    
    return 1

# --- Execução ---
# Aplicando a lógica de sorologia antes de renomear as colunas originais
aLbs1['sorologias_encam'] = aLbs1.apply(categorizar_sorologia, axis=1)

# Renomeando as colunas (CD_PACIENTE vira record_id aqui)
aLbs1 = aLbs1.rename(columns=mapping)

# 2. AJUSTE: Forçar para inteiro (Int64 garante que não haverá .0 mesmo com nulos)
aLbs1['sorologias_encam'] = aLbs1['sorologias_encam'].astype('Int64')

# Garantindo a data das sorologias
aLbs1['data_sorologias_encam'] = aLbs1['dt_lab']

# 6. Seleção final das colunas (Ordenação RedCap style)
colunas_finais = [
    'record_id', 'cd_atendimento', 'ANO', 'dt_lab', 
    'hb_pre', 'leucocitos_pre', 'plaquetas_pre', 
    'creatinina_pre', 'ureia_pre', 'bnp_pre', 
    'hba1c_encam', 'tsh_encam', 'sorologias_encam', 
    'data_sorologias_encam'
]

# Criar o DF final aLbs2
aLbs2 = aLbs1[[c for c in colunas_finais if c in aLbs1.columns]].copy()




In [83]:
aLbs2.columns

Index(['record_id', 'cd_atendimento', 'ANO', 'dt_lab', 'hb_pre',
       'leucocitos_pre', 'plaquetas_pre', 'creatinina_pre', 'ureia_pre',
       'bnp_pre', 'hba1c_encam', 'tsh_encam', 'sorologias_encam',
       'data_sorologias_encam'],
      dtype='object', name='NM_CAMPO')

In [88]:
# =====================================================================
# PRÓXIMOS PASSOS: Cruzamento com regvalv_cirurgia e definição da Instância
# =====================================================================

# 1. Garantir que os tipos de chaves estão iguais (str) para evitar falhas no merge
aLbs2['record_id'] = aLbs2['record_id'].astype(str)
aLbs2['cd_atendimento'] = aLbs2['cd_atendimento'].astype(str)

# Preparar o df de cirurgias (ajuste os nomes das colunas se necessário)
reg_cir = regvalv_cirurgia[['record_id', 'redcap_repeat_instance', 'cd_atendimento']].copy()
reg_cir['record_id'] = reg_cir['record_id'].astype(str)
reg_cir['cd_atendimento'] = reg_cir['cd_atendimento'].astype(str)

# ---------------------------------------------------------------------
# CENÁRIO 1: cd_atendimento IGUAL (Primeira Ocorrência)
# ---------------------------------------------------------------------
# Fazemos um merge exato pelo record_id e cd_atendimento
match_exato = aLbs2.merge(reg_cir, on=['record_id', 'cd_atendimento'], how='inner')

# Ordena por data CRESCENTE para pegar o PRIMEIRO exame desse atendimento
match_exato = match_exato.sort_values(by=['record_id', 'redcap_repeat_instance', 'dt_lab'], ascending=True)

# Pega a primeira ocorrência
df_match = match_exato.groupby(['record_id', 'redcap_repeat_instance'], as_index=False).first()


# ---------------------------------------------------------------------
# CENÁRIO 2: cd_atendimento DIFERENTE (Última Ocorrência)
# ---------------------------------------------------------------------
# Para esse cenário, precisamos da ÚLTIMA data de laboratório do paciente.
# Então ordenamos aLbs2 por data DECRESCENTE e pegamos o primeiro registro de cada paciente.
aLbs2_ultimos = aLbs2.sort_values(by=['record_id', 'dt_lab'], ascending=False).groupby('record_id', as_index=False).first()

# SUB-CENÁRIO 2A: Paciente TEM cirurgia, mas o cd_atendimento do exame não bateu
# Identificamos quais instâncias de cirurgia já foram preenchidas no Cenário 1
cirurgias_resolvidas = df_match[['record_id', 'redcap_repeat_instance']].drop_duplicates()

# Fazemos um merge para ver quais cirurgias sobraram (não tiveram match exato)
cirurgias_sem_match = reg_cir.merge(cirurgias_resolvidas, on=['record_id', 'redcap_repeat_instance'], how='left', indicator=True)
cirurgias_sem_match = cirurgias_sem_match[cirurgias_sem_match['_merge'] == 'left_only'].drop(columns=['_merge'])

# Trazemos os exames mais recentes (aLbs2_ultimos) para preencher essas instâncias de cirurgia
# Removemos o cd_atendimento do exame para não sobrescrever a informação da cirurgia
df_cir_fallback = cirurgias_sem_match[['record_id', 'redcap_repeat_instance', 'cd_atendimento']].merge(
    aLbs2_ultimos.drop(columns=['cd_atendimento', 'ANO'], errors='ignore'), 
    on='record_id', 
    how='inner'
)

# SUB-CENÁRIO 2B: Pacientes que NÃO TEM CIRURGIA NENHUMA (Aguardando vaga)
# Pega quem está no lab, mas não existe em regvalv_cirurgia
pacientes_com_cirurgia = reg_cir['record_id'].unique()
pacientes_sem_cirurgia = aLbs2_ultimos[~aLbs2_ultimos['record_id'].isin(pacientes_com_cirurgia)].copy()

# Como não têm cirurgia, não têm 'redcap_repeat_instance'. Setamos como 1 por padrão.
pacientes_sem_cirurgia['redcap_repeat_instance'] = 1


# ---------------------------------------------------------------------
# CONSOLIDAÇÃO FINAL
# ---------------------------------------------------------------------
# Juntamos os 3 blocos: 
# (1) Match exato, (2A) Tem cirurgia mas pegou exame recente, (2B) Não tem cirurgia
aLbs3 = pd.concat([df_match, df_cir_fallback, pacientes_sem_cirurgia], ignore_index=True)

# Ordena, reseta o índice e garante que a instância fique como string ou int
aLbs3 = aLbs3.sort_values(['record_id', 'dt_lab']).reset_index(drop=True)

# Opcional: definir instrument e event do REDCap
aLbs3['redcap_event_name'] = 'fase1preoperatorio_arm_1'
aLbs3['redcap_repeat_instrument'] = 'exames_laboratoriais'
aLbs3['redcap_event_name'] = aLbs3['redcap_event_name'].replace({
    "fase3procedcirurgi_arm_1": "fase1preoperatorio_arm_1"
})


In [89]:
aLbs4 = aLbs3.drop(columns=['ANO', 'data_cirurgia', 'cd_atendimento', 'data_sorologias_encam'], errors='ignore')

In [92]:
aLbs4.to_excel('C://Users/priscilla.sequetin/Downloads/aLbs4.xlsx', index=False)

In [ ]:
def extrair_numero(texto):
    """
    Extrai apenas o valor numérico de uma string, lidando com vírgulas e pontos.
    Ex: 'Inferior a 0,01' -> 0.01 | '< 20' -> 20.0
    """
    if pd.isna(texto):
        return np.nan
    
    texto_str = str(texto)
    # Busca por uma ou mais casas decimais, acompanhadas ou não de vírgula/ponto e mais números
    match = re.search(r'(\d+(?:[.,]\d+)?)', texto_str)
    
    if match:
        # Pega o valor encontrado, troca vírgula por ponto (para o Python entender) e converte para float
        numero_limpo = match.group(1).replace(',', '.')
        return float(numero_limpo)
    
    return np.nan

# Aplicando a função nas colunas desejadas do seu DataFrame (ex: aLbs1 ou aLbs2)
aLbs4['tsh_encam'] = aLbs4['tsh_encam'].apply(extrair_numero)
aLbs4['bnp_pre'] = aLbs4['bnp_pre'].apply(extrair_numero)

In [ ]:
# # 1. Definir a lista de colunas de dados reais (conforme sua especificação)
# cols_para_checar = [
#     'dt_lab', 'hb_pre', 'leucocitos_pre', 'plaquetas_pre', 
#     'creatinina_pre', 'ureia_pre', 'bnp_pre', 
#     'hba1c_encam', 'tsh_encam', 'sorologias_encam'
# ]



In [ ]:
results_labs = import_redcap(
            aLbs4, 
            api_url=api_url,
            api_key=api_key,
            batch_size=30,
            overwrite=False
        )

Batch 1/16 imported successfully. Records in this batch: 29. Total records imported: 29
Batch 2/16 imported successfully. Records in this batch: 29. Total records imported: 58
Batch 3/16 imported successfully. Records in this batch: 30. Total records imported: 88
Batch 4/16 imported successfully. Records in this batch: 30. Total records imported: 118
Batch 5/16 imported successfully. Records in this batch: 30. Total records imported: 148
Batch 6/16 imported successfully. Records in this batch: 29. Total records imported: 177
Batch 8/16 imported successfully. Records in this batch: 30. Total records imported: 207
Batch 9/16 imported successfully. Records in this batch: 30. Total records imported: 237
Batch 10/16 imported successfully. Records in this batch: 30. Total records imported: 267
Batch 11/16 imported successfully. Records in this batch: 30. Total records imported: 297
Batch 12/16 imported successfully. Records in this batch: 30. Total records imported: 327
Batch 13/16 imported 

# PLANILHA VALVULA

In [ ]:
# 1. CARREGAR OS ARQUIVOS
caminho_excel = r"C:\Users\priscilla.sequetin\Downloads\Fila Cirúrgica FTRISK.csv"
caminho_discrepancia = r"C:\Users\priscilla.sequetin\Downloads\RegValvDANTE-Teste2_DATA_2026-02-11_1811.csv"

valv_ftrisk = pd.read_csv(caminho_excel, sep=';')
# O CSV do REDCap costuma usar vírgula, mas verifique se o seu é ponto e vírgula
df_discrepancia = pd.read_csv(caminho_discrepancia, sep=';') 

# Limpeza de colunas
valv_ftrisk.columns = valv_ftrisk.columns.str.strip()
df_discrepancia.columns = df_discrepancia.columns.str.strip()



In [ ]:
df_discrepancia.columns

In [ ]:
valv_ftrisk.columns

In [ ]:
# 2. PADRONIZAÇÃO PARA O CRUZAMENTO
# Garantimos que record_id seja string em todos para evitar erro de match
valv_ftrisk['record_id'] = valv_ftrisk['record_id'].astype(str)
df_discrepancia['record_id'] = df_discrepancia['record_id'].astype(str)

# --- NOVO PASSO 3: FILTRAR valv_ftrisk COM BASE NAS DISCREPÂNCIAS ---
# Pegamos a lista de record_ids que o REDCap apontou como erro/discrepância
lista_discrepantes = df_discrepancia['record_id'].unique()

# Filtramos a valv_ftrisk: só segue quem está no relatório de discrepância
valv_ftrisk_1 = valv_ftrisk[valv_ftrisk['record_id'].isin(lista_discrepantes)].copy()

print(f"IDs em discrepância: {len(lista_discrepantes)}")
print(f"IDs encontrados no Excel da Anestesia: {len(valv_ftrisk_1)}")



In [ ]:
valv_ftrisk_2 = valv_ftrisk[~valv_ftrisk['record_id'].isin(lista_discrepantes)].copy()

In [ ]:
len(valv_ftrisk_2['record_id'].unique())

In [ ]:
valv_ftrisk_2['record_id'].unique().tolist()

In [ ]:
# Consolidação final preservando o REDCap
chaves_ftr = ['record_id']
valv_ftrisk_1_consolidado = mesclar_preservando_redcap(df_discrepancia, valv_ftrisk_1, chaves_ftr)



In [ ]:
valv_ftrisk_1_consolidado = valv_ftrisk_1_consolidado.dropna(subset=['cirurgia_proposta', 'classe_nyha', 'angina',
       'carga_sintomas_sincope', 'anticoagulacao', 'aceita_transfusao', 'hipertensao_pulmonar_encam'])

In [ ]:
# --- LIMPEZA FINAL PARA UPLOAD (REDCap) ---

# Garantimos que tudo seja tratado como string para permitir as substituições de texto
valv_ftrisk_1_consolidado = valv_ftrisk_1_consolidado.astype(str)

for col in valv_ftrisk_1_consolidado.columns:
    # 1. Substituir valores indesejados por string vazia
    # Nota: Removi o else perdido e limpei a lista de replace
    valv_ftrisk_1_consolidado[col] = valv_ftrisk_1_consolidado[col].replace(['nan', 'None', 'NaN', '<NA>'], '')
    
    # 2. Remover ".0" final (comum em conversões de float para string)
    # O regex r'\.0$' garante que só remove se o .0 estiver no fim da string
    valv_ftrisk_1_consolidado[col] = valv_ftrisk_1_consolidado[col].str.replace(r'\.0$', '', regex=True)
    
    # 3. Trim (remove espaços extras nas pontas)
    valv_ftrisk_1_consolidado[col] = valv_ftrisk_1_consolidado[col].str.strip()

# Opcional: Se após a limpeza sobrar algo que o Pandas ainda veja como nulo, forçamos vazio
valv_ftrisk_1_consolidado = valv_ftrisk_1_consolidado.fillna('')

In [ ]:
valv_ftrisk_1_consolidado.columns

In [ ]:
valv_ftrisk_1_consolidado = valv_ftrisk_1_consolidado.drop(columns='cirurgia_proposta',)

In [ ]:
import_redcap(
            valv_ftrisk_1_consolidado,  # <-- Importante usar o DF LIMPO aqui
            api_url=api_url,
            api_key=api_key,
            batch_size=20,
            overwrite=False
        )


# TESTE ANALISE KF

In [ ]:
# instrumentos = [
# 'procedimento',
#     'qualidade_vida',
#     'fisioterapia',
#     'psicologia',
#     'odontologia',
#     'pre_operatoria',
#     'ficha_complicacoes',
#     'fatores_risco',
#     'ecocardiograma',
#     'coronarias',
#     'anestesia',
#     'pos_operatorio',
#     'alta_hospitalar',
#     'ambulatorio',
#     'ficha_satisfacao',
#     'exames_laboratoriais',
#     'acompanhamento_nutricional',
# 'dados_demograficos'
# ]

# eventos = [
# 'fase1preoperatorio_arm_1',
# 'fase2preprocedimen_arm_1',
# 'fase3procedcirurgi_arm_1',
# 'fase4posprocedimed_arm_1',
# 'fase5recuperacaoin_arm_1',
# 'fase6segalta30dias_arm_1',
# 'encerramento_arm_1'
# ]

In [ ]:
import pandas as pd
import numpy as np

def merge_redcap_dfs_completos(dfs_dict, instrumentos, eventos):
    """
    Merge INNER sequencial respeitando a ORDEM do dict dfs_dict.
    regvalv_pcte entra APENAS no final, somente por record_id.
    """

    # Mapeamento DataFrame -> instrumento
    df_to_instrumento = {
        'regvalv_cirurgia': 'procedimento',
        'regvalv_anest': 'anestesia',
        'regvalv_ftrisk': 'fatores_risco',
        'regvalv_preop': 'pre_operatoria',
        'regvalv_obito': 'ficha_complicacoes',
        'regvalv_fisio': 'fisioterapia',
        'regvalv_posop': 'pos_operatorio',
        'regvalv_altah': 'alta_hospitalar',
        'regvalv_eco': 'ecocardiograma',
        'regvalv_coron': 'coronarias',
        'regvalv_geronto': 'qualidade_vida',
        'regvalv_ambval': 'ambulatorio',
        'regvalv_fcsatis': 'ficha_satisfacao',
        'regvalv_labs': 'exames_laboratoriais',
        'regvalv_psico': 'psicologia',
        'regvalv_odonto': 'odontologia',
        'regvalv_dnutri': 'acompanhamento_nutricional'
    }

    # =========================
    # 1️⃣ Padronizar record_id
    # =========================
    for nome_df, df in dfs_dict.items():
        if 'record_id' in df.columns:
            dfs_dict[nome_df] = df.copy()
            dfs_dict[nome_df]['record_id'] = dfs_dict[nome_df]['record_id'].astype(str)

    # =========================
    # 2️⃣ Separar regvalv_pcte
    # =========================
    df_pcte = dfs_dict['regvalv_pcte'].copy()
    df_pcte['record_id'] = df_pcte['record_id'].astype(str)

    # manter ordem original do dict
    dfs_repetidos = [
        (k, v) for k, v in dfs_dict.items()
        if k != 'regvalv_pcte'
    ]

    # =========================
    # 3️⃣ Preparar DFs limpos
    # =========================
    dfs_preparados = []

    for nome_df, df in dfs_repetidos:
        if nome_df not in df_to_instrumento or df.empty:
            continue

        instrumento = df_to_instrumento[nome_df]

        cols_problematicas = ['redcap_repeat_instrument', 'redcap_event_name']
        df_limpo = df.drop(columns=[c for c in cols_problematicas if c in df.columns])

        df_prep = pd.DataFrame({
            'record_id': df_limpo['record_id'].astype(str),
            'redcap_repeat_instance': df_limpo['redcap_repeat_instance'].astype(str),
            f'{instrumento}_instrumento': instrumento
        })

        for col in df_limpo.columns:
            if col not in ['record_id', 'redcap_repeat_instance']:
                novo_nome = f'{instrumento}_{col}'
                df_prep[novo_nome] = df_limpo[col]

        dfs_preparados.append(df_prep)

    # =========================
    # 4️⃣ Merge sequencial (INNER)
    # =========================
    print("Iniciando merges sequenciais...")

    if not dfs_preparados:
        df_repetidos = pd.DataFrame()
    else:
        df_repetidos = dfs_preparados[0]
        print(f"DF base: {df_repetidos.shape}")

        for i, df_next in enumerate(dfs_preparados[1:], start=2):
            print(f"Merge {i}/{len(dfs_preparados)}: {df_next.shape}")
            df_repetidos = pd.merge(
                df_repetidos,
                df_next,
                on=['record_id', 'redcap_repeat_instance'],
                how='inner'
            )
            print(f"Após merge: {df_repetidos.shape}")

    # =========================
    # 5️⃣ Merge FINAL com regvalv_pcte (somente record_id)
    # =========================
    print("Merge final com regvalv_pcte...")

    if not df_repetidos.empty:
        df_consolidado = pd.merge(
            df_repetidos,
            df_pcte,
            on='record_id',
            how='inner'
        )
    else:
        df_consolidado = df_pcte.copy()

    # =========================
    # 6️⃣ Fase consolidada
    # =========================
    fase_mapping = {
        'pre_operatoria': 'fase1preoperatorio_arm_1',
        'fatores_risco': 'fase1preoperatorio_arm_1',
        'procedimento': 'fase3procedcirurgi_arm_1',
        'ecocardiograma': 'fase1preoperatorio_arm_1',
        'coronarias': 'fase1preoperatorio_arm_1',
        'anestesia': 'fase3procedcirurgi_arm_1',
        'pos_operatorio': 'fase4posprocedimed_arm_1',
        'fisioterapia': 'fase4posprocedimed_arm_1',
        'fisioterapia': 'fase5recuperacaoin_arm_1',
        'psicologia': 'fase1preoperatorio_arm_1',
        'psicologia': 'fase6segalta30dias_arm_1',
        'odontologia': 'fase1preoperatorio_arm_1',
        'alta_hospitalar': 'fase5recuperacaoin_arm_1',
        'ambulatorio': 'fase6segalta30dias_arm_1',
        'ficha_satisfacao': 'encerramento_arm_1',
        'exames_laboratoriais': 'fase1preoperatorio_arm_1',
        'acompanhamento_nutricional': 'fase1preoperatorio_arm_1',
        'qualidade_vida': 'fase1preoperatorio_arm_1',
        'qualidade_vida': 'fase6segalta30dias_arm_1',
    }

    for inst in df_to_instrumento.values():
        col_inst = f'{inst}_instrumento'
        if col_inst in df_consolidado.columns:
            df_consolidado['fase'] = df_consolidado[col_inst].map(fase_mapping)
            break

    df_consolidado['fase'] = df_consolidado['fase'].fillna('sem_fase')

    # =========================
    # 7️⃣ Ordenar colunas
    # =========================
    cols_meta = ['record_id', 'redcap_repeat_instance', 'fase']
    cols_meta = [c for c in cols_meta if c in df_consolidado.columns]

    cols_vars = sorted([c for c in df_consolidado.columns if c not in cols_meta])

    df_final = df_consolidado[cols_meta + cols_vars]

    print(f"✅ SUCESSO! Shape final: {df_final.shape}")
    print(f"Records únicos: {df_final['record_id'].nunique()}")

    return df_final



# USO:
dfs = {
    'regvalv_cirurgia': regvalv_cirurgia,
    'regvalv_anest': regvalv_anest,
    'regvalv_ftrisk': regvalv_ftrisk,
    'regvalv_preop': regvalv_preop,
    'regvalv_obito': regvalv_obito,
    'regvalv_fisio': regvalv_fisio,
    'regvalv_posop': regvalv_posop,
    'regvalv_altah': regvalv_altah,
    'regvalv_eco': regvalv_eco,
    'regvalv_coron': regvalv_coron,
    'regvalv_geronto': regvalv_geronto,
    'regvalv_ambval': regvalv_ambval,
    'regvalv_fcsatis': regvalv_fcsatis,
    'regvalv_labs': regvalv_labs,
    'regvalv_psico': regvalv_psico,
    'regvalv_odonto': regvalv_odonto,
    'regvalv_dnutri': regvalv_dnutri
}


instrumentos = [
    'procedimento', 'qualidade_vida', 'fisioterapia', 'psicologia', 'odontologia',
    'pre_operatoria', 'ficha_complicacoes', 'fatores_risco', 'ecocardiograma',
    'coronarias', 'anestesia', 'pos_operatorio', 'alta_hospitalar', 'ambulatorio',
    'ficha_satisfacao', 'exames_laboratoriais', 'acompanhamento_nutricional',
    'dados_demograficos'
]

eventos = [
    'fase1preoperatorio_arm_1', 'fase2preprocedimen_arm_1', 'fase3procedcirurgi_arm_1',
    'fase4posprocedimed_arm_1', 'fase5recuperacaoin_arm_1', 'fase6segalta30dias_arm_1',
    'encerramento_arm_1'
]


# Executar
df_consolidado = merge_redcap_dfs_completos(dfs, instrumentos, eventos)

print(f"Shape final: {df_consolidado.shape}")
# print("\nColunas (primeiras 15):")
# print(df_consolidado.columns[:15].tolist())
# print("\nPrimeiras linhas:")
# print(df_consolidado[['record_id', 'redcap_repeat_instance', 'repeat_instrument', 'fase']].head())



In [ ]:
import pandas as pd
import re

def processar_checkboxes(df):
    """
    Processa todos os grupos de checkbox do codebook REDCap automaticamente.
    Transforma colunas wide em formato long para análise granular.
    """
    
    # 1. IDENTIFICAR TODOS OS GRUPOS DE CHECKBOX no DataFrame
    checkbox_patterns = [
        r'tipo_intervencao_previa___',
        r'medicacoes_atuais___',
        r'diagnostico_principal___',
        r'tipo_de_cirurgia_anterior___',
        r'tipo_arritmia___',
        r'valvula_afetada___',
        r'valvula_nativa___',
        r'protese_biologica___',
        r'protese_mecanica___',
        r'lesao_valvar_disc___',
        r'lesao_valvar_normo___',
        r'estenose_moderada___',
        r'estenose_importante___',
        r'disfuncao_moderada___',
        r'disfuncao_importante___',
        r'insuficiencia_moderada___',
        r'insuficiencia_importante___',
        r'tipo_cirurgia___',
        r'cirurgica_associada_encam___',
        r'tipo_complicacao_intraop___',
        r'tipos_de_drogas___',
        r'tipo_evento_tromboembolico___',
        r'tipo_arritmia_pos_op___',
        r'detalhes_complicacao_vascular___',
        r'tipo_avc___',
        r'tipo_nova_arritmia___',
        r'tp_new_arrtm_evento_30d___',
        r'limitacoes_persistentes___',
        r'tipo_disfuncao_estrutural___',
        r'tipo_disfuncao_nao_estrutural___',
        r'avaliacao_imagem_valvar___',
        r'tratamento_trombose_valvar___',
        r'biomarcadores_ic___',
        r'componentes_eficacia___',
        r'componentes_seguranca___',
        r'motivo_reintubacao___',
        r'tipo_intercorrencias___'
    ]
    
    # 2. MAPEAMENTO AUTOMÁTICO baseando-se no codebook
    mapeamentos = {
        # 71 - Tipo de Intervenção Prévia
        'tipo_intervencao_previa___1': 'Angioplastia',
        'tipo_intervencao_previa___2': 'Cirurgia de revascularização',
        'tipo_intervencao_previa___6': 'Cirurgia de aorta',
        'tipo_intervencao_previa___3': 'Cirurgia valvar',
        'tipo_intervencao_previa___4': 'Marcapasso',
        'tipo_intervencao_previa___5': 'Outra',
        
        # 77 - Medicações Atuais
        'medicacoes_atuais___1': 'Betabloqueadores',
        'medicacoes_atuais___2': 'IECA/BRA',
        'medicacoes_atuais___3': 'Antagonistas do cálcio',
        'medicacoes_atuais___4': 'Diuréticos',
        'medicacoes_atuais___5': 'Estatinas',
        'medicacoes_atuais___6': 'Anticoagulantes',
        'medicacoes_atuais___7': 'Antiplaquetários',
        'medicacoes_atuais___8': 'Antiarrítmicos',
        'medicacoes_atuais___9': 'Digital',
        'medicacoes_atuais___10': 'Outros',
        
        # 82 - Diagnóstico Principal
        'diagnostico_principal___1': 'EAO',
        'diagnostico_principal___2': 'IAO',
        'diagnostico_principal___3': 'EM',
        'diagnostico_principal___4': 'IM',
        'diagnostico_principal___5': 'Doença Tricúspide',
        'diagnostico_principal___6': 'Aneurisma de Aorta',
        'diagnostico_principal___7': 'Disfunção de Prótese Valvar',
        'diagnostico_principal___8': 'Endocardite',
        'diagnostico_principal___9': 'Trombose',
        
        # 86 - Tipo de Cirurgia Anterior
        'tipo_de_cirurgia_anterior___1': 'Revascularização Miocárdica',
        'tipo_de_cirurgia_anterior___2': 'Cirurgia de Aorta',
        'tipo_de_cirurgia_anterior___3': 'Cirurgia Valvar',
        'tipo_de_cirurgia_anterior___4': 'Outra',
        
        # 93 - Tipo de Arritmia
        'tipo_arritmia___7': 'Sinusal',
        'tipo_arritmia___1': 'Fibrilação atrial',
        'tipo_arritmia___2': 'Flutter atrial',
        'tipo_arritmia___3': 'Taquicardia ventricular',
        'tipo_arritmia___4': 'Fibrilação ventricular',
        'tipo_arritmia___6': 'Bloqueio Atrioventricular (BAV) Alto Grau',
        'tipo_arritmia___5': 'Outra',
        
        # 97-108 - Válvulas (padrão similar)
        'valvula_afetada___1': 'Aórtica',
        'valvula_afetada___2': 'Mitral',
        'valvula_afetada___3': 'Tricúspide',
        'valvula_nativa___1': 'Mitral',
        'valvula_nativa___2': 'Aórtica',
        'valvula_nativa___3': 'Tricúspide',
        'protese_biologica___1': 'Mitral',
        'protese_biologica___2': 'Aórtica',
        'protese_biologica___3': 'Tricúspide',
        'protese_mecanica___1': 'Mitral',
        'protese_mecanica___2': 'Aórtica',
        'protese_mecanica___3': 'Tricúspide',
        'lesao_valvar_disc___1': 'Mitral - Sem lesão e/ou Lesão Discreta',
        'lesao_valvar_disc___2': 'Aórtica - Sem lesão e/ou Lesão Discreta',
        'lesao_valvar_disc___3': 'Tricúspide - Sem lesão e/ou Lesão Discreta',
        # ... (continua para todos os grupos similares)
    }
    
    # Lista para armazenar todos os DataFrames processados
    dfs_processados = []
    
    # 3. PROCESSAR CADA GRUPO DE CHECKBOX
    for pattern in checkbox_patterns:
        cols_grupo = [col for col in df.columns if re.match(pattern, col)]
        
        if len(cols_grupo) > 0:
            # Identificar colunas ID (não checkboxes)
            id_vars = [col for col in df.columns if not any(re.match(p, col) for p in checkbox_patterns)]
            
            # Unpivot deste grupo
            df_grupo = pd.melt(
                df,
                id_vars=id_vars,
                value_vars=cols_grupo,
                var_name='campo_original',
                value_name='selecionado'
            )
            
            # Filtrar apenas selecionados (1 ou não nulo)
            df_grupo_filtrado = df_grupo[
                df_grupo['selecionado'].notna() & 
                (df_grupo['selecionado'] == 1)
            ].copy()
            
            # Extrair nome do grupo da coluna
            grupo_nome = re.match(r'(.+)___', pattern).group(1)
            df_grupo_filtrado['grupo_checkbox'] = grupo_nome
            
            # Mapear para descrição
            df_grupo_filtrado['descricao'] = df_grupo_filtrado['campo_original'].map(mapeamentos)
            
            # Colunas finais
            cols_finais = id_vars + ['grupo_checkbox', 'descricao', 'selecionado']
            df_final_grupo = df_grupo_filtrado[cols_finais].copy()
            
            dfs_processados.append(df_final_grupo)
    
    # 4. CONCATENAR TODOS OS GRUPOS
    df_final = pd.concat(dfs_processados, ignore_index=True)
    
    return df_final

# USO:
# df_long = processar_checkboxes(df)
# print(df_long.head(10))
# print(df_long['grupo_checkbox'].value_counts())


# XML - Lendo backup segurança REDCap

In [ ]:
# import xml.etree.ElementTree as ET

# # --- CAMINHOS DOS ARQUIVOS ---
# arq_xml_ontem = r"C:/Users/priscilla.sequetin/Downloads/RegValvDANTE_2026-01-19_1159.REDCap.xml"
# arq_xml_hoje  = r"C:/Users/priscilla.sequetin/Downloads/RegValvDANTE_2026-01-20_1413.REDCap.xml"

# def extrair_variaveis(caminho_arquivo):
#     """Lê um XML do REDCap e retorna um conjunto (set) com todas as variáveis (ItemOID)"""
#     try:
#         ns = {'odm': 'http://www.cdisc.org/ns/odm/v1.3'}
#         tree = ET.parse(caminho_arquivo)
#         root = tree.getroot()
        
#         variaveis = set()
#         # Busca todas as tags ItemData e pega o atributo ItemOID (nome da variável)
#         for item in root.findall('.//odm:ItemData', ns):
#             variaveis.add(item.get('ItemOID'))
#         return variaveis
#     except Exception as e:
#         print(f"Erro ao ler {caminho_arquivo}: {e}")
#         return set()

# print("--- Iniciando Comparação ---")

# # 1. Extrair variáveis de cada arquivo
# vars_ontem = extrair_variaveis(arq_xml_ontem)
# print(f"Arquivo de Ontem (19/01): {len(vars_ontem)} variáveis encontradas.")

# vars_hoje = extrair_variaveis(arq_xml_hoje)
# print(f"Arquivo de Hoje  (20/01): {len(vars_hoje)} variáveis encontradas.")

# # 2. Calcular as diferenças (Matemática de Conjuntos)
# # O que tem em HOJE que não tinha ONTEM (Novas ou Renomeadas)
# novas_criadas = vars_hoje - vars_ontem

# # O que tinha ONTEM e não tem HOJE (Removidas ou Renomeadas)
# antigas_removidas = vars_ontem - vars_hoje

# # 3. Gerar Relatório
# print("\n" + "="*60)
# print(f"RELATÓRIO DE ALTERAÇÕES ({len(novas_criadas)} adições, {len(antigas_removidas)} remoções)")
# print("="*60)

# print(f"\n>>> VARIÁVEIS NOVAS (Apareceram no arquivo de hoje):")
# if novas_criadas:
#     for v in sorted(novas_criadas):
#         print(f" [+] {v}")
# else:
#     print(" - Nenhuma variável nova encontrada.")

# print(f"\n>>> VARIÁVEIS REMOVIDAS (Sumiram do arquivo de hoje):")
# if antigas_removidas:
#     for v in sorted(antigas_removidas):
#         print(f" [-] {v}")
# else:
#     print(" - Nenhuma variável foi removida.")

# print("\n" + "-"*60)
# print("DICA DE INTERPRETAÇÃO:")
# print("1. Se você RENOMEOU uma variável (ex: 'obs' vira 'notas'), ela vai aparecer")
# print("   como REMOVIDA ('obs') e como NOVA ('notas').")
# print("2. Se você só MOVEU de instrumento mas manteve o nome, ela NÃO aparece aqui")
# print("   (pois a variável continua existindo no banco, apenas mudou de lugar).")


In [ ]:
# import pandas as pd

# df = pd.read_xml(
#     r'C:/Users/priscilla.sequetin/Downloads/RegValvDANTE_2026-01-19_1159.REDCap.xml',
#     parser='etree'  # <--- Adicione isto
# )
# print(df.head())


In [ ]:
# import pandas as pd
# import xml.etree.ElementTree as ET

# # --- CAMINHO DO ARQUIVO ANTIGO ---
# caminho_xml = r"C:/Users/priscilla.sequetin/Downloads/RegValvDANTE_2026-01-19_1159.REDCap.xml"

# print("Lendo XML com estrutura completa (Evento e Instância)...")

# tree = ET.parse(caminho_xml)
# root = tree.getroot()
# ns = {'odm': 'http://www.cdisc.org/ns/odm/v1.3'}

# dados_lista = []

# # 1. Nível Paciente
# for subject in root.findall('.//odm:SubjectData', ns):
#     record_id = subject.get('SubjectKey')
    
#     # 2. Nível Evento (redcap_event_name)
#     for event in subject.findall('./odm:StudyEventData', ns):
#         event_name = event.get('StudyEventOID') # Ex: baseline_arm_1, visita_2_arm_1
        
#         # 3. Nível Formulário
#         for form in event.findall('./odm:FormData', ns):
#             instrumento = form.get('FormOID')
            
#             # 4. Nível Grupo de Itens (redcap_repeat_instance fica aqui!)
#             for group in form.findall('./odm:ItemGroupData', ns):
#                 # O REDCap salva a instância no atributo 'ItemGroupRepeatKey'.
#                 # Se for vazio, geralmente assume-se '1' ou NaN.
#                 instancia = group.get('ItemGroupRepeatKey')
                
#                 # 5. Nível Dados (Variáveis)
#                 for item in group.findall('./odm:ItemData', ns):
#                     dados_lista.append({
#                         'record_id': record_id,
#                         'redcap_event_name': event_name,
#                         'redcap_repeat_instrument': instrumento,
#                         'redcap_repeat_instance': instancia,
#                         'variavel': item.get('ItemOID'),
#                         'valor': item.get('Value')
#                     })
# # 3. Criar o DataFrame final
# df = pd.DataFrame(dados_lista)

# print(f"Sucesso! DataFrame carregado com {len(df)} registros.")
# print(df.head())

In [ ]:
#df['redcap_repeat_instrument'].unique()

In [ ]:
# def transformar_redcap_para_tabela(df_input):
#     """
#     Recebe o DataFrame no formato Longo (uma linha por variável)
#     e transforma para o formato Largo (uma coluna por variável).
#     """
#     # 1. Fazemos uma cópia para não alterar o original sem querer
#     df = df_input.copy()

#     # --- TRANSFORMAÇÃO 1: Limpeza de Texto ---
#     # Remove 'Event.' da coluna de evento
#     if 'redcap_event_name' in df.columns:
#         df['redcap_event_name'] = df['redcap_event_name'].str.replace('Event.', '', regex=False)

#     # Remove 'Form.' da coluna de instrumento
#     if 'redcap_repeat_instrument' in df.columns:
#         df['redcap_repeat_instrument'] = df['redcap_repeat_instrument'].str.replace('Form.', '', regex=False)

#     # --- TRANSFORMAÇÃO 2: Pivotagem (Long to Wide) ---
#     # Definimos o que identifica uma linha única: ID + Evento + Form + Instância
#     index_cols = ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance']
    
#     # pivot_table é mais seguro que pivot pois lida com eventuais duplicatas (usando 'first')
#     df_pivot = df.pivot_table(
#         index=index_cols,
#         columns='variavel', 
#         values='valor',
#         aggfunc='first'  # Pega o primeiro valor caso haja duplicação (não deve haver no XML)
#     )

#     # Resetar o index para que record_id, event, etc. voltem a ser colunas normais
#     df_pivot = df_pivot.reset_index()

#     # Remove o nome da hierarquia de colunas (estética)
#     df_pivot.columns.name = None

#     return df_pivot

# # # ==========================================================
# # # APLICANDO NO SEU DATAFRAME ATUAL (df_completo)
# # # ==========================================================

# # # Supondo que seu dataframe anterior se chama 'df_completo'
# # df_final = transformar_redcap_para_tabela(df_completo)

# # print(f"Dimensões da nova tabela: {df_final.shape}")
# # print(df_final.head())

# # # Dica: O XML traz tudo como texto. Se quiser converter números automaticamente:
# # df_final = df_final.convert_dtypes()

In [ ]:
# # 1. Filtra apenas o instrumento desejado (usando o nome que vem no XML, ex: Form.pre_operatoria)
# preop = df[df['redcap_repeat_instrument'] == 'Form.pre_operatoria']

# # 2. Aplica a função mágica
# preop1 = transformar_redcap_para_tabela(preop)

# # 3. Resultado: Uma tabela limpa só com variáveis pré-operatórias nas colunas
# #print("Tabela Pré-Operatória Pronta:")
# #print(preop1.head())

In [ ]:
# # 1. Filtra apenas o instrumento desejado (usando o nome que vem no XML, ex: Form.pre_operatoria)
# eco = df[df['redcap_repeat_instrument'] == 'Form.ecocardiograma']

# # 2. Aplica a função mágica
# eco1 = transformar_redcap_para_tabela(eco)

# # 3. Resultado: Uma tabela limpa só com variáveis pré-operatórias nas colunas
# #print("Tabela Pré-Operatória Pronta:")
# #print(eco1.head())

In [ ]:
#dfcsv = pd.read_csv('C:\\Users\\priscilla.sequetin\\Downloads\\RegValvDANTE_DATA_2026-01-19_1159.csv', sep="|")


In [ ]:
# # --- 1. CONFIGURAÇÕES ---
# caminho_xml = arq_xml_hoje

# dict_fases_manual = {
#     'acompanhamento_nutricional': 1, 'alta_hospitalar': 1, 'ambulatorio': 1,
#     'anestesia': 1, 'dados_demograficos': 1, 'ecocardiograma': 1,
#     'evento_intrahospitalar': 1, 'exames_laboratoriais': 1, 'fatores_risco': 1,
#     'ficha_complicacoes': 1, 'ficha_satisfacao': 1, 'fisioterapia': 2,
#     'odontologia': 1, 'pos_operatorio': 1, 'pre_operatoria': 1,
#     'procedimento': 1, 'psicologia': 2, 'qfca': 1, 'qfca_internacao': 2,
#     'qualidade_vida': 2,
# }

# print("--- Lendo XML... ---")
# tree = ET.parse(caminho_xml)
# root = tree.getroot()
# ns = {'odm': 'http://www.cdisc.org/ns/odm/v1.3'}

# # ==============================================================================
# # ETAPA 1: CALCULAR AS BASES (PACIENTES vs PROCEDIMENTOS)
# # ==============================================================================
# print("--- Calculando Universos (Pacientes vs Procedimentos)... ---")

# set_pacientes = set()       # record_id únicos
# set_cd_atendimento = set()  # cd_atendimento únicos
# contagem_variaveis = {}     # contagem simples de preenchimento (ItemOID)

# # ItemOID da variável cd_atendimento no ODM
# oid_cd_atendimento = 'cd_atendimento'  # ajuste se o OID no XML for outro

# for subject in root.findall('.//odm:SubjectData', ns):
#     record_id = subject.get('SubjectKey')
#     set_pacientes.add(record_id)

#     # varrer todos os ItemData do sujeito
#     for item in subject.findall('.//odm:ItemData', ns):
#         var_oid = item.get('ItemOID')

#         # 1) contagem global para cada ItemOID
#         contagem_variaveis[var_oid] = contagem_variaveis.get(var_oid, 0) + 1

#         # 2) capturar cd_atendimento únicos
#         if var_oid == oid_cd_atendimento:
#             valor_atendimento = item.get('Value')
#             if valor_atendimento is not None:
#                 # chave composta record_id + cd_atendimento para garantir unicidade por paciente
#                 set_cd_atendimento.add((record_id, valor_atendimento))

# # --- DEFINIÇÃO FINAL DAS BASES ---
# qtde_pacientes_reais = len(set_pacientes)
# qtde_procedimentos_reais = len(set_cd_atendimento)

# print(f"BASE A (Pacientes Únicos): {qtde_pacientes_reais} -> Usado p/ Demográficos")
# print(f"BASE B (cd_atendimento Únicos): {qtde_procedimentos_reais} -> Usado p/ Outros")

# # Fallback se não encontrar nenhum cd_atendimento
# if qtde_procedimentos_reais == 0:
#     print("AVISO: Nenhum cd_atendimento único encontrado. Usando total de pacientes como base B.")
#     qtde_procedimentos_reais = qtde_pacientes_reais

# # ==============================================================================
# # ETAPA 2: CONSTRUIR METADATA DO XML
# # ==============================================================================
# dicionario_xml = []
# mapa_labels = {}

# # Mapear Labels
# for item_def in root.findall('.//odm:ItemDef', ns):
#     try:
#         label = item_def.find('.//odm:TranslatedText', ns).text
#     except:
#         label = item_def.get('Name')
#     mapa_labels[item_def.get('OID')] = str(label)[:60]

# # Mapear Forms e Variáveis
# for group in root.findall('.//odm:ItemGroupDef', ns):
#     form_name = group.get('OID')
#     for item_ref in group.findall('./odm:ItemRef', ns):
#         var_oid = item_ref.get('ItemOID')
#         dicionario_xml.append({
#             'Instrumento': form_name,
#             'Variavel_ID': var_oid,
#             'Pergunta': mapa_labels.get(var_oid, var_oid)
#         })

# df_dicionario = pd.DataFrame(dicionario_xml)

# # ==============================================================================
# # ETAPA 3: GERAR RELATÓRIO (Modelo Pandas adaptado para XML/ODM)
# # ==============================================================================
# print("\n--- Gerando Relatório... ---")
# relatorio = []

# # 1) Precisa mapear ItemGroupDef -> form_name equivalente ao 'redcap_repeat_instrument'
# # Mapeamento manual baseado na estrutura ODM do seu XML (ajuste conforme necessário)
# mapa_form_para_repeticao = {
#     'fisioterapia': True,
#     'psicologia': True, 
#     'qfca_internacao': True,
#     'qualidade_vida': True,
#     'acompanhamento_nutricional': True, 
#     'alta_hospitalar': True, 
#     'ambulatorio': True,
#     'anestesia': True, 
#     'ecocardiograma': True,
#     'evento_intrahospitalar': True, 
#     'exames_laboratoriais': True, 
#     'fatores_risco': True,
#     'ficha_complicacoes': True, 
#     'ficha_satisfacao': True, 
#     'odontologia': True, 
#     'pos_operatorio': True, 
#     'pre_operatoria': True,
#     'procedimento': True, 
# }

# # 2) Contadores por form_name (equivalente ao slice do pandas)
# contadores_por_form = {}
# for form_name in df_dicionario['Instrumento'].unique():
#     contadores_por_form[form_name] = {}

# # Preencher contadores por form_name
# for var_oid, total in contagem_variaveis.items():
#     # Encontra qual form_name contém esta variável
#     form_contendo_var = df_dicionario[df_dicionario['Variavel_ID'] == var_oid]['Instrumento'].iloc[0] if not df_dicionario[df_dicionario['Variavel_ID'] == var_oid].empty else None
    
#     if form_contendo_var:
#         if form_contendo_var not in contadores_por_form:
#             contadores_por_form[form_contendo_var] = {}
#         contadores_por_form[form_contendo_var][var_oid] = total

# for index, row in df_dicionario.iterrows():
#     var_name = row['Variavel_ID']
#     form_name = row['Instrumento']
#     label_base = str(row['Pergunta'])[:60]

#     # === APLICANDO LÓGICA DO DICT MANUAL ===
    
#     # 1. Pega o Multiplicador do seu DICT
#     mult_eventos = dict_fases_manual.get(form_name, 1)
    
#     # 2. Define a Base (Paciente ou Procedimento?) 
#     base_calculo = 0
    
#     if form_name == 'dados_demograficos':
#         base_calculo = qtde_pacientes_reais
        
#         # Para demográficos: usa TODOS os dados (não filtra repetição)
#         df_slice_equivalente = 'todos'  # flag especial
        
#     else:
#         base_calculo = qtde_procedimentos_reais
        
#         # Verifica se é repetitivo no XML
#         e_repetitivo = mapa_form_para_repeticao.get(form_name, False)
        
#         if e_repetitivo:
#             # Se repetitivo: usa só contagens DESSE form_name específico
#             df_slice_equivalente = form_name
#         else:
#             # Se não repetitivo: usa contagens de TODOS os forms (base)
#             df_slice_equivalente = 'base'

#     # 3. CALCULA O TOTAL ESPERADO FINAL
#     total_esperado_final = base_calculo * mult_eventos

#     # === CONTABILIZAÇÃO DO NUMERADOR ===
#     lista_vars = []
    
#     # Suporte a checkbox (mesma lógica var___1, var___2...)
#     if any(k.startswith(f"{var_name}___") for k in contagem_variaveis.keys()):
#         # É checkbox: soma todas as opções
#         cols_checkbox = [k for k in contagem_variaveis.keys() if k.startswith(f"{var_name}___")]
#         preenchidos = sum(contagem_variaveis[c] for c in cols_checkbox)
#         label_final = f"{label_base} (Checkbox)"
        
#         # ADICIONA UMA LINHA ÚNICA para todo checkbox
#         relatorio.append({
#             'Instrumento': form_name,
#             'Variavel_ID': var_name,
#             'Pergunta': label_final,
#             'Fases_Dict': mult_eventos,
#             'Total_Esperado': total_esperado_final,
#             'Qtd_Preenchidos': preenchidos,
#             'Qtd_Vazios': total_esperado_final - preenchidos,
#             'Preenchimento (%)': round((preenchidos / total_esperado_final * 100) if total_esperado_final > 0 else 0, 2)
#         })
#         continue  # pula o resto, checkbox já processado como linha única
    
#     # Caso normal (não checkbox)
#     else:
#         lista_vars.append({'var': var_name, 'lbl': label_base})

#     # Processa cada item da lista (normalmente só 1)
#     for item in lista_vars:
#         col_real = item['var']
#         lbl_final = item['lbl']
        
#         preenchidos = 0
        
#         # === LÓGICA DO SLICE (equivalente ao pandas) ===
#         if df_slice_equivalente == 'todos':
#             # Demográficos: conta tudo
#             preenchidos = contagem_variaveis.get(col_real, 0)
#         elif df_slice_equivalente == form_name:
#             # Form específico (repetitivo): conta só contagens DESTE form
#             form_contador = contadores_por_form.get(form_name, {})
#             preenchidos = form_contador.get(col_real, 0)
#         elif df_slice_equivalente == 'base':
#             # Base (não repetitivo): conta tudo (como no pandas)
#             preenchidos = contagem_variaveis.get(col_real, 0)
        
#         nulos = total_esperado_final - preenchidos
#         if nulos < 0: nulos = 0
#         pct = (preenchidos / total_esperado_final * 100) if total_esperado_final > 0 else 0

#         relatorio.append({
#             'Instrumento': form_name,
#             'Variavel_ID': col_real,
#             'Pergunta': lbl_final,
#             'Fases_Dict': mult_eventos,  # Para conferência
#             'Total_Esperado': total_esperado_final,
#             'Qtd_Preenchidos': preenchidos,
#             'Qtd_Vazios': nulos,
#             'Preenchimento (%)': round(pct, 2)
#         })

# # --- EXIBIÇÃO ---
# df_resultado = pd.DataFrame(relatorio)



# # Ordenação (Opcional, para ficar organizado)
# #df_resultado = df_resultado.sort_values(by=['Instrumento', 'Preenchimento (%)'], ascending=[True, False])

# #print(df_resultado.head(10))

# # Se quiser salvar
# #df_resultado.to_excel(f"C://Users/priscilla.sequetin/Downloads/dicionario_regvalv5.xlsx", index=False)

In [ ]:
# from collections import defaultdict

# ns = {
#     'odm': 'http://www.cdisc.org/ns/odm/v1.3'
# }

# OID_CD_ATENDIMENTO = 'cd_atendimento'
# INSTRUMENTO_DEMOGRAFICO = 'dados_demograficos'

# tree = ET.parse(caminho_xml)
# root = tree.getroot()

# set_pacientes = set()
# set_procedimentos = set()  # (record_id, cd_atendimento)


# for subject in root.findall('.//odm:SubjectData', ns):
#     record_id = subject.get('SubjectKey')
#     set_pacientes.add(record_id)

#     atendimentos_sujeito = set()

#     for item in subject.findall('.//odm:ItemData', ns):
#         if item.get('ItemOID') == OID_CD_ATENDIMENTO:
#             valor = item.get('Value')
#             if valor:
#                 atendimentos_sujeito.add(valor)

#     for at in atendimentos_sujeito:
#         set_procedimentos.add((record_id, at))

# qtde_pacientes = len(set_pacientes)
# qtde_procedimentos = len(set_procedimentos)

# if qtde_procedimentos == 0:
#     qtde_procedimentos = qtde_pacientes

# # Instrumento -> paciente -> set(instancias)
# instancias_por_instrumento = defaultdict(lambda: defaultdict(set))

# for subject in root.findall('.//odm:SubjectData', ns):
#     record_id = subject.get('SubjectKey')

#     for ig in subject.findall('.//odm:ItemGroupData', ns):
#         instrumento = ig.get('ItemGroupOID')
#         repeat_key = ig.get('ItemGroupRepeatKey', '1')

#         instancias_por_instrumento[instrumento][record_id].add(repeat_key)

# # Quantidade de fases por instrumento (máximo observado)
# dict_fases = {}

# for instrumento, sujeitos in instancias_por_instrumento.items():
#     dict_fases[instrumento] = max(
#         (len(insts) for insts in sujeitos.values()),
#         default=1
#     )

# # Preenchimento por procedimento (contagem bruta)
# contagem_por_procedimento = defaultdict(int)

# # Preenchimento por paciente (1 por paciente)
# contagem_por_paciente = defaultdict(set)

# for subject in root.findall('.//odm:SubjectData', ns):
#     record_id = subject.get('SubjectKey')

#     for item in subject.findall('.//odm:ItemData', ns):
#         var_oid = item.get('ItemOID')

#         contagem_por_procedimento[var_oid] += 1
#         contagem_por_paciente[var_oid].add(record_id)

# mapa_labels = {}

# for item_def in root.findall('.//odm:ItemDef', ns):
#     try:
#         label = item_def.find('.//odm:TranslatedText', ns).text
#     except:
#         label = item_def.get('Name')

#     mapa_labels[item_def.get('OID')] = str(label)[:80]

# linhas = []

# for group in root.findall('.//odm:ItemGroupDef', ns):
#     instrumento = group.get('OID')

#     for ref in group.findall('./odm:ItemRef', ns):
#         var_oid = ref.get('ItemOID')

#         linhas.append({
#             'Instrumento': instrumento,
#             'Variavel_ID': var_oid,
#             'Pergunta': mapa_labels.get(var_oid, var_oid)
#         })

# relatorio = []

# for _, row in df_dicionario.iterrows():
#     instrumento = row['Instrumento']
#     var = row['Variavel_ID']
#     pergunta = row['Pergunta'][:80]

#     instancias = dict_fases.get(instrumento, 1)

#     # BASE
#     if instrumento == INSTRUMENTO_DEMOGRAFICO:
#         base = qtde_pacientes
#         preenchidos = len(contagem_por_paciente.get(var, set()))
#     else:
#         base = qtde_procedimentos
#         preenchidos = contagem_por_procedimento.get(var, 0)

#     total_esperado = base * instancias
#     vazios = max(total_esperado - preenchidos, 0)
#     pct = round((preenchidos / total_esperado * 100), 2) if total_esperado else 0

#     relatorio.append({
#         'Instrumento': instrumento,
#         'Variavel_ID': var,
#         'Pergunta': pergunta,
#         'Instancias': instancias,
#         'Base_Calculo': base,
#         'Total_Esperado': total_esperado,
#         'Qtd_Preenchidos': preenchidos,
#         'Qtd_Vazios': vazios,
#         'Preenchimento (%)': pct
#     })

# df_resultado = pd.DataFrame(relatorio)



In [ ]:
#df_resultado.to_excel(f"C://Users/priscilla.sequetin/Downloads/dicionario_regvalv7.xlsx", index=False)

In [ ]:
# def base_por_instrumento(form_name):
#     if form_name == 'dados_demograficos':
#         return qtde_pacientes_reais
#     return qtde_procedimentos_reais


In [ ]:
# print("\n--- Gerando Relatório (corrigido) ---")
# relatorio = []

# for _, row in df_dicionario.iterrows():
#     form = row['Instrumento']
#     var = row['Variavel_ID']
#     label = row['Pergunta'][:60]

#     instancias = dict_fases_auto.get(form, 1)

#     if form == 'dados_demograficos':
#         base = qtde_pacientes_reais
#         preenchidos = len(contagem_por_paciente.get(var, set()))
#     else:
#         base = qtde_procedimentos_reais
#         preenchidos = contagem_por_procedimento.get(var, 0)

#     total_esperado = base * instancias
#     vazios = max(total_esperado - preenchidos, 0)
#     pct = round((preenchidos / total_esperado * 100), 2) if total_esperado else 0

#     relatorio.append({
#         'Instrumento': form,
#         'Variavel_ID': var,
#         'Pergunta': label,
#         'Instancias': instancias,
#         'Base_Calculo': base,
#         'Total_Esperado': total_esperado,
#         'Qtd_Preenchidos': preenchidos,
#         'Qtd_Vazios': vazios,
#         'Preenchimento (%)': pct
#     })


# df_resultado = pd.DataFrame(relatorio)


In [ ]:
#df_resultado.to_excel(f"C://Users/priscilla.sequetin/Downloads/dicionario_regvalv6.xlsx", index=False)

In [ ]:
# # Buscando hap por psap >= 50

# def find_psap_number(text):
#     match = re.search(r'PSAP.*?(\d+)', text)
#     if match:
#         return match.group(1)
#     return None

# df_PSAP = []

# query_PSAP =f"""
#     SELECT 
#         CD_PACIENTE,
#         CD_ATENDIMENTO,
#         DH_DOCUMENTO,
#         DS_CAMPO,
#         DS_RESPOSTA
#     FROM
#         ft_doc_eletronico_texto fdet 
#     WHERE
#         CD_DOCUMENTO in (1066, 1061, 1070, 1068, 910, 911, 939, 950, 951, 957, 959)
#         AND YEAR(DH_DOCUMENTO) >= 2025
# """ 
# df_PSAP = query_sql_to_dataframe(conn_string, query_PSAP)

# df_psap_hp = df_PSAP.copy()

# df_psap_hp = df_psap_hp[df_psap_hp['CD_ATENDIMENTO'].isin(df_com['CD_ATENDIMENTO'])]

# df_psap_hp.reset_index(inplace=True, drop=True)

# index_not_hap = []
# index_hap = []
# df_psap_hp = df_psap_hp[df_psap_hp['DS_CAMPO'] == "DESCRIÇÃO EXAMES COMPLEMENTARES"]
# df_psap_hp.reset_index(drop=True, inplace=True)
# for i in range(len(df_psap_hp)):
#     text = df_psap_hp['DS_RESPOSTA'][i]
#     if text:
#         psap = find_psap_number(text)
#         if psap:
#             psap = int(psap)
#             if psap >= 50:  #COLOCAR VALOR MÍMINO PARA HIPERTENSÃO ARTERIAL PULMONAR
#                 index_hap.append(i)
#             else: index_not_hap.append(i)
                    
# df_psap_hp['HAP'] = None
# df_psap_hp.loc[index_hap, 'HAP'] = 1 
# df_psap_hp.loc[index_not_hap, 'HAP'] = 0 

# df_psap_hp.dropna(subset=['HAP'], inplace=True, ignore_index=True)

In [ ]:
# df_hap = df_psap_hp.copy()

# # Trabalhando com o ano corrente
# df_hap = df_hap[df_hap['DH_DOCUMENTO'].dt.year >= 2025]

In [ ]:
# # Buscando obesidade

# query_OBES =f"""
#     SELECT 
#         CD_PACIENTE,
#         CD_ATENDIMENTO,
#         DH_DOCUMENTO,
#         DS_CAMPO,
#         DS_RESPOSTA
#     FROM
#         ft_doc_eletronico_texto fdet 
#     WHERE
#         CD_DOCUMENTO in (1066, 1061, 1070, 1068, 910, 911, 939, 950, 951, 957, 959)
#         AND YEAR(DH_DOCUMENTO) >= 2025
#         AND((DS_RESPOSTA LIKE '%obesidade%' COLLATE LATIN1_GENERAL_CI_AI) OR (DS_RESPOSTA LIKE '%obeso%' COLLATE LATIN1_GENERAL_CI_AI))
# """ 
# df_OBES = query_sql_to_dataframe(conn_string, query_OBES)



In [ ]:
# df_obs = df_OBES.copy()

# # Trabalhando com o ano corrente
# df_obs = df_obs[df_obs['DH_DOCUMENTO'].dt.year >= 2025]

In [ ]:
# df_com.columns

In [ ]:
# # Ajustando colunas da tabela comorbidade

# for key in tqdm(comorb_ls.keys(), position=0, leave=False):
#     df_com[key] = None
#     for item in comorb_ls[key]:
#         with ipython_io.capture_output() as captured:
#             df_com[key][df_com['CD_METADADO'] == item] = item

# df_com.drop(columns=['CD_METADADO'], inplace=True)

In [ ]:
# lista_keys = []
# for key in comorb_rn.keys():
#     lista_keys.append(np.int64(key))
# lista_values = []
# for val in comorb_rn.values():
#     lista_values.append(val)
# for column in df_com.columns:
#     if column not in ['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO']:
#         df_com[column].replace(lista_keys,lista_values, inplace=True)
            

In [ ]:
# # Organizando o df em linhas unicas por pacientes

# def organize_patient_data(df):
#     # Definir a ordem das colunas desejadas
#     desired_columns = [
#         'CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 
#         'TABAGISMO', 'AVC', 'HAS', 'DM', 'DLP', 'DRC', 'DPOC', 
#         'DAOP', 'DAC', 'DAC TIPOS', 'ICC', 'ICC TIPOS', 'ARRITMIA', 
#         'ARRITMIA TIPOS', 'ANGINA', 'ANGINA TIPOS', 'EDEMA', 
#         'DOR TORACICA', 'DOR TORACICA TIPOS', 'SINCOPE', 'DISPNEIA', 
#         'DISPNEIA TIPOS', 'PALPITACAO', 'VALVOPATIA', 'VALVOPATIA TIPOS', 
#         'VALVOPATIA PROTESE', 'TAVI PREVIA', 'IAO', 'EAO', 'IMI', 
#         'EMI', 'ITRI', 'ETRI', 'TONTURA', 'CLASSIFICACAO PS'
#     ]
    
#     # Agrupar o DataFrame por CD_ATENDIMENTO, CD_PACIENTE, DH_DOCUMENTO
#     grouped = df.groupby(['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO'], sort=False)
    
#     # Inicializar um dicionário para armazenar os dados organizados
#     organized_data = {
#         col: [] for col in desired_columns
#     }
    
#     # Iterar sobre cada grupo identificado pelas chaves
#     for group_key, group_df in grouped:
#         # Armazenar as colunas-chave para esta entrada
#         for key in ['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO']:
#             organized_data[key].append(group_key[desired_columns.index(key)])
        
#         # Preencher os dados para cada coluna desejada
#         for col in desired_columns[3:]:  # Ignorar as colunas chave
#             if col in group_df:
#                 if group_df[col].notnull().any():
#                     # Preencher com a primeira resposta não nula encontrada
#                     organized_data[col].append(group_df[col].dropna().iloc[0])
#                 else:
#                     # Preencher com None se não houver resposta
#                     organized_data[col].append(None)
#             else:
#                 # Preencher com None se a coluna não estiver presente no grupo
#                 organized_data[col].append(None)
    
#     # Criar um novo DataFrame a partir dos dados organizados
#     organized_df = pd.DataFrame(organized_data)
    
#     return organized_df

# # Aplicar a função ao DataFrame Comorb
# df_com = organize_patient_data(df_com)

In [ ]:
#df_hap.columns

In [ ]:
# # Hipertensão arterial pulmonar (HAP)
# df_hap = df_hap[['CD_PACIENTE', 'CD_ATENDIMENTO', 'HAP']]
# df_hap.drop_duplicates(subset=['CD_ATENDIMENTO', 'CD_PACIENTE', 'HAP'], inplace=True, ignore_index=True)

In [ ]:
#aHap = df_hap[df_hap['CD_ATENDIMENTO'].isin(aCir['CD_ATENDIMENTO'])].reset_index(drop=True)

In [ ]:
#df_obs.columns

In [ ]:
# # Manipulação de dados
# # Obesidade
# df_obs = df_obs[df_obs['CD_ATENDIMENTO'].isin(aCir['CD_ATENDIMENTO'])]
# df_obs.drop_duplicates(subset=['CD_ATENDIMENTO'], inplace=True, ignore_index=True)
# df_obs = df_obs.assign(OBESIDADE=pd.Series([1]*len(df_obs), index=df_obs.index))
# df_obs = df_obs[['CD_PACIENTE', 'CD_ATENDIMENTO', 'OBESIDADE']]
# #df_obs['OBESIDADE'] = df_obs['OBESIDADE'].replace(1, 'SIM')

In [ ]:
#aObs = df_obs[df_obs['CD_ATENDIMENTO'].isin(aCir['CD_ATENDIMENTO'])].reset_index(drop=True)

In [ ]:
#aObs1 = aObs[['CD_PACIENTE', 'CD_ATENDIMENTO', 'OBESIDADE']]

In [ ]:
#results2 = pd.merge(aHap, aObs1, on=['CD_PACIENTE', 'CD_ATENDIMENTO'], how='left')

In [ ]:
#results2.columns

In [ ]:
#results2 = results2.drop_duplicates(subset='CD_ATENDIMENTO')

In [ ]:
#results3 = pd.merge(aCom, results2, on=['CD_ATENDIMENTO', 'CD_PACIENTE'], how='left')

In [ ]:
#results4 = results3.drop_duplicates(subset='CD_ATENDIMENTO')

In [ ]:
#results4['CD_PACIENTE'].nunique()

In [ ]:
#results1.columns

In [ ]:
#results1['CD_PACIENTE'].nunique()

In [ ]:
#results4.columns

In [ ]:
#results5 = pd.merge(results1, results4, on=['CD_ATENDIMENTO', 'CD_PACIENTE'], how='left')

In [ ]:
#results6 = results5.sort_values(by='CD_CIRURGIA_AVISO', ascending=True)

In [ ]:
#results6['HR_ALTA'] = pd.to_datetime(results6['HR_ALTA'])

In [ ]:
# from datetime import timedelta

# # Adiciona uma coluna de data limite de retorno (30 dias após HR_ALTA)
# results6['DATA_RETORNO'] = results6['HR_ALTA'] + timedelta(days=30)

In [ ]:
#6643

In [ ]:
# # Obtém a data atual no formato desejado (dd.mm.aaaa)
# data_atual = datetime.now().strftime('%d.%m.%Y')

# # Define o caminho do arquivo com a data no nome
# caminho_cirurgia_2025 = f"C://Users/priscilla.sequetin/Documents/BasesDashs/cirurgia/cirurgia_{data_atual}.xlsx"


# # Salva o DataFrame no arquivo parquet
# results6.to_excel(caminho_cirurgia_2025, index=False)


# print(f"Arquivo salvo como: {caminho_cirurgia_2025}")


In [ ]:
# Listar colunas do dataframe final
col = results6.columns
for i in col:
    print(f'"{i}",')

In [ ]:
# # Buscando cirurgias na ft_descricao_cirurgica e ft_aviso_cirurgico


# query_dsATD = f"""
# SET DATEFORMAT ymd;
# WITH cid AS (
#     SELECT 
#         NK_CD_CID,
#         DS_SGRU_CID,
#         DT_ATUAL
#     FROM dim_cid cid1
#     WHERE DT_ATUAL = (
#         SELECT MAX(DT_ATUAL)
#         FROM dim_cid cid2
#         WHERE cid2.NK_CD_CID = cid1.NK_CD_CID
#     )
# ),
# doa AS (
#     SELECT 
#         NK_CD_ORI_ATE,
#         DS_ORI_ATE,
#         DT_ATUAL
#     FROM dim_origem_atendimento doa1
#     WHERE DT_ATUAL = (
#         SELECT MAX(DT_ATUAL)
#         FROM dim_origem_atendimento doa2
#         WHERE doa2.NK_CD_ORI_ATE = doa1.NK_CD_ORI_ATE
#     )
# ),
# dps AS (
#     SELECT 
#         NK_CD_PROCEDIMENTO,
#         DS_PROCEDIMENTO,
#         DT_ATUAL
#     FROM dim_procedimento_sus dps1
#     WHERE DT_ATUAL = (
#         SELECT MAX(DT_ATUAL)
#         FROM dim_procedimento_sus dps2
#         WHERE dps2.NK_CD_PROCEDIMENTO = dps1.NK_CD_PROCEDIMENTO
#     )
# )

# SELECT DISTINCT
#     fa.NK_CD_ATENDIMENTO,
#     fa.CD_PACIENTE,
#     fa.CD_CID as CID_ATEND,
#     cid.DS_SGRU_CID AS GRUPO_CID,
#     fau.DS_SINTOMA,
#     fau.DS_QUEIXA_PRINCIPAL,
#     fa.TP_ATENDIMENTO AS TPO_ATEND,
#     dps.DS_PROCEDIMENTO,
#     fa.HR_ATENDIMENTO AS DT_ATENDIMENTO
# FROM ft_atendimento fa
# LEFT JOIN ft_atendimento_urgencia fau ON fau.CD_ATENDIMENTO = fa.NK_CD_ATENDIMENTO
# LEFT JOIN cid ON fa.CD_CID = cid.NK_CD_CID
# LEFT JOIN doa ON fa.CD_ORI_ATE = doa.NK_CD_ORI_ATE
# LEFT JOIN dps ON fa.CD_PROCEDIMENTO = dps.NK_CD_PROCEDIMENTO
# WHERE fa.CD_PROCEDIMENTO NOT IN (202020134, 301010048)
# AND YEAR(HR_ATENDIMENTO) >= 2025

# ORDER BY fa.NK_CD_ATENDIMENTO DESC;


# """ 
# # Executa a função para buscar os dados
# dsATD = query_sql_to_dataframe(conn_string, query_dsATD)  

In [ ]:
#aAtd2 = dsATD[dsATD['CD_PACIENTE'].isin(aCir['CD_PACIENTE'])].reset_index(drop=True)

In [ ]:
#aAtd3 = aAtd2[(aAtd2['TPO_ATEND']=="I") | (aAtd2['TPO_ATEND']=="U")]

In [ ]:
# Realiza o merge com base no ID_PACIENTE
#results7 = results6[['CD_PACIENTE', 'CD_ATENDIMENTO', 'HR_ATENDIMENTO', 'HR_ALTA', 'DATA_RETORNO']]

In [ ]:
# Realiza o merge com base no ID_PACIENTE
#results8 = pd.merge(aAtd3, results7, on=['CD_PACIENTE'], how='left')

In [ ]:
# # Verifica a condição linha a linha com uma função lambda
# results8['Retorno'] = results8.apply(
#     lambda row: row['DT_ATENDIMENTO'] > row['HR_ALTA'] and row['DT_ATENDIMENTO'] <= row['DATA_RETORNO'],
#     axis=1
# )

In [ ]:
#results9 = results8[results8['Retorno']==True]

In [ ]:
# # Define o caminho do arquivo com a data no nome
# caminho_resultado_2025 = f"C://Users/priscilla.sequetin/Documents/BasesDashs/cirurgia/resultado_{data_atual}.xlsx"


# # Salva o DataFrame no arquivo parquet
# results8.to_excel(caminho_resultado_2025, index=False)


# print(f"Arquivo salvo como: {caminho_resultado_2025}")

In [ ]:
""" # Passo 4: Adicionar coluna de instância repetida
aAtd2_first_last['instance_repeat'] = aAtd2_first_last.groupby('record_id').cumcount() + 1 """

In [ ]:
# # Buscando cirurgias na ft_descricao_cirurgica e ft_aviso_cirurgico


# query_dsEVO = f"""
# SET DATEFORMAT ymd;
# SELECT 
#     fdt.CD_ATENDIMENTO,
#     fdt.CD_PACIENTE,
# 	fdt.DH_DOCUMENTO,
#     CONVERT(date, fee.HR_EVO_ENF) AS HR_EVO_ENF,
#     MAX(CASE WHEN dcd.CD_METADADO = 422853 THEN CONVERT(nvarchar(MAX), fdt.DS_RESPOSTA) END) AS DS_EVO_MED,
#     MAX(CASE WHEN dcd.CD_METADADO = 390962 THEN CONVERT(nvarchar(MAX), fdt.DS_RESPOSTA) END) AS DS_CON_MED,
#     CONVERT(nvarchar(MAX), fee.DS_EVO_ENF) AS DS_EVO_ENF
# FROM 
#     DATADANTE.dbo.ft_doc_eletronico_texto fdt
# JOIN 
#     ft_evolucao_enfermagem fee 
#     ON fee.CD_ATENDIMENTO = fdt.CD_ATENDIMENTO
# JOIN 
#     dim_campo_documento dcd 
#     ON dcd.NK_CD_CAMPO = fdt.NK_CD_CAMPO
# WHERE 
#     dcd.CD_METADADO IN (422853, 390962)
#     AND CONVERT(date, fdt.DH_DOCUMENTO) = CONVERT(date, fee.HR_EVO_ENF)
#     AND YEAR(DH_DOCUMENTO) >= 2025
# GROUP BY 
#     fdt.CD_ATENDIMENTO,
#     fdt.CD_PACIENTE,
# 	fdt.DH_DOCUMENTO,
#     CONVERT(date, fee.HR_EVO_ENF),
#     CONVERT(nvarchar(MAX), fee.DS_EVO_ENF)
# ORDER BY 
#     fdt.CD_ATENDIMENTO DESC;
# """ 
# # Executa a função para buscar os dados
# dsEVO = query_sql_to_dataframe(conn_string, query_dsEVO)  

In [ ]:
#aEvol = dsEVO[dsEVO['CD_PACIENTE'].isin(aCir['CD_PACIENTE'])].reset_index(drop=True)

In [ ]:
# # Definir o dicionário de termos de interesse
# terms_dict = {
#     'EXTUBACAO': ['EXTUBAÇÃO'],
#     'INFECCAO_FO': ['INFECÇÕES NO LOCAL DA CIRURGIA', 'INFECÇÃO DE FO', 'FERIDA OPERATÓRIA'],
#     'TROMBOS_EV': ['TROMBOEMBÓLICO'],
#     'SANGRAMENTOS': ['COMPLICAÇÕES HEMORRÁGICAS', 'HEMORRAGIA'],
#     'ARRITMIAS': ['ARRITMIAS'],
#     'INS_RENAL': ['INSUFICIENCIA RENAL', 'DISFUNÇÃO RENAL'],
#     'AVC_EV': ['ACIDENTE VASCULAR CEREBRAL', 'AVC HEMORRÁGICO'],
#     'DREN_MARFAN': ['DRENAGEM DE MARFAN', 'DERRAME PERICÁRDICO'],
#     'ENDOCARDITE': ['ENDOCARDITE']
# }

In [ ]:
# import re

# # Função para verificar a presença dos termos usando re.sub e re.search
# def check_terms(row, terms):
#     row_str = str(row)
#     return any(re.search(re.sub(r'\s', '', term), row_str, re.IGNORECASE) for term in terms)

# # Adicionar coluna ANEURISMA_STATUS
# aExt2['ANEURISMA_STATUS'] = aExt2.apply(
#     lambda row: any(check_terms(row[col], terms_dict['ANEURISMA_STATUS']) for col in aExt2.columns), 
#     axis=1
# ).astype(int)

In [ ]:
# # Função para verificar a presença dos padrões
# def check_patterns(cell, patterns):
#     if isinstance(cell, str):  # Verificar apenas células que são strings
#         cell_no_spaces = re.sub(r'\s', '', cell)  # Remover espaços da célula
#         for pattern in patterns:
#             pattern_no_spaces = re.sub(r'\s', '', pattern)  # Remover espaços do padrão
#             if re.search(pattern_no_spaces, cell_no_spaces, re.IGNORECASE):  # Ignorar maiúsculas e minúsculas
#                 return True
#     return False

# for termos in terms_dict.keys():
#     if termos != 'ANEURISMA_STATUS':
#         # Aplicar a função para criar a coluna 'EAO_STATUS'
#         aEvol[termos] = aEvol.applymap(
#             lambda x: check_patterns(x, terms_dict[termos])
#         ).any(axis=1).map({True: '1', False: '0'})

In [ ]:
#aEvol.columns

In [ ]:
# aEvol1 = aEvol[['CD_ATENDIMENTO', 'CD_PACIENTE', 'DH_DOCUMENTO', 'EXTUBACAO', 'INFECCAO_FO',
#         'TROMBOS_EV', 'SANGRAMENTOS', 'ARRITMIAS', 'INS_RENAL', 'AVC_EV',
#         'DREN_MARFAN', 'ENDOCARDITE']]

In [ ]:
#aEvol2 = aEvol1.sort_values(by='DH_DOCUMENTO', ascending=False)

In [ ]:
#aEvol3 = aEvol2.drop_duplicates(subset='DH_DOCUMENTO', keep='first').reset_index(drop=True)


In [ ]:
# # Define o caminho do arquivo com a data no nome
# caminho_eventos_2025 = f"C://Users/priscilla.sequetin/Documents/BasesDashs/cirurgia/eventos_{data_atual}.xlsx"


# # Salva o DataFrame no arquivo parquet
# aEvol3.to_excel(caminho_eventos_2025, index=False)


# print(f"Arquivo salvo como: {caminho_eventos_2025}")

In [ ]:
# # # Obtém a data atual no formato desejado (dd.mm.aaaa)
# data_atual = datetime.now().strftime('%d.%m.%Y')

# caminho_resul = f"C://Users/priscilla.sequetin/Downloads/valvulas_{data_atual}.parquet"

# # # Salva o DataFrame no arquivo parquet
# valvulas.to_parquet(caminho_resul, index=False)

# print(f"Arquivo salvo como: {caminho_resul}")

In [ ]:
import pandas as pd


inicio = pd.to_datetime('24/02/2025')
fim = pd.to_datetime('27/05/2025')

# Calcula e corrige se for negativo (fim < inicio) somando 1 dia
delta = (fim - inicio) if fim >= inicio else (fim - inicio) + pd.Timedelta(days=1)

minutos = delta.total_seconds() / 60
print(delta)
print(minutos)

In [ ]:
# colunas necessarias record_id e redcap_repeat_instance de cada df
# age esta em  regvalv_cirurgia['idade_calc']
# female esta em regvalv_pcte['genero']=2 # nao tem instancia, demografico não repete
# nyha esta em regvalv_ftrisk['classe_nyha']=(1, 2, 3, 4)
# ccs4 esta em regvalv_ftrisk['ccs']=4
# iddm esta em regvalv_ftrisk['diabetes_controle']=3
# eca esta em regvalv_ftrisk['arteriopatia_extracardiaca']=1
# cpd esta em regvalv_ftrisk['dpoc']=1
# mob esta em regvalv_ftrisk['mobilidade_severa_preju']=1 ou regvalv_geronto['eq5d_mobilidade']=3 ou regvalv_geronto['barthel_mobilidade']=0
# redo esta em regvalv_preop['cirurgia_anterior']=1 ou regvalv_ftrisk['tipo_intervencao_previa___2', 'tipo_intervencao_previa___3', 'tipo_intervencao_previa___6']=1
# creatinine_umol_l esta em regvalv_labs['creatinina_pre']
# weight_kg esta em regvalv_ftrisk['peso']
# dialysis esta em regvalv_posop['dialise_necessaria']=1 ou regvalv_ftrisk['drc_clcr']=4
# ae esta em regvalv_ftrisk['endocardite_ativa']=1 ou regvalv_preop['diagnostico_principal___8']=1 ou regvalv_obito['endocardite']=1
# critical esta em regvalv_ftrisk['estado_pr_operat_rio_cr_ti']=1
# lv_ef esta em regvalv_eco['disfuncao_ventricular_encam']=1 e regvalv_eco['feve_encam'] = (if([feve_encam] = "" or [feve_encam] = "", 0,
#   if([feve_encam] < 20, 3,     if([feve_encam] < 30, 2,      if([feve_encam] <= 50, 1, 0)     )   ) ))
# recent_mi esta em regvalv_ftrisk['iam_menor_90dias']=1
# pa_pressure esta em regvalv_ftrisk['valor_psap_encam']=(1, 2, 3)
# urgency esta em regvalv_preop['tipo_operacao']=(1, 2, 3, 4)
# n_procedures esta em regvalv_cirurgia['tipo_cirurgia','cirurgica_associada_encam']= (if([cirurgica_associada_encam___1] = "1" and 
#    (countChecked("[tipo_cirurgia___1]", "[tipo_cirurgia___10]") = "0" or [tipo_cirurgia___8] = "1"), 0,
#   if(countChecked("[tipo_cirurgia___1]", "[tipo_cirurgia___7]") <= "1", 1,     if(countChecked("[tipo_cirurgia___1]", "[tipo_cirurgia___7]") <= "2", 2, 3)   ) ) )
# thoracic_aorta esta em regvalv_ftrisk['tipo_intervencao_previa___6']=1 ou regvalv_preop['diagnostico_principal___6', 'tipo_de_cirurgia_anterior___2	']=1
# ou regvalv_cirurgia['cirurgica_associada_encam___2']=1 = (if([cirurgica_associada_encam___2] = "1" or     [tipo_de_cirurgia_anterior___2] = "1" or     
#         [tipo_intervencao_previa___6] = "1" or     [diagnostico_principal___6] = "1",     "1", "0"))

# AUDITORIA

In [ ]:
def extrair_fase(event_name: str) -> str:
    """
    Extrai a fase clínica a partir do redcap_event_name.
    Retorna valores padronizados para uso analítico.
    """
    event_name = event_name.lower()

    if 'fase1preoperatorio_arm_1' in event_name:
        return 'fase1'
    elif 'fase2preprocedimen_arm_1' in event_name:
        return 'fase2'
    elif 'fase3procedcirurgi_arm_1' in event_name:
        return 'fase3'
    elif 'fase4posprocedimed_arm_1' in event_name:
        return 'fase4'
    elif 'fase5recuperacaoin_arm_1' in event_name:
        return 'fase5'
    elif 'fase6segalta30dias_arm_1' in event_name:
        return 'fase'
    elif 'encerramento_arm_1' in event_name:
        return 'encerra'
    else:
        return 'outros'